<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap09/cap09_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 9 Apprendimento Profondo per la Visione Artificiale

I capitoli precedenti hanno stabilito le fondamenta della Visione Artificiale (VA) attraverso metodi classici di estrazione e rappresentazione delle caratteristiche. Nel **Capitolo 7**, descrittori come *Local Binary Patterns* (*LBP*) e *Histogram of Oriented Gradients* (*HOG*) hanno mostrato come trame e forme possano essere codificate da descrittori progettati manualmente. Nel **Capitolo 8**, algoritmi come *Oriented FAST and Rotated BRIEF* (*ORB*) e il rivelatore *Haar Cascade* hanno esteso questo principio a compiti di corrispondenza, rilevamento e riconoscimento di oggetti.

Queste tecniche rimangono rilevanti per la loro interpretabilità ed efficienza computazionale, ma dipendono da una fase preliminare di definizione manuale dei descrittori, denominata **ingegneria delle caratteristiche** (*feature engineering*). Questa dipendenza limita l'adattamento del modello a scenari per i quali il descrittore non è stato progettato.

L'**Apprendimento Profondo** (*Deep Learning*) propone un'alternativa: invece di specificare manualmente le caratteristiche rilevanti, il modello apprende automaticamente rappresentazioni dai dati durante l'addestramento — processo noto come **apprendimento di rappresentazioni** (*representation learning*) (GOODFELLOW, 2016). Nella VA, questa strategia è implementata principalmente dalle **Reti Neurali Convoluzionali** (*Convolutional Neural Networks* — CNN), nelle quali i filtri convoluzionali cessano di avere coefficienti fissi e vengono regolati da algoritmi di ottimizzazione.

Sebbene rappresentino un cambiamento nella costruzione di sistemi di riconoscimento di pattern, le CNN preservano concetti già studiati in questo libro: la convoluzione, presentata nel **Capitolo 3**, rimane l'operazione responsabile dell'estrazione locale delle caratteristiche, ora applicata con coefficienti appresi anziché progettati.

Va sottolineato che l'obiettivo di questo capitolo non è esplorare esaustivamente la teoria dell'Apprendimento Profondo, ma piuttosto offrire una panoramica dei suoi fondamenti e dimostrare come queste architetture vengano applicate nel contesto della VA. I lettori interessati a un approfondimento teorico e concettuale nell'area dovrebbero ricorrere a riferimenti specialistici della letteratura, come Goodfellow (2016) e Lecun (2015).

## 9.1 Obiettivi del Capitolo

Al termine di questo capitolo, lo studente dovrebbe essere in grado di:

- Collegare la convoluzione appresa dalle CNN con la convoluzione di *kernel* fissi presentata nel **Capitolo 3**;
- Descrivere l'architettura di base di una CNN e la funzione dei suoi strati principali;
- Implementare, addestrare e valutare modelli CNN per la classificazione delle immagini;
- Applicare il **transfer learning** (*transfer learning*) per adattare modelli pre-addestrati a nuovi problemi;
- Utilizzare modelli pre-addestrati in compiti di classificazione, rilevamento di oggetti e segmentazione;
- Implementare, addestrare e valutare un'architettura *U-Net* per la segmentazione semantica, confrontandola con approcci classici;
- Preparare set di dati annotati e integrarli in un *pipeline* di addestramento tramite piattaforme come **Roboflow**;
- Integrare geometria computazionale e Deep Learning in applicazioni di realtà aumentata e fotogrammetria.

La [Figura 9.1](#fig-09-infografo) sintetizza l'organizzazione dei concetti studiati in questo capitolo e le relazioni tra essi.

<figure id="fig-09-infografo" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-09-infografo.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figura 9.1:</strong> Panoramica dei principali concetti trattati in questo capitolo. **Fonte:** elaborato con l'ausilio di *Gemini Notebook* ({GOOGLE}, 2025).</figcaption>
</figure>

## 9.2 Panoramica: Classificazione, Rilevamento e Segmentazione

I compiti di VC si differenziano, soprattutto, per l'informazione prodotta come output. La **classificazione** assegna un'unica etichetta all'intera immagine; il **rilevamento di oggetti** localizza e identifica gli oggetti presenti nella scena; la **segmentazione** associa una classe a ogni pixel e, in alcuni approcci, distingue differenti istanze di una stessa categoria.

La [Tabela 9.1](#tbl-panorama-tarefas) riassume i compiti studiati lungo il libro, indicando la domanda a cui ciascuno risponde e la granularità dell'informazione prodotta.

<a id="tbl-panorama-tarefas"></a>

**Tabela 9.1:** Confronto tra i principali compiti di VC secondo la granularità dell'informazione prodotta.

| Compito | Domanda a cui risponde | Granularità dell'output | Capitolo |
|-----------------------------|-----------------------------------------------|-------------------------------------------------------------|:--------:|
| **Classificazione** | "Qual è la classe di questa immagine?" | Un'unica etichetta per l'intera immagine | 7 e 9 |
| **Rilevamento di oggetti** | "Quali oggetti esistono e dove si trovano?" | Classe e bounding box per ogni oggetto | 8 e 9 |
| **Segmentazione semantica** | "A quale classe appartiene ogni pixel?" | Un'etichetta di classe per ogni pixel | 8 e 9 |
| **Segmentazione delle istanze** | "Quali pixel appartengono a ogni oggetto?" | Un'etichetta per pixel per ogni istanza | 8 e 9 |
| **Segmentazione panottica** | "Qual è la classe e l'identità di ogni oggetto?" | Classe e identificatore di istanza per ogni pixel | 8 e 9 |


Questi compiti rappresentano livelli crescenti di interpretazione dell'immagine: la classificazione descrive la scena in modo globale, il rilevamento aggiunge la localizzazione degli oggetti e la segmentazione produce una rappresentazione spaziale dettagliata, consentendo di analizzare ciascuna regione individualmente. Questo capitolo si concentra, in primo luogo, sulla classificazione tramite CNN, per poi estendere gli stessi principi al rilevamento e alla segmentazione.

## 9.3 Configurazione dell'Ambiente

Gli esempi di questo capitolo utilizzano **PyTorch**, un *framework* ampiamente impiegato nello sviluppo e nell'addestramento di modelli di Deep Learning. Il codice seguente verifica la disponibilità delle librerie necessarie e installa automaticamente quelle non ancora presenti nell'ambiente di esecuzione.

Se **PyTorch** non è installato, viene selezionata automaticamente una versione compatibile con l'hardware disponibile: la versione con supporto **CUDA**, se è disponibile una GPU NVIDIA, oppure la versione per l'esecuzione su CPU, in caso contrario.

Successivamente, l'ambiente viene inizializzato con l'importazione delle librerie utilizzate nel corso del capitolo, la definizione di un seed casuale per favorire la riproducibilità degli esperimenti e il recupero del file `morph.py` — la libreria didattica di elaborazione morfologica già utilizzata nei capitoli precedenti —, qualora non sia ancora disponibile nella directory di lavoro.

In [1]:
import contextlib, importlib, importlib.metadata, importlib.util
import io, os, random, shutil, subprocess, sys, urllib.request, warnings

# Sopprime gli avvisi di PyTorch e i warning generali
warnings.filterwarnings("ignore", category=UserWarning)

url = ("https://raw.githubusercontent.com/fzampirolli/"
       "pdi-vc/master/morph/config.py")
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config

# Silenzia stdout e stderr sia a livello di Python che di File Descriptors del SO
def setup_silencioso():
    with open(os.devnull, "w") as fnull:
        old_out = os.dup(1)
        old_err = os.dup(2)
        try:
            os.dup2(fnull.fileno(), 1)
            os.dup2(fnull.fileno(), 2)
            with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
                config.setup()
        finally:
            os.dup2(old_out, 1)
            os.dup2(old_err, 2)
            os.close(old_out)
            os.close(old_err)

setup_silencioso()
from morph import mm


def setup_cap09():
    """Installa le librerie mancanti per questo capitolo in modo 100% silenzioso."""
    pkgs = {
        "skimage": "scikit-image", "numpy": "numpy",
        "sklearn": "scikit-learn", "matplotlib": "matplotlib",
        "torchviz": "torchviz", "ultralytics": "ultralytics",
        "roboflow": "roboflow",
    }
    for mod, pkg in pkgs.items():
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if importlib.util.find_spec("torch") is None:
        args = (["torch", "torchvision"] if shutil.which("nvidia-smi")
                 else ["--index-url",
                       "https://download.pytorch.org/whl/cpu",
                       "torch", "torchvision"])
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if not shutil.which("dot") and shutil.which("apt-get"):
        subprocess.run(["apt-get", "install", "-y", "-qq", "graphviz"],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)


setup_cap09()

import cv2, numpy as np, torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from skimage import data as skdata
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import OxfordIIITPet
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights)
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.transforms.functional import to_tensor
from torchviz import make_dot
from ultralytics import YOLO

torch.manual_seed(42)
FLAG_LIMPAR_DADOS = False
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
ver = importlib.metadata.version("ultralytics")
print(f"✅ Ambiente pronto. OpenCV {cv2.__version__} | "
      f"morph {getattr(mm, '__version__', 'local_file')} | "
      f"PyTorch {torch.__version__} | Ultralytics {ver} | {device}{gpu}")

✅ Ambiente pronto. OpenCV 5.0.0 | morph local_file | PyTorch 2.6.0+cu124 | Ultralytics 8.4.113 | cuda (NVIDIA GeForce GTX TITAN X)


## 9.4 Fondamenti di Deep Learning per la VC

Le CNN sono la principale architettura di Deep Learning applicata all'analisi delle immagini. Il loro funzionamento si basa sulla composizione di operazioni convoluzionali organizzate in strati successivi, in cui i filtri appresi durante l'addestramento trasformano l'immagine in rappresentazioni progressivamente più astratte. In questa sezione vengono presentati i concetti fondamentali che collegano la convoluzione spaziale studiata in precedenza ai moderni modelli di VC, inclusa l'estrazione gerarchica delle caratteristiche, il processo di addestramento e l'utilizzo di modelli pre-addestrati.

### 9.4.1 Dalla Convoluzione Fissa alla Convoluzione Appresa

Il **Capitolo 3** ha presentato la convoluzione spaziale con *kernel* fissi, come gli operatori di Sobel, progettati per evidenziare caratteristiche specifiche di un'immagine. Nei **Capitoli 7** e **8**, lo stesso principio ha sostenuto descrittori come *HOG*, *LBP* e *ORB*, oltre al rilevatore *Haar Cascade*: in tutti questi casi, i filtri sono definiti prima dell'esecuzione dell'algoritmo e rimangono invariati durante l'elaborazione.

La [Figura 9.2](#fig-09-sim-09-convolucao) riprende il funzionamento della convoluzione spaziale: il simulatore consente di selezionare diversi *kernel* e di seguire lo spostamento della finestra di convoluzione su un'immagine. In ogni posizione, i coefficienti del *kernel* si combinano con l'intorno locale dell'immagine — denominato **campo recettivo** (*receptive field*) — per produrre un valore della **mappa delle caratteristiche** (*feature map*), illustrando anche la condivisione dei pesi (*weight sharing*).

Le CNN preservano questa operazione, ma sostituiscono i *kernel* fissi con **filtri appresi**: invece di coefficienti definiti in precedenza, la rete aggiusta questi valori durante l'addestramento a partire da esempi etichettati, cercando di minimizzare una **funzione di perdita** (*loss function*), che misura la differenza tra le previsioni del modello e le risposte attese.

La differenza essenziale tra i metodi classici e le CNN, quindi, non sta nell'operazione di convoluzione in sé, ma nel modo in cui i filtri vengono ottenuti: mentre i primi utilizzano filtri progettati manualmente, le CNN apprendono, a partire dai dati di addestramento, rappresentazioni adeguate al compito.

In [2]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-convolucao" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-convolucao .cap09conv_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-convolucao .cap09conv_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-convolucao .cap09conv_btn {
      font-size:11px; font-weight:600; padding:6px 10px; border-radius:6px;
      border:1px solid #E4DCC8; background:#FFF; color:#26241D; cursor:pointer;
      transition:all .15s ease;
    }
    #sim-09-convolucao .cap09conv_btn:hover { background:#F1EAD7; }
    #sim-09-convolucao .cap09conv_modebtn {
      flex:1; text-align:center; padding:5px 8px; font-size:11px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:8px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-convolucao .cap09conv_modebtn.cap09conv_active { background:#26241D; color:#FBF7EE; }
    #sim-09-convolucao .cap09conv_btn.cap09conv_active { background:#26241D !important; color:#FBF7EE !important; border-color:#26241D !important; }
    #sim-09-convolucao .cap09conv_btn_primary { background:#2F6F9F; color:#FFF; border-color:#2F6F9F; }
    #sim-09-convolucao .cap09conv_btn_primary:hover { background:#245880; }
    #sim-09-convolucao .cap09conv_btn_success { background:#1E8F6F; color:#FFF; border-color:#1E8F6F; }
    #sim-09-convolucao .cap09conv_btn_success:hover { background:#166e55; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">🎯 Simulatore: Operazione di Convoluzione 2D Classica</span>
    <span class="cap09conv_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Input 12×12 · Kernel 3×3 · Stride 1</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Painel de Seleção de Imagens e Kernels -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:12px;">
      <div style="flex:1;min-width:230px;">
        <div class="cap09conv_grouplabel">IMMAGINE DI INPUT (12×12)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09conv_btnCasa" class="cap09conv_modebtn cap09conv_active">🏠 Casa</button>
          <button id="cap09conv_btnFeliz" class="cap09conv_modebtn">😊 Felice</button>
          <button id="cap09conv_btnTriste" class="cap09conv_modebtn">😢 Triste</button>
        </div>
      </div>

      <div style="flex:1;min-width:280px;">
        <div class="cap09conv_grouplabel">FILTRO (KERNEL 3×3)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09conv_btnSobelV" class="cap09conv_btn cap09conv_active">📐 Sobel V</button>
          <button id="cap09conv_btnSobelH" class="cap09conv_btn">📏 Sobel H</button>
          <button id="cap09conv_btnSharpen" class="cap09conv_btn">✨ Nitidezza</button>
          <button id="cap09conv_btnIdentidade" class="cap09conv_btn">🎯 Identità</button>
        </div>
      </div>
    </div>

    <!-- Controles do Passo a Passo -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-bottom:12px;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;justify-content:space-between;">
        <div style="display:flex;gap:6px;align-items:center;">
          <button id="cap09conv_btnPasso" class="cap09conv_btn cap09conv_btn_success">▶ Avanti di un Passo</button>
          <button id="cap09conv_btnTudo" class="cap09conv_btn cap09conv_btn_primary">⏭ Calcola Tutto</button>
          <button id="cap09conv_btnReset" class="cap09conv_btn">↺ Reimposta</button>
        </div>
        <span class="cap09conv_mono" style="font-size:11px;color:#5b5647;">Posizione Attuale: <b id="cap09conv_posTxt" style="color:#2F6F9F;">(0, 0)</b> [Output 10×10]</span>
      </div>
    </div>

    <!-- Área Gráfica: Entrada, Kernel e Saída -->
    <div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Input (12×12)</div>
        <canvas id="cap09conv_canvasEntrada" style="width:240px;height:240px;"></canvas>
      </div>

      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Kernel (3×3)</div>
        <canvas id="cap09conv_canvasKernel" style="width:105px;height:105px;"></canvas>
      </div>

      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:4px;color:#4b5563;">Mappa di Output (10×10)</div>
        <canvas id="cap09conv_canvasSaida" style="width:200px;height:200px;"></canvas>
      </div>
    </div>

    <!-- Terminal de Cálculo em Tempo Real -->
    <div id="cap09conv_calcTxt" class="cap09conv_mono" style="text-align:center;font-size:11px;margin-top:12px;color:#7EE7C6;background:#1B2430;padding:8px 10px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;overflow-x:auto;">
      ∑ (xᵢ × wᵢ) = calcolo della posizione attuale...
    </div>

  </div>
</div>

<script>
(function(){
  var cap09conv_IMAGENS = {
    casa: [
      "............", "....XXXX....", "...XXXXXX...", "..XXXXXXXX..",
      ".XXXXXXXXXX.", ".XXXXXXXXXX.", ".XXXXXXXXXX.", ".XXXXXXXXXX.",
      ".XXXX..XXXX.", ".XXXX..XXXX.", ".XXXX..XXXX.", "............"
    ],
    feliz: [
      "............", "....XXXX....", "..XXXXXXXX..", ".XXXXXXXXXX.",
      ".XXXXXXXXXX.", ".XXX.XX.XXX.", ".XXXXXXXXXX.", ".XX......XX.",
      ".XXX....XXX.", ".XXXXXXXXXX.", "..XXXXXXXX..", "............"
    ],
    triste: [
      "............", "....XXXX....", "..XXXXXXXX..", ".XXXXXXXXXX.",
      ".XXXXXXXXXX.", ".XXX.XX.XXX.", ".XXXXXXXXXX.", ".XXX....XXX.",
      ".XX......XX.", ".XXXXXXXXXX.", "..XXXXXXXX..", "............"
    ]
  };

  var cap09conv_KERNELS = {
    sobelV:     { nome: "Sobel Vertical", k: [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]] },
    sobelH:     { nome: "Sobel Horizontal", k: [[-1, -2, -1], [0, 0, 0], [1, 2, 1]] },
    sharpen:    { nome: "Nitidez (Sharpen)", k: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]] },
    identidade: { nome: "Identidade", k: [[0, 0, 0], [0, 1, 0], [0, 0, 0]] }
  };

  function cap09conv_converterLinhas(cap09conv_linhas){
    return cap09conv_linhas.map(function(cap09conv_l){
      var cap09conv_res = [];
      for(var cap09conv_i=0; cap09conv_i<cap09conv_l.length; cap09conv_i++) cap09conv_res.push(cap09conv_l[cap09conv_i]==='X' ? 1.0 : 0.0);
      return cap09conv_res;
    });
  }

  function cap09conv_init(cap09conv_root){
    if(!cap09conv_root || cap09conv_root.dataset.initConv) return;
    cap09conv_root.dataset.initConv = "1";

    var cap09conv_imgAtualId = "casa";
    var cap09conv_kernelAtualKey = "sobelV";

    var cap09conv_imgEntradaBase = cap09conv_converterLinhas(cap09conv_IMAGENS[cap09conv_imgAtualId]);
    var cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
    var cap09conv_pos = {r:0, c:0};

    var cap09conv_cvsEnt = cap09conv_root.querySelector('#cap09conv_canvasEntrada');
    var cap09conv_cvsKer = cap09conv_root.querySelector('#cap09conv_canvasKernel');
    var cap09conv_cvsSai = cap09conv_root.querySelector('#cap09conv_canvasSaida');

    var cap09conv_ctxEnt = cap09conv_cvsEnt.getContext('2d');
    var cap09conv_ctxKer = cap09conv_cvsKer.getContext('2d');
    var cap09conv_ctxSai = cap09conv_cvsSai.getContext('2d');

    function cap09conv_prepararCanvas(cap09conv_canvas, cap09conv_ctx, cap09conv_cssW, cap09conv_cssH){
      var cap09conv_dpr = window.devicePixelRatio || 1;
      cap09conv_canvas.width = cap09conv_cssW * cap09conv_dpr;
      cap09conv_canvas.height = cap09conv_cssH * cap09conv_dpr;
      cap09conv_ctx.scale(cap09conv_dpr, cap09conv_dpr);
    }
    cap09conv_prepararCanvas(cap09conv_cvsEnt, cap09conv_ctxEnt, 240, 240);
    cap09conv_prepararCanvas(cap09conv_cvsKer, cap09conv_ctxKer, 105, 105);
    cap09conv_prepararCanvas(cap09conv_cvsSai, cap09conv_ctxSai, 200, 200);

    var cap09conv_posTxt  = cap09conv_root.querySelector('#cap09conv_posTxt');
    var cap09conv_calcTxt = cap09conv_root.querySelector('#cap09conv_calcTxt');

    var cap09conv_btnCasa   = cap09conv_root.querySelector('#cap09conv_btnCasa');
    var cap09conv_btnFeliz  = cap09conv_root.querySelector('#cap09conv_btnFeliz');
    var cap09conv_btnTriste = cap09conv_root.querySelector('#cap09conv_btnTriste');

    var cap09conv_btnSobelV     = cap09conv_root.querySelector('#cap09conv_btnSobelV');
    var cap09conv_btnSobelH     = cap09conv_root.querySelector('#cap09conv_btnSobelH');
    var cap09conv_btnSharpen    = cap09conv_root.querySelector('#cap09conv_btnSharpen');
    var cap09conv_btnIdentidade = cap09conv_root.querySelector('#cap09conv_btnIdentidade');

    function cap09conv_desenharEntrada(){
      var cap09conv_tam = 20;
      cap09conv_ctxEnt.clearRect(0,0,240,240);
      for (var cap09conv_r=0; cap09conv_r<12; cap09conv_r++){
        for (var cap09conv_c=0; cap09conv_c<12; cap09conv_c++){
          var cap09conv_val = cap09conv_imgEntradaBase[cap09conv_r][cap09conv_c];
          var cap09conv_g = Math.round(cap09conv_val * 255);
          cap09conv_ctxEnt.fillStyle = "rgb(" + cap09conv_g + "," + cap09conv_g + "," + cap09conv_g + ")";
          cap09conv_ctxEnt.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
          cap09conv_ctxEnt.strokeStyle = "#E4DCC8";
          cap09conv_ctxEnt.lineWidth = 0.8;
          cap09conv_ctxEnt.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

          cap09conv_ctxEnt.fillStyle = cap09conv_g > 128 ? "#111827" : "#FFFFFF";
          cap09conv_ctxEnt.font = "600 8px 'JetBrains Mono', monospace";
          cap09conv_ctxEnt.textAlign = "center";
          cap09conv_ctxEnt.textBaseline = "middle";
          cap09conv_ctxEnt.fillText(cap09conv_val.toFixed(0), cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
        }
      }

      if (cap09conv_pos.r < 10) {
        cap09conv_ctxEnt.strokeStyle = "#C1443A";
        cap09conv_ctxEnt.lineWidth = 2.5;
        cap09conv_ctxEnt.strokeRect(cap09conv_pos.c*cap09conv_tam, cap09conv_pos.r*cap09conv_tam, cap09conv_tam*3, cap09conv_tam*3);
      }
    }

    function cap09conv_desenharKernel(){
      var cap09conv_tam = 35;
      cap09conv_ctxKer.clearRect(0,0,105,105);
      var cap09conv_k = cap09conv_KERNELS[cap09conv_kernelAtualKey].k;
      for (var cap09conv_i=0; cap09conv_i<3; cap09conv_i++){
        for (var cap09conv_j=0; cap09conv_j<3; cap09conv_j++){
          var cap09conv_val = cap09conv_k[cap09conv_i][cap09conv_j];
          cap09conv_ctxKer.fillStyle = cap09conv_val > 0 ? "#E6F4EA" : (cap09conv_val < 0 ? "#FCE8E6" : "#FAFAF7");
          cap09conv_ctxKer.fillRect(cap09conv_j*cap09conv_tam, cap09conv_i*cap09conv_tam, cap09conv_tam, cap09conv_tam);
          cap09conv_ctxKer.strokeStyle = "#E4DCC8";
          cap09conv_ctxKer.strokeRect(cap09conv_j*cap09conv_tam, cap09conv_i*cap09conv_tam, cap09conv_tam, cap09conv_tam);

          cap09conv_ctxKer.fillStyle = cap09conv_val > 0 ? "#1E8F6F" : (cap09conv_val < 0 ? "#C1443A" : "#8A8371");
          cap09conv_ctxKer.font = "600 10px 'JetBrains Mono', monospace";
          cap09conv_ctxKer.textAlign = "center";
          cap09conv_ctxKer.textBaseline = "middle";
          cap09conv_ctxKer.fillText(cap09conv_val.toString(), cap09conv_j*cap09conv_tam + cap09conv_tam/2, cap09conv_i*cap09conv_tam + cap09conv_tam/2);
        }
      }
    }

    function cap09conv_desenharSaida(){
      var cap09conv_tam = 20;
      cap09conv_ctxSai.clearRect(0,0,200,200);
      for (var cap09conv_r=0; cap09conv_r<10; cap09conv_r++){
        for (var cap09conv_c=0; cap09conv_c<10; cap09conv_c++){
          var cap09conv_v = cap09conv_saida[cap09conv_r][cap09conv_c];
          if (cap09conv_v === null) {
            cap09conv_ctxSai.fillStyle = "#F7F5EE";
            cap09conv_ctxSai.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
            cap09conv_ctxSai.strokeStyle = "#E4DCC8";
            cap09conv_ctxSai.lineWidth = 0.8;
            cap09conv_ctxSai.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

            cap09conv_ctxSai.fillStyle = "#B8AE94";
            cap09conv_ctxSai.font = "700 8px 'JetBrains Mono', monospace";
            cap09conv_ctxSai.textAlign = "center";
            cap09conv_ctxSai.textBaseline = "middle";
            cap09conv_ctxSai.fillText("·", cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
          } else {
            var cap09conv_normV = Math.max(0, Math.min(1, (cap09conv_v + 2.0) / 4.0));
            var cap09conv_g = Math.round(cap09conv_normV * 255);
            cap09conv_ctxSai.fillStyle = "rgb(" + cap09conv_g + "," + cap09conv_g + "," + cap09conv_g + ")";
            cap09conv_ctxSai.fillRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
            cap09conv_ctxSai.strokeStyle = "#E4DCC8";
            cap09conv_ctxSai.lineWidth = 0.8;
            cap09conv_ctxSai.strokeRect(cap09conv_c*cap09conv_tam, cap09conv_r*cap09conv_tam, cap09conv_tam, cap09conv_tam);

            cap09conv_ctxSai.fillStyle = cap09conv_g > 128 ? "#111827" : "#FFFFFF";
            cap09conv_ctxSai.font = "600 7.5px 'JetBrains Mono', monospace";
            cap09conv_ctxSai.textAlign = "center";
            cap09conv_ctxSai.textBaseline = "middle";
            cap09conv_ctxSai.fillText(cap09conv_v.toFixed(1), cap09conv_c*cap09conv_tam + cap09conv_tam/2, cap09conv_r*cap09conv_tam + cap09conv_tam/2);
          }
        }
      }

      if (cap09conv_pos.r < 10){
        cap09conv_ctxSai.strokeStyle = "#2F6F9F";
        cap09conv_ctxSai.lineWidth = 2;
        cap09conv_ctxSai.strokeRect(cap09conv_pos.c*cap09conv_tam, cap09conv_pos.r*cap09conv_tam, cap09conv_tam, cap09conv_tam);
      }
    }

    function cap09conv_calcularPosicao(cap09conv_r, cap09conv_c){
      var cap09conv_k = cap09conv_KERNELS[cap09conv_kernelAtualKey].k;
      var cap09conv_soma = 0;
      var cap09conv_termos = [];
      for (var cap09conv_i=0; cap09conv_i<3; cap09conv_i++){
        for (var cap09conv_j=0; cap09conv_j<3; cap09conv_j++){
          var cap09conv_valEnt = cap09conv_imgEntradaBase[cap09conv_r+cap09conv_i][cap09conv_c+cap09conv_j];
          var cap09conv_valKer = cap09conv_k[cap09conv_i][cap09conv_j];
          var cap09conv_prod = cap09conv_valEnt * cap09conv_valKer;
          cap09conv_soma += cap09conv_prod;
          cap09conv_termos.push(cap09conv_prod >= 0 ? cap09conv_prod.toFixed(0) : '(' + cap09conv_prod.toFixed(0) + ')');
        }
      }
      return { valFinal: cap09conv_soma, expressao: cap09conv_termos.join(" + ") };
    }

    function cap09conv_atualizarCalculoTexto(){
      if (cap09conv_pos.r >= 10) {
        cap09conv_calcTxt.textContent = "Convoluzione Completata! Tutti i 100 pixel della mappa di output sono stati generati.";
        return;
      }
      var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
      cap09conv_calcTxt.textContent = 'Posizione (' + cap09conv_pos.r + ',' + cap09conv_pos.c + ') → z = ' + cap09conv_obj.expressao + ' = ' + cap09conv_obj.valFinal.toFixed(2);
    }

    function cap09conv_avancarPasso(){
      if (cap09conv_pos.r >= 10) return;
      var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
      cap09conv_saida[cap09conv_pos.r][cap09conv_pos.c] = cap09conv_obj.valFinal;
      cap09conv_pos.c++;
      if (cap09conv_pos.c >= 10){ cap09conv_pos.c = 0; cap09conv_pos.r++; }
      cap09conv_render();
    }

    function cap09conv_calcularTudo(){
      while (cap09conv_pos.r < 10) {
        var cap09conv_obj = cap09conv_calcularPosicao(cap09conv_pos.r, cap09conv_pos.c);
        cap09conv_saida[cap09conv_pos.r][cap09conv_pos.c] = cap09conv_obj.valFinal;
        cap09conv_pos.c++;
        if (cap09conv_pos.c >= 10){ cap09conv_pos.c = 0; cap09conv_pos.r++; }
      }
      cap09conv_render();
    }

    function cap09conv_resetar(){
      cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
      cap09conv_pos = {r:0, c:0};
      cap09conv_render();
    }

    function cap09conv_resetarECalcular(){
      cap09conv_saida = Array.from({length:10}, function(){ return Array(10).fill(null); });
      cap09conv_pos = {r:0, c:0};
      cap09conv_calcularTudo();
    }

    function cap09conv_render(){
      cap09conv_desenharEntrada();
      cap09conv_desenharKernel();
      cap09conv_desenharSaida();
      cap09conv_posTxt.textContent = cap09conv_pos.r < 10 ? '(' + cap09conv_pos.r + ', ' + cap09conv_pos.c + ')' : 'concluído';
      cap09conv_atualizarCalculoTexto();
    }

    function cap09conv_trocarImagem(cap09conv_id, cap09conv_btn){
      [cap09conv_btnCasa, cap09conv_btnFeliz, cap09conv_btnTriste].forEach(function(b){ b.classList.remove('cap09conv_active'); });
      cap09conv_btn.classList.add('cap09conv_active');
      cap09conv_imgAtualId = cap09conv_id;
      cap09conv_imgEntradaBase = cap09conv_converterLinhas(cap09conv_IMAGENS[cap09conv_id]);
      cap09conv_resetarECalcular();
    }

    function cap09conv_trocarKernel(cap09conv_key, cap09conv_btn){
      [cap09conv_btnSobelV, cap09conv_btnSobelH, cap09conv_btnSharpen, cap09conv_btnIdentidade].forEach(function(b){ b.classList.remove('cap09conv_active'); });
      cap09conv_btn.classList.add('cap09conv_active');
      cap09conv_kernelAtualKey = cap09conv_key;
      cap09conv_resetarECalcular();
    }

    cap09conv_btnCasa.addEventListener('click', function(){ cap09conv_trocarImagem("casa", cap09conv_btnCasa); });
    cap09conv_btnFeliz.addEventListener('click', function(){ cap09conv_trocarImagem("feliz", cap09conv_btnFeliz); });
    cap09conv_btnTriste.addEventListener('click', function(){ cap09conv_trocarImagem("triste", cap09conv_btnTriste); });

    cap09conv_btnSobelV.addEventListener('click', function(){ cap09conv_trocarKernel("sobelV", cap09conv_btnSobelV); });
    cap09conv_btnSobelH.addEventListener('click', function(){ cap09conv_trocarKernel("sobelH", cap09conv_btnSobelH); });
    cap09conv_btnSharpen.addEventListener('click', function(){ cap09conv_trocarKernel("sharpen", cap09conv_btnSharpen); });
    cap09conv_btnIdentidade.addEventListener('click', function(){ cap09conv_trocarKernel("identidade", cap09conv_btnIdentidade); });

    cap09conv_root.querySelector('#cap09conv_btnPasso').addEventListener('click', cap09conv_avancarPasso);
    cap09conv_root.querySelector('#cap09conv_btnTudo').addEventListener('click', cap09conv_calcularTudo);
    cap09conv_root.querySelector('#cap09conv_btnReset').addEventListener('click', cap09conv_resetar);

    cap09conv_resetarECalcular();
  
  }

  function cap09conv_tryInit(){
    var cap09conv_root = document.getElementById('sim-09-convolucao');
    if(cap09conv_root) cap09conv_init(cap09conv_root); else setTimeout(cap09conv_tryInit, 200);
  }
  cap09conv_tryInit();
})();
</script>
''')

**Figura 9.2:** Simulatore interattivo di convoluzione 2D classica: scegli tra le tre immagini sintetiche di input 12×12 (casa, faccia felice o triste) e un filtro 3×3 (Sobel V, Sobel H, Nitidezza o Identità) e procedi passo dopo passo per osservare come i prodotti interni locali del campo recettivo costruiscono la mappa delle caratteristiche cella per cella.


<figure id="fig-09-sim-09-convolucao">
  <img src="imagens/fig-09-sim-09-convolucao.png" alt=" Simulatore interattivo di convoluzione 2D classica: scegli tra le tre immagini sintetiche di input 12×12 (casa, faccia felice o triste) e un filtro 3×3 (Sobel V, Sobel H, Nitidezza o Identità) e procedi passo dopo passo per osservare come i prodotti interni locali del campo recettivo costruiscono la mappa delle caratteristiche cella per cella. " style="max-width:80%" />
  <figcaption><strong>Figura 9.2:</strong>  Simulatore interattivo di convoluzione 2D classica: scegli tra le tre immagini sintetiche di input 12×12 (casa, faccia felice o triste) e un filtro 3×3 (Sobel V, Sobel H, Nitidezza o Identità) e procedi passo dopo passo per osservare come i prodotti interni locali del campo recettivo costruiscono la mappa delle caratteristiche cella per cella. </figcaption>
</figure>

Per comprendere come avviene questo apprendimento, è necessario studiare l'unità di base dell'elaborazione delle reti neurali: il **neurone artificiale**.

### 9.4.2 Neurone Artificiale

Il **neurone artificiale** (*artificial neuron*) è l'unità fondamentale di elaborazione di una rete neurale. Il suo primo modello matematico — un insieme di ingressi combinati e confrontati con una soglia — fu proposto da Mcculloch (1943), ancora privo di qualsiasi meccanismo di apprendimento. Il **Percettrone** (ROSENBLATT, 1958) fece avanzare questa formulazione introducendo una regola di aggiustamento dei pesi basata su esempi, diventando il primo modello di neurone artificiale capace di apprendere e la base delle architetture moderne di **Apprendimento Profondo** (*Deep Learning*). Il termine Apprendimento Profondo si riferisce all'uso di reti con più strati di elaborazione, in grado di apprendere rappresentazioni gerarchiche dei dati: i primi strati apprendono caratteristiche semplici, come bordi e trame, e gli strati più profondi combinano progressivamente queste rappresentazioni per identificare strutture e oggetti più complessi.

Ogni neurone riceve un insieme di ingressi, calcola una combinazione lineare di tali valori e applica una **funzione di attivazione** (*activation function*), producendo un unico valore di uscita. Matematicamente, la combinazione lineare è data da

$$
z=\sum_{i=1}^{n}w_i x_i+b,
$$

dove $x_i$ rappresentano gli ingressi, $w_i$ i pesi associati a ciascun ingresso e $b$ il **bias** (*bias*). L'uscita del neurone si ottiene applicando la funzione di attivazione:

$$
y=f(z).
$$

Nelle CNN, questo principio assume forme diverse a seconda dello strato. Negli **strati convoluzionali** (*convolutional layers*), ogni neurone elabora solo una piccola regione dell'ingresso, denominata **campo recettivo** (*receptive field*), preservando l'organizzazione spaziale dell'immagine. Negli **strati completamente connessi** (*fully connected layers*), ogni neurone riceve tutte le uscite dello strato precedente, combinando le caratteristiche estratte per produrre l'uscita finale della rete, come la classe attribuita all'immagine.

La [Figura 9.3](#fig-09-sim-09-neuronio) illustra il funzionamento di un neurone artificiale: il simulatore consente di modificare gli ingressi ($x_1$ e $x_2$), i pesi ($w_1$ e $w_2$), il bias ($b$) e la funzione di attivazione, osservando in tempo reale il calcolo della combinazione lineare e dell'uscita corrispondente.

In [3]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-neuronio" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-neuronio .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-neuronio .cn-modebtn {
      flex:1; text-align:center; padding:8px 10px; font-size:12px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:10px;
      transition:background .15s ease, color .15s ease;
    }
    #sim-09-neuronio .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-neuronio input[type=range] { accent-color:#2F6F9F; min-width:0; }
    #sim-09-neuronio .cn-tag {
      font-size:10px; font-weight:700; color:#8A8371; width:16px; text-align:center; flex-shrink:0;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulatore: Neurone Artificiale in una CNN</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">y = f(∑ wᵢxᵢ + b)</span>
  </div>

  <div style="padding:18px 20px;background:#FFFFFF;overflow:auto">

    <!-- Alternador de contexto: camada convolucional vs. totalmente conectada -->
    <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:12px;padding:4px;margin-bottom:14px;max-width:480px;margin-left:auto;margin-right:auto;">
      <button id="cap09neuronio_modeConv" class="cn-modebtn active">🧩 Strato Convoluzionale</button>
      <button id="cap09neuronio_modeFC" class="cn-modebtn">🔗 Strato Totalmente Connesso</button>
    </div>

    <!-- Painel de Controles -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));gap:12px;align-items:start;">

        <div>
          <label id="cap09neuronio_lblX1" style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Pixel x₁ (normalizzato) / Peso del kernel w₁</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">x₁</span>
            <input type="range" id="cap09neuronio_x1" min="-3" max="3" step="0.1" value="1.0" style="flex:1;">
            <span class="cn-tag cn-mono">w₁</span>
            <input type="range" id="cap09neuronio_w1" min="-3" max="3" step="0.1" value="0.8" style="flex:1;">
          </div>
        </div>

        <div>
          <label id="cap09neuronio_lblX2" style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Pixel x₂ (normalizzato) / Peso del kernel w₂</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">x₂</span>
            <input type="range" id="cap09neuronio_x2" min="-3" max="3" step="0.1" value="-1.5" style="flex:1;">
            <span class="cn-tag cn-mono">w₂</span>
            <input type="range" id="cap09neuronio_w2" min="-3" max="3" step="0.1" value="0.5" style="flex:1;">
          </div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:4px;">Bias (b) e Funzione f(z)</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <span class="cn-tag cn-mono">b</span>
            <input type="range" id="cap09neuronio_bias" min="-3" max="3" step="0.1" value="0.2" style="flex:1;">
            <select id="cap09neuronio_func" style="font-size:11px;padding:5px 6px;border-radius:6px;border:1px solid #ccc;background:#fff;">
              <option value="relu">ReLU</option>
              <option value="sigmoid">Sigmoide</option>
              <option value="step">Gradino (Step)</option>
              <option value="identity">Identità</option>
            </select>
          </div>
        </div>

      </div>
      <div id="cap09neuronio_notaEscala" style="font-size:10.5px;color:#8A8371;margin-top:10px;padding-top:8px;border-top:1px dashed #E9E3D3;">
        I valori di x variano da -3 a 3 perché, in una CNN, i pixel (0–255) vengono <b>normalizzati</b> prima di entrare nella rete. Il disegno accanto traduce questo valore normalizzato di nuovo in una tonalità di grigio, solo per dare un'intuizione visiva — i numeri che contano per il calcolo sono quelli delle barre.
      </div>
    </div>

    <!-- Área Gráfica: Esquema do Neurônio + Curva -->
    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div id="cap09neuronio_diagTitle" style="font-size:11px;font-weight:600;text-align:center;margin-bottom:6px;color:#4b5563;">Campo Recettivo → Convoluzione → Mappa delle Caratteristiche</div>
        <canvas id="cap09neuronio_canvasEsquema" style="width:320px;height:230px;"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:6px;color:#4b5563;">Attivazione in f(z)</div>
        <canvas id="cap09neuronio_canvasCurva" style="width:260px;height:230px;"></canvas>
      </div>
    </div>

    <!-- Legenda contextual -->
    <div id="cap09neuronio_caption" style="font-size:11.5px;line-height:1.5;color:#5b5647;background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:10px 12px;margin-top:14px;">
      <b>x₁, x₂</b> = intensità dei pixel nel campo recettivo · <b>w₁, w₂</b> = pesi del kernel (filtro) · <b>z</b> = risultato della convoluzione in questa posizione · <b>y</b> = valore del pixel prodotto nella mappa delle caratteristiche, dopo l'attivazione.
    </div>

    <!-- Status numérico estilo terminal -->
    <div id="cap09neuronio_valTxt" class="cn-mono" style="text-align:center;font-size:12px;margin-top:12px;color:#7EE7C6;background:#1B2430;padding:10px 12px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;">
      z = (1.00 × 0.80) + (-1.50 × 0.50) + 0.20 = 0.25 → y = 0.25
    </div>

  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var canvasEsquema = root.querySelector('#cap09neuronio_canvasEsquema');
    var canvasCurva   = root.querySelector('#cap09neuronio_canvasCurva');
    var ctxEsquema = canvasEsquema.getContext('2d');
    var ctxCurva   = canvasCurva.getContext('2d');

    // Escala para telas de alta resolução (retina)
    function prepararCanvas(canvas, ctx, cssW, cssH){
      var dpr = window.devicePixelRatio || 1;
      canvas.width = cssW * dpr;
      canvas.height = cssH * dpr;
      ctx.scale(dpr, dpr);
    }
    prepararCanvas(canvasEsquema, ctxEsquema, 320, 230);
    prepararCanvas(canvasCurva, ctxCurva, 260, 230);

    var inX1 = root.querySelector('#cap09neuronio_x1');
    var inW1 = root.querySelector('#cap09neuronio_w1');
    var inX2 = root.querySelector('#cap09neuronio_x2');
    var inW2 = root.querySelector('#cap09neuronio_w2');
    var inB  = root.querySelector('#cap09neuronio_bias');
    var selF = root.querySelector('#cap09neuronio_func');
    var valTxt = root.querySelector('#cap09neuronio_valTxt');
    var caption = root.querySelector('#cap09neuronio_caption');
    var diagTitle = root.querySelector('#cap09neuronio_diagTitle');
    var lblX1 = root.querySelector('#cap09neuronio_lblX1');
    var lblX2 = root.querySelector('#cap09neuronio_lblX2');
    var notaEscala = root.querySelector('#cap09neuronio_notaEscala');
    var btnConv = root.querySelector('#cap09neuronio_modeConv');
    var btnFC   = root.querySelector('#cap09neuronio_modeFC');

    var modo = 'conv'; // 'conv' | 'fc'

    var CORES = {
      pos: "#1E8F6F",      // peso/conexão positiva
      neg: "#C1443A",      // peso/conexão negativa
      bias: "#C08A2E",     // viés
      saida: "#2F5FA8",    // sinal de saída
      texto: "#26241D",
      textoSuave: "#6b7280",
      grade: "#EFEAdd"
    };

    function calcularAtivacao(z, tipo){
      if (tipo === 'relu') return Math.max(0, z);
      if (tipo === 'sigmoid') return 1 / (1 + Math.exp(-z));
      if (tipo === 'step') return z >= 0 ? 1 : 0;
      return z; // identity
    }

    // Converte um valor de entrada (-3..3) em um tom de cinza 0-255 (intensidade de pixel)
    function valorParaCinza(v){
      var t = Math.max(0, Math.min(1, (v + 3) / 6));
      return Math.round(t * 255);
    }
    // Converte a saída y (que pode ter faixas diferentes conforme f) em cinza 0-255
    function saidaParaCinza(y, tipo){
      var t;
      if (tipo === 'sigmoid' || tipo === 'step') t = y;
      else t = Math.max(0, Math.min(1, (y + 3) / 6));
      return Math.round(Math.max(0, Math.min(1, t)) * 255);
    }

    function desenharPixel(ctx, cx, cy, lado, cinza, corBorda){
      var g = "rgb(" + cinza + "," + cinza + "," + cinza + ")";
      ctx.fillStyle = g;
      ctx.fillRect(cx - lado/2, cy - lado/2, lado, lado);
      ctx.strokeStyle = corBorda || "#9ca3af";
      ctx.lineWidth = 1.2;
      ctx.strokeRect(cx - lado/2, cy - lado/2, lado, lado);
    }

    function chipPeso(ctx, cx, cy, valor){
      var cor = valor >= 0 ? CORES.pos : CORES.neg;
      ctx.font = "600 10px 'JetBrains Mono', monospace";
      var texto = (valor>=0?"+":"") + valor.toFixed(1);
      var w = ctx.measureText(texto).width + 10;
      ctx.fillStyle = cor;
      roundRect(ctx, cx - w/2, cy - 9, w, 18, 9);
      ctx.fill();
      ctx.fillStyle = "#fff";
      ctx.textAlign = "center";
      ctx.textBaseline = "middle";
      ctx.fillText(texto, cx, cy+1);
    }

    function roundRect(ctx, x, y, w, h, r){
      ctx.beginPath();
      ctx.moveTo(x+r, y);
      ctx.arcTo(x+w, y, x+w, y+h, r);
      ctx.arcTo(x+w, y+h, x, y+h, r);
      ctx.arcTo(x, y+h, x, y, r);
      ctx.arcTo(x, y, x+w, y, r);
      ctx.closePath();
    }

    function seta(ctx, x1,y1,x2,y2,cor){
      ctx.strokeStyle = cor; ctx.lineWidth = 2;
      ctx.beginPath(); ctx.moveTo(x1,y1); ctx.lineTo(x2,y2); ctx.stroke();
      var ang = Math.atan2(y2-y1, x2-x1);
      ctx.fillStyle = cor;
      ctx.beginPath();
      ctx.moveTo(x2,y2);
      ctx.lineTo(x2 - 7*Math.cos(ang-0.4), y2 - 7*Math.sin(ang-0.4));
      ctx.lineTo(x2 - 7*Math.cos(ang+0.4), y2 - 7*Math.sin(ang+0.4));
      ctx.closePath(); ctx.fill();
    }

    // ---------- MODO CONVOLUÇÃO ----------
    function desenharConvolucao(x1, w1, x2, w2, b, z, y, tipo){
      var W=320, H=230;
      ctxEsquema.clearRect(0,0,W,H);
      ctxEsquema.textAlign = "center"; ctxEsquema.textBaseline = "middle";

      var cinzaX1 = valorParaCinza(x1), cinzaX2 = valorParaCinza(x2);
      var cinzaY  = saidaParaCinza(y, tipo);

      // Campo receptivo (patch de imagem de entrada) — dois pixels empilhados
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Campo recettivo", 48, 14);

      desenharPixel(ctxEsquema, 48, 48, 44, cinzaX1, "#9ca3af");
      desenharPixel(ctxEsquema, 48, 118, 44, cinzaX2, "#9ca3af");
      ctxEsquema.fillStyle = cinzaX1 > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("x₁=" + x1.toFixed(1), 48, 48);
      ctxEsquema.fillStyle = cinzaX2 > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.fillText("x₂=" + x2.toFixed(1), 48, 118);

      // CORREÇÃO AQUI: As setas agora param na borda esquerda do bloco do kernel (x=118)
      seta(ctxEsquema, 70, 48, 118, 72, w1 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 70, 118, 118, 96, w2 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 140, 45, 140, 62, b >= 0 ? CORES.bias : CORES.neg);

      // CORREÇÃO AQUI: Chips dos pesos centralizados no meio da seta (x=94)
      chipPeso(ctxEsquema, 94, 60, w1);
      chipPeso(ctxEsquema, 94, 107, w2);

      // Viés
      ctxEsquema.fillStyle = "#FDF0DA";
      ctxEsquema.beginPath(); ctxEsquema.arc(140, 26, 19, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.bias; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#8a5a12"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("b=" + b.toFixed(1), 140, 26);

      // Nó de convolução (kernel * patch) — ocupa de x=118 a x=162
      ctxEsquema.fillStyle = "#DCE8F5";
      roundRect(ctxEsquema, 118, 62, 44, 44, 10); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#2F5FA8"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#1e3a5f"; ctxEsquema.font = "600 12px Inter, sans-serif";
      ctxEsquema.fillText("⊛", 140, 78);
      ctxEsquema.font = "9px Inter, sans-serif";
      ctxEsquema.fillText("kernel", 140, 94);

      // Seta para o mapa de características
      seta(ctxEsquema, 162, 84, 202, 84, CORES.saida);

      // Mapa de características (mini tira com o pixel de saída em destaque)
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Mappa delle caratteristiche", 265, 14);

      var vizinhos = [190, 190];
      desenharPixel(ctxEsquema, 224, 84, 26, vizinhos[0], "#d1d5db");
      ctxEsquema.save();
      desenharPixel(ctxEsquema, 265, 84, 40, cinzaY, "#2F5FA8");
      ctxEsquema.lineWidth = 2.4; ctxEsquema.strokeStyle = CORES.saida;
      ctxEsquema.strokeRect(265-21, 84-21, 42, 42);
      ctxEsquema.fillStyle = cinzaY > 140 ? "#111827" : "#f3f4f6";
      ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("y=" + y.toFixed(1), 265, 84);
      ctxEsquema.restore();
      desenharPixel(ctxEsquema, 306, 84, 26, vizinhos[1], "#d1d5db");

      ctxEsquema.fillStyle = "#9ca3af"; ctxEsquema.font = "8px Inter, sans-serif";
      ctxEsquema.fillText("(il kernel scivola →)", 265, 212);
    }

    // ---------- MODO TOTALMENTE CONECTADA ----------
    function desenharFC(x1, w1, x2, w2, b, z, y){
      var W=320, H=230;
      ctxEsquema.clearRect(0,0,W,H);
      ctxEsquema.textAlign = "center"; ctxEsquema.textBaseline = "middle";

      seta(ctxEsquema, 55, 60, 155, 118, w1 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 55, 176, 155, 118, w2 >= 0 ? CORES.pos : CORES.neg);
      seta(ctxEsquema, 170, 45, 170, 100, b  >= 0 ? CORES.bias : CORES.neg);
      seta(ctxEsquema, 195, 118, 260, 118, CORES.saida);

      // Entrada x1
      ctxEsquema.fillStyle = "#EDEDE7"; ctxEsquema.beginPath(); ctxEsquema.arc(45, 60, 24, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#9ca3af"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = CORES.texto; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("x₁=" + x1.toFixed(1), 45, 60);

      // Entrada x2
      ctxEsquema.fillStyle = "#EDEDE7"; ctxEsquema.beginPath(); ctxEsquema.arc(45, 176, 24, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#9ca3af"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = CORES.texto;
      ctxEsquema.fillText("x₂=" + x2.toFixed(1), 45, 176);

      chipPeso(ctxEsquema, 108, 92, w1);
      chipPeso(ctxEsquema, 108, 148, w2);

      // Viés
      ctxEsquema.fillStyle = "#FDF0DA"; ctxEsquema.beginPath(); ctxEsquema.arc(170, 30, 20, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.bias; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#8a5a12"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("b=" + b.toFixed(1), 170, 30);

      // Soma / ativação
      ctxEsquema.fillStyle = "#DCE8F5"; ctxEsquema.beginPath(); ctxEsquema.arc(172, 118, 28, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = "#2F5FA8"; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#1e3a5f"; ctxEsquema.font = "12px Inter, sans-serif";
      ctxEsquema.fillText("∑, f", 172, 118);

      // Saída
      ctxEsquema.fillStyle = "#DCEEE6"; ctxEsquema.beginPath(); ctxEsquema.arc(280, 118, 26, 0, 2*Math.PI); ctxEsquema.fill();
      ctxEsquema.strokeStyle = CORES.pos; ctxEsquema.stroke();
      ctxEsquema.fillStyle = "#14532d"; ctxEsquema.font = "600 10px 'JetBrains Mono', monospace";
      ctxEsquema.fillText("y=" + y.toFixed(2), 280, 118);

      ctxEsquema.fillStyle = CORES.textoSuave; ctxEsquema.font = "9px Inter, sans-serif";
      ctxEsquema.fillText("neurone dello strato totalmente connesso", 172, 210);
    }

    function desenharCurva(z, y, tipo){
      var W = 260, H = 230;
      var origemX = 130, origemY = 165;
      var escalaX = 20, escalaY = 40;

      ctxCurva.clearRect(0,0,W,H);

      // Grade sutil
      ctxCurva.strokeStyle = CORES.grade; ctxCurva.lineWidth = 1;
      for (var gx = 10; gx <= W-10; gx += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(gx,10); ctxCurva.lineTo(gx,H-10); ctxCurva.stroke(); }
      for (var gy = 10; gy <= H-10; gy += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(10,gy); ctxCurva.lineTo(W-10,gy); ctxCurva.stroke(); }

      // Eixos
      ctxCurva.strokeStyle = "#9ca3af"; ctxCurva.lineWidth = 1.3;
      ctxCurva.beginPath();
      ctxCurva.moveTo(10, origemY); ctxCurva.lineTo(W-10, origemY);
      ctxCurva.moveTo(origemX, 10); ctxCurva.lineTo(origemX, H-10);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280"; ctxCurva.font = "10px Inter, sans-serif"; ctxCurva.textAlign = "left";
      ctxCurva.fillText("z", W-18, origemY-6);
      ctxCurva.fillText("y", origemX+6, 16);

      // Curva da função
      ctxCurva.strokeStyle = "#2F5FA8"; ctxCurva.lineWidth = 2.4;
      ctxCurva.beginPath();
      var primeiro = true;
      for (var px = -5; px <= 5; px += 0.1) {
        var valY = calcularAtivacao(px, tipo);
        var cx = origemX + px * escalaX;
        var cy = origemY - valY * escalaY;
        if (primeiro) { ctxCurva.moveTo(cx, cy); primeiro = false; }
        else { ctxCurva.lineTo(cx, cy); }
      }
      ctxCurva.stroke();

      // Ponto atual
      var ptX = Math.max(10, Math.min(W-10, origemX + z * escalaX));
      var ptY = Math.max(10, Math.min(H-10, origemY - y * escalaY));

      ctxCurva.strokeStyle = "#C1443A"; ctxCurva.setLineDash([3,3]); ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(ptX, origemY); ctxCurva.lineTo(ptX, ptY); ctxCurva.lineTo(origemX, ptY);
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);

      ctxCurva.fillStyle = "#C1443A";
      ctxCurva.beginPath(); ctxCurva.arc(ptX, ptY, 5, 0, 2*Math.PI); ctxCurva.fill();
      ctxCurva.strokeStyle = "#fff"; ctxCurva.lineWidth = 1.5; ctxCurva.stroke();
    }

    function atualizarLegendas(){
      if (modo === 'conv'){
        lblX1.textContent = "Pixel x₁ (normalizzato) / Peso del kernel w₁";
        lblX2.textContent = "Pixel x₂ (normalizzato) / Peso del kernel w₂";
        diagTitle.textContent = "Campo Recettivo → Convoluzione → Mappa delle Caratteristiche";
        caption.innerHTML = "<b>🧩 Preservação Espacial:</b> Na camada convolucional, a operação ocorre localmente via campo receptivo. O resultado (y) mantém uma posição bem definida no mapa de características 2D, preservando a vizinhança e a estrutura geométrica dos pixels.";
        notaEscala.innerHTML = "Os valores de x variam de -3 a 3 porque, em uma CNN, os pixels (0–255) são <b>normalizados</b> antes de entrar na rede. O desenho ao lado traduz esse valor normalizado de volta em um tom de cinza, só para dar intuição visual — os números que valem para a conta são os das barras.";
      } else {
        lblX1.textContent = "Attributo x₁ (feature) / Peso w₁";
        lblX2.textContent = "Attributo x₂ (feature) / Peso w₂";
        diagTitle.textContent = "Vettore di Attributi → Neurone → Uscita";
        caption.innerHTML = "<b>🔗 Perda da Informação Espacial:</b> Na camada totalmente conectada, os mapas de características são achatados (flatten) em um vetor 1D. Como o neurônio se conecta a todas as entradas indiferenciadamente, a noção de 'vizinho de cima/lado' é destruída em prol de uma decisão global.";
        notaEscala.innerHTML = "Aqui x₁, x₂ representam atributos já extraídos (não pixels), tipicamente padronizados para uma faixa pequena como esta antes de entrarem na camada.";
      }
    }

    function atualizar(){
      var x1 = parseFloat(inX1.value);
      var w1 = parseFloat(inW1.value);
      var x2 = parseFloat(inX2.value);
      var w2 = parseFloat(inW2.value);
      var b  = parseFloat(inB.value);
      var tipo = selF.value;

      var z = (x1 * w1) + (x2 * w2) + b;
      var y = calcularAtivacao(z, tipo);

      if (modo === 'conv') desenharConvolucao(x1, w1, x2, w2, b, z, y, tipo);
      else desenharFC(x1, w1, x2, w2, b, z, y);

      desenharCurva(z, y, tipo);

      valTxt.textContent = 'z = (' + x1.toFixed(2) + ' × ' + w1.toFixed(2) + ') + (' +
                           x2.toFixed(2) + ' × ' + w2.toFixed(2) + ') + (' + b.toFixed(2) +
                           ') = ' + z.toFixed(2) + '  →  y = ' + selF.options[selF.selectedIndex].text + '(z) = ' + y.toFixed(2);
    }

    function definirModo(novoModo){
      modo = novoModo;
      btnConv.classList.toggle('active', modo === 'conv');
      btnFC.classList.toggle('active', modo === 'fc');
      atualizarLegendas();
      atualizar();
    }

    btnConv.addEventListener('click', function(){ definirModo('conv'); });
    btnFC.addEventListener('click', function(){ definirModo('fc'); });

    inX1.addEventListener('input', atualizar);
    inW1.addEventListener('input', atualizar);
    inX2.addEventListener('input', atualizar);
    inW2.addEventListener('input', atualizar);
    inB.addEventListener('input', atualizar);
    selF.addEventListener('change', atualizar);

    atualizarLegendas();
    atualizar();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-neuronio');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.3:** Simulatore interattivo del neurone artificiale in un contesto di CNN: passa da un neurone di un livello convoluzionale (dove x_i sono intensità di pixel in un campo ricettivo e w_i sono pesi del *kernel*) a un neurone di un livello completamente connesso, regolando input, pesi, bias e funzione di attivazione per visualizzare il calcolo di z e dell


<figure id="fig-09-sim-09-neuronio">
  <img src="imagens/fig-09-sim-09-neuronio.png" alt=" Simulatore interattivo del neurone artificiale in un contesto di CNN: passa da un neurone di un livello convoluzionale (dove x_i sono intensità di pixel in un campo ricettivo e w_i sono pesi del *kernel*) a un neurone di un livello completamente connesso, regolando input, pesi, bias e funzione di attivazione per visualizzare il calcolo di z e dell'output y in tempo reale. " style="max-width:80%" />
  <figcaption><strong>Figura 9.3:</strong>  Simulatore interattivo del neurone artificiale in un contesto di CNN: passa da un neurone di un livello convoluzionale (dove x_i sono intensità di pixel in un campo ricettivo e w_i sono pesi del *kernel*) a un neurone di un livello completamente connesso, regolando input, pesi, bias e funzione di attivazione per visualizzare il calcolo di z e dell'output y in tempo reale. </figcaption>
</figure>

In una CNN, migliaia di neuroni si organizzano in strati con funzioni specifiche: i primi sono responsabili dell'estrazione delle caratteristiche tramite la convoluzione, mentre gli ultimi eseguono la classificazione a partire dalle caratteristiche apprese.

### 9.4.3 Livello Convoluzionale

Il **livello convoluzionale** (*convolutional layer*) è responsabile dell'estrazione delle caratteristiche dell'immagine. Ogni filtro genera una **mappa delle caratteristiche** (*feature map*), la cui intensità in ciascuna posizione indica la risposta del filtro alla regione corrispondente dell'input.

L'operazione eseguita segue lo stesso principio di scorrimento e combinazione locale presentato nel **Capitolo 3** per la convoluzione spaziale. Considerando un *kernel* $K$ di dimensione $k \times k$, il valore prodotto nella posizione $(i,j)$ è dato da

$$
F(i,j)=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1}K(u,v)\,I(i+u,j+v).
$$

Vale la pena registrare una distinzione terminologica: l'espressione sopra corrisponde, formalmente, a una **correlazione incrociata** (*cross-correlation*), e non alla convoluzione matematica rigorosa, che richiede la riflessione del *kernel* prima della combinazione. La maggior parte dei *framework* di Deep Learning, incluso **PyTorch**, implementa questa operazione senza riflessione e la designa, per convenzione, come convoluzione — convenzione adottata anche in questo capitolo. Questa differenza non ha alcun effetto pratico sull'allenamento, poiché i coefficienti del *kernel* vengono appresi e non imposti in anticipo.

La differenza principale rispetto ai metodi classici risiede quindi nell'ottenimento del *kernel* $K$: nei filtri tradizionali, i suoi coefficienti sono definiti manualmente per evidenziare caratteristiche specifiche dell'immagine; nelle CNN, i coefficienti vengono inizializzati automaticamente e regolati durante l'allenamento tramite la **retropropagazione dell'errore** (*backpropagation*), rendendo ogni filtro specializzato nell'identificare pattern rilevanti per il compito in esame.

Due concetti caratterizzano questo livello:

- **Condivisione dei pesi** (*weight sharing*): lo stesso filtro viene applicato in tutte le posizioni dell'immagine, riducendo significativamente il numero di parametri del modello.
- **Campo recettivo** (*receptive field*): ogni neurone convoluzionale processa solo un piccolo intorno dell'immagine, preservando la struttura spaziale dei dati.

Impilando più livelli convoluzionali, la rete apprende una **gerarchia di caratteristiche**: i primi livelli tendono a rilevare pattern semplici, come bordi e trame, e i livelli più profondi combinano queste informazioni per rappresentare strutture progressivamente più complesse. Dopo la convoluzione, la mappa delle caratteristiche viene sottoposta a una funzione di attivazione, introducendo non linearità nel modello e ampliando la sua capacità di rappresentare relazioni complesse tra le variabili di input.

La [Figura 9.4](#fig-09-sim-09-camada-conv) presenta questo livello in modo interattivo.

In [4]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-camada-conv" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-camada-conv .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-camada-conv .cn-grouplabel {
      font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px;
    }
    #sim-09-camada-conv .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-camada-conv .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-camada-conv input[type=range] { accent-color:#2F6F9F; min-width:0; }
    #sim-09-camada-conv .cn-navbtn {
      width:26px;height:26px;border-radius:8px;border:1px solid #E4DCC8;background:#FAFAF7;
      color:#26241D;font-size:12px;cursor:pointer;display:flex;align-items:center;justify-content:center;
      transition:background .15s ease; flex-shrink:0;
    }
    #sim-09-camada-conv .cn-navbtn:hover { background:#F1EAD7; }
    #sim-09-camada-conv .cn-playbtn {
      padding:0 10px;height:26px;border-radius:8px;border:1px solid #2F6F9F;background:#EAF2FA;
      color:#2F6F9F;font-size:10.5px;font-weight:700;cursor:pointer;white-space:nowrap;flex-shrink:0;
    }
    #sim-09-camada-conv .cn-playbtn:hover { background:#DCEEFB; }
    #sim-09-camada-conv .cn-prodcell {
      border-radius:6px;padding:3px 2px;text-align:center;border:1px solid #e5e7eb;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulador: Operação de Convolução & Mapa de Características</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">F(i,j) = f(∑ K(u,v) · I(i+u, j+v))</span>
  </div>

  <div style="padding:12px 14px;background:#FFFFFF;overflow:auto">

    <!-- Seletores: Imagem de Entrada + Kernel, lado a lado para compactar -->
    <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px;">
      <div style="flex:1;min-width:230px;">
        <div class="cn-grouplabel">IMAGEM DE ENTRADA</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09camdconvimg_btnImgCasa" class="cn-modebtn active">🏠 Casa</button>
          <button id="cap09camdconvimg_btnImgFeliz" class="cn-modebtn">😊 Feliz</button>
          <button id="cap09camdconvimg_btnImgTriste" class="cn-modebtn">😢 Triste</button>
        </div>
      </div>
      <div style="flex:1;min-width:280px;">
        <div class="cn-grouplabel">KERNEL (FILTRO FIXO)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09camdconvimg_btnSobelV" class="cn-modebtn active">📐 Vertical</button>
          <button id="cap09camdconvimg_btnSobelH" class="cn-modebtn">📏 Horizontal</button>
          <button id="cap09camdconvimg_btnSharpen" class="cn-modebtn">✨ Nitidez</button>
          <button id="cap09camdconvimg_btnIdentity" class="cn-modebtn">🎯 Identidade</button>
        </div>
      </div>
    </div>

    <!-- Painel de Controles -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-bottom:10px;">
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));gap:10px;align-items:start;">

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Posição do Campo Receptivo</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <button id="cap09camdconvimg_btnAnterior" class="cn-navbtn" title="Passo anterior">◀</button>
            <input type="range" id="cap09camdconvimg_step" min="0" max="8" step="1" value="0" style="flex:1;">
            <button id="cap09camdconvimg_btnProximo" class="cn-navbtn" title="Próximo passo">▶</button>
            <button id="cap09camdconvimg_btnPlay" class="cn-playbtn">⏵ Auto</button>
            <button id="cap09camdconvimg_btnReiniciar" class="cn-navbtn" title="Reiniciar varredura">↺</button>
          </div>
          <div id="cap09camdconvimg_posLabel" class="cn-mono" style="font-size:10px;color:#8A8371;margin-top:4px;"></div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Função de Ativação f(z)</label>
          <div style="display:flex;gap:6px;align-items:center;">
            <select id="cap09camdconvimg_func" style="font-size:11px;padding:5px 6px;border-radius:6px;border:1px solid #ccc;background:#fff;width:100%;">
              <option value="relu">ReLU</option>
              <option value="identity">Identidade (Linear)</option>
              <option value="sigmoid">Sigmoide</option>
            </select>
          </div>
        </div>

        <div>
          <label style="font-size:11px;font-weight:600;color:#374151;display:block;margin-bottom:3px;">Preenchimento (Padding)</label>
          <div style="display:flex;gap:6px;align-items:center;padding-top:3px;">
            <input type="checkbox" id="cap09camdconvimg_padding" style="accent-color:#2F6F9F;">
            <span style="font-size:11px;color:#374151;font-weight:500;">Zero-Padding (p = 1)</span>
          </div>
          <div style="display:flex;gap:8px;align-items:center;margin-top:6px;font-size:9.5px;color:#6b7280;flex-wrap:wrap;">
            <span><span style="display:inline-block;width:10px;height:10px;background:#555;border:1px solid #999;vertical-align:middle;"></span> pixel real</span>
            <span><span style="display:inline-block;width:10px;height:10px;background:#111;border:1px dashed #C98A2E;vertical-align:middle;"></span> margem fixa da imagem (0)</span>
            <span><span style="display:inline-block;width:10px;height:10px;background:#DCEEFB;border:1px dashed #8AB4D8;vertical-align:middle;"></span> padding do algoritmo (0)</span>
          </div>
        </div>

      </div>
      <div id="cap09camdconvimg_notaEscala" style="font-size:10.5px;color:#8A8371;margin-top:8px;padding-top:6px;border-top:1px dashed #E9E3D3;">
        O mesmo kernel desliza sobre toda a imagem reutilizando seus coeficientes (<b>compartilhamento de pesos</b>). Cada imagem já vem cercada por uma margem fixa de 1 pixel de fundo (zeros, contorno tracejado âmbar), isolando a forma nos quatro lados. Escolha uma imagem e um kernel fixo acima, depois use ◀ ▶ ou "Auto" para percorrer o campo receptivo — o <b>Feature Map</b> à direita é preenchido célula a célula, na mesma ordem em que a convolução é calculada (as células ainda não visitadas aparecem como "···").
      </div>
      <div id="cap09camdconvimg_notaFormula" class="cn-mono" style="font-size:10.5px;color:#2F6F9F;margin-top:5px;"></div>
      <div id="cap09camdconvimg_notaOffset" style="font-size:10.5px;color:#8A8371;margin-top:4px;">
        📌 A saída F(i,j) vem do campo receptivo entre (i,j) e (i+2,j+2); seu centro real é (i+1,j+1) — <b>1 linha e 1 coluna abaixo/à direita</b> do índice usado para rotular a célula, sempre nas duas direções. Esse deslocamento só fica visível no eixo em que o kernel diferencia a imagem (por isso o Sobel V parece deslocar só para o lado, e o Sobel H, só para baixo).
      </div>
    </div>

    <!-- Área Gráfica: Esquema da Convolução + Curva de Ativação -->
    <div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <canvas id="cap09camdconvimg_canvasEsquema" style="width:580px;height:260px;"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Ativação em f(z)</div>
        <canvas id="cap09camdconvimg_canvasCurva" style="width:200px;height:190px;"></canvas>
      </div>
    </div>

    <!-- Painel de Cálculo Detalhado -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px 12px;margin-top:10px;">
      <div style="font-size:11px;font-weight:600;color:#4b5563;margin-bottom:8px;">🔍 Cálculo Detalhado no Campo Receptivo Atual</div>
      <div style="display:flex;gap:18px;align-items:center;flex-wrap:wrap;">
        <div id="cap09camdconvimg_gradeProdutos" style="display:grid;grid-template-columns:repeat(3,44px);gap:3px;"></div>
        <div id="cap09camdconvimg_expressaoSoma" style="font-size:11px;color:#374151;line-height:1.6;"></div>
      </div>
    </div>

    <!-- Legenda contextual (dinâmica: imagem escolhida + kernel escolhido) -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;margin-top:10px;">
      <div id="cap09camdconvimg_legendaImagem" style="flex:1;min-width:250px;font-size:11px;line-height:1.5;color:#5b5647;background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:8px 10px;"></div>
      <div id="cap09camdconvimg_legendaKernel" style="flex:1;min-width:250px;font-size:11px;line-height:1.5;color:#5b5647;background:#F1F6FB;border:1px solid #DCEEFB;border-radius:10px;padding:8px 10px;"></div>
    </div>

    <!-- Status numérico estilo terminal -->
    <div id="cap09camdconvimg_valTxt" class="cn-mono" style="text-align:center;font-size:11px;margin-top:10px;color:#7EE7C6;background:#1B2430;padding:8px 10px;border-radius:8px;box-shadow:inset 0 0 0 1px #2B3644;letter-spacing:.2px;overflow-x:auto;">
      z = 0.00 → y = ReLU(z) = 0.00
    </div>

  </div>
</div>

<script>
(function(){
  function initCamadaConvImg(root){
    if(!root || root.dataset.initConvImg) return;
    root.dataset.initConvImg = "1";

    var canvasEsquema = root.querySelector('#cap09camdconvimg_canvasEsquema');
    var canvasCurva   = root.querySelector('#cap09camdconvimg_canvasCurva');
    var ctxEsquema = canvasEsquema.getContext('2d');
    var ctxCurva   = canvasCurva.getContext('2d');

    function prepararCanvas(canvas, ctx, cssW, cssH){
      var dpr = window.devicePixelRatio || 1;
      canvas.width = cssW * dpr;
      canvas.height = cssH * dpr;
      ctx.scale(dpr, dpr);
    }
    prepararCanvas(canvasEsquema, ctxEsquema, 580, 260);
    prepararCanvas(canvasCurva, ctxCurva, 200, 190);

    var inStep   = root.querySelector('#cap09camdconvimg_step');
    var selF     = root.querySelector('#cap09camdconvimg_func');
    var chkP     = root.querySelector('#cap09camdconvimg_padding');
    var valTxt   = root.querySelector('#cap09camdconvimg_valTxt');
    var posLabel = root.querySelector('#cap09camdconvimg_posLabel');
    var notaFormula = root.querySelector('#cap09camdconvimg_notaFormula');
    var legendaKernel = root.querySelector('#cap09camdconvimg_legendaKernel');
    var legendaImagem = root.querySelector('#cap09camdconvimg_legendaImagem');
    var gradeProdutos = root.querySelector('#cap09camdconvimg_gradeProdutos');
    var expressaoSoma = root.querySelector('#cap09camdconvimg_expressaoSoma');

    var btnSobelV   = root.querySelector('#cap09camdconvimg_btnSobelV');
    var btnSobelH   = root.querySelector('#cap09camdconvimg_btnSobelH');
    var btnSharpen  = root.querySelector('#cap09camdconvimg_btnSharpen');
    var btnIdentity = root.querySelector('#cap09camdconvimg_btnIdentity');

    var btnImgCasa   = root.querySelector('#cap09camdconvimg_btnImgCasa');
    var btnImgFeliz  = root.querySelector('#cap09camdconvimg_btnImgFeliz');
    var btnImgTriste = root.querySelector('#cap09camdconvimg_btnImgTriste');

    var btnAnterior  = root.querySelector('#cap09camdconvimg_btnAnterior');
    var btnProximo   = root.querySelector('#cap09camdconvimg_btnProximo');
    var btnPlay      = root.querySelector('#cap09camdconvimg_btnPlay');
    var btnReiniciar = root.querySelector('#cap09camdconvimg_btnReiniciar');

    var CORES = {
      pos: "#1E8F6F",
      neg: "#C1443A",
      saida: "#2F5FA8",
      textoSuave: "#6b7280",
      padding: "#DCEEFB",
      paddingBorda: "#8AB4D8",
      paddingTexto: "#2F6F9F",
      margemBase: "#C98A2E"
    };

    var KERNELS = {
      sobelV: [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
      sobelH: [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
      sharpen: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
      identity: [[0, 0, 0], [0, 1, 0], [0, 0, 0]]
    };

    var KERNEL_INFO = {
      sobelV: { emoji: "📐", nome: "Sobel Vertical", desc: "responde fortemente a mudanças bruscas de intensidade na direção horizontal — por isso realça <b>bordas verticais</b> da imagem." },
      sobelH: { emoji: "📏", nome: "Sobel Horizontal", desc: "responde a mudanças bruscas de intensidade na direção vertical — por isso realça <b>bordas horizontais</b> da imagem." },
      sharpen: { emoji: "✨", nome: "Nitidez (Sharpen)", desc: "amplifica o pixel central em relação aos vizinhos, aumentando o contraste local e destacando detalhes finos." },
      identity: { emoji: "🎯", nome: "Identidade", desc: "reproduz o valor original do pixel central sem alterá-lo — útil como referência de que a convolução não introduz distorção por si só." }
    };

    // Três imagens de entrada 12×12 desenhadas como "arte ASCII": 'X' = pixel
    // aceso (1.0), '.' = pixel apagado (0.0). Todas com o mesmo tamanho fixo,
    // para que o simulador continue mostrando apenas a operação de convolução
    // (sem qualquer etapa de classificação).
    var IMAGENS = {
      casa: {
        emoji: "🏠",
        nome: "Casa",
        desc: "combina bordas diagonais no telhado, bordas verticais retas nas paredes e uma porta recortada no centro. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "...XXXXXX...",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXXX..XXXX.",
          ".XXXX..XXXX.",
          ".XXXX..XXXX.",
          "............"
        ]
      },
      feliz: {
        emoji: "😊",
        nome: "Rosto Feliz",
        desc: "um contorno arredondado com dois olhos e uma boca que se abre mais na parte de cima e se fecha em direção ao queixo. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXX.XX.XXX.",
          ".XXXXXXXXXX.",
          ".XX......XX.",
          ".XXX....XXX.",
          ".XXXXXXXXXX.",
          "..XXXXXXXX..",
          "............"
        ]
      },
      triste: {
        emoji: "😢",
        nome: "Rosto Triste",
        desc: "mesmo contorno arredondado do rosto feliz, mas com a boca invertida: mais estreita perto do nariz e mais larga perto do queixo, simulando cantos da boca virados para baixo. Toda a forma fica cercada por uma margem de fundo (zeros), para que o deslocamento entre o índice de saída e a borda real fique visível nos quatro lados.",
        linhas: [
          "............",
          "....XXXX....",
          "..XXXXXXXX..",
          ".XXXXXXXXXX.",
          ".XXXXXXXXXX.",
          ".XXX.XX.XXX.",
          ".XXXXXXXXXX.",
          ".XXX....XXX.",
          ".XX......XX.",
          ".XXXXXXXXXX.",
          "..XXXXXXXX..",
          "............"
        ]
      }
    };

    var kernelAtual = KERNELS.sobelV;
    var kernelAtualId = "sobelV";
    var imagemAtualId = "casa";
    var imgEntradaBase = null;
    var N_BASE = 12;
    var autoplayInterval = null;

    function converterLinhasParaMatriz(linhas){
      return linhas.map(function(linha){
        var pixels = [];
        for (var i = 0; i < linha.length; i++){
          pixels.push(linha[i] === 'X' ? 1.0 : 0.0);
        }
        return pixels;
      });
    }

    function calcularAtivacao(z, tipo){
      if (tipo === 'relu') return Math.max(0, z);
      if (tipo === 'sigmoid') return 1 / (1 + Math.exp(-z));
      return z;
    }

    // Retorna a matriz de entrada (com ou sem padding) e um mapa booleano
    // indicando quais células são padding artificial (para não confundi-las
    // com pixels reais de valor 0).
    function obterMatrizEntrada(comPadding){
      var dim = comPadding ? N_BASE + 2 : N_BASE;
      var matriz = [], mapaPad = [];
      for (var r = 0; r < dim; r++){
        var linhaVal = [], linhaPad = [];
        for (var c = 0; c < dim; c++){
          if (comPadding && (r === 0 || r === dim - 1 || c === 0 || c === dim - 1)){
            linhaVal.push(0.0);
            linhaPad.push(true);
          } else {
            var ri = comPadding ? r - 1 : r;
            var ci = comPadding ? c - 1 : c;
            linhaVal.push(imgEntradaBase[ri][ci]);
            linhaPad.push(false);
          }
        }
        matriz.push(linhaVal);
        mapaPad.push(linhaPad);
      }
      return { matriz: matriz, pad: mapaPad };
    }

    function desenharEsquema(passoIdx, tipoFunc, comPadding){
      ctxEsquema.clearRect(0, 0, 580, 260);

      var entrada = obterMatrizEntrada(comPadding);
      var img = entrada.matriz, mapaPad = entrada.pad;
      var dimImg = img.length;
      var dimOut = dimImg - 3 + 1;

      var maxSteps = (dimOut * dimOut) - 1;
      inStep.max = maxSteps;
      if (passoIdx > maxSteps) {
        passoIdx = maxSteps;
        inStep.value = maxSteps;
      }
      var rowOut = Math.floor(passoIdx / dimOut);
      var colOut = passoIdx % dimOut;

      var startX = 30, startY = 46;
      var cellSize = comPadding ? 12.5 : 14.5;

      ctxEsquema.textAlign = "center";
      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Entrada I (" + dimImg + "×" + dimImg + ")", startX + (dimImg * cellSize) / 2, startY - 16);

      ctxEsquema.font = "7px 'JetBrains Mono', monospace";
      for (var c0 = 0; c0 < dimImg; c0++){
        ctxEsquema.fillText(String(c0), startX + c0 * cellSize + cellSize / 2, startY - 5);
      }
      ctxEsquema.textAlign = "right";
      for (var r0 = 0; r0 < dimImg; r0++){
        ctxEsquema.fillText(String(r0), startX - 5, startY + r0 * cellSize + cellSize / 2 + 3);
      }
      ctxEsquema.textAlign = "center";

      for (var r = 0; r < dimImg; r++) {
        for (var c = 0; c < dimImg; c++) {
          var val = img[r][c];
          var x = startX + c * cellSize, y = startY + r * cellSize;

          if (mapaPad[r][c]) {
            ctxEsquema.fillStyle = CORES.padding;
            ctxEsquema.fillRect(x, y, cellSize, cellSize);
            ctxEsquema.strokeStyle = CORES.paddingBorda;
            ctxEsquema.setLineDash([2, 2]);
            ctxEsquema.lineWidth = 1;
            ctxEsquema.strokeRect(x, y, cellSize, cellSize);
            ctxEsquema.setLineDash([]);
            ctxEsquema.fillStyle = CORES.paddingTexto;
          } else {
            var g = Math.round(val * 255);
            ctxEsquema.fillStyle = "rgb(" + g + "," + g + "," + g + ")";
            ctxEsquema.fillRect(x, y, cellSize, cellSize);

            // Distingue a margem fixa da própria imagem (1px de zeros nos
            // quatro lados, embutida em imgEntradaBase) do padding opcional
            // do algoritmo: mesmo contorno tracejado, mas em âmbar.
            var riBase = comPadding ? r - 1 : r;
            var ciBase = comPadding ? c - 1 : c;
            var ehMargemBase = (riBase === 0 || riBase === N_BASE - 1 || ciBase === 0 || ciBase === N_BASE - 1);

            if (ehMargemBase) {
              ctxEsquema.strokeStyle = CORES.margemBase;
              ctxEsquema.setLineDash([2, 2]);
              ctxEsquema.lineWidth = 1;
              ctxEsquema.strokeRect(x, y, cellSize, cellSize);
              ctxEsquema.setLineDash([]);
            } else {
              ctxEsquema.strokeStyle = "#d1d5db";
              ctxEsquema.lineWidth = 1;
              ctxEsquema.strokeRect(x, y, cellSize, cellSize);
            }
            ctxEsquema.fillStyle = g > 140 ? "#374151" : "#f3f4f6";
          }
          ctxEsquema.font = "600 7px 'JetBrains Mono', monospace";
          ctxEsquema.fillText(val.toFixed(0), x + cellSize / 2, y + cellSize / 2 + 2.5);
        }
      }

      // Destacar Campo Receptivo
      var krX = startX + colOut * cellSize;
      var krY = startY + rowOut * cellSize;
      ctxEsquema.strokeStyle = "#C1443A";
      ctxEsquema.lineWidth = 2.2;
      ctxEsquema.strokeRect(krX, krY, 3 * cellSize, 3 * cellSize);

      // Calcular z, y e os 9 termos do produto no ponto atual
      var termos = [];
      var z = 0;
      for (var kr = 0; kr < 3; kr++) {
        for (var kc = 0; kc < 3; kc++) {
          var iv = img[rowOut + kr][colOut + kc];
          var kv = kernelAtual[kr][kc];
          var prod = iv * kv;
          termos.push({ i: iv, k: kv, p: prod });
          z += prod;
        }
      }
      var y = calcularAtivacao(z, tipoFunc);

      // Desenhar Kernel (K)
      var kCell = 17;
      var kStartX = startX + (dimImg * cellSize) + 16;
      var kStartY = startY + (dimImg * cellSize) / 2 - (3 * kCell) / 2;

      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Kernel K", kStartX + (3 * kCell) / 2, kStartY - 9);

      for (var kr2 = 0; kr2 < 3; kr2++) {
        for (var kc2 = 0; kc2 < 3; kc2++) {
          var kv2 = kernelAtual[kr2][kc2];
          var kx = kStartX + kc2 * kCell, ky = kStartY + kr2 * kCell;
          ctxEsquema.fillStyle = kv2 > 0 ? "#E6F4EA" : (kv2 < 0 ? "#FCE8E6" : "#F3F4F6");
          ctxEsquema.fillRect(kx, ky, kCell, kCell);
          ctxEsquema.strokeStyle = "#9ca3af";
          ctxEsquema.strokeRect(kx, ky, kCell, kCell);

          ctxEsquema.fillStyle = kv2 > 0 ? CORES.pos : (kv2 < 0 ? CORES.neg : "#374151");
          ctxEsquema.font = "600 9px 'JetBrains Mono', monospace";
          ctxEsquema.fillText(String(kv2), kx + kCell / 2, ky + kCell / 2 + 3);
        }
      }

      // Desenhar Feature Map (F) — revelado progressivamente, na mesma ordem
      // (varredura linha a linha) em que a convolução realmente é calculada.
      // Células além do passo atual ainda não foram "computadas" e aparecem
      // como pendentes ("···"), reforçando que o mapa é construído aos poucos.
      var outCell = comPadding ? 13.5 : 16;
      var outStartX = kStartX + 3 * kCell + 52;
      var outStartY = startY + (dimImg * cellSize) / 2 - (dimOut * outCell) / 2;

      ctxEsquema.font = "600 10px Inter, sans-serif";
      ctxEsquema.fillStyle = CORES.textoSuave;
      ctxEsquema.fillText("Feature Map F (" + dimOut + "×" + dimOut + ")", outStartX + (dimOut * outCell) / 2, outStartY - 16);

      ctxEsquema.font = "7px 'JetBrains Mono', monospace";
      for (var oc0 = 0; oc0 < dimOut; oc0++){
        ctxEsquema.fillText(String(oc0), outStartX + oc0 * outCell + outCell / 2, outStartY - 5);
      }
      ctxEsquema.textAlign = "right";
      for (var or0 = 0; or0 < dimOut; or0++){
        ctxEsquema.fillText(String(or0), outStartX - 5, outStartY + or0 * outCell + outCell / 2 + 3);
      }
      ctxEsquema.textAlign = "center";

      for (var orr = 0; orr < dimOut; orr++) {
        for (var occ = 0; occ < dimOut; occ++) {
          var linIdx = orr * dimOut + occ;
          var jaCalculado = linIdx <= passoIdx;
          var cx = outStartX + occ * outCell;
          var cy = outStartY + orr * outCell;
          var isAtual = (orr === rowOut && occ === colOut);

          if (!jaCalculado) {
            // Célula ainda pendente: ainda não "visitada" pela varredura.
            ctxEsquema.fillStyle = "#F7F5EE";
            ctxEsquema.fillRect(cx, cy, outCell, outCell);
            ctxEsquema.strokeStyle = "#d9d2bd";
            ctxEsquema.setLineDash([2, 2]);
            ctxEsquema.lineWidth = 1;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);
            ctxEsquema.setLineDash([]);
            ctxEsquema.fillStyle = "#b8ae94";
            ctxEsquema.font = "700 8px 'JetBrains Mono', monospace";
            ctxEsquema.fillText("·", cx + outCell / 2, cy + outCell / 2 + 2.5);
          } else {
            var oz = 0;
            for (var kr3 = 0; kr3 < 3; kr3++) {
              for (var kc3 = 0; kc3 < 3; kc3++) {
                oz += img[orr + kr3][occ + kc3] * kernelAtual[kr3][kc3];
              }
            }
            var oy = calcularAtivacao(oz, tipoFunc);
            var normY = tipoFunc === 'sigmoid' ? oy : Math.max(0, Math.min(1, (oy + 2) / 4));
            var og = Math.round(normY * 255);

            ctxEsquema.fillStyle = "rgb(" + og + "," + og + "," + og + ")";
            ctxEsquema.fillRect(cx, cy, outCell, outCell);
            ctxEsquema.strokeStyle = isAtual ? CORES.saida : "#d1d5db";
            ctxEsquema.lineWidth = isAtual ? 2.4 : 1;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);

            ctxEsquema.fillStyle = og > 140 ? "#374151" : "#f3f4f6";
            ctxEsquema.font = "600 6.5px 'JetBrains Mono', monospace";
            ctxEsquema.fillText(oy.toFixed(1), cx + outCell / 2, cy + outCell / 2 + 2.2);
          }

          if (isAtual) {
            ctxEsquema.strokeStyle = CORES.saida;
            ctxEsquema.lineWidth = 2.4;
            ctxEsquema.strokeRect(cx, cy, outCell, outCell);
          }
        }
      }

      return { z: z, y: y, posR: rowOut, posC: colOut, dimImg: dimImg, dimOut: dimOut, termos: termos, p: comPadding ? 1 : 0 };
    }

    function desenharCurva(z, y, tipo){
      var W = 200, H = 190;
      var origemX = 100, origemY = 135;
      var escalaX = 17, escalaY = 32;

      ctxCurva.clearRect(0,0,W,H);

      ctxCurva.strokeStyle = "#EFEAdd"; ctxCurva.lineWidth = 1;
      for (var gx = 10; gx <= W-10; gx += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(gx,10); ctxCurva.lineTo(gx,H-10); ctxCurva.stroke(); }
      for (var gy = 10; gy <= H-10; gy += 20){ ctxCurva.beginPath(); ctxCurva.moveTo(10,gy); ctxCurva.lineTo(W-10,gy); ctxCurva.stroke(); }

      ctxCurva.strokeStyle = "#9ca3af"; ctxCurva.lineWidth = 1.2;
      ctxCurva.beginPath();
      ctxCurva.moveTo(10, origemY); ctxCurva.lineTo(W-10, origemY);
      ctxCurva.moveTo(origemX, 10); ctxCurva.lineTo(origemX, H-10);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280"; ctxCurva.font = "9.5px Inter, sans-serif"; ctxCurva.textAlign = "left";
      ctxCurva.fillText("z", W-16, origemY-5);
      ctxCurva.fillText("y", origemX+5, 14);

      ctxCurva.strokeStyle = "#2F5FA8"; ctxCurva.lineWidth = 2.2;
      ctxCurva.beginPath();
      var primeiro = true;
      for (var px = -5; px <= 5; px += 0.1) {
        var valY = calcularAtivacao(px, tipo);
        var cx = origemX + px * escalaX;
        var cy = origemY - valY * escalaY;
        if (primeiro) { ctxCurva.moveTo(cx, cy); primeiro = false; }
        else { ctxCurva.lineTo(cx, cy); }
      }
      ctxCurva.stroke();

      var ptX = Math.max(10, Math.min(W-10, origemX + z * escalaX));
      var ptY = Math.max(10, Math.min(H-10, origemY - y * escalaY));

      ctxCurva.strokeStyle = "#C1443A"; ctxCurva.setLineDash([3,3]); ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(ptX, origemY); ctxCurva.lineTo(ptX, ptY); ctxCurva.lineTo(origemX, ptY);
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);

      ctxCurva.fillStyle = "#C1443A";
      ctxCurva.beginPath(); ctxCurva.arc(ptX, ptY, 4.5, 0, 2*Math.PI); ctxCurva.fill();
      ctxCurva.strokeStyle = "#fff"; ctxCurva.lineWidth = 1.4; ctxCurva.stroke();
    }

    function atualizarPainelCalculo(res){
      var htmlGrade = '';
      for (var idx = 0; idx < res.termos.length; idx++) {
        var t = res.termos[idx];
        var corFundo = t.p > 0 ? "#E6F4EA" : (t.p < 0 ? "#FCE8E6" : "#F3F4F6");
        var corTxt = t.p > 0 ? CORES.pos : (t.p < 0 ? CORES.neg : "#374151");
        htmlGrade += '<div class="cn-prodcell" style="background:' + corFundo + ';">' +
          '<div class="cn-mono" style="font-size:9px;color:#6b7280;">' + t.i.toFixed(0) + '×' + t.k + '</div>' +
          '<div class="cn-mono" style="font-size:11px;font-weight:700;color:' + corTxt + ';">' + t.p.toFixed(0) + '</div></div>';
      }
      gradeProdutos.innerHTML = htmlGrade;

      var partes = res.termos.map(function(t){
        return t.p >= 0 ? t.p.toFixed(0) : '(' + t.p.toFixed(0) + ')';
      });
      var nomeFunc = selF.options[selF.selectedIndex].text;
      var htmlExpr = '<div class="cn-mono">z = ' + partes.join(' + ') + ' = <b>' + res.z.toFixed(2) + '</b></div>' +
        '<div class="cn-mono" style="margin-top:4px;">y = ' + nomeFunc + '(z) = <b>' + res.y.toFixed(2) + '</b></div>';
      if (res.p) {
        htmlExpr += '<div style="margin-top:6px;color:#2F6F9F;font-size:10.5px;">💡 Termos com fundo azul tracejado no diagrama vêm de <b>padding</b> — zeros adicionados artificialmente na borda, que não fazem parte da imagem original.</div>';
      }
      expressaoSoma.innerHTML = htmlExpr;
    }

    function atualizar(){
      var passoIdx = parseInt(inStep.value);
      var tipo = selF.value;
      var comPadding = chkP.checked;

      var res = desenharEsquema(passoIdx, tipo, comPadding);
      desenharCurva(res.z, res.y, tipo);
      atualizarPainelCalculo(res);

      var maxSteps = res.dimOut * res.dimOut - 1;
      var passoAtual = res.posR * res.dimOut + res.posC;
      posLabel.textContent = 'Passo ' + (passoAtual + 1) + ' de ' + (maxSteps + 1) +
        '  •  posição (i=' + res.posR + ', j=' + res.posC + ')';

      var nomeFunc = selF.options[selF.selectedIndex].text;
      valTxt.textContent = 'Posição (' + res.posR + ',' + res.posC + '): z = ' + res.z.toFixed(2) +
                           '  →  y = ' + nomeFunc + '(z) = ' + res.y.toFixed(2);

      notaFormula.textContent = '📏 Dimensão da saída: n_saída = (n + 2p − k)/s + 1 = (' + N_BASE + ' + 2×' + res.p + ' − 3)/1 + 1 = ' + res.dimOut;

      var infoKernel = KERNEL_INFO[kernelAtualId];
      legendaKernel.innerHTML = '<b>' + infoKernel.emoji + ' ' + infoKernel.nome + ':</b> este filtro ' + infoKernel.desc;

      var infoImagem = IMAGENS[imagemAtualId];
      legendaImagem.innerHTML = '<b>' + infoImagem.emoji + ' ' + infoImagem.nome + ':</b> ' + infoImagem.desc;
    }

    function pararAutoplay(){
      if (autoplayInterval) {
        clearInterval(autoplayInterval);
        autoplayInterval = null;
        btnPlay.textContent = '⏵ Auto';
      }
    }

    function setKernel(k, id, btn){
      [btnSobelV, btnSobelH, btnSharpen, btnIdentity].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      kernelAtual = k;
      kernelAtualId = id;
      pararAutoplay();
      atualizar();
    }

    function setImagem(id, btn){
      [btnImgCasa, btnImgFeliz, btnImgTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      imagemAtualId = id;
      imgEntradaBase = converterLinhasParaMatriz(IMAGENS[id].linhas);
      pararAutoplay();
      atualizar();
    }

    btnSobelV.addEventListener('click', function(){ setKernel(KERNELS.sobelV, 'sobelV', btnSobelV); });
    btnSobelH.addEventListener('click', function(){ setKernel(KERNELS.sobelH, 'sobelH', btnSobelH); });
    btnSharpen.addEventListener('click', function(){ setKernel(KERNELS.sharpen, 'sharpen', btnSharpen); });
    btnIdentity.addEventListener('click', function(){ setKernel(KERNELS.identity, 'identity', btnIdentity); });

    btnImgCasa.addEventListener('click', function(){ setImagem('casa', btnImgCasa); });
    btnImgFeliz.addEventListener('click', function(){ setImagem('feliz', btnImgFeliz); });
    btnImgTriste.addEventListener('click', function(){ setImagem('triste', btnImgTriste); });

    inStep.addEventListener('input', function(){ pararAutoplay(); atualizar(); });
    selF.addEventListener('change', function(){ pararAutoplay(); atualizar(); });
    chkP.addEventListener('change', function(){ pararAutoplay(); atualizar(); });

    btnAnterior.addEventListener('click', function(){
      pararAutoplay();
      var max = parseInt(inStep.max);
      var v = parseInt(inStep.value) - 1;
      inStep.value = v < 0 ? max : v;
      atualizar();
    });
    btnProximo.addEventListener('click', function(){
      pararAutoplay();
      var max = parseInt(inStep.max);
      var v = parseInt(inStep.value) + 1;
      inStep.value = v > max ? 0 : v;
      atualizar();
    });
    btnPlay.addEventListener('click', function(){
      if (autoplayInterval) { pararAutoplay(); return; }
      btnPlay.textContent = '⏸ Pausar';
      autoplayInterval = setInterval(function(){
        var max = parseInt(inStep.max);
        var v = parseInt(inStep.value) + 1;
        if (v > max) { pararAutoplay(); v = max; }
        inStep.value = v;
        atualizar();
      }, 130);
    });
    btnReiniciar.addEventListener('click', function(){
      pararAutoplay();
      inStep.value = 0;
      atualizar();
    });

    imgEntradaBase = converterLinhasParaMatriz(IMAGENS[imagemAtualId].linhas);
    atualizar();
  }

  function tryInitCamadaConvImg(){
    var root = document.getElementById('sim-09-camada-conv');
    if(root) initCamadaConvImg(root); else setTimeout(tryInitCamadaConvImg, 200);
  }
  tryInitCamadaConvImg();
})();
</script>
''')

**Figura 9.4:** Simulador interativo da Camada Convolucional: escolha entre três imagens de entrada 12×12 (casa, rosto feliz ou rosto triste) para observar como os mesmos *kernels* fixos reagem a diferentes bordas e formas. Navegue pelo campo receptivo com os botões ou reprodução automática, ajuste a função de ativação e alterne o zero-padding, acompanhando o mapa de características sendo revelado célula a célula, com o cálculo detalhado termo a termo e a fórmula da dimensão de saída em tempo real.


<figure id="fig-09-sim-09-camada-conv">
  <img src="imagens/fig-09-sim-09-camada-conv.png" alt=" Simulador interativo da Camada Convolucional: escolha entre três imagens de entrada 12×12 (casa, rosto feliz ou rosto triste) para observar como os mesmos *kernels* fixos reagem a diferentes bordas e formas. Navegue pelo campo receptivo com os botões ou reprodução automática, ajuste a função de ativação e alterne o zero-padding, acompanhando o mapa de características sendo revelado célula a célula, com o cálculo detalhado termo a termo e a fórmula da dimensão de saída em tempo real. " style="max-width:80%" />
  <figcaption><strong>Figura 9.4:</strong>  Simulador interativo da Camada Convolucional: escolha entre três imagens de entrada 12×12 (casa, rosto feliz ou rosto triste) para observar como os mesmos *kernels* fixos reagem a diferentes bordas e formas. Navegue pelo campo receptivo com os botões ou reprodução automática, ajuste a função de ativação e alterne o zero-padding, acompanhando o mapa de características sendo revelado célula a célula, com o cálculo detalhado termo a termo e a fórmula da dimensão de saída em tempo real. </figcaption>
</figure>

### 9.4.4 Funzione di Attivazione

La convoluzione è un'operazione lineare. Affinché la rete possa modellare relazioni non lineari tra ingressi e uscite, si applica una **funzione di attivazione** (*activation function*) dopo ogni strato convoluzionale.

La funzione più utilizzata nelle CNN è la **ReLU** (*Rectified Linear Unit*), definita da

$$
\mathrm{ReLU}(x)=\max(0,x).
$$

Questa funzione preserva i valori positivi e sostituisce con zero i valori negativi, introducendo non linearità nel modello e favorendo l'addestramento di reti profonde con basso costo computazionale.

La [Figura 9.5](#fig-09-sim-09-relu) illustra il funzionamento della **ReLU** applicata sia a valori individuali sia a una mappa delle caratteristiche, consentendo di confrontare l'uscita prima e dopo l'attivazione.

In [5]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-relu" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>⚡ Simulatore: Funzione di Attivazione ReLU</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">ReLU(x) = max(0, x)</span>
  </div>
  <div style="padding:20px;background:white;overflow:auto">

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:10px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:600;color:#374151;">x =</span>
        <input type="range" id="cap09relu_slider" min="-5" max="5" step="0.1" value="-2.5" style="flex:1;min-width:160px;">
        <span id="cap09relu_valTxt" style="font-family:monospace;font-size:12px;min-width:190px;color:#374151;">x = -2.50  →  ReLU(x) = 0.00</span>
      </div>
    </div>

    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Curva della funzione ReLU</div>
        <canvas id="cap09relu_canvasCurva" width="280" height="220"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Mappa delle caratteristiche: prima / dopo</div>
        <canvas id="cap09relu_canvasMapa" width="260" height="220"></canvas>
        <div style="display:flex;gap:8px;justify-content:center;margin-top:8px;">
          <button id="cap09relu_btnAplicar" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #16a34a;background:#16a34a;color:#fff;cursor:pointer;">Applica ReLU alla mappa</button>
          <button id="cap09relu_btnReset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">↺ Ripristina</button>
        </div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  var cap09relu_MAPA = [
    [ 1.2, -0.8,  3.4, -2.1, 0.5],
    [-1.5,  2.7, -0.3,  1.1, -4.0],
    [ 0.9, -2.9,  4.8, -0.6,  2.2],
    [-3.3,  0.2, -1.1,  3.9, -0.4],
    [ 2.0, -1.7,  0.8, -2.6,  1.4]
  ];

  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var aplicado = false;

    var ctxCurva = root.querySelector('#cap09relu_canvasCurva').getContext('2d');
    var ctxMapa  = root.querySelector('#cap09relu_canvasMapa').getContext('2d');
    var slider   = root.querySelector('#cap09relu_slider');
    var valTxt   = root.querySelector('#cap09relu_valTxt');

    var W = 280, H = 220;
    var origemX = 40, origemY = H - 30;
    var escala = 22;

    function xParaPixel(x){ return origemX + x*escala; }
    function yParaPixel(y){ return origemY - y*escala; }

    function desenharCurva(x){
      ctxCurva.clearRect(0,0,W,H);

      ctxCurva.strokeStyle = "#9ca3af";
      ctxCurva.lineWidth = 1;
      ctxCurva.beginPath();
      ctxCurva.moveTo(0, origemY); ctxCurva.lineTo(W, origemY);
      ctxCurva.moveTo(origemX, 0); ctxCurva.lineTo(origemX, H);
      ctxCurva.stroke();

      ctxCurva.fillStyle = "#6b7280";
      ctxCurva.font = "10px sans-serif";
      ctxCurva.fillText("x", W-12, origemY-4);
      ctxCurva.fillText("ReLU(x)", origemX+4, 10);

      ctxCurva.strokeStyle = "#4f46e5";
      ctxCurva.lineWidth = 2.5;
      ctxCurva.beginPath();
      ctxCurva.moveTo(xParaPixel(-5), yParaPixel(0));
      ctxCurva.lineTo(xParaPixel(0), yParaPixel(0));
      ctxCurva.lineTo(xParaPixel(5), yParaPixel(5));
      ctxCurva.stroke();

      var y = Math.max(0, x);
      ctxCurva.fillStyle = "#dc2626";
      ctxCurva.beginPath();
      ctxCurva.arc(xParaPixel(x), yParaPixel(y), 5, 0, 2*Math.PI);
      ctxCurva.fill();

      ctxCurva.strokeStyle = "#fca5a5";
      ctxCurva.setLineDash([3,3]);
      ctxCurva.beginPath();
      ctxCurva.moveTo(xParaPixel(x), origemY);
      ctxCurva.lineTo(xParaPixel(x), yParaPixel(y));
      ctxCurva.lineTo(origemX, yParaPixel(y));
      ctxCurva.stroke();
      ctxCurva.setLineDash([]);
    }

    function corValor(v, apl){
      if (apl && v < 0) v = 0;
      if (v < 0){
        var inten = Math.min(1, Math.abs(v)/5);
        var c = Math.round(255 - inten*180);
        return 'rgb('+c+','+c+',255)';
      } else {
        var inten2 = Math.min(1, v/5);
        var c2 = Math.round(255 - inten2*200);
        return 'rgb('+c2+',255,'+c2+')';
      }
    }

    function desenharMapa(){
      var tam = 42, offX = 20, offY = 10;
      ctxMapa.clearRect(0,0,260,220);
      for (var r=0;r<5;r++){
        for (var c=0;c<5;c++){
          var vOrig = cap09relu_MAPA[r][c];
          var v = aplicado ? Math.max(0, vOrig) : vOrig;
          ctxMapa.fillStyle = corValor(vOrig, aplicado);
          ctxMapa.fillRect(offX+c*tam, offY+r*tam, tam, tam);
          ctxMapa.strokeStyle = "#d1d5db";
          ctxMapa.strokeRect(offX+c*tam, offY+r*tam, tam, tam);
          ctxMapa.fillStyle = "#1f2937";
          ctxMapa.font = "10px monospace";
          ctxMapa.textAlign = "center";
          ctxMapa.fillText(v.toFixed(1), offX+c*tam+tam/2, offY+r*tam+tam/2+4);
        }
      }
      ctxMapa.fillStyle = "#6b7280";
      ctxMapa.font = "10px sans-serif";
      ctxMapa.textAlign = "left";
      ctxMapa.fillText(aplicado ? "Depois da ReLU (negativos → 0)" : "Antes da ReLU (valores brutos da convolução)", offX, 215);
    }

    function atualizarSlider(){
      var x = parseFloat(slider.value);
      var y = Math.max(0, x);
      valTxt.textContent = 'x = ' + x.toFixed(2) + '  →  ReLU(x) = ' + y.toFixed(2);
      desenharCurva(x);
    }

    slider.addEventListener('input', atualizarSlider);

    root.querySelector('#cap09relu_btnAplicar').addEventListener('click', function(){
      aplicado = true;
      desenharMapa();
    });
    root.querySelector('#cap09relu_btnReset').addEventListener('click', function(){
      aplicado = false;
      desenharMapa();
    });

    atualizarSlider();
    desenharMapa();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-relu');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.5:** Simulatore interattivo della funzione di attivazione ReLU: trascina il controllo per vedere come i valori negativi vengono azzerati e i valori positivi vengono preservati, sia nella curva che in una mappa delle caratteristiche reale.


<figure id="fig-09-sim-09-relu">
  <img src="imagens/fig-09-sim-09-relu.png" alt=" Simulatore interattivo della funzione di attivazione ReLU: trascina il controllo per vedere come i valori negativi vengono azzerati e i valori positivi vengono preservati, sia nella curva che in una mappa delle caratteristiche reale. " style="max-width:80%" />
  <figcaption><strong>Figura 9.5:</strong>  Simulatore interattivo della funzione di attivazione ReLU: trascina il controllo per vedere come i valori negativi vengono azzerati e i valori positivi vengono preservati, sia nella curva che in una mappa delle caratteristiche reale. </figcaption>
</figure>

I mappe di caratteristiche risultanti dalla convoluzione e dall'attivazione preservano la struttura spaziale dell'immagine. In molte architetture, la fase successiva ne riduce la risoluzione tramite un'operazione di *pooling*.

### 9.4.5 *Pooling*

Il livello di ***pooling*** riduce la risoluzione spaziale delle mappe delle caratteristiche, preservando le informazioni più rilevanti per le fasi successive dell'elaborazione. L'operazione più utilizzata è il ***max-pooling***, che seleziona il valore massimo in ogni finestra dell'immagine:

$$
P(i,j)=\max_{(u,v)\in\text{finestra}(i,j)}F(u,v).
$$

Questa riduzione diminuisce il costo computazionale dei livelli successivi e rende la rappresentazione più robusta a piccole variazioni nella posizione dei pattern presenti nell'immagine.

La [Figura 9.6](#fig-09-sim-09-pooling) presenta questa operazione su una mappa delle caratteristiche di 8×8 pixel, ridotta a 4×4 tramite finestre di 2×2 con passo pari a 2, alternando tra **max-pooling** e **average-pooling** — che calcola, invece del massimo, la media dei valori della finestra corrispondente.

In [6]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-pooling" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🔻 Simulatore: Pooling</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">finestra 2×2, stride 2</span>
  </div>
  <div style="padding:20px;background:white;overflow:auto">

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:14px;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;margin-bottom:10px;">
        <span style="font-size:11px;font-weight:600;color:#374151;">Tipo:</span>
        <button id="cap09pool_btnMax" class="cap09pool_active" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #4f46e5;background:#4f46e5;color:#fff;cursor:pointer;">Max-pooling</button>
        <button id="cap09pool_btnAvg" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">Average-pooling</button>
      </div>
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <button id="cap09pool_btnPasso" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #16a34a;background:#16a34a;color:#fff;cursor:pointer;">▶ Avanti di 1 Passo</button>
        <button id="cap09pool_btnTudo" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">⏭ Calcola Tutto</button>
        <button id="cap09pool_btnReset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #d1d5db;background:#fff;cursor:pointer;">↺ Azzera</button>
        <span style="font-size:11px;color:#6b7280;">Finestra attuale: <b id="cap09pool_posTxt">(0, 0)</b> di 4×4</span>
      </div>
      <div id="cap09pool_explicacao" style="font-size:10.5px;color:#6b7280;margin-top:8px;line-height:1.4;">O <b>max-pooling</b> mantiene solo il valore più alto di ciascuna finestra 2×2, riducendo la risoluzione spaziale della metà e preservando le risposte più forti della mappa delle caratteristiche.</div>
    </div>

    <div style="display:flex;gap:24px;align-items:flex-start;flex-wrap:wrap;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Mappa di ingresso (8×8) — finestra attuale evidenziata</div>
        <canvas id="cap09pool_canvasEntrada" width="240" height="240"></canvas>
      </div>
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#4b5563;">Mappa ridotta (4×4)</div>
        <canvas id="cap09pool_canvasSaida" width="160" height="160"></canvas>
      </div>
    </div>

  </div>
</div>

<style>
  #sim-09-pooling button.cap09pool_active { background: #4f46e5 !important; color: #fff !important; border-color: #4f46e5 !important; }
</style>

<script>
(function(){
  var cap09pool_ENTRADA = [
    [1, 3, 2, 8,  5, 1, 0, 2],
    [4, 6, 1, 2,  3, 9, 1, 0],
    [0, 1, 9, 3,  1, 2, 8, 4],
    [2, 5, 4, 7,  0, 1, 3, 6],
    [3, 8, 1, 0,  6, 2, 5, 1],
    [1, 2, 6, 4,  9, 0, 2, 3],
    [7, 0, 3, 1,  2, 8, 1, 4],
    [2, 4, 1, 5,  3, 1, 6, 9]
  ];
  var MAX_GLOBAL = 9;

  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var tipo = "max";
    var saida = Array.from({length:4}, function(){ return Array(4).fill(null); });
    var pos = {r:0, c:0};

    var ctxEnt = root.querySelector('#cap09pool_canvasEntrada').getContext('2d');
    var ctxSai = root.querySelector('#cap09pool_canvasSaida').getContext('2d');
    var posTxt = root.querySelector('#cap09pool_posTxt');
    var explicacao = root.querySelector('#cap09pool_explicacao');

    function corEscala(v, max){
      var inten = Math.min(1, v/max);
      var c = Math.round(245 - inten*160);
      return 'rgb('+c+','+(c+8)+',255)';
    }

    function desenharEntrada(){
      var tam = 30;
      ctxEnt.clearRect(0,0,240,240);
      for (var r=0;r<8;r++){
        for (var c=0;c<8;c++){
          var v = cap09pool_ENTRADA[r][c];
          ctxEnt.fillStyle = corEscala(v, MAX_GLOBAL);
          ctxEnt.fillRect(c*tam, r*tam, tam, tam);
          ctxEnt.strokeStyle = "#e5e7eb";
          ctxEnt.strokeRect(c*tam, r*tam, tam, tam);
          ctxEnt.fillStyle = "#1f2937";
          ctxEnt.font = "11px monospace";
          ctxEnt.textAlign = "center";
          ctxEnt.fillText(v, c*tam+tam/2, r*tam+tam/2+4);
        }
      }
      if (pos.r < 4){
        ctxEnt.strokeStyle = "#dc2626";
        ctxEnt.lineWidth = 3;
        ctxEnt.strokeRect(pos.c*2*tam, pos.r*2*tam, tam*2, tam*2);
        ctxEnt.lineWidth = 1;
      }
    }

    function desenharSaida(){
      var tam = 40;
      ctxSai.clearRect(0,0,160,160);
      for (var r=0;r<4;r++){
        for (var c=0;c<4;c++){
          var v = saida[r][c];
          ctxSai.fillStyle = (v === null) ? "#f3f4f6" : corEscala(v, MAX_GLOBAL);
          ctxSai.fillRect(c*tam, r*tam, tam, tam);
          ctxSai.strokeStyle = "#e5e7eb";
          ctxSai.strokeRect(c*tam, r*tam, tam, tam);
          if (v !== null){
            ctxSai.fillStyle = "#1f2937";
            ctxSai.font = "11px monospace";
            ctxSai.textAlign = "center";
            ctxSai.fillText(v.toFixed(1), c*tam+tam/2, r*tam+tam/2+4);
          }
        }
      }
      if (pos.r < 4){
        ctxSai.strokeStyle = "#dc2626";
        ctxSai.lineWidth = 2;
        ctxSai.strokeRect(pos.c*tam, pos.r*tam, tam, tam);
        ctxSai.lineWidth = 1;
      }
    }

    function calcularJanela(r, c){
      var vals = [];
      for (var i=0;i<2;i++) for (var j=0;j<2;j++) vals.push(cap09pool_ENTRADA[r*2+i][c*2+j]);
      if (tipo === "max") return Math.max.apply(null, vals);
      return vals.reduce(function(a,b){return a+b;},0) / vals.length;
    }

    function avancarPasso(){
      if (pos.r >= 4) return;
      saida[pos.r][pos.c] = calcularJanela(pos.r, pos.c);
      pos.c++;
      if (pos.c >= 4){ pos.c = 0; pos.r++; }
      render();
    }

    function calcularTudo(){
      while (pos.r < 4) avancarPasso();
    }

    function resetar(){
      saida = Array.from({length:4}, function(){ return Array(4).fill(null); });
      pos = {r:0, c:0};
      render();
    }

    function render(){
      desenharEntrada();
      desenharSaida();
      posTxt.textContent = pos.r < 4 ? '(' + pos.r + ', ' + pos.c + ')' : 'concluído';
    }

    function selecionarTipo(t){
      tipo = t;
      root.querySelector('#cap09pool_btnMax').classList.toggle('cap09pool_active', t === "max");
      root.querySelector('#cap09pool_btnAvg').classList.toggle('cap09pool_active', t === "avg");
      explicacao.innerHTML = t === "max"
        ? "O <b>max-pooling</b> mantém apenas o maior valor de cada janela 2×2, reduzindo a resolução espacial pela metade e preservando as respostas mais fortes do mapa de características."
        : "O <b>average-pooling</b> calcula a média dos quatro valores de cada janela 2×2, suavizando a informação em vez de preservar apenas o pico de resposta.";
      resetar();
    }

    root.querySelector('#cap09pool_btnMax').addEventListener('click', function(){ selecionarTipo("max"); });
    root.querySelector('#cap09pool_btnAvg').addEventListener('click', function(){ selecionarTipo("avg"); });
    root.querySelector('#cap09pool_btnPasso').addEventListener('click', avancarPasso);
    root.querySelector('#cap09pool_btnTudo').addEventListener('click', calcularTudo);
    root.querySelector('#cap09pool_btnReset').addEventListener('click', resetar);

    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-09-pooling');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.6:** Simulatore interattivo di *pooling*: scegli tra *max-pooling* e *average-pooling* e procedi passo dopo passo per osservare la riduzione della risoluzione spaziale della mappa delle caratteristiche.


<figure id="fig-09-sim-09-pooling">
  <img src="imagens/fig-09-sim-09-pooling.png" alt=" Simulatore interattivo di *pooling*: scegli tra *max-pooling* e *average-pooling* e procedi passo dopo passo per osservare la riduzione della risoluzione spaziale della mappa delle caratteristiche. " style="max-width:80%" />
  <figcaption><strong>Figura 9.6:</strong>  Simulatore interattivo di *pooling*: scegli tra *max-pooling* e *average-pooling* e procedi passo dopo passo per osservare la riduzione della risoluzione spaziale della mappa delle caratteristiche. </figcaption>
</figure>

Complessivamente, convoluzione, funzione di attivazione e *pooling* formano il blocco fondamentale utilizzato nella costruzione di una CNN.

### 9.4.6 Addestramento delle Reti Neurali: Come le CNN Imparano

Una CNN impara regolando automaticamente i propri parametri — i coefficienti dei filtri convoluzionali, i pesi dei livelli completamente connessi e i bias — a partire da esempi etichettati. Questo addestramento è iterativo e coinvolge tre fasi: misurare l'errore prodotto dalla rete tramite una **funzione di perdita** (*loss function*), calcolare come tale errore dipende da ciascun parametro tramite la **retropropagazione** (*backpropagation*) e aggiornare i parametri con un **algoritmo di ottimizzazione** (*optimizer*).

#### 9.4.6.1 Funzione di Perdita (*Loss Function*)

La **funzione di perdita** (*loss function*) quantifica la differenza tra la previsione della rete e la risposta corretta, denominata **verità di riferimento** (*ground truth*). Il risultato è uno scalare $L$: minore è la perdita, più la previsione è vicina alla risposta attesa.

Nei problemi di classificazione multiclasse, la funzione più utilizzata è l'**Entropia Incrociata** (*Cross-Entropy Loss*), applicata alle probabilità prodotte dal livello **Softmax**:

$$
L=-\sum_{c=1}^{C} y_c \log(\hat{y}_c),
$$

dove $C$ è il numero di classi, $y_c$ è l'etichetta reale in codifica *one-hot* e $\hat{y}_c$ è la probabilità prevista per la classe $c$. La perdita si avvicina a zero quando la rete attribuisce alta probabilità alla classe corretta e cresce rapidamente man mano che tale probabilità diminuisce.

La [Figura 9.7](#fig-09-sim-09-loss) illustra questo comportamento: il simulatore consente di selezionare la classe corretta e modificare le probabilità prodotte dalla *Softmax*, mostrando in tempo reale la variazione della funzione di perdita.

In [7]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-loss" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-loss .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-loss .cn-grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-loss .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-loss .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-loss input[type=range] { accent-color:#2F6F9F; width:100%; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">📉 Simulatore: Funzione di Perdita (Entropia Incrociata)</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">L = -log(ŷ_target)</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletores de Rótulo Real (Ground Truth) -->
    <div style="margin-bottom:12px;">
      <div class="cn-grouplabel">CLASSE REALE DELL'IMMAGINE (GROUND TRUTH: y_c = 1)</div>
      <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;max-width:340px;">
        <button id="cap09loss_btnCasa" class="cn-modebtn active">🏠 Casa</button>
        <button id="cap09loss_btnFeliz" class="cn-modebtn">😊 Felice</button>
        <button id="cap09loss_btnTriste" class="cn-modebtn">😢 Triste</button>
      </div>
    </div>

    <!-- Ajuste de Probabilidades Preditas (Softmax ŷ) -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:12px;margin-bottom:14px;">
      <div class="cn-grouplabel" style="margin-bottom:8px;">PROBABILITÀ STIMATE DALLA SOFTMAX (ŷ_c)</div>
      <div style="display:grid;grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));gap:12px;">
        
        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>🏠 Casa (ŷ_1):</span>
            <span id="cap09loss_txtProbCasa" class="cn-mono" style="color:#2F6F9F;">0.70</span>
          </div>
          <input type="range" id="cap09loss_rangeCasa" min="0.01" max="0.98" step="0.01" value="0.70">
        </div>

        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>😊 Felice (ŷ_2):</span>
            <span id="cap09loss_txtProbFeliz" class="cn-mono" style="color:#2F6F9F;">0.20</span>
          </div>
          <input type="range" id="cap09loss_rangeFeliz" min="0.01" max="0.98" step="0.01" value="0.20">
        </div>

        <div>
          <div style="display:flex;justify-content:space-between;font-size:11px;font-weight:600;margin-bottom:2px;">
            <span>😢 Triste (ŷ_3):</span>
            <span id="cap09loss_txtProbTriste" class="cn-mono" style="color:#2F6F9F;">0.10</span>
          </div>
          <input type="range" id="cap09loss_rangeTriste" min="0.01" max="0.98" step="0.01" value="0.10">
        </div>

      </div>
    </div>

    <!-- Curva da Função Logarítmica & Resultado -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:18px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <div>
        <div style="font-size:11px;font-weight:600;text-align:center;margin-bottom:4px;color:#5E5A4A;">Curva di Penalizzazione L = -log(ŷ_target)</div>
        <canvas id="cap09loss_canvasCurva" width="260" height="170" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
      </div>

      <div style="flex:1;min-width:240px;">
        <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:12px;margin-bottom:8px;">
          <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:4px;">CALCOLO DELLA PERDITA:</div>
          <div id="cap09loss_exprCalc" class="cn-mono" style="font-size:11.5px;color:#374151;line-height:1.6;"></div>
          <div style="margin-top:6px;font-size:14px;font-weight:700;color:#C1443A;">
            Perdita L = <span id="cap09loss_valTotal" class="cn-mono">0.3567</span>
          </div>
        </div>
        <div id="cap09loss_explicacao" style="font-size:10.5px;color:#8A8371;line-height:1.4;"></div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function initLoss(root){
    if(!root || root.dataset.initLoss) return;
    root.dataset.initLoss = "1";

    var classeAlvo = "casa"; // "casa", "feliz", "triste"
    var probs = { casa: 0.70, feliz: 0.20, triste: 0.10 };

    var btnCasa   = root.querySelector('#cap09loss_btnCasa');
    var btnFeliz  = root.querySelector('#cap09loss_btnFeliz');
    var btnTriste = root.querySelector('#cap09loss_btnTriste');

    var rangeCasa   = root.querySelector('#cap09loss_rangeCasa');
    var rangeFeliz  = root.querySelector('#cap09loss_rangeFeliz');
    var rangeTriste = root.querySelector('#cap09loss_rangeTriste');

    var txtProbCasa   = root.querySelector('#cap09loss_txtProbCasa');
    var txtProbFeliz  = root.querySelector('#cap09loss_txtProbFeliz');
    var txtProbTriste = root.querySelector('#cap09loss_txtProbTriste');

    var exprCalc   = root.querySelector('#cap09loss_exprCalc');
    var valTotal   = root.querySelector('#cap09loss_valTotal');
    var explicacao = root.querySelector('#cap09loss_explicacao');

    var canvas = root.querySelector('#cap09loss_canvasCurva');
    var ctx    = canvas.getContext('2d');

    function normalizarProbs(modificado){
      var somaOutros = 0;
      var chaves = ["casa", "feliz", "triste"];
      chaves.forEach(function(k){ if(k !== modificado) somaOutros += probs[k]; });
      
      var restante = 1.0 - probs[modificado];
      if(somaOutros > 0){
        chaves.forEach(function(k){
          if(k !== modificado) probs[k] = (probs[k] / somaOutros) * restante;
        });
      } else {
        chaves.forEach(function(k){
          if(k !== modificado) probs[k] = restante / 2.0;
        });
      }

      rangeCasa.value   = probs.casa;
      rangeFeliz.value  = probs.feliz;
      rangeTriste.value = probs.triste;

      txtProbCasa.textContent   = probs.casa.toFixed(2);
      txtProbFeliz.textContent  = probs.feliz.toFixed(2);
      txtProbTriste.textContent = probs.triste.toFixed(2);
    }

    function desenharCurvaLog(){
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0,0,W,H);

      // Eixos
      ctx.strokeStyle = "#E4DCC8"; ctx.lineWidth = 1;
      ctx.beginPath();
      ctx.moveTo(30, 10); ctx.lineTo(30, H-20); ctx.lineTo(W-10, H-20);
      ctx.stroke();

      // Curva -log(x)
      ctx.strokeStyle = "#2F6F9F"; ctx.lineWidth = 2;
      ctx.beginPath();
      for(var x=0.002; x<=0.98; x+=0.01){
        var loss = -Math.log(x);
        var cx = 30 + x * (W - 40);
        var cy = (H - 20) - (loss / 4.0) * (H - 30);
        cy = Math.max(10, Math.min(H-20, cy));
        if(x === 0.002) ctx.moveTo(cx, cy); else ctx.lineTo(cx, cy);
      }
      ctx.stroke();

      // Ponto Atual
      var probAlvo = probs[classeAlvo];
      var lossAlvo = -Math.log(probAlvo);
      var ptX = 30 + probAlvo * (W - 40);
      var ptY = (H - 20) - (lossAlvo / 4.0) * (H - 30);
      ptY = Math.max(10, Math.min(H-20, ptY));

      ctx.strokeStyle = "#C1443A"; ctx.setLineDash([3,3]);
      ctx.beginPath();
      ctx.moveTo(ptX, H-20); ctx.lineTo(ptX, ptY); ctx.lineTo(30, ptY);
      ctx.stroke(); ctx.setLineDash([]);

      ctx.fillStyle = "#C1443A";
      ctx.beginPath(); ctx.arc(ptX, ptY, 4.5, 0, 2*Math.PI); ctx.fill();
    }

    function atualizar(){
      desenharCurvaLog();
      var probAlvo = probs[classeAlvo];
      var lossVal  = -Math.log(probAlvo);

      var nomes = { casa: "🏠 Casa", feliz: "😊 Feliz", triste: "😢 Triste" };
      exprCalc.innerHTML = 'L = -log(ŷ_' + classeAlvo + ') = -log(' + probAlvo.toFixed(2) + ')';
      valTotal.textContent = lossVal.toFixed(4);

      if(probAlvo > 0.8) {
        explicacao.innerHTML = "<b>Excelente precisão:</b> A rede atribuiu alta probabilidade à classe correta (" + nomes[classeAlvo] + "), gerando uma perda muito próxima de zero.";
      } else if(probAlvo > 0.4) {
        explicacao.innerHTML = "<b>Incerteza moderada:</b> A probabilidade da classe correta (" + nomes[classeAlvo] + ") é mediana, resultando em uma penalização moderada sobre a rede.";
      } else {
        explicacao.innerHTML = "<b>Erro alto (Confusão):</b> A rede atribuiu baixa probabilidade à classe real (" + nomes[classeAlvo] + "). A função logarítmica penaliza fortemente esse erro, gerando um alto valor de perda $L$.";
      }
    }

    function selecionarClasse(c, btn){
      [btnCasa, btnFeliz, btnTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      classeAlvo = c;
      atualizar();
    }

    btnCasa.addEventListener('click', function(){ selecionarClasse("casa", btnCasa); });
    btnFeliz.addEventListener('click', function(){ selecionarClasse("feliz", btnFeliz); });
    btnTriste.addEventListener('click', function(){ selecionarClasse("triste", btnTriste); });

    rangeCasa.addEventListener('input', function(){ probs.casa = parseFloat(this.value); normalizarProbs("casa"); atualizar(); });
    rangeFeliz.addEventListener('input', function(){ probs.feliz = parseFloat(this.value); normalizarProbs("feliz"); atualizar(); });
    rangeTriste.addEventListener('input', function(){ probs.triste = parseFloat(this.value); normalizarProbs("triste"); atualizar(); });

    atualizar();
  }

  function tryInitLoss(){
    var root = document.getElementById('sim-09-loss');
    if(root) initLoss(root); else setTimeout(tryInitLoss, 200);
  }
  tryInitLoss();
})();
</script>
''')

**Figura 9.7:** Simulatore interattivo della Funzione di Perdita (*Cross-Entropy*): seleziona la classe reale dell


<figure id="fig-09-sim-09-loss">
  <img src="imagens/fig-09-sim-09-loss.png" alt=" Simulatore interattivo della Funzione di Perdita (*Cross-Entropy*): seleziona la classe reale dell'immagine (Casa, Felice o Triste) e regola le probabilità stimate dalla Softmax per visualizzare il calcolo della penalizzazione scalare e il grafico del logaritmo negativo in tempo reale. " style="max-width:80%" />
  <figcaption><strong>Figura 9.7:</strong>  Simulatore interattivo della Funzione di Perdita (*Cross-Entropy*): seleziona la classe reale dell'immagine (Casa, Felice o Triste) e regola le probabilità stimate dalla Softmax per visualizzare il calcolo della penalizzazione scalare e il grafico del logaritmo negativo in tempo reale. </figcaption>
</figure>

#### 9.4.6.2 Retropropagazione (*Backpropagation*)

Dopo il calcolo della perdita, è necessario determinare come ciascun parametro della rete contribuisca a questo risultato. Questa fase è realizzata dalla **retropropagazione** (*backpropagation*), che applica la **Regola della Catena** del calcolo differenziale per ottenere il gradiente della funzione di perdita rispetto a ciascun parametro.

Per un parametro $w$, questo gradiente è dato da

$$
\frac{\partial L}{\partial w}.
$$

Il gradiente indica come varia la perdita in relazione a piccole modifiche di $w$: un gradiente positivo indica che aumentare $w$ aumenta la perdita, mentre un gradiente negativo indica l'effetto opposto.

La [Figura 9.8](#fig-09-sim-09-backprop) presenta questo processo in modo visivo, mostrando la propagazione del gradiente dallo strato di output fino ai primi strati convoluzionali.

In [8]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-backprop" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-backprop .cap09backprop_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-backprop .cap09backprop_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-backprop .cap09backprop_navbtn {
      padding:6px 12px; font-size:11px; font-weight:700; border-radius:8px; border:1px solid #E4DCC8;
      background:#FAF6EC; color:#374151; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-backprop .cap09backprop_navbtn:hover { background:#F1EAD7; }
    
    /* Blocos Interativos do Fluxo */
    #sim-09-backprop .cap09backprop_node {
      flex: 1;
      min-width: 90px;
      padding: 10px 6px;
      border-radius: 10px;
      border: 1px solid #E4DCC8;
      background: #FFFFFF;
      text-align: center;
      transition: all 0.25s ease;
      box-shadow: 0 1px 2px rgba(0,0,0,0.02);
      cursor: pointer;
    }
    #sim-09-backprop .cap09backprop_node_title {
      font-size: 11px;
      font-weight: 700;
      color: #374151;
    }
    #sim-09-backprop .cap09backprop_node_sub {
      font-size: 9px;
      font-weight: 600;
      color: #8A8371;
      margin-top: 3px;
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active {
      border: 2px solid #C1443A;
      background: #FCE8E6;
      box-shadow: 0 3px 8px rgba(193,68,58,0.15);
      transform: translateY(-2px);
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active .cap09backprop_node_title {
      color: #C1443A;
    }
    #sim-09-backprop .cap09backprop_node.cap09backprop_active .cap09backprop_node_sub {
      color: #A8322A;
    }
    
    /* Seta do Fluxo Reverso */
    #sim-09-backprop .cap09backprop_arrow {
      font-size: 14px;
      font-weight: bold;
      color: #D1D5DB;
      transition: color 0.2s ease;
      padding: 0 2px;
    }
    #sim-09-backprop .cap09backprop_arrow.cap09backprop_active_arrow {
      color: #C1443A;
    }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⬅️ Simulatore: Retropropagazione (Backpropagation)</span>
    <span class="cap09backprop_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">∂L/∂w = (∂L/∂y) · (∂y/∂z) · (∂z/∂w)</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Controles do Passo a Passo do Backprop -->
    <div style="display:flex;gap:10px;align-items:center;justify-content:space-between;margin-bottom:14px;flex-wrap:wrap;">
      <div style="display:flex;gap:6px;">
        <button id="cap09backprop_btnVoltar" class="cap09backprop_navbtn">◀ Passo Indietro</button>
        <button id="cap09backprop_btnAvancar" class="cap09backprop_navbtn" style="border-color:#C1443A;color:#C1443A;background:#FCE8E6;">Passo Inverso (Backprop) ◀</button>
        <button id="cap09backprop_btnReset" class="cap09backprop_navbtn">↺ Riavvia</button>
      </div>
      <span class="cap09backprop_mono" id="cap09backprop_txtEtapa" style="font-size:11px;color:#2F6F9F;font-weight:700;">Passo 1 di 4: Output (Loss & Softmax)</span>
    </div>

    <!-- Fluxo Visual das Camadas (Flexbox em alta resolução) -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:16px 12px;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8A8371;margin-bottom:8px;letter-spacing:.3px;text-align:center;">
        DIREZIONE DELLA PROPAGAZIONE DELL'ERRORE (FLUSSO INVERSO ⟵)
      </div>
      <div style="display:flex;align-items:center;justify-content:space-between;gap:4px;max-width:620px;margin:0 auto;">
        
        <div id="cap09backprop_node3" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Kernel Conv1</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂K</div>
        </div>

        <div id="cap09backprop_arrow2" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node2" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Max-Pooling</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂X_pool</div>
        </div>

        <div id="cap09backprop_arrow1" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node1" class="cap09backprop_node">
          <div class="cap09backprop_node_title">Layer FC</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂W_fc</div>
        </div>

        <div id="cap09backprop_arrow0" class="cap09backprop_arrow">⟵</div>

        <div id="cap09backprop_node0" class="cap09backprop_node cap09backprop_active">
          <div class="cap09backprop_node_title">Loss / Softmax</div>
          <div class="cap09backprop_node_sub cap09backprop_mono">∂L/∂y_pred</div>
        </div>

      </div>
    </div>

    <!-- Painel da Regra da Cadeia Detalhada -->
    <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:12px;padding:14px;">
      <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:6px;display:flex;align-items:center;gap:6px;">
        <span>🔗 Regola della Catena nel Layer Corrente:</span>
      </div>
      <div id="cap09backprop_exprCadeia" class="cap09backprop_mono" style="font-size:12px;font-weight:700;color:#C1443A;line-height:1.6;margin-bottom:8px;background:#FFF;padding:8px 10px;border-radius:8px;border:1px solid #E4DCC8;"></div>
      <div id="cap09backprop_descPasso" style="font-size:11.5px;color:#374151;line-height:1.5;"></div>
    </div>

  </div>
</div>

<script>
(function(){
  function cap09backprop_init(cap09backprop_root){
    if(!cap09backprop_root || cap09backprop_root.dataset.initBackprop) return;
    cap09backprop_root.dataset.initBackprop = "1";

    var cap09backprop_passoAtual = 0; // 0: Loss/Softmax, 1: Camada FC, 2: Pooling, 3: Conv1 Kernels

    var cap09backprop_btnVoltar  = cap09backprop_root.querySelector('#cap09backprop_btnVoltar');
    var cap09backprop_btnAvancar = cap09backprop_root.querySelector('#cap09backprop_btnAvancar');
    var cap09backprop_btnReset   = cap09backprop_root.querySelector('#cap09backprop_btnReset');

    var cap09backprop_txtEtapa   = cap09backprop_root.querySelector('#cap09backprop_txtEtapa');
    var cap09backprop_exprCadeia = cap09backprop_root.querySelector('#cap09backprop_exprCadeia');
    var cap09backprop_descPasso  = cap09backprop_root.querySelector('#cap09backprop_descPasso');

    var cap09backprop_ETAPAS = [
      {
        nome: "Passo 1 de 4: Saída (Loss & Softmax)",
        expressao: "∂L/∂y_pred = y_pred - y_real = 0.85 - 1.00 = -0.15",
        desc: "O algoritmo de Backpropagation começa no final do pipeline, calculando a derivada direta da função de perda por Entropia Cruzada em relação à probabilidade gerada pela Softmax."
      },
      {
        nome: "Passo 2 de 4: Camadas Densas (Fully Connected)",
        expressao: "∂L/∂w_fc = (∂L/∂y_pred) · (∂y_pred/∂z_fc) = (-0.15) · (0.42) = -0.063",
        desc: "O sinal de erro retropropaga pelas camadas totalmente conectadas através de multiplicadores de matrizes, definindo quanto cada peso denso contribuiu para o desvio final."
      },
      {
        nome: "Passo 3 de 4: Camada de Max-Pooling",
        expressao: "∂L/∂x_pool = (∂L/∂y_fc) · Mútil_max  ➔  Roteado integralmente para a posição do valor máximo",
        desc: "Na camada de Max-Pooling, não há pesos treináveis. O gradiente é repassado sem alteração exatamente para o pixel que forneceu o valor máximo no Forward Pass, enquanto os demais pixels recebem gradiente zero."
      },
      {
        nome: "Passo 4 de 4: Filtros Convolucionais (Conv1 Kernels)",
        expressao: "∂L/∂K(u,v) = ∑ (∂L/∂F) · I(i+u, j+v)  ➔  Gradiente acumulado do filtro 3×3",
        desc: "O erro atinge os coeficientes numéricos dos filtros originais. Como o mesmo kernel foi reutilizado sobre várias regiões da imagem, os gradientes de todas as posições do campo receptivo são somados para atualizar o filtro."
      }
    ];

    function cap09backprop_atualizarUI(){
      var cap09backprop_info = cap09backprop_ETAPAS[cap09backprop_passoAtual];
      cap09backprop_txtEtapa.textContent = cap09backprop_info.nome;
      cap09backprop_exprCadeia.innerHTML = cap09backprop_info.expressao;
      cap09backprop_descPasso.innerHTML  = cap09backprop_info.desc;

      // Atualizar nós ativos
      for(var cap09backprop_i=0; cap09backprop_i<4; cap09backprop_i++){
        var cap09backprop_node = cap09backprop_root.querySelector('#cap09backprop_node' + cap09backprop_i);
        if(cap09backprop_node){
          if(cap09backprop_i === cap09backprop_passoAtual){
            cap09backprop_node.classList.add('cap09backprop_active');
          } else {
            cap09backprop_node.classList.remove('cap09backprop_active');
          }
        }
      }

      // Atualizar setas ativas
      for(var cap09backprop_j=0; cap09backprop_j<3; cap09backprop_j++){
        var cap09backprop_arrow = cap09backprop_root.querySelector('#cap09backprop_arrow' + cap09backprop_j);
        if(cap09backprop_arrow){
          if(cap09backprop_j < cap09backprop_passoAtual){
            cap09backprop_arrow.classList.add('cap09backprop_active_arrow');
          } else {
            cap09backprop_arrow.classList.remove('cap09backprop_active_arrow');
          }
        }
      }
    }

    // Permitir clicar nos nós diretamente
    for(var cap09backprop_k=0; cap09backprop_k<4; cap09backprop_k++){
      (function(idx){
        var cap09backprop_n = cap09backprop_root.querySelector('#cap09backprop_node' + idx);
        if(cap09backprop_n){
          cap09backprop_n.addEventListener('click', function(){
            cap09backprop_passoAtual = idx;
            cap09backprop_atualizarUI();
          });
        }
      })(cap09backprop_k);
    }

    cap09backprop_btnAvancar.addEventListener('click', function(){
      if(cap09backprop_passoAtual < cap09backprop_ETAPAS.length - 1){
        cap09backprop_passoAtual++;
        cap09backprop_atualizarUI();
      }
    });

    cap09backprop_btnVoltar.addEventListener('click', function(){
      if(cap09backprop_passoAtual > 0){
        cap09backprop_passoAtual--;
        cap09backprop_atualizarUI();
      }
    });

    cap09backprop_btnReset.addEventListener('click', function(){
      cap09backprop_passoAtual = 0;
      cap09backprop_atualizarUI();
    });

    cap09backprop_atualizarUI();
  }

  function cap09backprop_tryInit(){
    var cap09backprop_root = document.getElementById('sim-09-backprop');
    if(cap09backprop_root) cap09backprop_init(cap09backprop_root); else setTimeout(cap09backprop_tryInit, 200);
  }
  cap09backprop_tryInit();
})();
</script>
''')

**Figura 9.8:** Simulatore interattivo di *Backpropagation*: procedi attraverso i passaggi della Regola della Catena per seguire il flusso del segnale di errore nella direzione inversa della rete, osservando il calcolo delle derivate parziali del gradiente in ogni strato.


<figure id="fig-09-sim-09-backprop">
  <img src="imagens/fig-09-sim-09-backprop.png" alt=" Simulatore interattivo di *Backpropagation*: procedi attraverso i passaggi della Regola della Catena per seguire il flusso del segnale di errore nella direzione inversa della rete, osservando il calcolo delle derivate parziali del gradiente in ogni strato. " style="max-width:80%" />
  <figcaption><strong>Figura 9.8:</strong>  Simulatore interattivo di *Backpropagation*: procedi attraverso i passaggi della Regola della Catena per seguire il flusso del segnale di errore nella direzione inversa della rete, osservando il calcolo delle derivate parziali del gradiente in ogni strato. </figcaption>
</figure>

#### 9.4.6.3 Algoritmi di Ottimizzazione

Dopo il calcolo dei gradienti, un **algoritmo di ottimizzazione** (*optimizer*) aggiorna i parametri della rete per ridurre la funzione di perdita. Nelle reti profonde, questa ricerca avviene in uno spazio ad alta dimensionalità e, in generale, **non convesso**, il che rende l'ottimizzazione un problema complesso.

Per facilitare la comprensione, la [Figura 9.9](#fig-09-sim-09-opt) utilizza una **superficie di perdita semplificata**, con un minimo globale, un minimo locale e una barriera tra queste regioni. Il **minimo globale** corrisponde al valore più basso della funzione di perdita e rappresenta il miglior insieme di parametri della rete; un **minimo locale** presenta anch'esso una bassa perdita, ma può essere distante dalla soluzione ottimale. Quando l'ottimizzazione rimane intrappolata in un minimo locale, gli aggiustamenti di filtri, pesi e bias diventano molto piccoli, e l'addestramento si ferma prima di raggiungere un modello con errore inferiore.

##### Discesa del Gradiente Stocastica (*SGD*)

La **Discesa del Gradiente Stocastica** (*Stochastic Gradient Descent* — *SGD*) aggiorna i parametri nella direzione opposta al gradiente:

$$
w_{\text{novo}} = w_{\text{atual}} - \eta \frac{\partial L}{\partial w},
$$

dove $\eta$ è il **tasso di apprendimento** (*learning rate*), responsabile del controllo dell'ampiezza dell'aggiornamento. La *SGD* utilizza solo il gradiente dell'iterazione corrente; quando la ricerca raggiunge un minimo locale, i gradienti diventano molto piccoli e gli aggiornamenti cessano praticamente.

##### Ottimizzatori Adattivi: *Adam*

L'**Adam** (*Adaptive Moment Estimation*) combina stime adattative dei primi e secondi momenti dei gradienti (KINGMA, 2015), adattando il tasso di apprendimento di ogni parametro individualmente. Questa adattazione favorisce, in molti casi, il superamento dei minimi locali che trattenerebbero il *SGD*.

La [Figura 9.9](#fig-09-sim-09-opt) confronta la traiettoria del *SGD* e dell'*Adam* sulla stessa superficie di perdita non convessa.

In [9]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-opt" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-opt .cap09opt_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-opt .cap09opt_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-opt .cap09opt_modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:8px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-opt .cap09opt_modebtn.cap09opt_active { background:#26241D; color:#FBF7EE; }
    #sim-09-opt .cap09opt_playbtn {
      padding:6px 12px; font-size:11px; font-weight:700; border-radius:8px; border:1px solid #2F6F9F;
      background:#EAF2FA; color:#2F6F9F; cursor:pointer; transition:background .15s ease;
    }
    #sim-09-opt .cap09opt_playbtn:hover { background:#DCEEFB; }
    #sim-09-opt .cap09opt_navbtn {
      font-size:11px; font-weight:600; padding:6px 10px; border-radius:8px;
      border:1px solid #E4DCC8; background:#FFF; color:#26241D; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-opt .cap09opt_navbtn:hover { background:#F1EAD7; }
    #sim-09-opt .cap09opt_infobox {
      background:#EAF2FA; border:1px solid #CFE2F3; border-radius:10px; padding:9px 12px;
      font-size:11px; color:#2c4a63; line-height:1.5; margin-bottom:12px;
    }
    #sim-09-opt .cap09opt_legendrow { display:flex; gap:14px; flex-wrap:wrap; align-items:center; margin-top:10px; font-size:10.5px; color:#5E5A4A; }
    #sim-09-opt .cap09opt_legenditem { display:flex; align-items:center; gap:5px; }
    #sim-09-opt .cap09opt_swatch { width:14px; height:3px; border-radius:2px; display:inline-block; }
    #sim-09-opt .cap09opt_dot { width:9px; height:9px; border-radius:50%; display:inline-block; }
    #sim-09-opt .cap09opt_checklbl { display:flex; align-items:center; gap:5px; font-size:10.5px; font-weight:600; color:#374151; cursor:pointer; user-select:none; }
    #sim-09-opt .cap09opt_statgrid { display:grid; grid-template-columns:1fr 1fr; gap:6px 14px; margin-top:6px; }
    #sim-09-opt .cap09opt_statlbl { font-size:9.5px; color:#8A8371; font-weight:700; letter-spacing:.2px; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:10px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚡ Simulador: Otimização com Curvas de Nível (SGD vs. Adam)</span>
    <span class="cap09opt_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Relevo Não Convexo: Mínimo Local vs. Global</span>
  </div>

  <div style="padding:16px 18px;background:#FFFFFF;overflow:auto">

    <!-- Caixa de contexto didático -->

<!-- Texto atualizado na infobox do simulador -->
<div class="cap09opt_infobox">
  💡 <b>Como ler este mapa:</b> a <b>seta amarela</b> aponta na direção de <b>descida</b> (&minus;∇L), que é o sentido oposto ao vetor gradiente (∇L). O otimizador avança nessa direção para reduzir a perda <i>L</i>(<i>w</i><sub>1</sub>, <i>w</i><sub>2</sub>) até atingir as regiões mais profundas (tons mais escuros).
</div>

    <!-- Seletores de Otimizador e Controles -->
    <div style="display:flex;gap:12px;flex-wrap:wrap;align-items:flex-end;margin-bottom:12px;">
      <div style="flex:1;min-width:200px;">
        <div class="cap09opt_grouplabel">ALGORITMO PRINCIPAL (linha sólida)</div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:10px;padding:3px;">
          <button id="cap09opt_btnSGD" class="cap09opt_modebtn cap09opt_active">SGD (Sem Momento)</button>
          <button id="cap09opt_btnAdam" class="cap09opt_modebtn">Adam (Com Momento)</button>
        </div>
      </div>

      <div style="flex:1;min-width:160px;">
        <div class="cap09opt_grouplabel">TAXA DE APRENDIZADO (η)</div>
        <select id="cap09opt_selLR" style="font-size:11px;padding:5px 8px;border-radius:8px;border:1px solid #E4DCC8;background:#FAF6EC;width:100%;font-weight:600;color:#374151;">
          <option value="0.12" selected>0.12 (Alta)</option>
          <option value="0.05">0.05 (Ideal)</option>
          <option value="0.01">0.01 (Lenta)</option>
        </select>
      </div>

      <div style="display:flex;gap:6px;">
        <button id="cap09opt_btnPasso" class="cap09opt_playbtn">▶ Passo</button>
        <button id="cap09opt_btnAuto" class="cap09opt_playbtn">⏵ Executar Auto</button>
        <button id="cap09opt_btnReset" class="cap09opt_navbtn">↺ Resetar</button>
      </div>
    </div>

    <label class="cap09opt_checklbl">
      <input type="checkbox" id="cap09opt_chkComparar" checked style="accent-color:#2F6F9F;cursor:pointer;">
      👻 Mostrar trajetória-fantasma do outro otimizador (comparação instantânea)
    </label>

    <!-- Mapa Topográfico / Superfície da Perda -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:18px;flex-wrap:wrap;align-items:center;justify-content:center;margin-top:12px;">
      <div style="text-align:center;">
        <div style="font-size:11px;font-weight:600;margin-bottom:6px;color:#5E5A4A;">Mapa de Calor da Perda L(w₁, w₂) — clique para escolher o início</div>
        <canvas id="cap09opt_canvasContorno" style="width:320px;height:240px;border:1px solid #E4DCC8;border-radius:10px;cursor:crosshair;box-shadow:0 2px 4px rgba(0,0,0,0.04);"></canvas>

        <div class="cap09opt_legendrow">
          <span class="cap09opt_legenditem"><span class="cap09opt_dot" style="background:#1E8F6F;"></span> Mínimo Global</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_dot" style="background:#C1443A;"></span> Mínimo Local</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" style="background:#F5B301;"></span> Gradiente (↓ descida)</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" id="cap09opt_legSolidLine" style="background:#C1443A;"></span> Trajetória principal</span>
          <span class="cap09opt_legenditem"><span class="cap09opt_swatch" id="cap09opt_legGhostLine" style="background:#2F6F9F;opacity:.5;background-image:repeating-linear-gradient(90deg,#2F6F9F 0 4px,transparent 4px 7px);"></span> Fantasma (outro otimizador)</span>
        </div>
      </div>

      <div style="flex:1;min-width:230px;">
        <div style="background:#FAF6EC;border:1px solid #EEE6D2;border-radius:10px;padding:12px;margin-bottom:10px;">
          <div style="font-size:11px;font-weight:700;color:#5E5A4A;margin-bottom:4px;">ESTADO DA OTIMIZAÇÃO:</div>
          <div class="cap09opt_mono" style="font-size:11px;color:#374151;">w₁ = <span id="cap09opt_txtW1">1.80</span>, w₂ = <span id="cap09opt_txtW2">0.20</span></div>
          <div class="cap09opt_mono" style="font-size:13px;font-weight:700;color:#C1443A;margin-top:4px;">Loss L = <span id="cap09opt_txtLoss">2.450</span></div>

          <div class="cap09opt_statgrid">
            <div>
              <div class="cap09opt_statlbl">PASSO</div>
              <div class="cap09opt_mono" style="font-size:12px;color:#26241D;font-weight:700;" id="cap09opt_txtPasso">0</div>
            </div>
            <div>
              <div class="cap09opt_statlbl">|∇L| (MAGNITUDE)</div>
              <div class="cap09opt_mono" style="font-size:12px;color:#26241D;font-weight:700;" id="cap09opt_txtGrad">0.000</div>
            </div>
          </div>

          <div id="cap09opt_txtStatusRegiao" class="cap09opt_mono" style="font-size:10px;color:#1E8F6F;margin-top:8px;font-weight:600;">Status: Ponto Inicial</div>
        </div>

        <div id="cap09opt_descOpt" style="font-size:11px;color:#6B7280;line-height:1.55;"></div>
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function cap09opt_hexToRgb(h){
    var v = parseInt(h.slice(1),16);
    return { r:(v>>16)&255, g:(v>>8)&255, b:v&255 };
  }
  function cap09opt_lerp(a,b,t){ return a + (b-a)*t; }
  function cap09opt_lerpColor(c1,c2,t){
    var A=cap09opt_hexToRgb(c1), B=cap09opt_hexToRgb(c2);
    return "rgb(" + Math.round(cap09opt_lerp(A.r,B.r,t)) + "," + Math.round(cap09opt_lerp(A.g,B.g,t)) + "," + Math.round(cap09opt_lerp(A.b,B.b,t)) + ")";
  }
  var cap09opt_STOPS = [
    { l:0.5, c:"#16202B" },
    { l:1.4, c:"#2F6F9F" },
    { l:2.3, c:"#8FB8C9" },
    { l:3.0, c:"#E8DEC4" },
    { l:3.8, c:"#FBF7EE" }
  ];
  function cap09opt_lossToColor(loss){
    for(var i=0;i<cap09opt_STOPS.length-1;i++){
      var a = cap09opt_STOPS[i], b = cap09opt_STOPS[i+1];
      if(loss >= a.l && loss <= b.l){
        var t = (loss - a.l) / (b.l - a.l);
        return cap09opt_lerpColor(a.c, b.c, t);
      }
    }
    return loss < cap09opt_STOPS[0].l ? cap09opt_STOPS[0].c : cap09opt_STOPS[cap09opt_STOPS.length-1].c;
  }

  function cap09opt_init(cap09opt_root){
    if(!cap09opt_root || cap09opt_root.dataset.initOpt) return;
    cap09opt_root.dataset.initOpt = "1";

    var cap09opt_BETA1 = 0.8, cap09opt_BETA2 = 0.99;

    var cap09opt_algoritmo = "sgd";
    var cap09opt_posInicial = { w1: 1.8, w2: 0.2 };
    var cap09opt_posW = { w1: 1.8, w2: 0.2 };
    var cap09opt_trajetoria = [{ w1: 1.8, w2: 0.2 }];
    var cap09opt_trajFantasma = [];
    var cap09opt_passoAtual = 0;

    var cap09opt_m = { w1: 0, w2: 0 };
    var cap09opt_v = { w1: 0, w2: 0 };
    var cap09opt_tStep = 0;
    var cap09opt_autoInterval = null;

    var cap09opt_canvas = cap09opt_root.querySelector('#cap09opt_canvasContorno');
    var cap09opt_ctx    = cap09opt_canvas.getContext('2d');

    function cap09opt_prepararCanvas(cap09opt_cvs, cap09opt_c, cap09opt_cssW, cap09opt_cssH){
      var cap09opt_dpr = window.devicePixelRatio || 1;
      cap09opt_cvs.width = cap09opt_cssW * cap09opt_dpr;
      cap09opt_cvs.height = cap09opt_cssH * cap09opt_dpr;
      cap09opt_c.scale(cap09opt_dpr, cap09opt_dpr);
    }
    cap09opt_prepararCanvas(cap09opt_canvas, cap09opt_ctx, 320, 240);

    var cap09opt_btnSGD    = cap09opt_root.querySelector('#cap09opt_btnSGD');
    var cap09opt_btnAdam   = cap09opt_root.querySelector('#cap09opt_btnAdam');
    var cap09opt_selLR     = cap09opt_root.querySelector('#cap09opt_selLR');
    var cap09opt_chkComp   = cap09opt_root.querySelector('#cap09opt_chkComparar');

    var cap09opt_btnPasso  = cap09opt_root.querySelector('#cap09opt_btnPasso');
    var cap09opt_btnAuto   = cap09opt_root.querySelector('#cap09opt_btnAuto');
    var cap09opt_btnReset  = cap09opt_root.querySelector('#cap09opt_btnReset');

    var cap09opt_txtW1     = cap09opt_root.querySelector('#cap09opt_txtW1');
    var cap09opt_txtW2     = cap09opt_root.querySelector('#cap09opt_txtW2');
    var cap09opt_txtLoss   = cap09opt_root.querySelector('#cap09opt_txtLoss');
    var cap09opt_txtPasso  = cap09opt_root.querySelector('#cap09opt_txtPasso');
    var cap09opt_txtGrad   = cap09opt_root.querySelector('#cap09opt_txtGrad');
    var cap09opt_txtStatus = cap09opt_root.querySelector('#cap09opt_txtStatusRegiao');
    var cap09opt_descOpt   = cap09opt_root.querySelector('#cap09opt_descOpt');
    var cap09opt_legGhost  = cap09opt_root.querySelector('#cap09opt_legGhostLine');
    var cap09opt_legSolid  = cap09opt_root.querySelector('#cap09opt_legSolidLine');

    function cap09opt_wToPx(w1, w2){
      return { x: 160 + w1 * 55, y: 120 - w2 * 45 };
    }
    function cap09opt_pxToW(x, y){
      return { w1: (x - 160) / 55.0, w2: (120 - y) / 45.0 };
    }

    function cap09opt_calcLoss(w1, w2){
      var gGlobal = 3.0 * Math.exp(-((w1 + 1.5)*(w1 + 1.5)*0.8 + w2*w2*1.5));
      var gLocal  = 1.6 * Math.exp(-((w1 - 1.5)*(w1 - 1.5)*1.2 + w2*w2*1.5));
      var parabola = 0.18 * (w1*w1 + w2*w2);
      return 3.5 - gGlobal - gLocal + parabola;
    }

    function cap09opt_calcGrad(w1, w2){
      var eps = 0.001;
      var l0 = cap09opt_calcLoss(w1, w2);
      var dw1 = (cap09opt_calcLoss(w1 + eps, w2) - l0) / eps;
      var dw2 = (cap09opt_calcLoss(w1, w2 + eps) - l0) / eps;
      return { g1: dw1, g2: dw2 };
    }

    function cap09opt_passoGenerico(algo, estado, lr){
      var grad = cap09opt_calcGrad(estado.w1, estado.w2);
      if(algo === "sgd"){
        estado.w1 -= lr * grad.g1;
        estado.w2 -= lr * grad.g2;
      } else {
        estado.t = (estado.t||0) + 1;
        estado.m1 = cap09opt_BETA1 * (estado.m1||0) + (1-cap09opt_BETA1) * grad.g1;
        estado.m2 = cap09opt_BETA1 * (estado.m2||0) + (1-cap09opt_BETA1) * grad.g2;
        estado.v1 = cap09opt_BETA2 * (estado.v1||0) + (1-cap09opt_BETA2) * (grad.g1*grad.g1);
        estado.v2 = cap09opt_BETA2 * (estado.v2||0) + (1-cap09opt_BETA2) * (grad.g2*grad.g2);
        var mHat1 = estado.m1 / (1 - Math.pow(cap09opt_BETA1, estado.t));
        var mHat2 = estado.m2 / (1 - Math.pow(cap09opt_BETA1, estado.t));
        var vHat1 = estado.v1 / (1 - Math.pow(cap09opt_BETA2, estado.t));
        var vHat2 = estado.v2 / (1 - Math.pow(cap09opt_BETA2, estado.t));
        estado.w1 -= (lr / (Math.sqrt(vHat1) + 1e-4)) * mHat1 * 1.5;
        estado.w2 -= (lr / (Math.sqrt(vHat2) + 1e-4)) * mHat2 * 1.5;
      }
      return grad;
    }

    function cap09opt_computarFantasma(){
      var outroAlgo = cap09opt_algoritmo === "sgd" ? "adam" : "sgd";
      var lr = parseFloat(cap09opt_selLR.value);
      var estado = { w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 };
      var caminho = [{ w1: estado.w1, w2: estado.w2 }];
      for(var i=0; i<150; i++){
        var antesW1 = estado.w1, antesW2 = estado.w2;
        cap09opt_passoGenerico(outroAlgo, estado, lr);
        caminho.push({ w1: estado.w1, w2: estado.w2 });
        var delta = Math.hypot(estado.w1-antesW1, estado.w2-antesW2);
        if(delta < 0.0008 && i > 6) break;
      }
      return caminho;
    }

    function cap09opt_desenharSeta(x, y, ang, comprimento, cor){
      var x2 = x + Math.cos(ang) * comprimento;
      var y2 = y + Math.sin(ang) * comprimento;
      cap09opt_ctx.strokeStyle = cor; cap09opt_ctx.fillStyle = cor; cap09opt_ctx.lineWidth = 2;
      cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(x,y); cap09opt_ctx.lineTo(x2,y2); cap09opt_ctx.stroke();
      var cabeca = 6, angSeta = Math.atan2(y2-y, x2-x);
      cap09opt_ctx.beginPath();
      cap09opt_ctx.moveTo(x2, y2);
      cap09opt_ctx.lineTo(x2 - cabeca*Math.cos(angSeta-0.45), y2 - cabeca*Math.sin(angSeta-0.45));
      cap09opt_ctx.lineTo(x2 - cabeca*Math.cos(angSeta+0.45), y2 - cabeca*Math.sin(angSeta+0.45));
      cap09opt_ctx.closePath(); cap09opt_ctx.fill();
    }

    function cap09opt_desenharMapa(){
      var W = 320, H = 240, gridRes = 5;
      cap09opt_ctx.clearRect(0,0,W,H);

      for(var py=0; py<H; py+=gridRes){
        for(var px=0; px<W; px+=gridRes){
          var wC = cap09opt_pxToW(px + gridRes/2, py + gridRes/2);
          var lC = cap09opt_calcLoss(wC.w1, wC.w2);
          cap09opt_ctx.fillStyle = cap09opt_lossToColor(lC);
          cap09opt_ctx.fillRect(px, py, gridRes+0.5, gridRes+0.5);
        }
      }

      var niveisLoss = [0.8, 1.2, 1.6, 2.0, 2.4, 2.8, 3.2, 3.8];
      for(var nIdx=0; nIdx<niveisLoss.length; nIdx++){
        var alvoL = niveisLoss[nIdx];
        cap09opt_ctx.strokeStyle = "rgba(20, 24, 30, 0.18)";
        cap09opt_ctx.lineWidth = 1;
        for(var qy=0; qy<H; qy+=gridRes){
          for(var qx=0; qx<W; qx+=gridRes){
            var wA = cap09opt_pxToW(qx, qy);
            var lA = cap09opt_calcLoss(wA.w1, wA.w2);
            var wB = cap09opt_pxToW(qx + gridRes, qy);
            var lB = cap09opt_calcLoss(wB.w1, wB.w2);
            var wCc = cap09opt_pxToW(qx, qy + gridRes);
            var lCc = cap09opt_calcLoss(wCc.w1, wCc.w2);
            if((lA <= alvoL && lB >= alvoL) || (lA >= alvoL && lB <= alvoL)){
              cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(qx, qy); cap09opt_ctx.lineTo(qx + gridRes, qy); cap09opt_ctx.stroke();
            }
            if((lA <= alvoL && lCc >= alvoL) || (lA >= alvoL && lCc <= alvoL)){
              cap09opt_ctx.beginPath(); cap09opt_ctx.moveTo(qx, qy); cap09opt_ctx.lineTo(qx, qy + gridRes); cap09opt_ctx.stroke();
            }
          }
        }
      }

      cap09opt_ctx.strokeStyle = "rgba(255,255,255,0.35)";
      cap09opt_ctx.lineWidth = 1;
      cap09opt_ctx.beginPath();
      cap09opt_ctx.moveTo(0, 120); cap09opt_ctx.lineTo(W, 120);
      cap09opt_ctx.moveTo(160, 0); cap09opt_ctx.lineTo(160, H);
      cap09opt_ctx.stroke();

      cap09opt_ctx.font = "600 9px 'JetBrains Mono', monospace";
      cap09opt_ctx.fillStyle = "rgba(38,36,29,0.55)";
      cap09opt_ctx.textAlign = "center";
      for(var wv=-2; wv<=2; wv++){
        if(wv===0) continue;
        var px1 = cap09opt_wToPx(wv, 0);
        cap09opt_ctx.fillText(wv.toString(), px1.x, 132);
        var py1 = cap09opt_wToPx(0, wv*0.9);
        cap09opt_ctx.fillText(wv.toString(), 172, py1.y+3);
      }
      cap09opt_ctx.font = "700 10px 'Inter', sans-serif";
      cap09opt_ctx.fillText("w₁ →", 300, 134);
      cap09opt_ctx.save(); cap09opt_ctx.translate(150, 14); cap09opt_ctx.fillText("w₂ ↑", 0, 0); cap09opt_ctx.restore();

      var posGlobal = cap09opt_wToPx(-1.4, 0);
      var posLocal  = cap09opt_wToPx(1.3, 0);

      [ [posGlobal, "#1E8F6F", "Mínimo Global ★"], [posLocal, "#C1443A", "Mínimo Local ⚠️"] ].forEach(function(item){
        var p = item[0];
        cap09opt_ctx.fillStyle = "rgba(255,255,255,0.65)";
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(p.x, p.y, 8, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.fillStyle = item[1];
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(p.x, p.y, 4.5, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.strokeStyle = "#FFF"; cap09opt_ctx.lineWidth = 1.5; cap09opt_ctx.stroke();
        cap09opt_ctx.font = "700 9px 'JetBrains Mono', monospace";
        cap09opt_ctx.fillStyle = "#26241D";
        cap09opt_ctx.textAlign = "center";
        cap09opt_ctx.fillText(item[2], p.x, p.y - 12);
      });

      if(cap09opt_chkComp.checked && cap09opt_trajFantasma.length > 1){
        var corFantasma = cap09opt_algoritmo === "sgd" ? "#2F6F9F" : "#C1443A";
        cap09opt_ctx.save();
        cap09opt_ctx.setLineDash([5,4]);
        cap09opt_ctx.strokeStyle = corFantasma;
        cap09opt_ctx.globalAlpha = 0.55;
        cap09opt_ctx.lineWidth = 2;
        cap09opt_ctx.beginPath();
        for(var fi=0; fi<cap09opt_trajFantasma.length; fi++){
          var fp = cap09opt_wToPx(cap09opt_trajFantasma[fi].w1, cap09opt_trajFantasma[fi].w2);
          if(fi===0) cap09opt_ctx.moveTo(fp.x, fp.y); else cap09opt_ctx.lineTo(fp.x, fp.y);
        }
        cap09opt_ctx.stroke();
        cap09opt_ctx.restore();
        var fEnd = cap09opt_wToPx(cap09opt_trajFantasma[cap09opt_trajFantasma.length-1].w1, cap09opt_trajFantasma[cap09opt_trajFantasma.length-1].w2);
        cap09opt_ctx.fillStyle = corFantasma; cap09opt_ctx.globalAlpha = 0.7;
        cap09opt_ctx.beginPath(); cap09opt_ctx.arc(fEnd.x, fEnd.y, 4, 0, 2*Math.PI); cap09opt_ctx.fill();
        cap09opt_ctx.globalAlpha = 1;
      }

      if(cap09opt_trajetoria.length > 1){
        var corPrincipal = cap09opt_algoritmo === "sgd" ? "#C1443A" : "#2F6F9F";
        cap09opt_ctx.strokeStyle = corPrincipal;
        cap09opt_ctx.lineWidth = 2.5;
        cap09opt_ctx.beginPath();
        for(var i=0; i<cap09opt_trajetoria.length; i++){
          var p = cap09opt_wToPx(cap09opt_trajetoria[i].w1, cap09opt_trajetoria[i].w2);
          if(i === 0) cap09opt_ctx.moveTo(p.x, p.y); else cap09opt_ctx.lineTo(p.x, p.y);
        }
        cap09opt_ctx.stroke();
        for(var j=0; j<cap09opt_trajetoria.length; j++){
          var pj = cap09opt_wToPx(cap09opt_trajetoria[j].w1, cap09opt_trajetoria[j].w2);
          cap09opt_ctx.fillStyle = corPrincipal;
          cap09opt_ctx.beginPath(); cap09opt_ctx.arc(pj.x, pj.y, 2, 0, 2*Math.PI); cap09opt_ctx.fill();
        }
      }

      var pInicio = cap09opt_wToPx(cap09opt_posInicial.w1, cap09opt_posInicial.w2);
      cap09opt_ctx.fillStyle = "#FBF7EE";
      cap09opt_ctx.beginPath(); cap09opt_ctx.arc(pInicio.x, pInicio.y, 7, 0, 2*Math.PI); cap09opt_ctx.fill();
      cap09opt_ctx.strokeStyle = "#26241D"; cap09opt_ctx.lineWidth = 1.5; cap09opt_ctx.stroke();
      cap09opt_ctx.fillStyle = "#26241D"; cap09opt_ctx.font = "700 8px 'JetBrains Mono', monospace";
      cap09opt_ctx.textAlign = "center"; cap09opt_ctx.textBaseline = "middle";
      cap09opt_ctx.fillText("S", pInicio.x, pInicio.y);

      // Ponto atual + seta do gradiente DESCENDENTE (CORRIGIDO)
      var ptAtual = cap09opt_wToPx(cap09opt_posW.w1, cap09opt_posW.w2);
      var gradAtual = cap09opt_calcGrad(cap09opt_posW.w1, cap09opt_posW.w2);
      
      // Mapeia 1 passo na direção de DESCIDA ( - gradiente )
      var ptDescida = cap09opt_wToPx(cap09opt_posW.w1 - gradAtual.g1 * 0.2, cap09opt_posW.w2 - gradAtual.g2 * 0.2);
      var angTela = Math.atan2(ptDescida.y - ptAtual.y, ptDescida.x - ptAtual.x);
      var magGrad = Math.hypot(gradAtual.g1, gradAtual.g2);

      if(magGrad > 0.01){
        cap09opt_desenharSeta(ptAtual.x, ptAtual.y, angTela, 22, "#F5B301");
      }

      cap09opt_ctx.fillStyle = "#26241D";
      cap09opt_ctx.beginPath(); cap09opt_ctx.arc(ptAtual.x, ptAtual.y, 6, 0, 2*Math.PI); cap09opt_ctx.fill();
      cap09opt_ctx.strokeStyle = "#FFFFFF"; cap09opt_ctx.lineWidth = 2; cap09opt_ctx.stroke();

      return magGrad;
    }

    function cap09opt_darPasso(){
      var lr = parseFloat(cap09opt_selLR.value);
      var estadoTmp = { w1: cap09opt_posW.w1, w2: cap09opt_posW.w2, m1: cap09opt_m.w1, m2: cap09opt_m.w2, v1: cap09opt_v.w1, v2: cap09opt_v.w2, t: cap09opt_tStep };
      cap09opt_passoGenerico(cap09opt_algoritmo, estadoTmp, lr);
      cap09opt_posW.w1 = estadoTmp.w1; cap09opt_posW.w2 = estadoTmp.w2;
      cap09opt_m.w1 = estadoTmp.m1||0; cap09opt_m.w2 = estadoTmp.m2||0;
      cap09opt_v.w1 = estadoTmp.v1||0; cap09opt_v.w2 = estadoTmp.v2||0;
      cap09opt_tStep = estadoTmp.t||0;

      cap09opt_trajetoria.push({ w1: cap09opt_posW.w1, w2: cap09opt_posW.w2 });
      cap09opt_passoAtual++;
      cap09opt_atualizar();
    }

    function cap09opt_atualizar(){
      var magGrad = cap09opt_desenharMapa();
      var loss = cap09opt_calcLoss(cap09opt_posW.w1, cap09opt_posW.w2);
      cap09opt_txtW1.textContent   = cap09opt_posW.w1.toFixed(2);
      cap09opt_txtW2.textContent   = cap09opt_posW.w2.toFixed(2);
      cap09opt_txtLoss.textContent = loss.toFixed(3);
      cap09opt_txtPasso.textContent = cap09opt_passoAtual;
      cap09opt_txtGrad.textContent = magGrad.toFixed(3);

      var lrVal = parseFloat(cap09opt_selLR.value);
      var conseguiuEscapar = cap09opt_posW.w1 < -0.5;
      var estaPresoLocal = cap09opt_posW.w1 > 0.5;
      var corPrincipal = cap09opt_algoritmo === "sgd" ? "#C1443A" : "#2F6F9F";
      cap09opt_legSolid.style.background = corPrincipal;
      var corFantasma = cap09opt_algoritmo === "sgd" ? "#2F6F9F" : "#C1443A";
      cap09opt_legGhost.style.backgroundImage = "repeating-linear-gradient(90deg,"+corFantasma+" 0 4px,transparent 4px 7px)";

      if(estaPresoLocal){
        cap09opt_txtStatus.textContent = "Região: Preso no Mínimo Local ⚠️";
        cap09opt_txtStatus.style.color = "#C1443A";
      } else if(conseguiuEscapar){
        cap09opt_txtStatus.textContent = "Região: Convergiu para Mínimo Global ★";
        cap09opt_txtStatus.style.color = "#1E8F6F";
      } else {
        cap09opt_txtStatus.textContent = "Região: Aclive / Transposição de Barreira";
        cap09opt_txtStatus.style.color = "#2F6F9F";
      }

      if(cap09opt_algoritmo === "sgd"){
        cap09opt_descOpt.innerHTML = "<b>SGD (Sem Momento):</b> a cada passo, o SGD olha apenas para o gradiente <i>local e instantâneo</i> — sem memória do que veio antes. Por isso, ao partir do lado direito, ele fica <b>preso no Mínimo Local</b>: não tem energia acumulada para subir o aclive até a barreira central. Compare com a linha tracejada azul (Adam) ao lado.";
      } else {
        if(conseguiuEscapar){
          cap09opt_descOpt.innerHTML = "<b>Adam (Sucesso):</b> o Adam acumula <i>momento</i> (uma média móvel dos gradientes recentes) e ajusta a taxa de cada peso adaptativamente. Com η=" + lrVal + ", esse impulso acumulado foi suficiente para vencer a barreira e alcançar o <b>Mínimo Global ★</b>. Note como a linha tracejada vermelha (SGD) fica presa antes disso.";
        } else if(estaPresoLocal && cap09opt_trajetoria.length > 8){
          cap09opt_descOpt.innerHTML = "<b>Adam (Retido no Mínimo Local):</b> mesmo acumulando momento, com η=" + lrVal + " o impulso não foi suficiente para transpor a elevação. <i>Isso mostra que nem mesmo o Adam garante escapar de poços profundos sem ajuste fino da taxa de aprendizado ou de uma inicialização melhor.</i>";
        } else {
          cap09opt_descOpt.innerHTML = "<b>Adam (Em movimento):</b> acumulando momento e ajustando o tamanho do passo adaptativamente conforme percorre o relevo...";
        }
      }
    }

    function cap09opt_resetar(){
      if(cap09opt_autoInterval) { clearInterval(cap09opt_autoInterval); cap09opt_autoInterval = null; cap09opt_btnAuto.textContent = "⏵ Executar Auto"; }
      cap09opt_posW = { w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 };
      cap09opt_trajetoria = [{ w1: cap09opt_posInicial.w1, w2: cap09opt_posInicial.w2 }];
      cap09opt_m = { w1: 0, w2: 0 }; cap09opt_v = { w1: 0, w2: 0 }; cap09opt_tStep = 0;
      cap09opt_passoAtual = 0;
      cap09opt_trajFantasma = cap09opt_computarFantasma();
      cap09opt_atualizar();
    }

    cap09opt_canvas.addEventListener('click', function(evt){
      var rect = cap09opt_canvas.getBoundingClientRect();
      var scaleX = 320 / rect.width, scaleY = 240 / rect.height;
      var clickX = (evt.clientX - rect.left) * scaleX;
      var clickY = (evt.clientY - rect.top) * scaleY;
      var ptW = cap09opt_pxToW(clickX, clickY);
      cap09opt_posInicial = { w1: ptW.w1, w2: ptW.w2 };
      cap09opt_resetar();
    });

    cap09opt_btnSGD.addEventListener('click', function(){
      cap09opt_btnSGD.classList.add('cap09opt_active'); cap09opt_btnAdam.classList.remove('cap09opt_active');
      cap09opt_algoritmo = "sgd"; cap09opt_resetar();
    });
    cap09opt_btnAdam.addEventListener('click', function(){
      cap09opt_btnAdam.classList.add('cap09opt_active'); cap09opt_btnSGD.classList.remove('cap09opt_active');
      cap09opt_algoritmo = "adam"; cap09opt_resetar();
    });

    cap09opt_btnPasso.addEventListener('click', cap09opt_darPasso);
    cap09opt_btnReset.addEventListener('click', cap09opt_resetar);
    cap09opt_chkComp.addEventListener('change', cap09opt_atualizar);

    cap09opt_btnAuto.addEventListener('click', function(){
      if(cap09opt_autoInterval){
        clearInterval(cap09opt_autoInterval); cap09opt_autoInterval = null; cap09opt_btnAuto.textContent = "⏵ Executar Auto";
      } else {
        cap09opt_btnAuto.textContent = "⏸ Pausar";
        cap09opt_autoInterval = setInterval(cap09opt_darPasso, 120);
      }
    });

    cap09opt_selLR.addEventListener('change', cap09opt_resetar);

    cap09opt_resetar();
  }

  function cap09opt_tryInit(){
    var cap09opt_root = document.getElementById('sim-09-opt');
    if(cap09opt_root) cap09opt_init(cap09opt_root); else setTimeout(cap09opt_tryInit, 200);
  }
  cap09opt_tryInit();
})();
</script>
''')

**Figura 9.9:** Simulador interativo dos Algoritmos de Otimização: compare a trajetória do SGD e do Adam sobre uma superfície de perda não convexa com mapa de calor e curvas de nível. A linha sólida mostra o otimizador selecionado avançando passo a passo; a linha tracejada mostra, para comparação instantânea, o caminho completo que o outro otimizador percorreria a partir do mesmo ponto inicial. Observe como o SGD fica retido no Mínimo Local à direita, enquanto o Adam pode ou não transpor a barreira central dependendo do impulso acumulado e da taxa de aprendizado. Clique em qualquer ponto do mapa para redefinir o ponto inicial dos pesos.


<figure id="fig-09-sim-09-opt">
  <img src="imagens/fig-09-sim-09-opt.png" alt=" Simulador interativo dos Algoritmos de Otimização: compare a trajetória do SGD e do Adam sobre uma superfície de perda não convexa com mapa de calor e curvas de nível. A linha sólida mostra o otimizador selecionado avançando passo a passo; a linha tracejada mostra, para comparação instantânea, o caminho completo que o outro otimizador percorreria a partir do mesmo ponto inicial. Observe como o SGD fica retido no Mínimo Local à direita, enquanto o Adam pode ou não transpor a barreira central dependendo do impulso acumulado e da taxa de aprendizado. Clique em qualquer ponto do mapa para redefinir o ponto inicial dos pesos. " style="max-width:80%" />
  <figcaption><strong>Figura 9.9:</strong>  Simulador interativo dos Algoritmos de Otimização: compare a trajetória do SGD e do Adam sobre uma superfície de perda não convexa com mapa de calor e curvas de nível. A linha sólida mostra o otimizador selecionado avançando passo a passo; a linha tracejada mostra, para comparação instantânea, o caminho completo que o outro otimizador percorreria a partir do mesmo ponto inicial. Observe como o SGD fica retido no Mínimo Local à direita, enquanto o Adam pode ou não transpor a barreira central dependendo do impulso acumulado e da taxa de aprendizado. Clique em qualquer ponto do mapa para redefinir o ponto inicial dos pesos. </figcaption>
</figure>

### 9.4.7 Architettura di una CNN

Una CNN per la classificazione delle immagini combina i livelli presentati nelle sezioni precedenti. Durante il **passo in avanti** (*forward pass*), l'immagine attraversa successivamente i livelli convoluzionali, le funzioni di attivazione, le operazioni di *pooling*, la fase di **Flatten**, i livelli completamente connessi e, infine, il livello **Softmax**, che produce le probabilità delle classi. Durante l'addestramento, questa previsione viene confrontata con l'etichetta corretta per calcolare la funzione di perdita, eseguire la retropropagazione e aggiornare i parametri tramite un algoritmo di ottimizzazione.

La [Figura 9.10](#fig-09-cnn-arquitetura) presenta questo flusso di elaborazione e addestramento.

<figure id="fig-09-cnn-arquitetura" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-09-cnn-arquitetura.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figura 9.10:</strong> Architettura semplificata di una CNN per la classificazione delle immagini, evidenziando il *forward pass* e le fasi di addestramento tramite la funzione di perdita, la retropropagazione e l'algoritmo di ottimizzazione.</figcaption>
</figure>

Dopo l'ultimo blocco convoluzionale, l'operazione **Flatten** riorganizza le mappe delle caratteristiche in un vettore unidimensionale, che alimenta i **livelli completamente connessi** (*fully connected layers*), responsabili di combinare le caratteristiche estratte per produrre i punteggi (*logits*) di ciascuna classe. Il livello **Softmax** converte questi punteggi in una distribuzione di probabilità, utilizzata sia per la classificazione sia per il calcolo della funzione di perdita durante l'addestramento.

La [Figura 9.11](#fig-09-sim-09-arquitetura) presenta una versione interattiva di questa architettura, consentendo di eseguire fasi successive di addestramento e di osservare la riduzione della perdita, la retropropagazione dei gradienti e l'aggiornamento dei filtri della rete.

In [10]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-arquitetura" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-arquitetura .cn-mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-arquitetura .cn-grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-arquitetura .cn-modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-modebtn.active { background:#26241D; color:#FBF7EE; }
    #sim-09-arquitetura .cn-navbtn2 {
      padding:0 10px; height:26px; border-radius:8px; border:1px solid #E4DCC8; background:#FAFAF7;
      color:#26241D; font-size:10.5px; font-weight:600; cursor:pointer; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-navbtn2:hover { background:#F1EAD7; }
    #sim-09-arquitetura .cn-playbtn {
      padding:0 12px; height:26px; border-radius:8px; border:1px solid #2F6F9F; background:#EAF2FA;
      color:#2F6F9F; font-size:10.5px; font-weight:700; cursor:pointer; white-space:nowrap;
    }
    #sim-09-arquitetura .cn-playbtn:hover { background:#DCEEFB; }
    #sim-09-arquitetura .cn-kcell {
      width:30px;height:30px;border-radius:5px;border:1px solid #e5e7eb;display:flex;
      align-items:center;justify-content:center;font-size:8.5px;font-weight:700;
    }
    #sim-09-arquitetura .cn-ciclo { font-size:10.5px; transition: color .3s ease, background .3s ease; padding:3px 6px; border-radius:6px; }
    #sim-09-arquitetura .cn-ciclo.pulso { background:#FCE8E6; color:#C1443A; font-weight:700; }
    #sim-09-arquitetura .cn-graflabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:4px; text-align:center; letter-spacing:.3px; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">⚙️ Simulatore: Architettura Completa di una CNN</span>
    <span class="cn-mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">Input (12×12) → Conv → Pool → FC → Softmax</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletor de Imagem de Entrada para manter a mesma didática do simulador anterior -->
    <div style="margin-bottom:12px;max-width:320px;">
      <div class="cn-grouplabel">IMMAGINE DI INPUT DEL PIPELINE (12×12)</div>
      <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
        <button id="cap09arch_btnCasa" class="cn-modebtn active">🏠 Casa</button>
        <button id="cap09arch_btnFeliz" class="cn-modebtn">😊 Felice</button>
        <button id="cap09arch_btnTriste" class="cn-modebtn">😢 Triste</button>
      </div>
    </div>

    <!-- Fluxo das Camadas -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:10px;margin-bottom:14px;overflow-x:auto;">
      <div id="cap09arch_flow" style="display:flex;align-items:center;gap:2px;padding:4px 2px;min-width:680px;"></div>
    </div>

    <!-- Detalhe da Visualização da Camada Selecionada -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;display:flex;gap:20px;flex-wrap:wrap;align-items:center;margin-bottom:14px;">
      <div style="flex:0 0 auto;text-align:center;">
        <canvas id="cap09arch_canvasVis" width="280" height="200" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;box-shadow:0 1px 3px rgba(0,0,0,0.03);"></canvas>
        <div id="cap09arch_barsWrap" style="display:none;align-items:flex-end;gap:24px;height:140px;margin-top:10px;justify-content:center;padding:0 10px;"></div>
        <div id="cap09arch_kernelsPanel" style="display:none;margin-top:10px;"></div>
        <div id="cap09arch_legenda" style="font-size:10.5px;color:#8A8371;margin-top:8px;max-width:280px;line-height:1.4;"></div>
      </div>
      <div id="cap09arch_desc" style="flex:1;min-width:240px;font-size:12px;line-height:1.6;color:#374151;"></div>
    </div>

    <!-- Painel de Treinamento: Forward Pass + Retropropagação de verdade -->
    <div style="background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:14px;">
      <div style="font-size:11.5px;font-weight:600;color:#4b5563;margin-bottom:10px;">🎯 Addestramento (Forward Pass + Retropropagazione)</div>

      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;">
        <div style="min-width:230px;">
          <div id="cap09arch_cicloForward" class="cn-mono cn-ciclo" style="color:#374151;">Input ▸ Conv+ReLU ▸ Pool ▸ Flatten ▸ FC ▸ Softmax ▸ Previsione</div>
          <div id="cap09arch_cicloBackward" class="cn-mono cn-ciclo" style="color:#8A8371;margin-top:3px;">Perdita ◂ Ottimizzatore ◂ Retropropagazione ◂ (a ogni passo)</div>
        </div>
        <div>
          <div class="cn-graflabel">PERDITA (LOSS)</div>
          <canvas id="cap09arch_canvasLoss" width="260" height="110" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div>
          <div class="cn-graflabel">ACCURATEZZA</div>
          <canvas id="cap09arch_canvasAcc" width="260" height="110" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div style="min-width:170px;">
          <div id="cap09arch_lossTxt" class="cn-mono" style="font-size:11px;color:#7EE7C6;background:#1B2430;padding:6px 10px;border-radius:6px;">Loss: —</div>
          <div id="cap09arch_accTxt" class="cn-mono" style="font-size:11px;color:#FFD98E;background:#1B2430;padding:6px 10px;border-radius:6px;margin-top:5px;">Accuratezza: —</div>
          <div id="cap09arch_stepTxt" class="cn-mono" style="font-size:10.5px;color:#8A8371;margin-top:5px;">Passo di addestramento: 0</div>
          <div style="display:flex;gap:5px;margin-top:8px;flex-wrap:wrap;">
            <button id="cap09arch_btnPassoUnico" class="cn-navbtn2">Passo Singolo</button>
            <button id="cap09arch_btnTreinar" class="cn-playbtn">▶ Addestra</button>
            <button id="cap09arch_btnReiniciarPesos" class="cn-navbtn2">↺ Nuovi Pesi</button>
          </div>
        </div>
      </div>

      <div id="cap09arch_notaTreino" style="font-size:10.5px;color:#8A8371;margin-top:10px;padding-top:8px;border-top:1px dashed #E9E3D3;line-height:1.5;">
        I <i>kernels</i> e i pesi iniziano <b>casuali</b> (non sono più i filtri fissi del simulatore precedente). A ogni passo, la rete esegue il <i>forward pass</i> sulle 3 immagini, calcola la perdita (<i>cross-entropy</i>) e a <b>accuratezza</b> (quante delle 3 immagini sono classificate correttamente), retropropaga l'errore e regola tutti i pesi (inclusi i <i>kernels</i> della convoluzione) tramite discesa del gradiente. ⚠️ Poiché il "set di addestramento" ha solo 3 esempi, ciò dimostra il <b>meccanismo</b> dell'addestramento (perdita in calo, accuratezza in aumento, pesi che cambiano) — non la capacità di generalizzare a immagini nuove, che richiederebbe molti più dati.
      </div>
    </div>

  </div>
</div>

<script>
(function(){
  function clamp01(v){ return Math.max(0, Math.min(1, v)); }
  // Transforma uma ativação ReLU (não-negativa, sem limite superior) em algo
  // sempre entre 0 e 1 apenas para fins de exibição em cor — os valores brutos
  // usados no forward/backward NÃO passam por essa saturação.
  function saturar(v){ return 1 - Math.exp(-Math.max(0, v)); }

  // Gerador pseudoaleatório determinístico (mesma semente = mesmo resultado
  // inicial), para que o comportamento do simulador seja reprodutível.
  function criarRng(seed){
    return function(){
      seed |= 0; seed = (seed + 0x6D2B79F5) | 0;
      var t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
      t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
      return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
    };
  }

  // Mesmas matrizes 12x12 em formato ASCII usadas no simulador da camada convolucional
  var IMAGENS_ASCII = {
    casa: [
      "............",
      "....XXXX....",
      "...XXXXXX...",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXXX..XXXX.",
      ".XXXX..XXXX.",
      ".XXXX..XXXX.",
      "............"
    ],
    feliz: [
      "............",
      "....XXXX....",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXX.XX.XXX.",
      ".XXXXXXXXXX.",
      ".XX......XX.",
      ".XXX....XXX.",
      ".XXXXXXXXXX.",
      "..XXXXXXXX..",
      "............"
    ],
    triste: [
      "............",
      "....XXXX....",
      "..XXXXXXXX..",
      ".XXXXXXXXXX.",
      ".XXXXXXXXXX.",
      ".XXX.XX.XXX.",
      ".XXXXXXXXXX.",
      ".XXX....XXX.",
      ".XX......XX.",
      ".XXXXXXXXXX.",
      "..XXXXXXXX..",
      "............"
    ]
  };
  var ORDEM_CLASSES = ["casa", "feliz", "triste"];
  var ROTULOS = { casa: "🏠 Casa", feliz: "😊 Feliz", triste: "😢 Triste" };
  var ONE_HOTS = { casa: [1,0,0], feliz: [0,1,0], triste: [0,0,1] };

  function converterLinhas(linhas){
    return linhas.map(function(l){
      var res = [];
      for (var i = 0; i < l.length; i++) res.push(l[i] === 'X' ? 1.0 : 0.0);
      return res;
    });
  }
  var IMAGENS = {};
  ORDEM_CLASSES.forEach(function(id){ IMAGENS[id] = converterLinhas(IMAGENS_ASCII[id]); });

  function corMapa(v){
    var r = Math.round(255 - v*(255-47));
    var g = Math.round(255 - v*(255-111));
    var b = Math.round(255 - v*(255-159));
    return 'rgb('+r+','+g+','+b+')';
  }

  // ---------------------------------------------------------------------
  // Rede: Conv1 (4 filtros 3×3) → ReLU → MaxPool 2×2 → Flatten(100) →
  // Densa1 (16, ReLU) → Densa2/saída (3 logits) → Softmax.
  // Implementação manual de forward e backward (sem bibliotecas), pensada
  // para ficar pequena o bastante para caber num simulador didático.
  // ---------------------------------------------------------------------

  function inicializarParametros(rng){
    var K = [], bConv = [];
    for (var f = 0; f < 4; f++){
      var k = [];
      for (var r = 0; r < 3; r++){
        var row = [];
        for (var c = 0; c < 3; c++) row.push((rng() - 0.5) * 1.0);
        k.push(row);
      }
      K.push(k); bConv.push(0);
    }
    var W1 = [], b1 = [];
    for (var i = 0; i < 16; i++){
      var row1 = [];
      for (var j = 0; j < 100; j++) row1.push((rng() - 0.5) * 0.2);
      W1.push(row1); b1.push(0);
    }
    var W2 = [], b2 = [];
    for (var i2 = 0; i2 < 3; i2++){
      var row2 = [];
      for (var j2 = 0; j2 < 16; j2++) row2.push((rng() - 0.5) * 0.3);
      W2.push(row2); b2.push(0);
    }
    return { K: K, bConv: bConv, W1: W1, b1: b1, W2: W2, b2: b2 };
  }

  function relu(x){ return Math.max(0, x); }
  function reluDeriv(x){ return x > 0 ? 1 : 0; }
  function softmax(logits){
    var m = Math.max.apply(null, logits);
    var exps = logits.map(function(v){ return Math.exp(v - m); });
    var soma = exps.reduce(function(a,b){ return a+b; }, 0);
    return exps.map(function(v){ return v / soma; });
  }

  function forwardPassRede(p, img){
    var Z1 = [], A1 = [];
    for (var f = 0; f < 4; f++) {
      var zf = [], af = [];
      for (var r = 0; r < 10; r++) {
        var zr = [], ar = [];
        for (var c = 0; c < 10; c++) {
          var s = p.bConv[f];
          for (var kr = 0; kr < 3; kr++)
            for (var kc = 0; kc < 3; kc++)
              s += img[r+kr][c+kc] * p.K[f][kr][kc];
          zr.push(s); ar.push(relu(s));
        }
        zf.push(zr); af.push(ar);
      }
      Z1.push(zf); A1.push(af);
    }

    var P1 = [], argMax = [];
    for (var f2 = 0; f2 < 4; f2++) {
      var pf = [], amf = [];
      for (var r2 = 0; r2 < 5; r2++) {
        var pr = [], amr = [];
        for (var c2 = 0; c2 < 5; c2++) {
          var v00=A1[f2][2*r2][2*c2], v01=A1[f2][2*r2][2*c2+1];
          var v10=A1[f2][2*r2+1][2*c2], v11=A1[f2][2*r2+1][2*c2+1];
          var mv=v00, dr=0, dc=0;
          if (v01>mv){mv=v01;dr=0;dc=1;}
          if (v10>mv){mv=v10;dr=1;dc=0;}
          if (v11>mv){mv=v11;dr=1;dc=1;}
          pr.push(mv); amr.push({dr:dr,dc:dc});
        }
        pf.push(pr); amf.push(amr);
      }
      P1.push(pf); argMax.push(amf);
    }

    var flat = [];
    for (var f3 = 0; f3 < 4; f3++)
      for (var r3 = 0; r3 < 5; r3++)
        for (var c3 = 0; c3 < 5; c3++)
          flat.push(P1[f3][r3][c3]);

    var Z2 = [], A2 = [];
    for (var i = 0; i < 16; i++) {
      var s2 = p.b1[i];
      for (var j = 0; j < 100; j++) s2 += p.W1[i][j] * flat[j];
      Z2.push(s2); A2.push(relu(s2));
    }

    var logits = [];
    for (var o = 0; o < 3; o++) {
      var s3 = p.b2[o];
      for (var j2 = 0; j2 < 16; j2++) s3 += p.W2[o][j2] * A2[j2];
      logits.push(s3);
    }
    var probs = softmax(logits);

    return { Z1:Z1, A1:A1, P1:P1, argMax:argMax, flat:flat, Z2:Z2, A2:A2, logits:logits, probs:probs };
  }

  function backwardPassRede(p, cache, img, oneHot){
    var dLogits = cache.probs.map(function(v,i){ return v - oneHot[i]; });

    var gW2 = [], gb2 = dLogits.slice();
    for (var o = 0; o < 3; o++) {
      var row = [];
      for (var j = 0; j < 16; j++) row.push(dLogits[o] * cache.A2[j]);
      gW2.push(row);
    }
    var dA2 = [];
    for (var j = 0; j < 16; j++) {
      var s = 0;
      for (var o2 = 0; o2 < 3; o2++) s += p.W2[o2][j] * dLogits[o2];
      dA2.push(s);
    }
    var dZ2 = dA2.map(function(v,i){ return v * reluDeriv(cache.Z2[i]); });

    var gW1 = [], gb1 = dZ2.slice();
    for (var i = 0; i < 16; i++) {
      var row1 = [];
      for (var j2 = 0; j2 < 100; j2++) row1.push(dZ2[i] * cache.flat[j2]);
      gW1.push(row1);
    }
    var dFlat = [];
    for (var j3 = 0; j3 < 100; j3++) {
      var s2 = 0;
      for (var i2 = 0; i2 < 16; i2++) s2 += p.W1[i2][j3] * dZ2[i2];
      dFlat.push(s2);
    }

    var dP1 = []; var idx = 0;
    for (var f = 0; f < 4; f++) {
      var pf = [];
      for (var r = 0; r < 5; r++) {
        var pr = [];
        for (var c = 0; c < 5; c++) pr.push(dFlat[idx++]);
        pf.push(pr);
      }
      dP1.push(pf);
    }

    var dA1 = [];
    for (var f2 = 0; f2 < 4; f2++) {
      var af = [];
      for (var r2 = 0; r2 < 10; r2++) af.push(new Array(10).fill(0));
      for (var r3 = 0; r3 < 5; r3++)
        for (var c3 = 0; c3 < 5; c3++) {
          var am = cache.argMax[f2][r3][c3];
          af[2*r3+am.dr][2*c3+am.dc] += dP1[f2][r3][c3];
        }
      dA1.push(af);
    }

    var dZ1 = [];
    for (var f3 = 0; f3 < 4; f3++) {
      var zf = [];
      for (var r4 = 0; r4 < 10; r4++) {
        var row2 = [];
        for (var c4 = 0; c4 < 10; c4++)
          row2.push(dA1[f3][r4][c4] * reluDeriv(cache.Z1[f3][r4][c4]));
        zf.push(row2);
      }
      dZ1.push(zf);
    }

    var gK = [], gbConv = [];
    for (var f4 = 0; f4 < 4; f4++) {
      var gk = [[0,0,0],[0,0,0],[0,0,0]]; var gb = 0;
      for (var r5 = 0; r5 < 10; r5++)
        for (var c5 = 0; c5 < 10; c5++) {
          var d = dZ1[f4][r5][c5]; gb += d;
          for (var kr = 0; kr < 3; kr++)
            for (var kc = 0; kc < 3; kc++)
              gk[kr][kc] += d * img[r5+kr][c5+kc];
        }
      gK.push(gk); gbConv.push(gb);
    }

    var loss = -Math.log(Math.max(cache.probs[oneHot.indexOf(1)], 1e-9));
    return { gK:gK, gbConv:gbConv, gW1:gW1, gb1:gb1, gW2:gW2, gb2:gb2, loss:loss };
  }

  function zeros3(f,r,c){
    var a = [];
    for (var i=0;i<f;i++){ var b=[]; for(var j=0;j<r;j++){ b.push(new Array(c).fill(0)); } a.push(b); }
    return a;
  }

  // Um passo de treinamento em lote (as 3 imagens de uma vez): calcula o
  // forward+backward para cada uma, faz a média dos gradientes e atualiza
  // todos os parâmetros (kernels inclusive) via gradiente descendente.
  function treinarPassoLote(params, lr){
    var gK = zeros3(4,3,3), gbConv = [0,0,0,0];
    var gW1 = [], gb1 = new Array(16).fill(0);
    for (var i=0;i<16;i++) gW1.push(new Array(100).fill(0));
    var gW2 = [], gb2 = [0,0,0];
    for (var o=0;o<3;o++) gW2.push(new Array(16).fill(0));
    var totalLoss = 0;

    ORDEM_CLASSES.forEach(function(id){
      var cache = forwardPassRede(params, IMAGENS[id]);
      var grads = backwardPassRede(params, cache, IMAGENS[id], ONE_HOTS[id]);
      totalLoss += grads.loss;
      for (var f=0;f<4;f++){
        for (var kr=0;kr<3;kr++) for (var kc=0;kc<3;kc++) gK[f][kr][kc] += grads.gK[f][kr][kc];
        gbConv[f] += grads.gbConv[f];
      }
      for (var ii=0;ii<16;ii++){
        for (var jj=0;jj<100;jj++) gW1[ii][jj] += grads.gW1[ii][jj];
        gb1[ii] += grads.gb1[ii];
      }
      for (var oo=0;oo<3;oo++){
        for (var jj2=0;jj2<16;jj2++) gW2[oo][jj2] += grads.gW2[oo][jj2];
        gb2[oo] += grads.gb2[oo];
      }
    });

    var nB = ORDEM_CLASSES.length;
    for (var f2=0;f2<4;f2++){
      for (var kr2=0;kr2<3;kr2++) for (var kc2=0;kc2<3;kc2++) params.K[f2][kr2][kc2] -= lr*gK[f2][kr2][kc2]/nB;
      params.bConv[f2] -= lr*gbConv[f2]/nB;
    }
    for (var i2=0;i2<16;i2++){
      for (var j2=0;j2<100;j2++) params.W1[i2][j2] -= lr*gW1[i2][j2]/nB;
      params.b1[i2] -= lr*gb1[i2]/nB;
    }
    for (var o2=0;o2<3;o2++){
      for (var j3=0;j3<16;j3++) params.W2[o2][j3] -= lr*gW2[o2][j3]/nB;
      params.b2[o2] -= lr*gb2[o2]/nB;
    }
    return totalLoss / nB;
  }

  // Calcula a fração de acertos do lote de 3 imagens com os parâmetros
  // atuais: para cada imagem, roda o forward pass e verifica se a classe
  // de maior probabilidade (argmax do Softmax) coincide com a classe correta.
  function calcularAcuraciaLote(params){
    var acertos = 0;
    ORDEM_CLASSES.forEach(function(id){
      var cache = forwardPassRede(params, IMAGENS[id]);
      var maxProb = Math.max.apply(null, cache.probs);
      var idxPredito = cache.probs.indexOf(maxProb);
      var idxCorreto = ONE_HOTS[id].indexOf(1);
      if (idxPredito === idxCorreto) acertos += 1;
    });
    return acertos / ORDEM_CLASSES.length;
  }

  function construirPipeline(params, imgMatriz, imgId){
    var cache = forwardPassRede(params, imgMatriz);

    var conv1 = cache.A1.map(function(mapa){
      return mapa.map(function(row){ return row.map(saturar); });
    });
    var pool1 = cache.P1.map(function(mapa){
      return mapa.map(function(row){ return row.map(saturar); });
    });
    var flatten = cache.flat.map(saturar);
    var fc = cache.A2.map(saturar);
    var softmaxBars = ORDEM_CLASSES.map(function(id, i){
      return { rotulo: ROTULOS[id], valor: cache.probs[i] };
    });

    return { conv1: conv1, pool1: pool1, flatten: flatten, fc: fc, softmax: softmaxBars, probsCrus: cache.probs };
  }

  function initArch(root){
    if (!root || root.dataset.initArch) return;
    root.dataset.initArch = "1";

    var imgAtualId = "casa";
    var imgMatriz = IMAGENS[imgAtualId];

    var rngInicial = criarRng(42);
    var PARAMS = inicializarParametros(rngInicial);
    var historicoLoss = [];
    var historicoAcuracia = [];
    var passoTreino = 0;
    var autoplayInterval = null;

    var PIPE = construirPipeline(PARAMS, imgMatriz, imgAtualId);

    var btnCasa   = root.querySelector('#cap09arch_btnCasa');
    var btnFeliz  = root.querySelector('#cap09arch_btnFeliz');
    var btnTriste = root.querySelector('#cap09arch_btnTriste');

    var flowEl     = root.querySelector('#cap09arch_flow');
    var descEl     = root.querySelector('#cap09arch_desc');
    var barsWrapEl = root.querySelector('#cap09arch_barsWrap');
    var kernelsPanelEl = root.querySelector('#cap09arch_kernelsPanel');
    var legendaEl  = root.querySelector('#cap09arch_legenda');
    var canvas     = root.querySelector('#cap09arch_canvasVis');
    var ctx        = canvas.getContext('2d');
    var W = canvas.width, H = canvas.height;

    var canvasLoss = root.querySelector('#cap09arch_canvasLoss');
    var ctxLoss = canvasLoss.getContext('2d');
    var canvasAcc = root.querySelector('#cap09arch_canvasAcc');
    var ctxAcc = canvasAcc.getContext('2d');
    var lossTxt = root.querySelector('#cap09arch_lossTxt');
    var accTxt = root.querySelector('#cap09arch_accTxt');
    var stepTxt = root.querySelector('#cap09arch_stepTxt');
    var cicloBackward = root.querySelector('#cap09arch_cicloBackward');

    var btnPassoUnico    = root.querySelector('#cap09arch_btnPassoUnico');
    var btnTreinar       = root.querySelector('#cap09arch_btnTreinar');
    var btnReiniciarPesos = root.querySelector('#cap09arch_btnReiniciarPesos');

    var ETAPAS = [
      {
        id: "entrada", nome: "Entrada", forma: "12×12", tipo: "imagem",
        legenda: "Imagem em escala de cinza 12×12 com margem fixa de zeros.",
        desc: "A <b>imagem de entrada</b> é representada como uma matriz de pixels 12×12. É a mesma estrutura (casa, rosto feliz ou triste) do simulador anterior."
      },
      {
        id: "conv1", nome: "Conv1 + ReLU", forma: "10×10×4", tipo: "mapas", tam: 10, dadosKey: "conv1",
        legenda: "4 mapas de características 10×10, um por filtro aprendido.",
        desc: "A primeira <b>camada convolucional</b> aplica 4 <i>kernels</i> 3×3 <b>aprendidos por treinamento</b> — diferente do simulador anterior, aqui eles começam aleatórios e vão sendo ajustados a cada passo de treinamento (veja os valores abaixo do mapa). Nesta versão simplificada usamos apenas 1 bloco convolucional (N=1); redes reais costumam empilhar vários."
      },
      {
        id: "pool1", nome: "Pooling1", forma: "5×5×4", tipo: "mapas", tam: 5, dadosKey: "pool1",
        legenda: "Os 4 mapas reduzidos para 5×5 via Max-Pooling (2×2, stride 2).",
        desc: "A camada de <b>Max-Pooling</b> reduz a resolução espacial de 10×10 para 5×5, mantendo apenas a ativação máxima de cada janela 2×2 — o que reduz a dimensão dos dados e dá alguma tolerância a pequenos deslocamentos."
      },
      {
        id: "flatten", nome: "Flatten", forma: "100 valores", tipo: "vetor", dadosKey: "flatten",
        legenda: "Vetor linearizado com 4 × 5 × 5 = 100 elementos.",
        desc: "A operação <b>Flatten</b> 'achata' os 4 mapas 2D em um único vetor 1D de 100 valores, preparando a informação para entrar nas camadas densas."
      },
      {
        id: "fc", nome: "Camada Densa (FC)", forma: "16 neurônios", tipo: "vetor", dadosKey: "fc",
        legenda: "16 neurônios (com ReLU) combinando o vetor de características.",
        desc: "A <b>camada totalmente conectada</b> tem 16 neurônios com pesos também aprendidos, combinando todas as características locais extraídas antes. Uma segunda camada densa de saída (16→3, não desenhada separadamente aqui) produz os valores brutos (<i>logits</i>) que alimentam o Softmax."
      },
      {
        id: "softmax", nome: "Softmax", forma: "3 classes", tipo: "softmax",
        legenda: "Distribuição de probabilidade final, calculada a partir dos pesos atuais da rede.",
        desc: "A função <b>Softmax</b> converte os <i>logits</i> em probabilidades que somam 1.0 (100%). Estes valores são <b>calculados de verdade</b> a partir dos pesos atuais — antes de treinar, tendem a ficar próximos de 33%/33%/33%; depois de alguns passos de treinamento, devem convergir para a classe correta."
      }
    ];

    var etapaSelecionada = 0;

    function estiloBloco(el, selecionado){
      el.style.flex = '1';
      el.style.minWidth = '95px';
      el.style.textAlign = 'center';
      el.style.padding = '8px 4px';
      el.style.borderRadius = '8px';
      el.style.cursor = 'pointer';
      el.style.fontSize = '11px';
      el.style.fontWeight = '600';
      if (selecionado){
        el.style.border = '2px solid #2F6F9F';
        el.style.background = '#EAF2FA';
        el.style.color = '#2F6F9F';
      } else {
        el.style.border = '1px solid #E4DCC8';
        el.style.background = '#FFFFFF';
        el.style.color = '#374151';
      }
    }

    function montarFluxo(){
      flowEl.innerHTML = '';
      ETAPAS.forEach(function(etapa, idx){
        var bloco = document.createElement('div');
        bloco.id = 'cap09arch_bloco_' + etapa.id;
        estiloBloco(bloco, false);

        var linha1 = document.createTextNode(etapa.nome);
        var linha2 = document.createElement('span');
        linha2.textContent = etapa.forma;
        linha2.style.display = 'block';
        linha2.style.fontSize = '9px';
        linha2.style.fontWeight = '500';
        linha2.style.color = '#8A8371';
        linha2.style.marginTop = '2px';

        bloco.appendChild(linha1);
        bloco.appendChild(linha2);
        bloco.addEventListener('click', function(){ selecionar(idx); });
        flowEl.appendChild(bloco);

        if (idx < ETAPAS.length - 1){
          var seta = document.createElement('div');
          seta.textContent = '➔';
          seta.style.color = '#C5BC9D';
          seta.style.fontSize = '12px';
          seta.style.padding = '0 2px';
          flowEl.appendChild(seta);
        }
      });
    }

    function desenharImagemEntrada(){
      ctx.clearRect(0,0,W,H);
      var n = 12, tam = 13;
      var offX = Math.round((W - n*tam)/2), offY = Math.round((H - n*tam)/2);
      for (var r=0; r<n; r++){
        for (var c=0; c<n; c++){
          var val = imgMatriz[r][c];
          var g = Math.round(val * 255);
          ctx.fillStyle = "rgb(" + g + "," + g + "," + g + ")";
          ctx.fillRect(offX + c*tam, offY + r*tam, tam, tam);
          ctx.strokeStyle = "#D1D5DB";
          ctx.strokeRect(offX + c*tam, offY + r*tam, tam, tam);
        }
      }
    }

    function desenharMapas(mapasArr, tamEspacial){
      ctx.clearRect(0,0,W,H);
      var count = mapasArr.length;
      var cols = 2, rows = 2;
      var pad = 12;
      var thumb = 65;
      var totalW = cols*thumb + (cols-1)*pad;
      var totalH = rows*thumb + (rows-1)*pad;
      var offX = Math.round((W-totalW)/2), offY = Math.round((H-totalH)/2);
      var px = thumb/tamEspacial;

      var nomesFiltros = ["Filtro 1", "Filtro 2", "Filtro 3", "Filtro 4"];

      for (var f=0; f<count; f++){
        var col = f % cols, row = Math.floor(f/cols);
        var bx = offX + col*(thumb+pad);
        var by = offY + row*(thumb+pad);
        var mapa = mapasArr[f];
        for (var yy=0; yy<tamEspacial; yy++){
          for (var xx=0; xx<tamEspacial; xx++){
            ctx.fillStyle = corMapa(mapa[yy][xx]);
            ctx.fillRect(bx+xx*px, by+yy*px, px+0.5, px+0.5);
          }
        }
        ctx.strokeStyle = '#2F6F9F';
        ctx.lineWidth = 1;
        ctx.strokeRect(bx, by, thumb, thumb);

        ctx.fillStyle = "#5E5A4A";
        ctx.font = "9px Inter, sans-serif";
        ctx.fillText(nomesFiltros[f], bx, by - 3);
      }
    }

    function desenharVetor(vals){
      ctx.clearRect(0,0,W,H);
      var count = vals.length;
      var cols = count > 20 ? 10 : 4;
      var rows = Math.ceil(count/cols);
      var cellW = Math.min(22, (W - 40)/cols);
      var cellH = Math.min(22, (H - 40)/rows);
      var offX = Math.round((W - cols*cellW)/2);
      var offY = Math.round((H - rows*cellH)/2);

      for (var i=0; i<count; i++){
        var c = i % cols, r = Math.floor(i/cols);
        ctx.fillStyle = corMapa(vals[i]);
        ctx.fillRect(offX + c*cellW, offY + r*cellH, cellW-1, cellH-1);
        ctx.strokeStyle = "#E4DCC8";
        ctx.strokeRect(offX + c*cellW, offY + r*cellH, cellW-1, cellH-1);
      }
    }

    function desenharBarras(barras){
      barsWrapEl.innerHTML = '';
      barsWrapEl.style.display = 'flex';
      var alturaMax = 100;
      barras.forEach(function(b){
        var wrap = document.createElement('div');
        wrap.style.display = 'flex';
        wrap.style.flexDirection = 'column';
        wrap.style.alignItems = 'center';
        wrap.style.fontSize = '11px';
        wrap.style.color = '#374151';

        var bar = document.createElement('div');
        bar.style.width = '38px';
        bar.style.height = Math.max(1, Math.round(b.valor * alturaMax)) + 'px';
        bar.style.background = 'linear-gradient(#2F6F9F, #1E8F6F)';
        bar.style.borderRadius = '4px 4px 0 0';

        var legendaBar = document.createElement('div');
        legendaBar.style.marginTop = '6px';
        legendaBar.style.fontWeight = '600';
        legendaBar.textContent = b.rotulo;

        var valBar = document.createElement('div');
        valBar.className = 'cn-mono';
        valBar.style.fontSize = '10px';
        valBar.style.color = '#2F6F9F';
        valBar.textContent = (b.valor * 100).toFixed(1) + '%';

        wrap.appendChild(bar);
        wrap.appendChild(legendaBar);
        wrap.appendChild(valBar);
        barsWrapEl.appendChild(wrap);
      });
    }

    function desenharPainelKernels(){
      var html = '<div class="cn-grouplabel" style="text-align:left;">KERNELS APRENDIDOS (VALORES ATUAIS)</div>' +
        '<div style="display:flex;gap:10px;flex-wrap:wrap;justify-content:center;">';
      for (var f=0; f<4; f++){
        html += '<div style="display:grid;grid-template-columns:repeat(3,30px);gap:2px;">';
        for (var r=0; r<3; r++){
          for (var c=0; c<3; c++){
            var v = PARAMS.K[f][r][c];
            var cor = v > 0 ? "#E6F4EA" : (v < 0 ? "#FCE8E6" : "#F3F4F6");
            var corTxt = v > 0 ? "#1E8F6F" : (v < 0 ? "#C1443A" : "#374151");
            html += '<div class="cn-kcell cn-mono" style="background:' + cor + ';color:' + corTxt + ';">' + v.toFixed(1) + '</div>';
          }
        }
        html += '</div>';
      }
      html += '</div>';
      kernelsPanelEl.innerHTML = html;
    }

    function desenharGraficoLoss(){
      var Wc = canvasLoss.width, Hc = canvasLoss.height;
      ctxLoss.clearRect(0,0,Wc,Hc);
      ctxLoss.strokeStyle = "#E4DCC8"; ctxLoss.lineWidth = 1;
      ctxLoss.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoLoss.length < 2){
        ctxLoss.fillStyle = "#B8AE94";
        ctxLoss.font = "10px Inter, sans-serif";
        ctxLoss.textAlign = "center";
        ctxLoss.fillText("la perdita apparirà qui", Wc/2, Hc/2 + 3);
        return;
      }
      var maxN = 200;
      var dados = historicoLoss.slice(-maxN);
      var maxLoss = Math.max.apply(null, dados);
      maxLoss = Math.max(maxLoss, 0.05);
      var padL = 8, padR = 8, padT = 8, padB = 8;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxLoss.strokeStyle = "#2F5FA8";
      ctxLoss.lineWidth = 1.6;
      ctxLoss.beginPath();
      dados.forEach(function(v, i){
        var x = padL + (dados.length === 1 ? 0 : (i/(dados.length-1)) * plotW);
        var y = padT + (1 - v/maxLoss) * plotH;
        if (i===0) ctxLoss.moveTo(x,y); else ctxLoss.lineTo(x,y);
      });
      ctxLoss.stroke();
    }

    // Mesmo padrão visual do gráfico de perda, mas com eixo Y fixo em [0,1]
    // (a acurácia do lote é sempre 0, 1/3, 2/3 ou 1, já que só há 3 imagens).
    function desenharGraficoAcuracia(){
      var Wc = canvasAcc.width, Hc = canvasAcc.height;
      ctxAcc.clearRect(0,0,Wc,Hc);
      ctxAcc.strokeStyle = "#E4DCC8"; ctxAcc.lineWidth = 1;
      ctxAcc.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoAcuracia.length < 2){
        ctxAcc.fillStyle = "#B8AE94";
        ctxAcc.font = "10px Inter, sans-serif";
        ctxAcc.textAlign = "center";
        ctxAcc.fillText("l'accuratezza apparirà qui", Wc/2, Hc/2 + 3);
        return;
      }
      var maxN = 200;
      var dados = historicoAcuracia.slice(-maxN);
      var padL = 8, padR = 8, padT = 8, padB = 8;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxAcc.strokeStyle = "#1E8F6F";
      ctxAcc.lineWidth = 1.6;
      ctxAcc.beginPath();
      dados.forEach(function(v, i){
        var x = padL + (dados.length === 1 ? 0 : (i/(dados.length-1)) * plotW);
        var y = padT + (1 - v) * plotH;
        if (i===0) ctxAcc.moveTo(x,y); else ctxAcc.lineTo(x,y);
      });
      ctxAcc.stroke();
    }

    function estimarClassePredita(probs){
      var maxV = Math.max.apply(null, probs);
      return ORDEM_CLASSES[probs.indexOf(maxV)];
    }

    function atualizarVisual(etapa){
      kernelsPanelEl.style.display = 'none';
      if (etapa.tipo === 'softmax'){
        canvas.style.display = 'none';
        desenharBarras(PIPE.softmax);
      } else {
        canvas.style.display = 'block';
        barsWrapEl.style.display = 'none';
        if (etapa.tipo === 'imagem') desenharImagemEntrada();
        else if (etapa.tipo === 'mapas') desenharMapas(PIPE[etapa.dadosKey], etapa.tam);
        else if (etapa.tipo === 'vetor') desenharVetor(PIPE[etapa.dadosKey]);

        if (etapa.id === 'conv1'){
          kernelsPanelEl.style.display = 'block';
          desenharPainelKernels();
        }
      }
      legendaEl.textContent = etapa.legenda || '';
    }

    function selecionar(idx){
      etapaSelecionada = idx;
      var etapa = ETAPAS[idx];
      ETAPAS.forEach(function(e){
        var el = root.querySelector('#cap09arch_bloco_' + e.id);
        if (el) estiloBloco(el, false);
      });
      var atual = root.querySelector('#cap09arch_bloco_' + etapa.id);
      if (atual) estiloBloco(atual, true);
      descEl.innerHTML = '<b>' + etapa.nome + '</b> — dimensão: <code class="cn-mono" style="background:#EAF2FA;color:#2F6F9F;padding:2px 6px;border-radius:4px;">' + etapa.forma + '</code><br><br>' + etapa.desc;
      atualizarVisual(etapa);
    }

    function recomputarPipeline(){
      PIPE = construirPipeline(PARAMS, imgMatriz, imgAtualId);
    }

    function trocarImagem(id, btn){
      [btnCasa, btnFeliz, btnTriste].forEach(function(b){ b.classList.remove('active'); });
      btn.classList.add('active');
      imgAtualId = id;
      imgMatriz = IMAGENS[id];
      recomputarPipeline();
      selecionar(etapaSelecionada);
    }

    function pulsarBackward(){
      cicloBackward.classList.add('pulso');
      setTimeout(function(){ cicloBackward.classList.remove('pulso'); }, 350);
    }

    function passoDeTreinamento(){
      var lr = 0.3;
      var loss = treinarPassoLote(PARAMS, lr);
      var acuracia = calcularAcuraciaLote(PARAMS);
      passoTreino += 1;
      historicoLoss.push(loss);
      historicoAcuracia.push(acuracia);
      recomputarPipeline();

      lossTxt.textContent = 'Loss: ' + loss.toFixed(4);
      accTxt.textContent = 'Accuratezza: ' + Math.round(acuracia * 100) + '%';
      stepTxt.textContent = 'Passo di addestramento: ' + passoTreino;
      desenharGraficoLoss();
      desenharGraficoAcuracia();
      pulsarBackward();
      selecionar(etapaSelecionada);
      return { loss: loss, acuracia: acuracia };
    }

    function pararAutoplay(){
      if (autoplayInterval){
        clearInterval(autoplayInterval);
        autoplayInterval = null;
        btnTreinar.textContent = '▶ Treinar';
      }
    }

    btnCasa.addEventListener('click', function(){ trocarImagem('casa', btnCasa); });
    btnFeliz.addEventListener('click', function(){ trocarImagem('feliz', btnFeliz); });
    btnTriste.addEventListener('click', function(){ trocarImagem('triste', btnTriste); });

    btnPassoUnico.addEventListener('click', function(){
      pararAutoplay();
      passoDeTreinamento();
    });

    btnTreinar.addEventListener('click', function(){
      if (autoplayInterval){ pararAutoplay(); return; }
      btnTreinar.textContent = '⏸ Pausar';
      autoplayInterval = setInterval(function(){
        var resultado = passoDeTreinamento();
        if (resultado.loss < 0.002 && resultado.acuracia === 1){ pararAutoplay(); }
      }, 120);
    });

    btnReiniciarPesos.addEventListener('click', function(){
      pararAutoplay();
      PARAMS = inicializarParametros(criarRng(Date.now() % 2147483647));
      historicoLoss = [];
      historicoAcuracia = [];
      passoTreino = 0;
      lossTxt.textContent = 'Loss: —';
      accTxt.textContent = 'Acurácia: —';
      stepTxt.textContent = 'Passo de treinamento: 0';
      recomputarPipeline();
      desenharGraficoLoss();
      desenharGraficoAcuracia();
      selecionar(etapaSelecionada);
    });

    montarFluxo();
    desenharGraficoLoss();
    desenharGraficoAcuracia();
    selecionar(0);
  }

  function tryInitArch(){
    var root = document.getElementById('sim-09-arquitetura');
    if (root) initArch(root); else setTimeout(tryInitArch, 200);
  }
  tryInitArch();
})();
</script>
''')


**Figura 9.11:** Simulatore interattivo dell


<figure id="fig-09-sim-09-arquitetura">
  <img src="imagens/fig-09-sim-09-arquitetura.png" alt=" Simulatore interattivo dell'architettura di una CNN: scegli una delle immagini di ingresso 12×12 (casa, viso felice o triste), clicca su ciascun blocco del *pipeline* — Ingresso, *Conv+ReLU*, *Pooling*, *Flatten*, *FC* e *Softmax* — ed esegui passi di addestramento reali (*forward pass* + retropropagazione) per osservare la perdita e l'accuratezza evolversi, i *kernel* essere regolati e il *Softmax* iniziare a puntare alla classe corretta. " style="max-width:80%" />
  <figcaption><strong>Figura 9.11:</strong>  Simulatore interattivo dell'architettura di una CNN: scegli una delle immagini di ingresso 12×12 (casa, viso felice o triste), clicca su ciascun blocco del *pipeline* — Ingresso, *Conv+ReLU*, *Pooling*, *Flatten*, *FC* e *Softmax* — ed esegui passi di addestramento reali (*forward pass* + retropropagazione) per osservare la perdita e l'accuratezza evolversi, i *kernel* essere regolati e il *Softmax* iniziare a puntare alla classe corretta. </figcaption>
</figure>

### 9.4.8 Come il Gradiente Regola i *Kernel* della Convoluzione

La comprensione del processo di apprendimento in una Rete Neurale Convoluzionale (CNN) richiede la chiarificazione di un meccanismo fondamentale: **come i coefficienti casuali di un filtro iniziale si trasformano in rilevatori precisi di bordi, texture e pattern complessi.**

La risposta risiede nel principio della **condivisione dei pesi (*weight sharing*)**. Durante la fase di propagazione in avanti (*forward pass*), lo stesso filtro di dimensione $3\times3$ scorre su tutta l'estensione dell'immagine di input. Di conseguenza, ogni peso del *kernel* — come l'elemento $K[0][0]$ nell'angolo superiore sinistro — viene riutilizzato più volte lungo le diverse regioni spaziali del dato di input.

Durante la fase di retropropagazione (*backpropagation*), questo riutilizzo stabilisce una dinamica diretta: **ogni posizione spaziale elaborata dal filtro genera un contributo individuale ("voto") per l'aggiornamento del rispettivo peso.**

#### 9.4.8.1 L'Intuizione Dietro il Calcolo

Sia $Z[r][c]$ la mappa delle caratteristiche pre-attivazione nella posizione $(r,c)$ della finestra scorrevole, ottenuta tramite la correlazione incrociata tra il *kernel* $K$ e l'input $X$:

$$
Z[r][c] = \sum_{k_r} \sum_{k_c} K[k_r][k_c] \cdot X[r + k_r][c + k_c]
$$

Applicando la regola della catena per determinare il contributo di un peso specifico $K[k_r][k_c]$ nella funzione di perdita $L$, si ottengono i seguenti passaggi:

1. **Errore Locale ($dZ$):** In ogni posizione $(r,c)$, si calcola la derivata parziale della funzione di perdita rispetto alla pre-attivazione:
   $$dZ[r][c] = \frac{\partial L}{\partial Z[r][c]}$$
   che quantifica la responsabilità di quella specifica posizione nell'errore totale della rete ($L$).

2. **Contributo del Peso:** Poiché $\frac{\partial Z[r][c]}{\partial K[k_r][k_c]} = X[r + k_r][c + k_c]$, l'influenza di un peso specifico $K[k_r][k_c]$ sull'errore della posizione $(r,c)$ si ottiene moltiplicando l'errore locale $dZ[r][c]$ per il valore del pixel di input allineato a quel peso nel momento del calcolo:
   $$dZ[r][c] \cdot X[r + k_r][c + k_c]$$

3. **Accumulo dei Gradienti:** Il gradiente finale del peso corrisponde alla **somma dei contributi ("voti") di tutte le posizioni** percorse dalla finestra scorrevole:

$$
\frac{\partial L}{\partial K[k_r][k_c]} = \sum_{(r,c)} dZ[r][c] \cdot X[r + k_r][c + k_c]
$$

Questa formulazione assicura una parità diretta tra la derivazione analitica e i valori calcolati nel simulatore di ispezione del gradiente ([Figura 9.12](#fig-09-sim-09-gradiente-kernel)).

#### 9.4.8.2 Il Ruolo della Funzione ReLU come "Filtro di Rilevanza"

L'applicazione della funzione di attivazione **ReLU** ($\max(0, z)$) immediatamente dopo la convoluzione introduce una proprietà di selettività al gradiente:

* **Attivazione Positiva ($Z[r][c] > 0$):** La derivata della ReLU è $1$. L'errore locale viene propagato integralmente ($dZ \neq 0$), consentendo alla posizione di contribuire all'aggiornamento dei pesi del *kernel*.
* **Attivazione Inattiva ($Z[r][c] \le 0$):** La derivata della ReLU è $0$. L'errore locale viene annullato ($dZ = 0$), sopprimendo il contributo della posizione al gradiente finale.

> **Nota didattica:** La ReLU garantisce che solo le regioni spaziali che hanno prodotto risposte attive durante la propagazione in avanti abbiano la capacità di modificare i pesi del *kernel* nel processo di retropropagazione.

#### 9.4.8.3 Aggiornamento dei Pesi tramite Discesa del Gradiente

Dopo il consolidamento dei gradienti accumulati da tutte le posizioni, l'aggiornamento del peso avviene secondo l'algoritmo della Discesa del Gradiente Stocastica (SGD):

$$
K[k_r][k_c] \leftarrow K[k_r][k_c] - \eta \cdot \frac{\partial L}{\partial K[k_r][k_c]}
$$

dove $\eta$ denota il **tasso di apprendimento (*learning rate*)**. 

* Se la somma dei gradienti è **positiva**, il valore del peso viene ridotto.
* Se la somma è **negativa**, il valore del peso viene incrementato.

#### 9.4.8.4 Esplorando il Simulatore Interattivo

> ### 📝 🔗 Dall'Architettura Globale all'Ispezione del Gradiente
>
> Nel simulatore di architettura ([Figura 9.11](#fig-09-sim-09-arquitetura)), si osserva l'errore $dZ$ derivato dalla completa retropropagazione multistrato, originato dalla perdita di entropia incrociata (*Softmax*) sulle immagini di input $12\times12$.
>
> Per rendere possibile la verifica analitica del gradiente senza il sovraccarico di $100$ posizioni di convoluzione e retropropagazione multistrato, il simulatore di apprendimento del *kernel* ([Figura 9.12](#fig-09-sim-09-gradiente-kernel)) adotta un modello di ispezione ridotto ($6\times6$). In questo scenario, si semplifica il problema sostituendo la classificazione complessa con un **meta-dato di calibrazione scalare**: si regola il filtro per produrre una risposta accumulata predefinita ($\text{alvo} = 9$) quando identifica un pattern specifico (come un bordo a 45 gradi). Il meccanismo di accumulo dei gradienti ($dZ \cdot X$) rimane rigorosamente identico in entrambe le formulazioni.

Per ispezionare questa dinamica a livello numerico, si utilizza il simulatore nella [Figura 9.12](#fig-09-sim-09-gradiente-kernel):

* **Immagine di Input ($X$):** Matrice $6\times6$.
* **Filtro Convoluzionale ($K$):** Matrice $3\times3$ (9 pesi).
* **Mappa di Output ($Z$ / $A$):** Matrice $4\times4$ (16 posizioni della finestra).
* **Funzione di Perdita ($L$):** Definita da $L = \frac{1}{2}(S - \text{alvo})^2$, dove $S = \sum A[r][c]$ rappresenta la somma globale delle attivazioni post-ReLU.

> **Il ruolo del $\text{alvo} = 9$:** Il valore scalare $\text{alvo} = 9$ rappresenta l'"energia di attivazione" ideale stabilita per l'immagine con bordo diagonale. Poiché la mappa $A$ ha 16 posizioni, questo valore equivale a cercare una risposta media di $\frac{9}{16} \approx 0,56$ per pixel attivato. Quando $S > 9$, la rete identifica che il filtro sta reagendo con intensità eccessiva al pattern, generando un errore $dZ > 0$ che forza la riduzione dei pesi $K$. Quando $S < 9$, i pesi vengono incrementati per amplificare il segnale.

##### Percorso Suggerito per la Sperimentazione:

1. **Selezione del Peso:** Nella griglia $3\times3$, scegli il peso da analizzare (es.: $K[0][0]$).
2. **Scansione della Finestra:** Utilizza il pulsante **"▶ Avanza posizione"** per seguire lo spostamento della finestra attraverso le 16 posizioni spaziali. Nota l'evidenziazione visiva nella cella della mappa di input che allinea il pixel $X$ al peso selezionato.
3. **Analisi del Voto Locale:** Esamina il prodotto dell'errore locale per il pixel di input ($dZ \cdot X$) nel pannello di calcolo della posizione.
4. **Verifica della Cronologia:** Segui il consolidamento dei 16 risultati parziali organizzati nelle quattro colonne della cronologia, osservando l'accumulo del gradiente finale.
5. **Aggiornamento del Kernel:** Clicca su **"▶ Applica passo di discesa del gradiente"** per visualizzare la convergenza della curva di perdita e l'adattamento del *kernel* casuale al pattern di input selezionato.

In [11]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-09-gradiente-kernel" style="background-color:#FBF7EE;border-radius:18px;border:1px solid #E4DCC8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;700&display=swap');
    #sim-09-gradiente-kernel .cap09kgrad_mono { font-family:'JetBrains Mono', ui-monospace, monospace; }
    #sim-09-gradiente-kernel .cap09kgrad_grouplabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:3px; letter-spacing:.3px; }
    #sim-09-gradiente-kernel .cap09kgrad_modebtn {
      flex:1; text-align:center; padding:6px 8px; font-size:11.5px; font-weight:600;
      border:none; background:transparent; color:#8A8371; cursor:pointer; border-radius:9px;
      transition:background .15s ease, color .15s ease; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_modebtn.cap09kgrad_active { background:#26241D; color:#FBF7EE; }
    #sim-09-gradiente-kernel .cap09kgrad_lrbtn {
      padding:3px 8px; font-size:10px; font-weight:600; border:1px solid #E4DCC8; background:#FFFFFF;
      color:#5E5A4A; border-radius:6px; cursor:pointer; transition:all .15s ease;
    }
    #sim-09-gradiente-kernel .cap09kgrad_lrbtn.cap09kgrad_active { background:#2F6F9F; color:#FFFFFF; border-color:#2F6F9F; }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn {
      padding:0 10px; height:26px; border-radius:8px; border:1px solid #E4DCC8; background:#FAFAF7;
      color:#26241D; font-size:10.5px; font-weight:600; cursor:pointer; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn:hover { background:#F1EAD7; }
    #sim-09-gradiente-kernel .cap09kgrad_navbtn:disabled { opacity:0.4; cursor:not-allowed; }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn {
      padding:0 12px; height:26px; border-radius:8px; border:1px solid #2F6F9F; background:#EAF2FA;
      color:#2F6F9F; font-size:10.5px; font-weight:700; cursor:pointer; white-space:nowrap;
    }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn:hover { background:#DCEEFB; }
    #sim-09-gradiente-kernel .cap09kgrad_playbtn:disabled { opacity:0.4; cursor:not-allowed; }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn {
      height:24px; border-radius:5px; border:1px solid #E4DCC8; background:#FFFFFF;
      color:#374151; cursor:pointer; font-weight:600;
    }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn:hover { background:#F1EAD7; }
    #sim-09-gradiente-kernel .cap09kgrad_pesobtn_ativo { background:#26241D !important; color:#FBF7EE !important; border-color:#26241D !important; }
    #sim-09-gradiente-kernel .cap09kgrad_linhavoto { font-size:8.5px; padding:2px 3px; border-bottom:1px dashed #EDE7D6; border-radius:4px; transition:background .15s ease; white-space:nowrap; text-align:center; }
    #sim-09-gradiente-kernel .cap09kgrad_graflabel { font-size:9.5px; font-weight:700; color:#8A8371; margin-bottom:4px; text-align:center; letter-spacing:.3px; }
    #sim-09-gradiente-kernel .cap09kgrad_painel { background:#FAFAF7;border:1px solid #E9E3D3;border-radius:12px;padding:12px; }
    #sim-09-gradiente-kernel .cap09kgrad_cartao {
      flex:1; min-width:140px; background:#FFFFFF; border:1px solid #E4DCC8; border-radius:10px;
      padding:10px 12px; box-sizing:border-box;
    }
    #sim-09-gradiente-kernel .cap09kgrad_barraProgresso {
      width:100%; height:6px; background:#EDE7D6; border-radius:4px; overflow:hidden; margin-top:6px;
    }
    #sim-09-gradiente-kernel .cap09kgrad_barraProgressoFill {
      height:100%; background:#2F6F9F; border-radius:4px; transition:width .2s ease;
    }

    #sim-09-gradiente-kernel .cap09kgrad_labelinfo {
      position:relative; display:inline-flex; align-items:center; gap:4px; cursor:help;
    }
    #sim-09-gradiente-kernel .cap09kgrad_icone {
      width:12px; height:12px; min-width:12px; border-radius:50%; background:#E4DCC8; color:#5E5A4A;
      font-size:8px; font-weight:700; display:inline-flex; align-items:center; justify-content:center;
      font-family:'Inter',sans-serif;
    }
    #sim-09-gradiente-kernel .cap09kgrad_labelinfo:hover .cap09kgrad_icone { background:#2F6F9F; color:#FBF7EE; }
    #sim-09-gradiente-kernel .cap09kgrad_tooltip {
      visibility:hidden; opacity:0; position:absolute; top:calc(100% + 6px); left:0;
      width:240px; max-width:60vw; background:#26241D; color:#FBF7EE; font-size:10px; font-weight:400;
      line-height:1.55; padding:9px 11px; border-radius:8px; z-index:999; letter-spacing:0;
      box-shadow:0 6px 16px rgba(0,0,0,.2); transition:opacity .15s ease, visibility .15s ease;
      pointer-events:none; text-align:left;
    }
    #sim-09-gradiente-kernel .cap09kgrad_graflabel .cap09kgrad_tooltip { left:50%; transform:translateX(-50%); }
    #sim-09-gradiente-kernel .cap09kgrad_labelinfo:hover .cap09kgrad_tooltip { visibility:visible; opacity:1; }

    #sim-09-gradiente-kernel .cap09kgrad_grid_historico {
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 4px 6px;
      max-height: 140px;
      overflow-y: auto;
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 8px;
      padding: 6px;
      margin-top: 4px;
    }

    #sim-09-gradiente-kernel .cap09kgrad_eq_box {
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 6px;
      padding: 5px 8px;
      margin: 4px 0;
      display: block;
      text-align: left;
      font-family: 'JetBrains Mono', monospace;
      font-size: 10px;
      color: #26241D;
    }

    #sim-09-gradiente-kernel .cap09kgrad_matriz_details {
      margin-top: 8px;
      max-width: 175px;
      font-size: 10px;
      color: #5E5A4A;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_summary {
      font-weight: 600;
      font-size: 9.5px;
      color: #2F6F9F;
      cursor: pointer;
      user-select: none;
      outline: none;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_summary:hover {
      text-decoration: underline;
    }
    #sim-09-gradiente-kernel .cap09kgrad_matriz_expl {
      margin-top: 4px;
      padding: 6px;
      background: #FFFFFF;
      border: 1px solid #E4DCC8;
      border-radius: 6px;
      line-height: 1.4;
    }

    /* ---- Hover das células (contas detalhadas + destaque entre camadas) ---- */
    #sim-09-gradiente-kernel .cap09kgrad_cell_hoverable { cursor: help; }
    #sim-09-gradiente-kernel .cap09kgrad_hlPrincipal {
      outline: 2px solid #2F6F9F !important;
      outline-offset: -1px;
      box-shadow: 0 0 0 3px rgba(47,111,159,0.15) inset;
      position: relative;
      z-index: 2;
    }
    #sim-09-gradiente-kernel .cap09kgrad_hlSecundario {
      outline: 2px dashed #2F6F9F !important;
      outline-offset: -2px;
      position: relative;
      z-index: 1;
    }
    .cap09kgrad_hovertip {
      position: fixed;
      background: #26241D;
      color: #FBF7EE;
      font-family: 'JetBrains Mono', ui-monospace, monospace;
      font-size: 10px;
      line-height: 1.65;
      padding: 9px 11px;
      border-radius: 8px;
      box-shadow: 0 6px 18px rgba(0,0,0,.25);
      z-index: 99999;
      pointer-events: none;
      max-width: 260px;
      white-space: normal;
      display: none;
    }
    .cap09kgrad_hovertip b { color: #7EE7C6; }
  </style>

  <!-- Cabeçalho -->
  <div style="background:#F1EAD7;padding:8px 14px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #E4DCC8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241D;">🧮 Simulatore: Gradiente di un Peso del Kernel</span>
    <span class="cap09kgrad_mono" style="background:#26241D;color:#7EE7C6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;">&part;Perdita / &part;K[kr][kc] = &Sigma; dZ &middot; X</span>
  </div>

  <div style="padding:16px;background:#FFFFFF;overflow:auto">

    <!-- Seletor de padrão e Taxa de Aprendizado -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;margin-bottom:12px;">
      <div style="max-width:320px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">MODELLO DI INPUT (IMMAGINE 6&times;6)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Sceglie quale immagine 6&times;6 alimenta la convoluzione.</span>
        </div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09kgrad_btnDiagonal" class="cap09kgrad_modebtn cap09kgrad_active">↘ Bordo diagonale</button>
          <button id="cap09kgrad_btnVertical" class="cap09kgrad_modebtn">▍ Bordo verticale</button>
        </div>
      </div>

      <div>
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">TASSO DI APPRENDIMENTO (&eta;)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Tasso di apprendimento. Regola per vedere la differenza tra convergenza fluida (0.002) e collasso per overshooting (0.02).</span>
        </div>
        <div style="display:flex;gap:4px;background:#F1EAD7;border:1px solid #E4DCC8;border-radius:11px;padding:3px;">
          <button id="cap09kgrad_lr0005" class="cap09kgrad_lrbtn">0.0005 (Lento)</button>
          <button id="cap09kgrad_lr002" class="cap09kgrad_lrbtn cap09kgrad_active">0.002 (Ideale)</button>
          <button id="cap09kgrad_lr02" class="cap09kgrad_lrbtn">0.02 (Alto)</button>
        </div>
      </div>
    </div>

    <div style="font-size:11.5px;line-height:1.6;color:#5E5A4A;background:#FAFAF7;border:1px solid #E9E3D3;border-radius:10px;padding:10px 14px;margin-bottom:14px;">
      Esempio <b>ridotto</b>: immagine 6&times;6 e filtro 3&times;3 che generano mappe 4&times;4. Fai clic sulle schede <b>"🔍 Come viene calcolato?"</b> sotto ogni matrice per capire i calcoli passo dopo passo. <b>Passa il mouse su qualsiasi cella</b> di X, Z, A, dZ o K per vedere il calcolo esatto di quel valore, con gli elementi usati nei livelli correlati evidenziati con contorno tratteggiato/blu.
    </div>

    <div style="display:flex;gap:10px;flex-wrap:wrap;align-items:flex-start;">

      <!-- PAINEL 1: Seleção do Peso e Kernel K -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;min-width:160px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">1. PESO DEL KERNEL<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Seleziona quale peso del kernel vuoi analizzare singolarmente.</span>
        </div>
        <div id="cap09kgrad_seletorPeso"></div>
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo" style="margin-top:10px;">KERNEL ATTUALE (K)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Valori del filtro 3&times;3. Il peso selezionato è evidenziato in blu. Passa il mouse su un peso per vedere dove viene usato.</span>
        </div>
        <div id="cap09kgrad_gradeK"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Come viene aggiornato?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Regola del gradiente:</b><br>
            <div class="cap09kgrad_eq_box">K &larr; K &minus; &eta; &middot; &nabla;K</div>
            • <b>&eta;</b> = tasso di apprendimento.<br>
            • <b>&nabla;K</b> = somma dei 16 voti dZ &times; X.
          </div>
        </details>
      </div>

      <!-- PAINEL 2: Entrada X -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">INPUT X (6&times;6)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Immagine 6&times;6. Pixel blu = sovrapposizione con il peso K selezionato nella finestra corrente. Passa il mouse su un pixel per vedere in quali posizioni di Z viene usato.</span>
        </div>
        <div id="cap09kgrad_gradeX"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Come funziona X?</summary>
          <div class="cap09kgrad_matriz_expl">
            Matrice di input. Nella posizione (r,c), il peso K moltiplica il pixel:
            <div class="cap09kgrad_eq_box">X[r + k<sub>r</sub>][c + k<sub>c</sub>]</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 3: Saída Z (Pré-ativação) -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">PRE-ATTIVAZIONE Z (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Risultato della convoluzione prima del ReLU: Z = &Sigma; K &middot; X. Passa il mouse su una cella per vedere i 9 termini della somma, evidenziando la finestra in X e tutto il kernel K.</span>
        </div>
        <div id="cap09kgrad_gradeZ"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Come calcola Z?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Correlazione incrociata:</b><br>
            Moltiplicazione punto per punto del filtro 3&times;3 su X:
            <div class="cap09kgrad_eq_box">Z[r][c] = &Sigma; K &middot; X</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 4: Ativação A (Pós-ReLU) -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ATTIVAZIONE A (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Risultato post-ReLU: A = max(0, Z). Se Z &le; 0, l'attivazione viene azzerata. Passa il mouse su una cella per evidenziare il Z corrispondente.</span>
        </div>
        <div id="cap09kgrad_gradeA"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Come calcola A?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>Funzione ReLU:</b><br>
            <div class="cap09kgrad_eq_box">A[r][c] = max(0, Z[r][c])</div>
            <b>Somma globale (S):</b><br>
            <div class="cap09kgrad_eq_box">S = &Sigma; A[r][c]</div>
          </div>
        </details>
      </div>

      <!-- PAINEL 5: Erro dZ -->
      <div class="cap09kgrad_painel" style="flex:0 0 auto;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">ERRORE dZ (4&times;4)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Errore propagato: dZ = (S - obiettivo) &middot; I(Z > 0). Dove A=0, anche l'errore dZ è 0. Passa il mouse su una cella per vedere il calcolo completo, evidenziando il Z corrispondente e tutte le 16 celle di A che formano S.</span>
        </div>
        <div id="cap09kgrad_gradeDZ"></div>
        <div id="cap09kgrad_textoS" class="cap09kgrad_mono" style="font-size:9.5px;color:#374151;margin-top:6px;"></div>
        <div id="cap09kgrad_textoLoss" class="cap09kgrad_mono" style="font-size:9.5px;color:#374151;margin-top:2px;"></div>

        <details class="cap09kgrad_matriz_details">
          <summary class="cap09kgrad_matriz_summary">🔍 Come calcola dZ, S e Loss?</summary>
          <div class="cap09kgrad_matriz_expl">
            <b>1. Perdita (Loss L):</b><br>
            <div class="cap09kgrad_eq_box">L = &frac12; (S &minus; obiettivo)&sup2;</div>
            <b>2. Errore propagato dZ:</b><br>
            <div class="cap09kgrad_eq_box">dZ = (S &minus; obiettivo) &middot; deriv_ReLU(Z)</div>
          </div>
        </details>
      </div>

    </div>

    <!-- PAINEL SECUNDÁRIO: Cálculo dos Votos e Gradiente -->
    <div style="display:flex;gap:10px;flex-wrap:wrap;margin-top:12px;">
      <div class="cap09kgrad_painel" style="flex:1;min-width:340px;">
        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">2. CALCOLO E SOMMA DEI "VOTI" DI OGNI POSIZIONE<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Ogni posizione (r,c) genera un voto = dZ[r][c] &times; X[r+kr][c+kc]. La somma di tutti i 16 voti forma il gradiente del peso.</span>
        </div>
        <div id="cap09kgrad_formula" style="font-size:11.5px;color:#26241D;margin-bottom:8px;line-height:1.5;"></div>
        
        <div style="display:flex;gap:6px;flex-wrap:wrap;margin-bottom:10px;">
          <button id="cap09kgrad_btnVoltarPos" class="cap09kgrad_navbtn">⏮ Torna alla posizione</button>
          <button id="cap09kgrad_btnAvancar" class="cap09kgrad_playbtn">▶ Avanza posizione</button>
          <button id="cap09kgrad_btnSomarTudo" class="cap09kgrad_navbtn">Somma tutto</button>
          <button id="cap09kgrad_btnReiniciarPos" class="cap09kgrad_navbtn">↺ Reimposta posizioni</button>
        </div>

        <div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:10px;">
          <div class="cap09kgrad_cartao">
            <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">CALCOLO DI QUESTA POSIZIONE<span class="cap09kgrad_icone">?</span>
              <span class="cap09kgrad_tooltip">Mostra l'errore locale (dZ) e il pixel di input (X) moltiplicati nella posizione corrente della finestra scorrevole.</span>
            </div>
            <div id="cap09kgrad_calcAtual" class="cap09kgrad_mono" style="font-size:11px;color:#374151;line-height:1.6;"></div>
          </div>
          
          <div class="cap09kgrad_cartao">
            <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">SOMMA ACCUMULATA (GRADIENTE)<span class="cap09kgrad_icone">?</span>
              <span class="cap09kgrad_tooltip">Il valore accumulato dei prodotti dZ &times; X di tutte le posizioni già percorse. Quando raggiunge 16/16, questo è il gradiente finale del peso.</span>
            </div>
            <div id="cap09kgrad_somaAtual" class="cap09kgrad_mono" style="font-size:11px;color:#374151;line-height:1.6;"></div>
            <div class="cap09kgrad_barraProgresso"><div id="cap09kgrad_barraFill" class="cap09kgrad_barraProgressoFill" style="width:0%;"></div></div>
          </div>
        </div>

        <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo" style="margin-top:2px;">STORICO DELLE 16 POSIZIONI (COLONNE c=0, c=1, c=2, c=3)<span class="cap09kgrad_icone">?</span>
          <span class="cap09kgrad_tooltip">Segui l'elenco di tutte le 16 posizioni organizzate in 4 colonne per corrispondere al movimento della finestra sull'immagine di output.</span>
        </div>
        <div id="cap09kgrad_listaVotos" class="cap09kgrad_grid_historico"></div>
      </div>
    </div>

    <!-- PAINEL TERCIÁRIO: Atualização e Gráfico -->
    <div class="cap09kgrad_painel" style="margin-top:12px;">
      <div class="cap09kgrad_grouplabel cap09kgrad_labelinfo">3. USA IL GRADIENTE PER AGGIORNARE IL KERNEL<span class="cap09kgrad_icone">?</span>
        <span class="cap09kgrad_tooltip">Applica la regola della Discesa del Gradiente (K &larr; K &minus; &eta; &middot; gradiente) per tutti i 9 pesi.</span>
      </div>
      <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;">
        <div style="display:flex;gap:6px;flex-wrap:wrap;">
          <button id="cap09kgrad_btnAtualizarPesos" class="cap09kgrad_playbtn">▶ Applica passo di discesa del gradiente</button>
          <button id="cap09kgrad_btnDesfazerPasso" class="cap09kgrad_navbtn">⏮ Annulla passo</button>
          <button id="cap09kgrad_btnNovoKernel" class="cap09kgrad_navbtn">🎲 Nuovo kernel casuale</button>
        </div>
        <div>
          <div class="cap09kgrad_graflabel cap09kgrad_labelinfo">PERDITA NEL CORSO DEGLI AGGIORNAMENTI<span class="cap09kgrad_icone">?</span>
            <span class="cap09kgrad_tooltip">
              <b>Evoluzione dell'errore L = ½(S &minus; obiettivo)²:</b><br>
              • <b>Obiettivo:</b> L &rarr; 0 (S &rarr; obiettivo).<br>
              • <b>Se si blocca su L = 40.5:</b> Si è verificato "overshooting" (salto eccessivo). I pesi sono diventati molto negativi, generando Z &le; 0 (morte della ReLU). Con S = 0, la perdita si blocca su ½(0 &minus; 9)&sup2; = 40.5.
            </span>
          </div>
          <canvas id="cap09kgrad_canvasLoss" width="220" height="75" style="background:#fff;border:1px solid #E4DCC8;border-radius:8px;"></canvas>
        </div>
        <div id="cap09kgrad_textoEpoca" class="cap09kgrad_mono" style="font-size:10.5px;color:#8A8371;"></div>
      </div>

      <div id="cap09kgrad_notaAtualizacao" style="font-size:10.5px;color:#374151;margin-top:8px;padding-top:6px;border-top:1px dashed #E9E3D3;line-height:1.6;text-align:left;"></div>
    </div>

  </div>
</div>

<script>
(function cap09kgrad_scope(){
  function cap09kgrad_clamp01(v){ return Math.max(0, Math.min(1, v)); }
  function cap09kgrad_relu(x){ return Math.max(0, x); }
  function cap09kgrad_reluDeriv(x){ return x > 0 ? 1 : 0; }

  function cap09kgrad_criarRng(seed){
    return function(){
      seed |= 0; seed = (seed + 0x6D2B79F5) | 0;
      var t = Math.imul(seed ^ (seed >>> 15), 1 | seed);
      t = (t + Math.imul(t ^ (t >>> 7), 61 | t)) ^ t;
      return ((t ^ (t >>> 14)) >>> 0) / 4294967296;
    };
  }

  var cap09kgrad_PADROES = {
    diagonal: [
      [1,0,0,0,0,0],[1,1,0,0,0,0],[1,1,1,0,0,0],
      [1,1,1,1,0,0],[1,1,1,1,1,0],[1,1,1,1,1,1]
    ],
    vertical: [
      [1,1,1,0,0,0],[1,1,1,0,0,0],[1,1,1,0,0,0],
      [1,1,1,0,0,0],[1,1,1,0,0,0],[1,1,1,0,0,0]
    ]
  };
  var cap09kgrad_ALVOS = { diagonal: 9, vertical: 8 };

  function cap09kgrad_forward(K, X){
    var Z = [], A = [];
    for (var r = 0; r < 4; r++){
      var zr = [], ar = [];
      for (var c = 0; c < 4; c++){
        var s = 0;
        for (var kr = 0; kr < 3; kr++)
          for (var kc = 0; kc < 3; kc++)
            s += X[r+kr][c+kc] * K[kr][kc];
        zr.push(s); ar.push(cap09kgrad_relu(s));
      }
      Z.push(zr); A.push(ar);
    }
    var S = 0;
    for (var r2 = 0; r2 < 4; r2++)
      for (var c2 = 0; c2 < 4; c2++) S += A[r2][c2];
    return { Z: Z, A: A, S: S };
  }

  function cap09kgrad_inicializarKernelVivo(startSeed, X){
    var seedAtual = startSeed;
    var K, cache;
    for (var tentativas = 0; tentativas < 500; tentativas++){
      var rng = cap09kgrad_criarRng(seedAtual);
      K = [];
      for (var r = 0; r < 3; r++){
        var row = [];
        for (var c = 0; c < 3; c++) row.push(Number(((rng() - 0.5)).toFixed(2)));
        K.push(row);
      }
      cache = cap09kgrad_forward(K, X);
      if (cache.S > 0) return K;
      seedAtual++;
    }
    return K;
  }

  function cap09kgrad_calcularErro(cache, alvo){
    var dS = cache.S - alvo;
    var dZ = [];
    for (var r = 0; r < 4; r++){
      var row = [];
      for (var c = 0; c < 4; c++) row.push(dS * cap09kgrad_reluDeriv(cache.Z[r][c]));
      dZ.push(row);
    }
    var loss = 0.5 * dS * dS;
    return { dZ: dZ, loss: loss, dS: dS };
  }

  function cap09kgrad_gradientePeso(dZ, X, kr0, kc0){
    var votos = [];
    var soma = 0;
    for (var r = 0; r < 4; r++){
      for (var c = 0; c < 4; c++){
        var xVal = X[r+kr0][c+kc0];
        var dVal = dZ[r][c];
        var voto = dVal * xVal;
        soma += voto;
        votos.push({ r: r, c: c, x: xVal, dz: dVal, voto: voto, somaParcial: soma });
      }
    }
    return { votos: votos, gradiente: soma };
  }

  function cap09kgrad_gradienteKernelCompleto(dZ, X){
    var gK = [[0,0,0],[0,0,0],[0,0,0]];
    for (var kr = 0; kr < 3; kr++)
      for (var kc = 0; kc < 3; kc++)
        gK[kr][kc] = cap09kgrad_gradientePeso(dZ, X, kr, kc).gradiente;
    return gK;
  }

  function cap09kgrad_atualizarKernel(K, gK, lr){
    var novo = [[0,0,0],[0,0,0],[0,0,0]];
    for (var kr = 0; kr < 3; kr++)
      for (var kc = 0; kc < 3; kc++)
        novo[kr][kc] = K[kr][kc] - lr * gK[kr][kc];
    return novo;
  }

  function cap09kgrad_corValor(v, maxAbs){
    var m = maxAbs || 1;
    var t = cap09kgrad_clamp01(Math.abs(v) / m);
    if (v >= 0){
      var g = Math.round(230 - t*90);
      return "rgb(" + Math.round(235-t*120) + "," + g + "," + Math.round(220-t*90) + ")";
    } else {
      var r2 = Math.round(252 - t*20);
      return "rgb(" + r2 + "," + Math.round(232-t*130) + "," + Math.round(230-t*130) + ")";
    }
  }

  function cap09kgrad_initSim(root){
    if (!root || root.dataset.cap09kgradInit) return;
    root.dataset.cap09kgradInit = "1";

    var padraoAtual = "diagonal";
    var X = cap09kgrad_PADROES[padraoAtual];
    var alvo = cap09kgrad_ALVOS[padraoAtual];
    var lr = 0.002;

    var K = cap09kgrad_inicializarKernelVivo(7, X);
    var historicoKernel = [];

    var cacheForward = cap09kgrad_forward(K, X);
    var cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);

    var pesoSelKr = 0, pesoSelKc = 0;
    var posicaoIdx = 0;
    var votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
    var historicoLoss = [cacheErro.loss];
    var epocaAtual = 0;
    var ultimoGradiente = null;
    var ultimoKAntigo = null;

    // ---- Estado do sistema de hover (contas + destaque entre camadas) ----
    var celulasDestacadas = [];
    var elHoverTip = null;

    var elBtnDiagonal = root.querySelector('#cap09kgrad_btnDiagonal');
    var elBtnVertical = root.querySelector('#cap09kgrad_btnVertical');
    var elLr0005 = root.querySelector('#cap09kgrad_lr0005');
    var elLr002 = root.querySelector('#cap09kgrad_lr002');
    var elLr02 = root.querySelector('#cap09kgrad_lr02');
    var elGradeX = root.querySelector('#cap09kgrad_gradeX');
    var elGradeZ = root.querySelector('#cap09kgrad_gradeZ');
    var elGradeA = root.querySelector('#cap09kgrad_gradeA');
    var elGradeDZ = root.querySelector('#cap09kgrad_gradeDZ');
    var elGradeK = root.querySelector('#cap09kgrad_gradeK');
    var elSeletorPeso = root.querySelector('#cap09kgrad_seletorPeso');
    var elFormula = root.querySelector('#cap09kgrad_formula');
    var elListaVotos = root.querySelector('#cap09kgrad_listaVotos');
    var elCalcAtual = root.querySelector('#cap09kgrad_calcAtual');
    var elSomaAtual = root.querySelector('#cap09kgrad_somaAtual');
    var elBarraFill = root.querySelector('#cap09kgrad_barraFill');
    var elBtnVoltarPos = root.querySelector('#cap09kgrad_btnVoltarPos');
    var elBtnAvancar = root.querySelector('#cap09kgrad_btnAvancar');
    var elBtnSomarTudo = root.querySelector('#cap09kgrad_btnSomarTudo');
    var elBtnReiniciarPos = root.querySelector('#cap09kgrad_btnReiniciarPos');
    var elBtnAtualizarPesos = root.querySelector('#cap09kgrad_btnAtualizarPesos');
    var elBtnDesfazerPasso = root.querySelector('#cap09kgrad_btnDesfazerPasso');
    var elBtnNovoKernel = root.querySelector('#cap09kgrad_btnNovoKernel');
    var elTextoS = root.querySelector('#cap09kgrad_textoS');
    var elTextoLoss = root.querySelector('#cap09kgrad_textoLoss');
    var elTextoEpoca = root.querySelector('#cap09kgrad_textoEpoca');
    var elCanvasLoss = root.querySelector('#cap09kgrad_canvasLoss');
    var ctxLoss = elCanvasLoss.getContext('2d');
    var elNotaAtualizacao = root.querySelector('#cap09kgrad_notaAtualizacao');

    function montarSeletorPeso(){
      elSeletorPeso.innerHTML = '';
      elSeletorPeso.style.display = 'grid';
      elSeletorPeso.style.gridTemplateColumns = 'repeat(3, 30px)';
      elSeletorPeso.style.gap = '3px';
      for (var kr = 0; kr < 3; kr++){
        for (var kc = 0; kc < 3; kc++){
          (function(kr2, kc2){
            var bt = document.createElement('button');
            bt.className = 'cap09kgrad_pesobtn';
            bt.textContent = 'K[' + kr2 + '][' + kc2 + ']';
            bt.style.fontSize = '7.5px';
            bt.addEventListener('click', function(){ selecionarPeso(kr2, kc2); });
            elSeletorPeso.appendChild(bt);
          })(kr, kc);
        }
      }
    }

    function estiloCelula(el, ativo, corFundo, tam){
      var t = tam || '24px';
      el.style.width = t; el.style.height = t;
      el.style.display = 'flex'; el.style.alignItems = 'center'; el.style.justifyContent = 'center';
      el.style.fontSize = '7.5px'; el.style.fontWeight = '700'; el.style.borderRadius = '4px';
      el.style.background = corFundo;
      el.style.border = ativo ? '2px solid #2F6F9F' : '1px solid #E4DCC8';
      el.style.boxSizing = 'border-box';
    }

    function desenharGradeX(){
      elGradeX.innerHTML = '';
      elGradeX.style.display = 'grid';
      elGradeX.style.gridTemplateColumns = 'repeat(6, 24px)';
      elGradeX.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var janelaAtiva = posicaoIdx < 16;
      for (var rr = 0; rr < 6; rr++){
        for (var cc = 0; cc < 6; cc++){
          var cel = document.createElement('div');
          var v = X[rr][cc];
          var cor = v > 0 ? '#EFE9D8' : '#FFFFFF';
          var dentroJanela = janelaAtiva && rr >= r && rr <= r+2 && cc >= c && cc <= c+2;
          var ehPixelDoVoto = janelaAtiva && rr === r+pesoSelKr && cc === c+pesoSelKc;
          estiloCelula(cel, false, cor, '24px');
          if (dentroJanela){ cel.style.border = '1px solid #B8AE94'; }
          if (ehPixelDoVoto){ cel.style.border = '2px solid #2F6F9F'; cel.style.background = '#DCEEFB'; }
          cel.textContent = v; cel.style.color = '#5E5A4A';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeX.appendChild(cel);
        }
      }
    }

    function desenharGradeZ(){
      elGradeZ.innerHTML = '';
      elGradeZ.style.display = 'grid';
      elGradeZ.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeZ.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheForward.Z[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheForward.Z[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #2F6F9F'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeZ.appendChild(cel);
        }
      }
    }

    function desenharGradeA(){
      elGradeA.innerHTML = '';
      elGradeA.style.display = 'grid';
      elGradeA.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeA.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheForward.A[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheForward.A[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #2F6F9F'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeA.appendChild(cel);
        }
      }
    }

    function desenharGradeDZ(){
      elGradeDZ.innerHTML = '';
      elGradeDZ.style.display = 'grid';
      elGradeDZ.style.gridTemplateColumns = 'repeat(4, 26px)';
      elGradeDZ.style.gap = '2px';
      var r = Math.floor(posicaoIdx/4), c = posicaoIdx % 4;
      var maxAbs = 0;
      for (var i=0;i<4;i++) for (var j=0;j<4;j++) maxAbs = Math.max(maxAbs, Math.abs(cacheErro.dZ[i][j]));
      maxAbs = maxAbs || 1;
      for (var rr = 0; rr < 4; rr++){
        for (var cc = 0; cc < 4; cc++){
          var cel = document.createElement('div');
          var v = cacheErro.dZ[rr][cc];
          estiloCelula(cel, false, cap09kgrad_corValor(v, maxAbs), '26px');
          if (posicaoIdx < 16 && rr === r && cc === c){ cel.style.border = '2px solid #C1443A'; }
          cel.textContent = v.toFixed(2); cel.style.color = '#374151';
          cel.dataset.r = rr; cel.dataset.c = cc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeDZ.appendChild(cel);
        }
      }
    }

    function desenharGradeK(){
      elGradeK.innerHTML = '';
      elGradeK.style.display = 'grid';
      elGradeK.style.gridTemplateColumns = 'repeat(3, 30px)';
      elGradeK.style.gap = '3px';
      var maxAbs = 0;
      for (var i=0;i<3;i++) for (var j=0;j<3;j++) maxAbs = Math.max(maxAbs, Math.abs(K[i][j]));
      maxAbs = maxAbs || 1;
      for (var kr = 0; kr < 3; kr++){
        for (var kc = 0; kc < 3; kc++){
          var cel = document.createElement('div');
          var ativo = (kr === pesoSelKr && kc === pesoSelKc);
          estiloCelula(cel, ativo, cap09kgrad_corValor(K[kr][kc], maxAbs), '30px');
          cel.textContent = K[kr][kc].toFixed(2); cel.style.color = '#374151';
          cel.dataset.kr = kr; cel.dataset.kc = kc;
          cel.classList.add('cap09kgrad_cell_hoverable');
          elGradeK.appendChild(cel);
        }
      }
    }

    // ================= SISTEMA DE HOVER: contas detalhadas + destaque entre camadas =================

    function cap09kgrad_criarTooltipGlobal(){
      if (elHoverTip) return elHoverTip;
      elHoverTip = document.createElement('div');
      elHoverTip.className = 'cap09kgrad_hovertip';
      document.body.appendChild(elHoverTip);
      return elHoverTip;
    }

    function cap09kgrad_posicionarTooltip(evt){
      var tip = elHoverTip;
      if (!tip) return;
      var margem = 14;
      var x = evt.clientX + margem;
      var y = evt.clientY + margem;
      var larguraTip = tip.offsetWidth, alturaTip = tip.offsetHeight;
      if (x + larguraTip > window.innerWidth - 8) x = evt.clientX - larguraTip - margem;
      if (y + alturaTip > window.innerHeight - 8) y = evt.clientY - alturaTip - margem;
      tip.style.left = x + 'px';
      tip.style.top = y + 'px';
    }

    function cap09kgrad_mostrarTooltip(htmlConteudo, evt){
      var tip = cap09kgrad_criarTooltipGlobal();
      tip.innerHTML = htmlConteudo;
      tip.style.display = 'block';
      cap09kgrad_posicionarTooltip(evt);
    }

    function cap09kgrad_esconderTooltip(){
      if (elHoverTip) elHoverTip.style.display = 'none';
    }

    function cap09kgrad_limparDestaques(){
      celulasDestacadas.forEach(function(item){ item.el.classList.remove(item.classe); });
      celulasDestacadas = [];
    }

    function cap09kgrad_destacar(elementos, classe){
      elementos.forEach(function(el){
        if (!el) return;
        el.classList.add(classe);
        celulasDestacadas.push({ el: el, classe: classe });
      });
    }

    function cap09kgrad_celX(rr,cc){ return elGradeX.querySelector('[data-r="'+rr+'"][data-c="'+cc+'"]'); }
    function cap09kgrad_celZ(r,c){ return elGradeZ.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celA(r,c){ return elGradeA.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celDZ(r,c){ return elGradeDZ.querySelector('[data-r="'+r+'"][data-c="'+c+'"]'); }
    function cap09kgrad_celK(kr,kc){ return elGradeK.querySelector('[data-kr="'+kr+'"][data-kc="'+kc+'"]'); }
    function cap09kgrad_todasCelA(){ return Array.prototype.slice.call(elGradeA.querySelectorAll('[data-r]')); }

    function cap09kgrad_hoverZ(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      var termos = [];
      for (var kr=0; kr<3; kr++){
        for (var kc=0; kc<3; kc++){
          var xv = X[r+kr][c+kc];
          var kv = K[kr][kc];
          termos.push({ kr:kr, kc:kc, x:xv, k:kv, t:kv*xv });
          cap09kgrad_destacar([cap09kgrad_celX(r+kr, c+kc)], 'cap09kgrad_hlSecundario');
          cap09kgrad_destacar([cap09kgrad_celK(kr, kc)], 'cap09kgrad_hlSecundario');
        }
      }
      var linhas = termos.map(function(t){
        return 'K['+t.kr+']['+t.kc+']&middot;X['+(r+t.kr)+']['+(c+t.kc)+'] = '+t.k.toFixed(2)+'&times;'+t.x+' = <b>'+t.t.toFixed(3)+'</b>';
      }).join('<br>');
      var html =
        '<b>Z['+r+']['+c+']</b> = &Sigma; K &middot; X (9 termos)<br>' +
        '<div style="margin:4px 0;border-top:1px dashed #4A473A;padding-top:4px;">' + linhas + '</div>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">total = <b>'+cacheForward.Z[r][c].toFixed(3)+'</b></div>' +
        '<div style="margin-top:4px;color:#B8AE94;">contorno tracejado = janela em X e pesos em K usados</div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverA(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlPrincipal');
      var zv = cacheForward.Z[r][c], av = cacheForward.A[r][c];
      var html =
        '<b>A['+r+']['+c+']</b> = max(0, Z['+r+']['+c+'])<br>' +
        '= max(0, '+zv.toFixed(3)+') = <b>'+av.toFixed(3)+'</b>' +
        (zv <= 0 ? '<br><span style="color:#FF9A93;">ReLU zerou este valor.</span>' : '');
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverDZ(r, c, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlPrincipal');
      cap09kgrad_destacar(cap09kgrad_todasCelA(), 'cap09kgrad_hlSecundario');
      var zv = cacheForward.Z[r][c];
      var gate = cap09kgrad_reluDeriv(zv);
      var dS = cacheErro.dS;
      var dz = cacheErro.dZ[r][c];
      var html =
        '<b>dZ['+r+']['+c+']</b> = (S &minus; alvo) &middot; ReLU&prime;(Z['+r+']['+c+'])<br>' +
        'S = &Sigma; A (16 células, tracejado) = <b>'+cacheForward.S.toFixed(3)+'</b><br>' +
        'S &minus; alvo = '+cacheForward.S.toFixed(3)+' &minus; '+alvo+' = <b>'+dS.toFixed(3)+'</b><br>' +
        'ReLU&prime;(Z) = ' + (gate ? '1 (Z&gt;0)' : '0 (Z&le;0)') + '<br>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">dZ = '+dS.toFixed(3)+' &times; '+gate+' = <b>'+dz.toFixed(3)+'</b></div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverX(rr, cc, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      var contribs = [];
      for (var kr=0; kr<3; kr++){
        for (var kc=0; kc<3; kc++){
          var r = rr-kr, c = cc-kc;
          if (r>=0 && r<4 && c>=0 && c<4){
            contribs.push({ r:r, c:c, kr:kr, kc:kc, k:K[kr][kc] });
            cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlSecundario');
            cap09kgrad_destacar([cap09kgrad_celK(kr,kc)], 'cap09kgrad_hlSecundario');
          }
        }
      }
      var xv = X[rr][cc];
      var linhas = contribs.map(function(t){
        var termo = t.k*xv;
        return 'Z['+t.r+']['+t.c+'] usa K['+t.kr+']['+t.kc+']&times;X = '+t.k.toFixed(2)+'&times;'+xv+' = <b>'+termo.toFixed(3)+'</b>';
      }).join('<br>');
      var html =
        '<b>X['+rr+']['+cc+']</b> = '+xv+'<br>' +
        (contribs.length ?
          '<div style="margin:4px 0;border-top:1px dashed #4A473A;padding-top:4px;">Usado em '+contribs.length+' posição(ões) de Z:<br>'+linhas+'</div>'
          : '<span style="color:#B8AE94;">Fora do alcance de qualquer janela 3&times;3 sobre a saída atual.</span>');
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_hoverK(kr, kc, celEl, evt){
      cap09kgrad_limparDestaques();
      cap09kgrad_destacar([celEl], 'cap09kgrad_hlPrincipal');
      for (var r=0; r<4; r++){
        for (var c=0; c<4; c++){
          cap09kgrad_destacar([cap09kgrad_celZ(r,c)], 'cap09kgrad_hlSecundario');
          cap09kgrad_destacar([cap09kgrad_celX(r+kr,c+kc)], 'cap09kgrad_hlSecundario');
        }
      }
      var gAtual = cap09kgrad_gradientePeso(cacheErro.dZ, X, kr, kc).gradiente;
      var html =
        '<b>K['+kr+']['+kc+']</b> = '+K[kr][kc].toFixed(3)+'<br>' +
        'Participa das 16 posições de Z (destacadas), cada uma multiplicando um pixel X diferente (também destacado).<br>' +
        '<div style="border-top:1px dashed #4A473A;margin-top:4px;padding-top:4px;">Gradiente atual &nabla;K = &Sigma; dZ&middot;X = <b>'+gAtual.toFixed(3)+'</b></div>';
      cap09kgrad_mostrarTooltip(html, evt);
    }

    function cap09kgrad_configurarHover(container, seletor, manipulador){
      container.addEventListener('mouseover', function(e){
        var cel = e.target.closest(seletor);
        if (!cel || cel.parentElement !== container) return;
        manipulador(cel, e);
      });
      container.addEventListener('mousemove', function(e){
        var cel = e.target.closest(seletor);
        if (!cel || cel.parentElement !== container) return;
        cap09kgrad_posicionarTooltip(e);
      });
      container.addEventListener('mouseout', function(e){
        var cel = e.target.closest(seletor);
        if (!cel) return;
        var indoPara = e.relatedTarget;
        if (indoPara && cel.contains(indoPara)) return;
        cap09kgrad_limparDestaques();
        cap09kgrad_esconderTooltip();
      });
    }

    // ================= FIM DO SISTEMA DE HOVER =================

    function atualizarFormula(){
      elFormula.innerHTML =
        '<span class="cap09kgrad_mono">&part;L/&part;K[' + pesoSelKr + '][' + pesoSelKc + ']</span> = &Sigma;<sub>(r,c)</sub> ' +
        '<span class="cap09kgrad_mono">dZ[r][c] &middot; X[r+' + pesoSelKr + '][c+' + pesoSelKc + ']</span>';
    }

    function renderizarListaVotos(){
      elListaVotos.innerHTML = '';
      var elementoAtivo = null;
      var idxAtivo = posicaoIdx - 1;

      votosAtuais.forEach(function(v, idx){
        var linha = document.createElement('div');
        linha.className = 'cap09kgrad_linhavoto';
        var visivel = idx < posicaoIdx;
        linha.style.opacity = visivel ? '1' : '0.25';
        if (idx === idxAtivo){
          linha.style.background = '#EAF2FA';
          elementoAtivo = linha;
        }
        linha.innerHTML =
          '<span class="cap09kgrad_mono" style="color:#8A8371;">p(' + v.r + ',' + v.c + ')</span> ' +
          '<span class="cap09kgrad_mono">' + v.dz.toFixed(2) + '</span>&times;' +
          '<span class="cap09kgrad_mono">' + v.x + '</span>=' +
          '<span class="cap09kgrad_mono" style="color:' + (v.voto>=0 ? '#1E8F6F' : '#C1443A') + ';font-weight:700;">' + v.voto.toFixed(2) + '</span>';
        elListaVotos.appendChild(linha);
      });

      if (elementoAtivo){
        var alvoTop = elementoAtivo.offsetTop - (elListaVotos.clientHeight/2) + (elementoAtivo.clientHeight/2);
        elListaVotos.scrollTop = Math.max(0, alvoTop);
      } else {
        elListaVotos.scrollTop = 0;
      }
    }

    function atualizarCalcAtual(){
      if (posicaoIdx === 0){
        elCalcAtual.innerHTML =
          '<span style="color:#8A8371;">Clique em <b>"Avançar posição"</b> para ver a primeira conta.</span>';
        return;
      }
      var atual = votosAtuais[posicaoIdx - 1];
      var corVoto = atual.voto >= 0 ? '#1E8F6F' : '#C1443A';
      elCalcAtual.innerHTML =
        'posição <b>(' + atual.r + ',' + atual.c + ')</b><br>' +
        'dZ&nbsp;&nbsp;= <b>' + atual.dz.toFixed(3) + '</b><br>' +
        'X&nbsp;&nbsp;&nbsp;&nbsp;= <b>' + atual.x + '</b><br>' +
        '<span style="border-top:1px dashed #E4DCC8;display:block;margin:4px 0 2px;"></span>' +
        'voto = dZ &times; X = <b style="color:' + corVoto + ';">' + atual.voto.toFixed(3) + '</b>';
    }

    function atualizarSomaTexto(){
      var somaParcial = posicaoIdx > 0 ? votosAtuais[posicaoIdx-1].somaParcial : 0;
      var completo = posicaoIdx >= 16;

      elSomaAtual.innerHTML =
        'posições: <b>' + posicaoIdx + ' / 16</b><br>' +
        '<span style="border-top:1px dashed #E4DCC8;display:block;margin:4px 0 2px;"></span>' +
        'total = <b style="color:#2F6F9F;">' + somaParcial.toFixed(3) + '</b>' +
        (completo ? '<br><span style="color:#1E8F6F;font-weight:700;">✓ gradiente completo</span>' : '');

      elBarraFill.style.width = (posicaoIdx/16*100).toFixed(1) + '%';
      elBtnVoltarPos.disabled = (posicaoIdx === 0);
      elBtnAvancar.disabled = completo;
      elBtnSomarTudo.disabled = completo;
      elBtnDesfazerPasso.disabled = (historicoKernel.length === 0);
    }

    function desenharGraficoLoss(){
      var Wc = elCanvasLoss.width, Hc = elCanvasLoss.height;
      ctxLoss.clearRect(0,0,Wc,Hc);
      ctxLoss.strokeStyle = '#E4DCC8'; ctxLoss.lineWidth = 1;
      ctxLoss.strokeRect(0.5,0.5,Wc-1,Hc-1);

      if (historicoLoss.length < 2){
        ctxLoss.fillStyle = '#B8AE94';
        ctxLoss.font = '10px Inter, sans-serif';
        ctxLoss.textAlign = 'center';
        ctxLoss.fillText('perda aparecerá aqui', Wc/2, Hc/2+3);
        return;
      }

      var maxLoss = Math.max.apply(null, historicoLoss);
      maxLoss = Math.max(maxLoss, 0.01);

      var padL = 30, padR = 8, padT = 10, padB = 14;
      var plotW = Wc - padL - padR, plotH = Hc - padT - padB;

      ctxLoss.strokeStyle = '#EDE7D6';
      ctxLoss.lineWidth = 1;
      ctxLoss.beginPath();
      ctxLoss.moveTo(padL, padT); ctxLoss.lineTo(Wc - padR, padT);
      ctxLoss.moveTo(padL, Hc - padB); ctxLoss.lineTo(Wc - padR, Hc - padB);
      ctxLoss.stroke();

      ctxLoss.strokeStyle = '#1E8F6F';
      ctxLoss.setLineDash([2, 2]);
      ctxLoss.beginPath();
      ctxLoss.moveTo(padL, Hc - padB); ctxLoss.lineTo(Wc - padR, Hc - padB);
      ctxLoss.stroke();
      ctxLoss.setLineDash([]);

      ctxLoss.fillStyle = '#8A8371';
      ctxLoss.font = '8px JetBrains Mono, monospace';
      ctxLoss.textAlign = 'right';
      ctxLoss.fillText(maxLoss.toFixed(1), padL - 3, padT + 3);
      ctxLoss.fillText('0.0', padL - 3, Hc - padB + 2);

      ctxLoss.strokeStyle = '#2F5FA8'; 
      ctxLoss.lineWidth = 1.6;
      ctxLoss.beginPath();
      historicoLoss.forEach(function(v, i){
        var x = padL + (historicoLoss.length === 1 ? 0 : (i / (historicoLoss.length - 1)) * plotW);
        var y = padT + (1 - v / maxLoss) * plotH;
        if (i === 0) ctxLoss.moveTo(x, y); else ctxLoss.lineTo(x, y);
      });
      ctxLoss.stroke();
    }

    function renderizarNotaAtualizacao(){
      var gAtual = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).gradiente;
      var kr = pesoSelKr, kc = pesoSelKc;
      var kAtual = K[kr][kc];
      var passoPrevisto = lr * gAtual;
      var kNovoPrevisto = kAtual - passoPrevisto;

      if (cacheForward.S <= 0){
        elNotaAtualizacao.innerHTML = 
          '<div style="background:#FDF2F2;border:1px solid #F87171;border-radius:8px;padding:8px 10px;color:#991B1B;">' +
          '<b>⚡ Colapso por ReLU Morta (S = 0.000):</b><br>' +
          'Todas as ativações Z &le; 0 foram zeradas pela ReLU. O erro dZ = 0, zerando o gradiente (&nabla;K = 0).<br>' +
          '• A perda travou em L = &frac12;&middot;(0 &minus; ' + alvo + ')&sup2; = ' + cacheErro.loss.toFixed(1) + '.<br>' +
          '• <b>Como resolver:</b> Reduza a taxa &eta;, clique em <b>"⏮ Desfazer passo"</b> ou em <b>"🎲 Novo kernel aleatório"</b>.' +
          '</div>';
        return;
      }

      if (ultimoGradiente === null){
        elNotaAtualizacao.innerHTML = 
          '<div style="text-align:left;">' +
          '<b>Estado Inicial do peso K[' + kr + '][' + kc + '] = ' + kAtual.toFixed(3) + ':</b><br>' +
          '• <b>Gradiente visível acumulado (&nabla;K):</b> <b style="color:#2F6F9F;">' + gAtual.toFixed(3) + '</b> (soma dos 16 votos dZ &times; X)<br>' +
          '• <b>Passo de ajuste previsto (&eta; &times; &nabla;K):</b> ' + lr + ' &times; (' + gAtual.toFixed(3) + ') = <b>' + passoPrevisto.toFixed(4) + '</b><br>' +
          '• <b>Novo Peso previsto:</b> K &larr; ' + kAtual.toFixed(3) + ' &minus; (' + passoPrevisto.toFixed(4) + ') = <b>' + kNovoPrevisto.toFixed(3) + '</b><br>' +
          '<br><span style="color:#8A8371;">Clique em <b>"▶ Aplicar passo de gradiente descendente"</b> para atualizar a matriz.</span>' +
          '</div>';
        return;
      }

      var gValAnterior = ultimoGradiente[kr][kc];
      var kAntigoVal = ultimoKAntigo[kr][kc];
      var passoAplicado = lr * gValAnterior;

      elNotaAtualizacao.innerHTML = 
        '<div style="text-align:left;">' +
        '<b>Última atualização aplicada ao peso K[' + kr + '][' + kc + ']:</b><br>' +
        '1. <b>Taxa (&eta;):</b> <span class="cap09kgrad_mono">' + lr + '</span> | <b>&nabla;K aplicado:</b> <span class="cap09kgrad_mono">' + gValAnterior.toFixed(3) + '</span><br>' +
        '2. <b>Passo executado:</b> ' + lr + ' &times; (' + gValAnterior.toFixed(3) + ') = <b>' + passoAplicado.toFixed(4) + '</b><br>' +
        '3. <b>Resultado:</b> K &larr; ' + kAntigoVal.toFixed(3) + ' &minus; (' + passoAplicado.toFixed(4) + ') = <b>' + kAtual.toFixed(3) + '</b><br>' +
        '<br><span style="color:#2F6F9F;"><b>Novo gradiente pronto na matriz atual:</b> &nabla;K = <b>' + gAtual.toFixed(3) + '</b></span>' +
        '</div>';
    }

    function renderizarTudo(){
      cap09kgrad_limparDestaques();
      cap09kgrad_esconderTooltip();
      desenharGradeX();
      desenharGradeZ();
      desenharGradeA();
      desenharGradeDZ();
      desenharGradeK();
      atualizarFormula();
      renderizarListaVotos();
      atualizarCalcAtual();
      atualizarSomaTexto();
      desenharGraficoLoss();
      renderizarNotaAtualizacao();

      var valS = cacheForward.S.toFixed(3);
      var valLoss = cacheErro.loss.toFixed(4);

      elTextoS.innerHTML = 'S = <b>' + valS + '</b> (alvo = ' + alvo + ')';
      elTextoLoss.innerHTML = 'L = &frac12;&middot;(' + valS + ' &minus; ' + alvo + ')&sup2; = <b>' + valLoss + '</b>';
      elTextoEpoca.textContent = 'Aggiornamenti: ' + epocaAtual;
    }

    function selecionarPeso(kr, kc){
      pesoSelKr = kr; pesoSelKc = kc;
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, kr, kc).votos;
      var botoes = elSeletorPeso.querySelectorAll('.cap09kgrad_pesobtn');
      botoes.forEach(function(b){ b.classList.remove('cap09kgrad_pesobtn_ativo'); });
      var idxBotao = kr*3+kc;
      if (botoes[idxBotao]) botoes[idxBotao].classList.add('cap09kgrad_pesobtn_ativo');
      renderizarTudo();
    }

    function avancarPosicao(){
      if (posicaoIdx < 16) posicaoIdx += 1;
      renderizarTudo();
    }

    function voltarPosicao(){
      if (posicaoIdx > 0) posicaoIdx -= 1;
      renderizarTudo();
    }

    function somarTudoAutomatico(){
      posicaoIdx = 16;
      renderizarTudo();
    }

    function reiniciarPosicoes(){
      posicaoIdx = 0;
      renderizarTudo();
    }

    function atualizarPesosDoKernel(){
      var gK = cap09kgrad_gradienteKernelCompleto(cacheErro.dZ, X);
      historicoKernel.push({
        K: JSON.parse(JSON.stringify(K)),
        gK: ultimoGradiente,
        kAnt: ultimoKAntigo
      });

      ultimoKAntigo = JSON.parse(JSON.stringify(K));
      ultimoGradiente = gK;

      K = cap09kgrad_atualizarKernel(K, gK, lr);
      epocaAtual += 1;

      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;

      renderizarTudo();
    }

    function desfazerPasso(){
      if (historicoKernel.length === 0) return;
      var estadoAnt = historicoKernel.pop();
      K = estadoAnt.K;
      ultimoGradiente = estadoAnt.gK;
      ultimoKAntigo = estadoAnt.kAnt;
      historicoLoss.pop();
      epocaAtual -= 1;

      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function novoKernelAleatorio(){
      K = cap09kgrad_inicializarKernelVivo(Date.now() % 2147483647, X);
      historicoKernel = [];
      epocaAtual = 0;
      ultimoGradiente = null;
      ultimoKAntigo = null;
      historicoLoss = [];
      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function trocarPadrao(novoPadrao, botaoAtivo){
      [elBtnDiagonal, elBtnVertical].forEach(function(b){ b.classList.remove('cap09kgrad_active'); });
      botaoAtivo.classList.add('cap09kgrad_active');
      padraoAtual = novoPadrao;
      X = cap09kgrad_PADROES[padraoAtual];
      alvo = cap09kgrad_ALVOS[padraoAtual];

      var checagem = cap09kgrad_forward(K, X);
      if (checagem.S <= 0){
        K = cap09kgrad_inicializarKernelVivo(7, X);
      }

      historicoKernel = [];
      epocaAtual = 0;
      ultimoGradiente = null;
      ultimoKAntigo = null;
      historicoLoss = [];
      cacheForward = cap09kgrad_forward(K, X);
      cacheErro = cap09kgrad_calcularErro(cacheForward, alvo);
      historicoLoss.push(cacheErro.loss);
      posicaoIdx = 0;
      votosAtuais = cap09kgrad_gradientePeso(cacheErro.dZ, X, pesoSelKr, pesoSelKc).votos;
      renderizarTudo();
    }

    function alterarLr(novaLr, btnAtivo){
      [elLr0005, elLr002, elLr02].forEach(function(b){ b.classList.remove('cap09kgrad_active'); });
      btnAtivo.classList.add('cap09kgrad_active');
      lr = novaLr;
      renderizarTudo();
    }

    elBtnDiagonal.addEventListener('click', function(){ trocarPadrao('diagonal', elBtnDiagonal); });
    elBtnVertical.addEventListener('click', function(){ trocarPadrao('vertical', elBtnVertical); });
    elLr0005.addEventListener('click', function(){ alterarLr(0.0005, elLr0005); });
    elLr002.addEventListener('click', function(){ alterarLr(0.002, elLr002); });
    elLr02.addEventListener('click', function(){ alterarLr(0.02, elLr02); });

    elBtnAvancar.addEventListener('click', avancarPosicao);
    elBtnVoltarPos.addEventListener('click', voltarPosicao);
    elBtnSomarTudo.addEventListener('click', somarTudoAutomatico);
    elBtnReiniciarPos.addEventListener('click', reiniciarPosicoes);
    elBtnAtualizarPesos.addEventListener('click', atualizarPesosDoKernel);
    elBtnDesfazerPasso.addEventListener('click', desfazerPasso);
    elBtnNovoKernel.addEventListener('click', novoKernelAleatorio);

    cap09kgrad_configurarHover(elGradeZ, '[data-r]', function(cel, e){
      cap09kgrad_hoverZ(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeA, '[data-r]', function(cel, e){
      cap09kgrad_hoverA(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeDZ, '[data-r]', function(cel, e){
      cap09kgrad_hoverDZ(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeX, '[data-r]', function(cel, e){
      cap09kgrad_hoverX(parseInt(cel.dataset.r,10), parseInt(cel.dataset.c,10), cel, e);
    });
    cap09kgrad_configurarHover(elGradeK, '[data-kr]', function(cel, e){
      cap09kgrad_hoverK(parseInt(cel.dataset.kr,10), parseInt(cel.dataset.kc,10), cel, e);
    });

    montarSeletorPeso();
    selecionarPeso(0,0);
  }

  function cap09kgrad_tentarIniciar(){
    var root = document.getElementById('sim-09-gradiente-kernel');
    if (root) cap09kgrad_initSim(root); else setTimeout(cap09kgrad_tentarIniciar, 200);
  }
  cap09kgrad_tentarIniciar();
})();
</script>
'''
)

**Figura 9.12:** Simulatore interattivo del calcolo del gradiente di un peso del *kernel* convoluzionale.


<figure id="fig-09-sim-09-gradiente-kernel">
  <img src="imagens/fig-09-sim-09-gradiente-kernel.png" alt=" Simulatore interattivo del calcolo del gradiente di un peso del *kernel* convoluzionale. " style="max-width:80%" />
  <figcaption><strong>Figura 9.12:</strong>  Simulatore interattivo del calcolo del gradiente di un peso del *kernel* convoluzionale. </figcaption>
</figure>

> ### 📝 🧠 Sintesi — Dalla convoluzione all'apprendimento di rappresentazioni
>
> I simulatori di questa sezione dimostrano, in modo sequenziale, come una CNN trasformi un'immagine in ingresso in una stima probabilistica e come i suoi parametri vengano ottimizzati durante l'addestramento:
>
> - **Convoluzione:** applica filtri sull'immagine per estrarre caratteristiche locali, generando mappe di caratteristiche mediante la **condivisione dei pesi**.
> - **ReLU:** introduce non linearità nel sistema, consentendo la modellazione di relazioni complesse tra i dati.
> - ***Pooling*:** riduce la risoluzione spaziale delle mappe di caratteristiche, diminuendo il costo computazionale e conferendo invarianza a piccole traslazioni locali.
> - ***Flatten*:** riorganizza le mappe multidimensionali in un vettore unidimensionale per alimentare i layer successivi.
> - **Layer completamente connesso:** combina le caratteristiche estratte per produrre i punteggi grezzi (*logits*) associati a ciascuna classe.
> - **Softmax:** converte i *logits* in una distribuzione di probabilità normalizzata.
> - **Funzione di perdita:** confronta la distribuzione prevista con la verità di riferimento (*ground truth*), quantificando scalarmente l'errore della rete.
> - **Retropropagazione:** applica la regola della catena per calcolare la derivata parziale (gradiente) della funzione di perdita rispetto a ciascun parametro addestrabile.
> - **Ottimizzatore:** aggiorna i coefficienti dei filtri, i pesi e i bias nella direzione opposta al gradiente, riducendo la perdita a ogni iterazione.
>
> Nel corso delle iterazioni, i filtri convoluzionali si trasformano da valori stocastici in rilevatori specializzati: i layer iniziali apprendono primitive visive di basso livello (come bordi e trame), mentre i layer più profondi consolidano queste rappresentazioni in strutture astratte e semantiche.

## 9.5 Applicazioni Pratiche in VC

Dopo la consolidazione teorica dei fondamenti delle CNN e la verifica visiva di ciascuna delle loro operazioni elementari tramite i simulatori interattivi, diventa essenziale osservare l'integrazione di queste fasi in *pipeline* complete di programmazione.

Nelle sezioni seguenti, la teoria viene tradotta in codice eseguibile in **PyTorch**, esplorando i tre compiti fondamentali della VC: **classificazione**, **rilevamento di oggetti** e **segmentazione semantica**. Questa progressione pratica consente di analizzare dalla costruzione di un'architettura convoluzionale addestrata da zero fino all'applicazione di strategie avanzate di **transfer learning** (*transfer learning*) su modelli pre-addestrati per set di dati sintetici e reali.

### 9.5.1 Classificazione delle Immagini con CNN

La classificazione delle immagini è una delle applicazioni più tradizionali delle CNN. In questo compito, l'obiettivo è assegnare un'unica etichetta all'immagine in ingresso, come identificare una categoria di oggetto, una specie animale o una classe diagnostica. A tal fine, la CNN trasforma progressivamente i valori dei pixel in rappresentazioni a più alto livello di astrazione, combinando strati convoluzionali, funzioni di attivazione e operazioni di riduzione spaziale fino a produrre una distribuzione di probabilità tra le classi possibili. In questa sezione vengono presentati l'architettura di base di una CNN classificatrice, il flusso di trasformazione dei dati lungo la rete e il processo di addestramento per l'ajustamento dei parametri appresi.

#### 9.5.1.1 Addestramento di una CNN da Zero su Cifre

Per stabilire un confronto diretto con gli approcci presentati nel Capitolo 7, in questa sezione si sviluppa una CNN addestrata sullo stesso insieme di dati di cifre scritte a mano (`load_digits`). La differenza fondamentale risiede nella fase di rappresentazione: mentre i metodi classici dipendono da *pixel* grezzi o da descrittori calcolati manualmente, come l'*Histogram of Oriented Gradients* (*HOG*), la CNN apprende automaticamente i coefficienti dei filtri convoluzionali durante il processo di ottimizzazione.

I codici seguenti (consolidati nella [Figura 9.15](#fig-09-cnn-treinamento)) eseguono la preparazione dei dati, definiscono un'architettura convoluzionale semplice in **PyTorch**, eseguono il ciclo di addestramento tramite l'algoritmo *Adam* e generano le curve di evoluzione della funzione di perdita e dell'accuratezza.

##### Blocco 1: Preparazione e Strutturazione dei Dati

La fase iniziale di qualsiasi *pipeline* di Deep Learning consiste nella conversione e nell'adattamento dei dati di input al formato richiesto dal *framework* di calcolo scientifico.

###### Il Concetto di *Tensor*

Nel Deep Learning, la struttura fondamentale dei dati è il ***tensor***. Dal punto di vista computazionale, un *tensor* consiste in un array multidimensionale di numeri generalizzato a $n$ dimensioni:

* Un *tensor* di ordine 0 è uno scalare (un singolo valore).
* Un *tensor* di ordine 1 è un vettore (lunghezza).
* Un *tensor* di ordine 2 è una matrice (righe e colonne).
* Un *tensor* di ordine 3 o superiore rappresenta un volume o un iper-array di dati.

Nel contesto di **PyTorch**, la classe `torch.Tensor` estende la funzionalità degli array numerici multidimensionali (come quelli di **NumPy**) offrendo supporto a operazioni accelerate su hardware tramite *GPU* (*Graphics Processing Units*) e supporto al calcolo automatico delle derivate (*autograd*), essenziale per l'algoritmo di retropropagazione.

###### Analisi del Codice di Pre-elaborazione

1. **Caricamento e Normalizzazione delle Intensità:**
   Il dataset `load_digits` contiene $1.797$ campioni di cifre scritte a mano di $8 \times 8$ *pixel*, le cui intensità originali variano sulla scala intera da $0$ a $16$. La divisione per $16.0$ esegue la **normalizzazione** dei dati nell'intervallo $[0.0, 1.0]$. Questa trasformazione in scala floating-point (`float32`) è indispensabile nelle reti neurali per evitare la saturazione delle funzioni di attivazione e stabilizzare il calcolo dei gradienti nell'algoritmo di ottimizzazione.

2. **Divisione Stratificata (70% Addestramento / 30% Test):**
   La funzione `train_test_split` separa il $70\%$ dei campioni per l'adattamento dei parametri della rete e riserva il $30\%$ per la valutazione del modello su dati non visti. Il parametro `stratify=y` garantisce il campionamento stratificato, mantenendo la proporzione esatta di ciascuna delle 10 classi di cifre ($0$ a $9$) in entrambi gli insiemi, prevenendo distorsioni nella distribuzione.

3. **Adattamento Dimensionale per la Convoluzione 2D (`unsqueeze`):**
   Nelle CNN, i layer convoluzionali bidimensionali (`nn.Conv2d`) richiedono che il *tensor* di input possieda strettamente 4 dimensioni nell'ordinamento $(N, C, H, W)$:
   * $N$: numero di campioni (*batch size*).
   * $C$: numero di canali di colore ($1$ per scala di grigi, $3$ per *RGB*).
   * $H$: altezza dell'immagine in *pixel* ($8$).
   * $W$: larghezza dell'immagine in *pixel* ($8$).

   Poiché l'array originale possiede un formato $3\text{D}$ del tipo $(N, 8, 8)$, la chiamata `.unsqueeze(1)` inserisce una dimensione unitaria specificamente all'**indice 1** (la posizione riservata al canale di colore $C$), trasformando la struttura in un *tensor* $4\text{D}$ di formato $(N, 1, 8, 8)$, come richiesto da PyTorch.

4. **Conversione delle Etichette (`dtype=torch.long`):**
   Le etichette delle classi $y$ vengono convertite in *tensor* interi a 64 *bit* (`torch.long`). Questa specifica di tipo è un requisito della funzione di loss di Entropia Incrociata (`nn.CrossEntropyLoss`), che utilizza interi non negativi come indici per associare la classe corretta ai *logits* di output della rete.

> ### 📝 Nota
>
> **Attenzione alle Dimensioni:** La struttura finale è rappresentata dal **tensor `(N, 1, 8, 8)`**, dove `N` è il numero di campioni (*batch size*), `1` è il canale di colore (scala di grigi) e `8×8` è la risoluzione spaziale dell'immagine in *pixel*.

La [Figura 9.13](#fig-09-digits-amostra) illustra una sequenza di campioni dell'insieme di addestramento dopo la pre-elaborazione e l'adattamento dimensionale ai *tensor* di **PyTorch**. Nella fase di visualizzazione, la chiamata `img.squeeze().numpy()` combina due trasformazioni: il metodo `.squeeze()` elimina la dimensione unitaria ridondante del canale di colore, riducendo il *tensor* $3\text{D}$ di formato `(1, 8, 8)` a una matrice $2\text{D}$ di `(8, 8)`; successivamente, il metodo `.numpy()` converte la struttura di **PyTorch** in una matrice nativa di **NumPy**, formato richiesto dagli strumenti di rendering grafico come `mm.show()`.

In [12]:
# 1. Caricamento e pre-processamento dei dati
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0  # Normalizzazione all'intervallo [0, 1]
y = digits.target

# Divisione stratificata in set di addestramento (70%) e test (30%)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Adeguamento alla dimensione prevista da PyTorch: (N_campioni, Canali, Altezza, Larghezza)
X_treino_t = torch.tensor(X_treino).unsqueeze(1)   # Dimensione: (N, 1, 8, 8)
y_treino_t = torch.tensor(y_treino, dtype=torch.long)
X_teste_t = torch.tensor(X_teste).unsqueeze(1)
y_teste_t = torch.tensor(y_teste, dtype=torch.long)

# Mostrare un campione
n_amostras = 8
imgs = [img.squeeze().numpy() for img in X_treino_t[:n_amostras]]
imgs_titles = [str(label.item()) for label in y_treino_t[:n_amostras]]
mm.show(imgs, titles=imgs_titles, cols=n_amostras, figsize=(12, 2.5))

<Figure size 1800x375 with 8 Axes>

**Figura 9.13:** Campioni di cifre dal set di addestramento dopo la conversione in *tensori* PyTorch e normalizzazione.


##### Bloco 2: Definizione dell’Architettura Convoluzionale

La costruzione di modelli in **PyTorch** è strutturata secondo il paradigma dell’orientamento a oggetti, creando una classe specifica per rappresentare la rete neurale (in questo esempio, la classe `CNNDigitos`), che eredita tutte le funzionalità della classe base `nn.Module`. Il costruttore `__init__` è responsabile di istanziare i layer e dichiarare i loro parametri addestrabili, mentre il metodo `forward` stabilisce la sequenza numerica della propagazione in avanti (*forward pass*).

La [Figura 9.14](#fig-09-cnn-digitos-esquema) sintetizza le trasformazioni spaziali dei *tensor* e il flusso di dati lungo la classe `CNNDigitos`.

<figure id="fig-09-cnn-digitos-esquema" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-09-cnn-digitos-esquema.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figura 9.14:</strong> Rappresentazione del flusso delle trasformazioni dimensionali dei *tensor* lungo l’architettura *CNNDigitos*.</figcaption>
</figure>

1. **Costruttore (`__init__`) e Istanziazione dei Componenti:**
   * **Livello Convoluzionale 1 (`self.conv1`):** Applica $8$ filtri $3 \times 3$ con `padding=1` sull’input in scala di grigi ($1$ canale), preservando la risoluzione spaziale di $8 \times 8$ *pixel*.
   * **Livello Convoluzionale 2 (`self.conv2`):** Processa le $8$ mappe di caratteristiche ricevute dal livello precedente applicando $16$ filtri $3 \times 3$ con `padding=1`.
   * **Sottocampionamento (`self.pool`):** Istanzia l’operazione di *Max-Pooling* con finestra $2 \times 2$ e passo (*stride*) $2$, riducendo la dimensione spaziale (altezza e larghezza) della metà a ogni applicazione.
   * **Livelli Totalmente Connessi (`self.fc1` e `self.fc2`):** La prima proiezione densa riceve il *tensor* appiattito di dimensione $16 \times 2 \times 2 = 64$ e produce $32$ caratteristiche intermedie. La seconda proietta queste $32$ caratteristiche nei $10$ *logit* finali di output.

2. **Propagazione in Avanti nel Metodo `forward`:**
   * **Primo Blocco Convoluzionale:** Il *tensor* di input di formato $(N, 1, 8, 8)$ passa attraverso `conv1` + ReLU e viene sottocampionato da `pool`, risultando nel formato $(N, 8, 4, 4)$.
   * **Secondo Blocco Convoluzionale:** Il *tensor* $(N, 8, 4, 4)$ viene processato da `conv2` + ReLU e ridotto da `pool` al formato $(N, 16, 2, 2)$.
   * **Appiattimento (*Flatten*):** Il metodo `x.view(x.size(0), -1)` riconfigura la struttura $3\text{D}$ in un vettore $1\text{D}$ di $64$ elementi per campione, preservando la dimensione del batch $N$.
   * **Classificazione:** Il vettore di $64$ elementi alimenta `fc1` con attivazione ReLU ($32$ neuroni) e termina in `fc2`, producendo i $10$ *logit* non normalizzati per il calcolo della funzione di perdita.

In [13]:
# 2. Definizione dell'architettura convoluzionale
class CNNDigitos(nn.Module):
    """
    Architettura convoluzionale compatta:
    2 strati convoluzionali con ReLU e Max-Pooling + 2 strati densi.
    """
    def __init__(self, n_classes=10):
        super().__init__()
        
        # Conv1: 1 canale di ingresso, 8 filtri 3x3 con padding 1 (uscita: 8x8)
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        # Conv2: 8 canali di ingresso, 16 filtri 3x3 con padding 1 (uscita: 4x4)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)

        self.relu = nn.ReLU()

        # Max-Pooling 2x2 con passo (stride) 2
        self.pool = nn.MaxPool2d(2, 2)

        # Strati totalmente connessi (FC)
        self.fc1 = nn.Linear(16 * 2 * 2, 32)
        self.fc2 = nn.Linear(32, n_classes)

    def forward(self, x):
        # Primo blocco: Conv (8x8) -> ReLU -> Pool (4x4)
        x = self.pool(self.relu(self.conv1(x)))

        # Secondo blocco: Conv (4x4) -> ReLU -> Pool (2x2)
        x = self.pool(self.relu(self.conv2(x)))
        
        # Appiattimento (Flatten): riconfigura la matrice 3D (16, 2, 2) in vettore 1D (64)
        x = x.view(x.size(0), -1)

        # Strato denso intermedio con ReLU
        x = self.relu(self.fc1(x))

        # Strato finale di classificazione (logits)
        return self.fc2(x)

###### Analisi dei Strati e del Flusso della Classe `CNNDigiti`

1. **Costruttore (`__init__`) e Istanziazione dei Componenti:**
   * **Strato Convoluzionale 1 (`self.conv1`):** Applica $8$ filtri $3 \times 3$ con `padding=1` sull'ingresso in scala di grigi ($1$ canale), preservando la risoluzione di $8 \times 8$ *pixel*.
   * **Strato Convoluzionale 2 (`self.conv2`):** Processa le $8$ mappe di caratteristiche ricevute applicando $16$ filtri $3 \times 3$ con `padding=1`.
   * **Sottocampionamento (`self.pool`):** Istanzia l'operazione di *Max-Pooling* con finestra $2 \times 2$ e passo (*stride*) $2$, riducendo le dimensioni spaziali (altezza e larghezza) della metà a ogni applicazione.
   * **Strati Totalmente Connessi (`self.fc1` e `self.fc2`):** La prima proiezione densa riceve il *tensore* appiattito di dimensione $16 \times 2 \times 2 = 64$ e produce $32$ caratteristiche intermedie. La seconda proietta queste $32$ caratteristiche nei $10$ *logit* di uscita.

2. **Propagazione in Avanti nel Metodo `forward`:**
   * **Primo Blocco:** Il *tensore* $(N, 1, 8, 8)$ passa attraverso `conv1` + ReLU e viene ridotto da `pool` a $(N, 8, 4, 4)$.
   * **Secondo Blocco:** Il *tensore* $(N, 8, 4, 4)$ passa attraverso `conv2` + ReLU e viene ridotto da `pool` a $(N, 16, 2, 2)$.
   * **Appiattimento (*Flatten*):** Il metodo `x.view(x.size(0), -1)` converte la struttura $3\text{D}$ in un vettore $1\text{D}$ di $64$ elementi per campione.
   * **Classificazione:** Il vettore di $64$ elementi alimenta `fc1` con attivazione ReLU ($32$ neuroni) e termina in `fc2`, che produce i $10$ *logit* finali per il calcolo della perdita di Entropia Incrociata.

##### Bloco 3: Istanziazione e Parametri di Ottimizzazione

La fase di configurazione dell'apprendimento richiede l'istanziazione dell'architettura definita e la scelta di due componenti fondamentali: la **funzione di perdita**, che quantifica l'errore del modello, e l'**algoritmo di ottimizzazione**, responsabile dell'aggiustamento dei parametri verso il minimo di tale funzione.

1. **Istanziazione e Conteggio dei Parametri:**
   Il modello viene creato mediante l'istanziazione dell'oggetto `modello_cnn` della classe `CNNDigiti`. L'espressione `sum(p.numel() for p in modello_cnn.parameters())` esamina tutti i *tensor* dei parametri addestrabili della rete (pesi e bias di ogni livello) e calcola la cardinalità totale del modello, quantificandone la capacità di rappresentazione.

2. **Funzione di Perdita (`nn.CrossEntropyLoss`):**
   La perdita di Entropia Incrociata (*Cross-Entropy Loss*) rappresenta la scelta standard per i problemi di classificazione multiclasse. In PyTorch, questa implementazione combina internamente l'applicazione della funzione *LogSoftmax* con la Perdita di Log-Verosimiglianza Negativa (*NLLLoss*). Per tale ragione, il livello di output della rete produce *logit* grezzi, eliminando la necessità di applicare esplicitamente la funzione *Softmax* al termine del metodo `forward`.

3. **Ottimizzatore Adattivo (`optim.Adam`):**
   L'aggiornamento dei parametri utilizza l'algoritmo *Adam* (*Adaptive Moment Estimation*), con un tasso di apprendimento iniziale $\eta = 0,01$ (`lr=1e-2`). L'*Adam* combina i principi del momento con l'adattamento della dimensione del passo basato sulla media mobile delle derivate del primo e secondo ordine, regolando individualmente il tasso di apprendimento di ciascun parametro della rete.

In [14]:
# 3. Inizializzazione del Modello e Parametri di Ottimizzazione
modelo_cnn = CNNDigitos()
num_params = sum(p.numel() for p in modelo_cnn.parameters())
print(f"Parametri addestrabili del modello: {num_params}")

criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_cnn.parameters(), lr=1e-2)

Parametri addestrabili del modello: 3658


##### Blocco 4: Ciclo di Addestramento e Valutazione

L'addestramento di una CNN avviene in modo iterativo tramite l'algoritmo di Discesa del Gradiente Stocastica per *mini-batch* (*Mini-batch SGD*).

Le curve di apprendimento risultanti da questo processo sono presentate nella [Figura 9.15](#fig-09-cnn-treinamento), generata al termine dell'esecuzione.

1. **Fase di Addestramento (`modello_cnn.train()`):**
   Il ciclo principale esegue l'addestramento per $50$ epoche. In ogni epoca, si svolgono le seguenti fasi:
   * **Rimescolamento Stocastico:** La funzione `torch.randperm(n)` genera una permutazione casuale degli indici dei campioni, garantendo che l'ordinamento dei *mini-batch* vari a ogni epoca per evitare bias di campionamento.
   * **Divisione in *Mini-batch*:** L'insieme di addestramento viene suddiviso in lotti di $32$ campioni (`tam_lote = 32`).
   * **Azzera i Gradient (`otimizzatore.zero_grad()`):** Pulisce i gradienti accumulati nel *tensor* nell'iterazione precedente, evitando la somma indesiderata di derivate tra lotti distinti.
   * **Passo in Avanti e Perdita:** Il *forward pass* calcola le previsioni `saida`, e la chiamata `criterio(saida, y_treino_t[idx])` quantifica l'errore del lotto.
   * **Retropropagazione (`perda.backward()`):** Applica la regola della catena per calcolare le derivate parziali della perdita rispetto a ciascun parametro ($\frac{\partial L}{\partial w}$).
   * **Aggiornamento dei Pesi (`otimizzatore.step()`):** Aggiorna i parametri del modello secondo le equazioni dell'ottimizzatore *Adam*.

2. **Fase di Valutazione (`modello_cnn.eval()`):**
   Al termine di ogni epoca, il modello viene portato in modalità di valutazione. Il contesto `with torch.no_grad()` disattiva temporaneamente il motore di calcolo automatico delle derivate (*autograd*), riducendo il consumo di memoria e accelerando l'inferenza sull'insieme di test (`X_teste_t`). L'operazione `.argmax(dim=1)` estrae la classe di maggiore probabilità per ciascun campione, consentendo di calcolare l'accuratezza sul test.

3. **Visualizzazione con la Libreria `morph`:**
   La funzione `mm.showTrainCurves` della libreria didattica `morph` consolida lo storico della perdita di addestramento e l'accuratezza sul test in un unico pannello grafico, permettendo di diagnosticare la convergenza del modello e monitorare la stabilità dell'apprendimento lungo le epoche.

In [15]:
# 4. Ciclo di Training (Mini-batch SGD)
n = X_treino_t.size(0)                    # Numero di campioni
tam_lote = 32                             # Dimensione del mini-batch
epocas = 50                               # Totale epoche
historico_perda, historico_acc = [], []   # Storico delle metriche

for epoca in range(epocas):                  # Ripete per epoca
    modelo_cnn.train()                       # Modalità training
    perm = torch.randperm(n)                 # Mescola i campioni
    perda_epoca = 0.0                        # Accumula perdite
    for i in range(0, n, tam_lote):          # Scorre i mini-batch
        idx = perm[i:i + tam_lote]           # Indici del lotto
        otimizador.zero_grad()               # Azzera gradienti
        saida = modelo_cnn(X_treino_t[idx])  # Propagazione diretta
        perda = criterio(saida, y_treino_t[idx]) # Calcola perdita
        perda.backward()                         # Retropropagazione
        otimizador.step()                        # Aggiorna pesi
        perda_epoca += perda.item() * len(idx)   # Somma perdita

    # Valutazione del modello sul set di test alla fine di ogni epoca
    modelo_cnn.eval()                            # Modalità valutazione
    with torch.no_grad():                        # Senza gradienti
        pred_teste = modelo_cnn(X_teste_t).argmax(dim=1)  # Predizioni
        acc_teste = (pred_teste == y_teste_t).float().mean().item()  # Accuratezza
    historico_perda.append(perda_epoca / n)      # Registra perdita
    historico_acc.append(acc_teste)              # Registra accuratezza

acc_final_cnn = historico_acc[-1]                # Ultima accuratezza
print(f"Accuratezza finale della CNN sul set di test: {acc_final_cnn:.4f}")  # Mostra risultato

final = mm.showTrainCurves(                      # Traccia curve
    historico_perda, historico_acc,
    titulo="Evolução do Treinamento da CNN — Base de Dígitos",
    subtitulo=f"Acurácia final no teste: {acc_final_cnn:.4f}",
)

Accuratezza finale della CNN sul set di test: 0.9759


<Figure size 1190x714 with 2 Axes>

**Figura 9.15:** Curve di training e valutazione della CNN sul dataset di cifre: evoluzione della perdita di entropia incrociata sul set di training e dell


##### Bloco 5: Visualizzazione del Flusso di Attivazioni

L'ispezione della rete addestrata consente di osservare la trasformazione progressiva del *tensor* di ingresso attraverso i livelli dell'architettura `CNNDigitos`. La [Figura 9.16](#fig-09-cnn-ativacoes) illustra le dimensioni e le attivazioni intermedie ottenute elaborando un esempio reale della cifra $3$.

1. **Selezione e Preparazione del Campione:**
   Il seme stocastico viene fissato con `torch.manual_seed(7)` per garantire la riproducibilità dei risultati. La prima occorrenza della cifra $3$ nel dataset `load_digits` viene isolata, normalizzata nell'intervallo $[0.0, 1.0]$ e riconfigurata come un *tensor* `x` di dimensione $(1, 1, 8, 8)$.

2. **Ispezione Intermedia con `mm.showNet`:**
   La funzione `mm.showNet` della libreria `morph` esegue la propagazione in avanti (*forward pass*) del *tensor* `x` nell'istanza `modelo_cnn` precedentemente addestrata. Utilizzando *hook* di *forward*, la funzione intercetta lo stato numerico delle attivazioni nei livelli convoluzionali (`nn.Conv2d`), di pooling (`nn.MaxPool2d`) e completamente connessi (`nn.Linear`), restituendoli nel dizionario `acts`. Le funzioni di attivazione non lineare (`nn.ReLU`) non vengono registrate come stadi indipendenti, poiché la loro applicazione avviene direttamente sul *tensor* di uscita del livello corrispondente.

3. **Verifica dei Risultati:**
   L'istruzione `list(acts.keys())` visualizza la sequenza degli identificatori dei livelli monitorati, consentendo di confermare la riduzione dimensionale progressiva e la generazione del *logit* di valore massimo nell'indice corrispondente alla classe $3$, come dimostrato nella [Figura 9.16](#fig-09-cnn-ativacoes).

In [16]:
torch.manual_seed(7)
digits = load_digits()
idx = np.where(digits.target == 3)[0][0]
img = digits.images[idx] / 16.0
x = torch.tensor(img, dtype=torch.float32).view(1, 1, 8, 8)

# Riutilizzo dell'istanza del modello precedentemente addestrata
acts = mm.showNet(
    modelo_cnn,
    x,
    titulo="Fluxo de transformações dos tensors ao longo da arquitetura CNNDigitos",
    subtitulo=f"Exemplo real do dataset load_digits (classe verdadeira: {digits.target[idx]})",
)
print("Livelli catturati:", list(acts.keys()))

<Figure size 3213x578 with 7 Axes>

**Figura 9.16:** Flusso di attivazioni della CNN addestrata durante l


Livelli catturati: ['conv1', 'pool', 'conv2', 'pool #2', 'fc1', 'fc2']


##### Ispezionando il Grafo Computazionale con `torchviz`

Mentre `mm.showNet` privilegia la chiarezza didattica — mostrando una colonna per livello con parametri addestrabili —, la libreria `torchviz` proietta il **grafo di autograd** esattamente come PyTorch lo costruisce internamente per il calcolo dei gradienti. La [Figura 9.17](#fig-09-torchviz-grafo) illustra questa prospettiva rappresentando l'architettura `CNNDigitos`.

1. **Propagazione in Avanti Tracciata:** Con il modello addestrato in modalità `eval()`, il *forward pass* sul *tensor* `x` della cifra $3$ è sufficiente affinché il motore di *autograd* registri tutte le operazioni eseguite, incluse quelle senza parametri addestrabili, come la funzione di attivazione `ReLU` e la riconfigurazione dimensionale `view`.

2. **Generazione del Grafo (`make_dot`):** La funzione `make_dot(saida, params=...)` costruisce il grafo a partire dal *tensor* di uscita, percorrendo retroattivamente la cronologia delle operazioni fino ai nodi foglia (i parametri addestrabili del modello). Ogni nodo del diagramma rappresenta un'operazione del *backward pass* (come `ReluBackward` o `AddmmBackward`), e non solo un blocco concettuale del `nn.Module`.

3. **Esportazione e Renderizzazione (`.render`):** Il metodo `.render(..., format="png", cleanup=True)` richiama l'eseguibile `dot` di **Graphviz** per compilare l'immagine in formato PNG, eliminando automaticamente i file intermedi del codice sorgente.

La [Figura 9.17](#fig-09-torchviz-grafo) evidenzia come questo grafo computazionale, anche per un'architettura compatta, presenti una densità maggiore rispetto al pannello di `mm.showNet`, poiché dettaglia ogni operazione atomica responsabile del flusso dei gradienti.

In [17]:
# 1. Forward pass con tracciamento del gradiente abilitato
modelo_cnn.eval()
saida = modelo_cnn(x)  # Riutilizzo del tensore x (cifra 3)

# 2. Grafo di base: flusso delle operazioni fino all'uscita
grafo_simples = make_dot(saida, params=dict(modelo_cnn.named_parameters()))
caminho_simples = grafo_simples.render("cnn_digitos_grafo_simples", format="png", cleanup=True)

# 3. Visualizzazione diretta nell'ambiente Quarto/Jupyter
# Il .render() restituisce il percorso del file PNG generato; dobbiamo aprirlo come immagine
imagem_simples = np.array(Image.open(caminho_simples).convert("RGB"))
mm.show(imagem_simples, figsize=(5,10))

<Figure size 750x1500 with 1 Axes>

**Figura 9.17:** Grafo computacional della *CNNDigitos* generato tramite *torchviz*, che mostra le operazioni di *forward* e i nodi di gradiente (*backward*) associati a ciascun parametro addestrabile.


##### Visão Dettagliata del Grafo Computazionale con `torchviz`

Oltre alla rappresentazione semplificata, la libreria `torchviz` consente di espandere il grafo di *autograd* per ispezionare i dettagli interni dell'esecuzione della rete `CNNDigitos`. La [Figura 9.18](#fig-09-torchviz-grafo-detalhado) presenta questa struttura espansa per lo stesso *tensor* di input `x`.

1. **Tracciamento con Attributi di Operazione (`show_attrs=True`):**
   L'inclusione degli attributi visualizza le configurazioni iperparametriche associate a ciascun nodo computazionale durante la propagazione in avanti (*forward pass*), come le dimensioni del *kernel* (`kernel_size`), i passi (*stride*) e i riempimenti (*padding*) nelle convoluzioni e nei sottocampionamenti.

2. **Rilevamento dei *Tensor* Salvati in Memoria (`show_saved=True`):**
   Il parametro forza la visualizzazione esplicita dei *tensor* intermedi che PyTorch conserva in memoria durante il *forward pass*. Questi dati vengono preservati perché saranno strettamente necessari per il calcolo delle derivate parziali durante la fase di retropropagazione (*backward pass*).

3. **Generazione e Compilazione dei Grafi:**
   Mentre `grafo_simples` genera una visione diretta del flusso dei gradienti, `grafo_dettagliato` compila il grafo espanso nel file `cnn_digitos_grafo_detalhado.png` tramite l'eseguibile `dot` di **Graphviz**.

Come osservato nella [Figura 9.18](#fig-09-torchviz-grafo-detalhado), questa visualizzazione minuziosa è utile per eseguire il debug del consumo di memoria video (*VRAM*) e per verificare come il motore di PyTorch allochi internamente ciascun nodo della regola della catena.

In [18]:
# stesso codice precedente (make_dot + render)...

# 2. Grafo dettagliato: visualizzazione delle dimensioni e dei tensori salvati per la retropropagazione
grafo_detalhado = make_dot(
    saida,
    params=dict(modelo_cnn.named_parameters()),
    show_attrs=True,   # Mostra gli attributi delle operazioni (es.: kernel_size, stride)
    show_saved=True,   # Mostra i tensori salvati in memoria per la retropropagazione
)

caminho_detalhado = grafo_detalhado.render("cnn_digitos_grafo_detalhado", 
                                           format="png", cleanup=True)

# 3. Visualizzazione diretta nell'ambiente Quarto/Jupyter
imagem_detalhado = np.array(Image.open(caminho_detalhado).convert("RGB"))
mm.show(imagem_detalhado, figsize=(8, 16))

<Figure size 1200x2400 with 1 Axes>

**Figura 9.18:** Grafo dettagliato della classe CNNDigiti generato tramite *torchviz*.


##### Confronto con il Capitolo 7

La [Figura 9.19](#fig-09-comparativo-cap7) riunisce i risultati ottenuti sullo stesso insieme di dati (`load_digits`), stabilendo un parallelo diretto tra gli approcci classici esplorati in precedenza e la CNN sviluppata in questo capitolo.

1. **Prestazioni dei *Pixel* Grezzi vs. Descrittori Manuali:** Negli esperimenti del Capitolo 7, il classificatore $k\text{-NN}$ ($k=3$) ha raggiunto un'accuratezza del $98,4\%$ quando alimentato direttamente con i *pixel* grezzi delle immagini. Al contrario, l'estrazione preventiva delle caratteristiche tramite *Histogram of Oriented Gradients* (*HOG*) ha prodotto prestazioni significativamente inferiori ($75,8\%$). Questo calo si verifica perché il *HOG* è stato concepito per catturare i gradienti dei bordi in immagini a risoluzione più elevata; in matrici di soli $8 \times 8$ *pixel*, la risoluzione spaziale è insufficiente per formare istogrammi di orientamento informativi.

2. **Equivalenza della CNN e Apprendimento *End-to-End*:** La rete convoluzionale `CNNDigitos` raggiunge prestazioni competitive del $97,6\%$, avvicinandosi all'accuratezza del $k\text{-NN}$ con *pixel* grezzi su una base piccola e pre-allineata. Il grande vantaggio concettuale risiede nell'apprendimento della rappresentazione: invece di dipendere da descrittori progettati manualmente (*handcrafted features*) o di mantenere l'intero insieme di dati in memoria per la ricerca dei vicini al momento dell'inferenza, la CNN ottimizza automaticamente i propri filtri convoluzionali durante l'addestramento, generando un modello compatto in grado di eseguire l'estrazione delle caratteristiche e la classificazione in modo integrato (*end-to-end*).

In [19]:
import matplotlib.pyplot as plt

# Valori ottenuti nel Capitolo 7 (k-NN, k=3), riprodotti per confronto diretto
ACC_KNN_PIXELS_CAP7 = 0.9844
ACC_KNN_HOG_CAP7 = 0.7578

metodos = ["k-NN\n(pixels brutos)", "k-NN\n(HOG)", "CNN\n(este capítulo)"]
acuracias = [ACC_KNN_PIXELS_CAP7, ACC_KNN_HOG_CAP7, acc_final_cnn]

plt.figure(figsize=(5, 4))
cores = ["#6366f1", "#f97316", "#16a34a"]
plt.bar(metodos, acuracias, color=cores)
plt.ylim(0, max(acuracias) + 0.08)
plt.ylabel("Acurácia (conjunto de teste)")
plt.title("Cap. 7 vs. Cap. 9 — Base de Dígitos")

for i, v in enumerate(acuracias):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

<Figure size 1500x1200 with 1 Axes>

**Figura 9.19:** Confronto di accuratezza tra i classificatori classici del Capitolo 7 (pixel grezzi e HOG con k-NN) e la CNN addestrata in questo capitolo, sulla stessa base di cifre.


> ### 📝 Nota
>
> ###### 🧠 Perché funziona? — E perché la CNN non "vince" sempre
>
> Il risultato osservato qui **ripete il pattern già visto nel Capitolo 7**: la CNN, pur imparando automaticamente le proprie caratteristiche, non supera necessariamente il $k\text{-NN}$ con *pixel* grezzi in questa specifica base. La spiegazione è la stessa: `load_digits` è una base piccola (meno di $1.800$ esempi), con immagini già centralizzate, normalizzate e a bassissima risoluzione ($8 \times 8$) — condizioni in cui il confronto diretto delle intensità è già altamente informativo, e ci sono pochi dati perché la rete impari filtri realmente superiori ai descrittori semplici.
>
> Il vero valore distintivo delle CNN emerge in scenari che descrittori artigianali e classificatori semplici non riescono ad affrontare: immagini più grandi e più realistiche, con migliaia di categorie, variazione sostanziale di posa, illuminazione e sfondo, e insiemi di addestramento massicci — esattamente il regime in cui i modelli presentati nella sezione *"Applicazioni su Larga Scala"*, più avanti, sono stati addestrati. La lezione pedagogica che attraversa i Capitoli 7, 8 e 9 di questo libro è coerente: **la sofisticazione di un metodo deve essere proporzionale alla complessità del problema** — usare una CNN per un problema che un $k\text{-NN}$ risolve ugualmente bene è uno spreco di risorse computazionali, non una virtù.
>
> Questa stessa proporzionalità vale per gli strumenti di ispezione usati lungo il capitolo. Il `mm.showNet` è stato costruito per scopi **didattici** e funziona bene in reti poco profonde come la `CNNDigitos`, ma non scala a architetture profonde: ogni strato tracciato diventa una colonna nella figura, e gli strati convoluzionali con centinaia di canali generano mosaici troppo grandi per un'interpretazione visiva; inoltre, gli *hook* memorizzano tutte le attivazioni in memoria, e il *layout* assume un flusso sequenziale, non rappresentando fedelmente connessioni residue o ramificazioni (come in *ResNet* o moduli *Inception*). Pertanto, `showNet` deve essere inteso come una lente pedagogica per reti piccole — analoga al ruolo di `mm.showBoundBox` nel debug visivo delle rilevazioni — e non come sostituto di strumenti orientati alla produzione, come *TensorBoard* o *torchviz*.

#### 9.5.1.2 Apprendimento per Trasferimento

Addestrare una CNN da zero richiede generalmente una grande quantità di dati etichettati e notevoli risorse computazionali, poiché il processo di addestramento deve regolare tutti i parametri della rete. In molte applicazioni, tuttavia, è disponibile solo un insieme ridotto di dati per il compito di interesse. In questa situazione, l'**apprendimento per trasferimento** (*transfer learning*) riutilizza le rappresentazioni apprese da un modello precedentemente addestrato su un compito sorgente con un grande volume di dati, riducendo il costo di addestramento e la necessità di nuovi campioni.

Nella visione artificiale, questa strategia sfrutta l'organizzazione gerarchica delle CNN. I livelli iniziali apprendono caratteristiche visive di basso livello, come bordi, texture, gradienti di intensità e pattern di colore, che rimangono utili in diversi domini. I livelli più profondi combinano queste informazioni per formare rappresentazioni progressivamente più astratte e specializzate, legate alle classi presenti nel database di addestramento.

Questa sezione indaga in quali condizioni l'apprendimento per trasferimento produce buoni risultati. Il primo esperimento mostra che un estrattore piccolo, addestrato su un dominio ristretto, può condurre a un **trasferimento negativo** (*negative transfer*). Il secondo dimostra perché modelli profondi pre-addestrati su grandi basi di immagini raggiungono elevate prestazioni in nuovi compiti. Infine, il terzo applica questa strategia a un problema di diagnosi fitosanitaria, illustrando uno scenario vicino alle applicazioni reali.

##### Esperimento 1 — Limiti di un Estrattore Piccolo e Specializzato

Il primo esperimento mostra che il trasferimento dell'apprendimento non sempre migliora le prestazioni di un modello. A tale scopo, il set di cifre scritte a mano (`load_digits`) viene suddiviso in due domini disgiunti:

- **Dominio A (origine):** cifre da $0$ a $4$, utilizzate per addestrare una piccola CNN;
- **Dominio B (destinazione):** cifre da $5$ a $9$, rietichettate da $0$ a $4$, formando un nuovo compito con solo $20$ campioni di addestramento.

L'obiettivo consiste nel valutare l'effetto del riutilizzo dell'estrattore di caratteristiche appreso nel Dominio A senza consentirne l'adattamento al Dominio B.

###### Bloco 1: Divisione dei Domini, Scarsità e Visualizzazione dei Campioni

Questo blocco prepara il dataset per l'esperimento. Diversamente dal progetto precedente, che utilizzava tutte le cifre in un unico problema di classificazione, la base è suddivisa in due compiti indipendenti: un compito di origine (Dominio A) e un compito di destinazione (Dominio B).

La [Figura 9.20](#fig-09-transfer-amostras) presenta esempi dei due domini dopo la separazione delle classi, la conversione in *tensor* di **PyTorch** e il pre-processing.

1. **Separazione delle classi:** Le maschere booleane `mask_A` e `mask_B` separano gli esempi di ciascun dominio. Successivamente, il codice rietichetta i target del Dominio B (`y[mask_B] - 5`) nell'intervallo $[0,4]`, consentendo a entrambi i modelli di utilizzare cinque classi di output.

2. **Scarsità dei dati:** Il generatore `np.random.default_rng(0)` seleziona solo $20$ campioni per l'addestramento del Dominio B, circa quattro per classe, simulando uno scenario in cui l'addestramento da zero tende a soffrire di *overfitting*.

3. **Conversione in *tensor*:** La funzione `para_tensor` converte le immagini nel formato $(N,1,8,8)$ e i target in `torch.long`, compatibili con i layer `nn.Conv2d` e la funzione di loss.

4. **Visualizzazione dei campioni:** Il codice utilizza `.squeeze().numpy()` per convertire i *tensor* in array **NumPy**. La [Figura 9.20](#fig-09-transfer-amostras) presenta esempi dei due domini ed evidenzia la rietichettatura applicata ai target del Dominio B.

In [20]:
# 1. Divisione del dataset in due domini disgiunti
classes_A, classes_B = [0, 1, 2, 3, 4], [5, 6, 7, 8, 9]
mask_A, mask_B = np.isin(y, classes_A), np.isin(y, classes_B)

XA, yA = X[mask_A], y[mask_A]
XB, yB = X[mask_B], y[mask_B] - 5  # Reindicizzazione delle etichette nell'intervallo [0, 4]

# Divisione in addestramento e test per entrambi i domini
XA_tr, XA_te, yA_tr, yA_te = train_test_split(
    XA, yA, test_size=0.25, random_state=42, stratify=yA
)
XB_tr, XB_te, yB_tr, yB_te = train_test_split(
    XB, yB, test_size=0.25, random_state=42, stratify=yB
)

# Simulazione di estrema scarsità nel dominio di destinazione: solo 20 campioni di addestramento
rng = np.random.default_rng(0)
idx_poucos = rng.choice(len(XB_tr), size=20, replace=False)
XB_tr_poucos, yB_tr_poucos = XB_tr[idx_poucos], yB_tr[idx_poucos]

# Funzione ausiliaria per la conversione in tensori PyTorch
def para_tensor(Ximg, yarr):
    return torch.tensor(Ximg).unsqueeze(1), torch.tensor(yarr, dtype=torch.long)

XA_tr_t, yA_tr_t = para_tensor(XA_tr, yA_tr)
XA_te_t, yA_te_t = para_tensor(XA_te, yA_te)
XB_tr_t, yB_tr_t = para_tensor(XB_tr_poucos, yB_tr_poucos)
XB_te_t, yB_te_t = para_tensor(XB_te, yB_te)

# Visualizzazione di campioni di entrambi i domini
n_amostras = 5
imgs_A = [img.squeeze().numpy() for img in XA_tr_t[:n_amostras]]
titles_A = [f"A: {label.item()}" for label in yA_tr_t[:n_amostras]]

imgs_B = [img.squeeze().numpy() for img in XB_tr_t[:n_amostras]]
titles_B = [f"B: {label.item()} (orig: {label.item()+5})" for label in yB_tr_t[:n_amostras]]

mm.show(
    imgs_A + imgs_B,
    titles=titles_A + titles_B,
    cols=n_amostras,
    figsize=(12, 4.5)
)

<Figure size 1800x675 with 10 Axes>

**Figura 9.20:** Campioni dei set di addestramento dopo il preprocessamento e l


###### Blocco 2: Architettura Modulare e Routine Generiche

Per consentire il trasferimento dell'apprendimento, l'architettura convoluzionale e il ciclo di addestramento sono stati rifattorizzati rispetto alla classe `CNNDigitos` del progetto precedente.

1. **Modularizzazione dell'Architettura (Differenza rispetto a `CNNDigitos`):**
   * Nel progetto precedente, la classe `CNNDigitos` dichiarava tutti i layer (`conv1`, `conv2`, `pool`, `fc1`, `fc2`) come membri diretti di un'unica classe monolitica.
   * Qui, l'architettura è separata in due componenti: la classe `ExtratorConv` incapsula il blocco spaziale convoluzionale ($2$ convoluzioni $3 \times 3$, $2$ *Max-Pooling* $2 \times 2$ e l'appiattimento a $64$ elementi), mentre la classe `CNNCompleta` istanzia questo estrattore in `self.extrator` e vi aggiunge la "testa" classificatrice (`fc1` e `fc2`).
   * Questa separazione è ciò che consente di copiare lo stato interno dell'estrattore (`state_dict()`) da un modello all'altro in modo isolato.

2. **Adeguamento del Numero di Classi di Uscita:**
   Mentre `CNNDigitos` nel progetto precedente possedeva $10$ *logit* nel layer di uscita (`self.fc2 = nn.Linear(32, 10)`), la classe `CNNCompleta` riceve `n_classes=5` nel costruttore per adattarsi alla suddivisione dei domini $A$ e $B$.

3. **Flessibilizzazione del Ciclo di Addestramento (`treinar`):**
   * Nel progetto precedente, il ciclo di addestramento iterava direttamente sugli attributi globali del modello (`modelo_cnn.parameters()`) e calcolava metriche specifiche in linea.
   * La funzione `treinar` astrae questo processo e introduce il parametro opzionale `parametros`. Se fornito, l'ottimizzatore *Adam* aggiorna **solo** i parametri di questa lista, ignorando i layer i cui gradienti sono stati disattivati. Questa flessibilità è cruciale per eseguire l'addestramento con congelamento parziale della rete.

4. **Isolamento della Valutazione (`calcular_acuracia`):**
   Come nella fase di test del progetto precedente, la funzione pone il modello in `eval()` e utilizza il contesto `torch.no_grad()` per disattivare *autograd*, calcolando l'accuratezza tramite `.argmax(dim=1)`.

In [21]:
# Definizione del blocco convoluzionale riutilizzabile (stessa estrazione del progetto precedente)
class ExtratorConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return x.view(x.size(0), -1)

# Architettura modulare che combina l'estrattore e la testa classificatrice
class CNNCompleta(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.extrator = ExtratorConv()
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, n_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.extrator(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Routine generica di addestramento con ottimizzazione selettiva dei parametri
def treinar(modelo, X_t, y_t, epocas, lr, tam_lote=16, parametros=None):
    # Parametri addestrabili
    params = parametros if parametros is not None else modelo.parameters()  
    otim = optim.Adam(params, lr=lr)               # Ottimizzatore Adam
    crit = nn.CrossEntropyLoss()                   # Funzione di perdita
    n_amostras = X_t.size(0)                       # Numero di campioni
    for _ in range(epocas):                        # Ripeti per epoca
        perm = torch.randperm(n_amostras)          # Mescola i campioni
        for i in range(0, n_amostras, tam_lote):   # Scorri mini-batch
            idx = perm[i:i + tam_lote]             # Indici del lotto
            otim.zero_grad()                       # Azzera i gradienti
            perda = crit(modelo(X_t[idx]), y_t[idx])  # Calcola la perdita
            perda.backward()                       # Retropropagazione
            otim.step()                            # Aggiorna i pesi

# Routine di valutazione
def calcular_acuracia(modelo, X_t, y_t):
    modelo.eval()                                  # Modalità di valutazione
    with torch.no_grad():                          # Senza gradienti
        pred = modelo(X_t).argmax(dim=1)           # Classi predette
    return (pred == y_t).float().mean().item()     # Restituisce l'accuratezza

###### Bloco 3: Pre-addestramento, Trasferimento e Analisi Comparativa

Questo blocco esegue il confronto tra l'addestramento del modello da zero e l'applicazione del trasferimento con il congelamento statico dell'estrattore. Per garantire piena trasparenza all'esperimento, le dimensioni dei set di addestramento e test di entrambi i domini vengono stampate sul terminale.

1. **Quantificazione dei Campioni per Dominio:**
   * **Dominio A (origine, cifre $0$ a $4$):** Dispone di $675$ campioni di addestramento ($75\%$) e $226$ di test ($25\%$), fornendo dati abbondanti affinché il `modello_origine` apprenda l'estrattore convoluzionale fino a raggiungere il $100\%$ di accuratezza.
   * **Dominio B (destinazione, cifre $5$ a $9$):** Possiede $224$ campioni di test in totale, ma il suo set di addestramento è intenzionalmente ridotto da $672$ a soli **$20$ campioni** (`XB_tr_poucos`), creando uno scenario severo di scarsità di dati.

2. **Fase 1: Pre-addestramento nel Dominio A (Origine):**
   Il `modello_origine` viene addestrato da zero sui $675$ campioni delle cifre $0$ a $4$. Durante $40$ epoche, l'estrattore convoluzionale regola i suoi filtri per identificare le caratteristiche distintive di queste prime cinque cifre, raggiungendo il $100\%$ di accuratezza sul set di test ($226$ campioni).

3. **Fase 2: Trasferimento dei Pesi e Congelamento:**  
   * Viene creato il `modello_trasferimento` per risolvere il compito del Dominio B (cifre $5$ a $9$).
   * I pesi appresi nel Dominio A vengono copiati tramite:
  
     - `load_state_dict(modello_origine.estrattore.state_dict())`
  
   * **Congelamento:** Il ciclo `for p in modello_trasferimento.estrattore.parameters(): p.requires_grad = False` disattiva il calcolo dei gradienti nei layer convoluzionali.
   * **Addestramento Selettivo:** La chiamata `treinar(...)` passa strettamente i parametri dei layer densi (`params_cabeca`), regolando la testa di classificazione con soli $20$ campioni di addestramento.

4. **Fase 3: Addestramento da Zero nel Dominio B (Controllo Sperimentale):**
   Il `modello_da_zero` possiede la stessa architettura, ma viene addestrato da zero sugli stessi $20$ campioni del Dominio B, senza alcun riutilizzo di pesi, per le stesse $40$ epoche.

5. **Analisi dei Risultati ([Figura 9.21](#fig-09-transfer-learning)):**
   * **Con Trasferimento Congelato ($72,77\%$):** Riutilizzando l'estrattore addestrato nel Dominio A e congelando i suoi parametri, la rete raggiunge il $72,77\%$ di accuratezza sul test ($224$ campioni) regolando solo i layer densi.
   * **Addestrato da Zero ($76,79\%$):** L'addestramento da zero supera il trasferimento congelato sul set di test del Dominio B.
   * **Causa della Differenza:** Trattandosi di un modello minuscolo (solo $16$ filtri convoluzionali in matrici di $8 \times 8$), l'estrattore addestrato nel Dominio A è diventato **iper-specializzato** nelle forme geometriche delle cifre $0$ a $4$. Congelando rigidamente questi pochi filtri, il modello di destinazione è rimasto limitato a rilevatori inadeguati per le cifre $5$ a $9$. La rete addestrata da zero, anche con soli $20$ campioni, è riuscita ad adattare i suoi $16$ filtri direttamente ai tratti del Dominio B.

In [22]:
# Fixar semente para reprodutibilidade
torch.manual_seed(42)

# Exibição do tamanho dos grupos de treino e teste
print("=== Detalhamento do Tamanho das Bases ===")
print(f"Domínio A (0-4) — Treino: {len(XA_tr_t)} amostras | Teste: {len(XA_te_t)} amostras")
print(f"Domínio B (5-9) — Treino completo: {len(XB_tr)} | Treino reduzido: {len(XB_tr_poucos)}",
      f"| Teste: {len(XB_te_t)} amostras\n")

# 1. Pré-treinamento na tarefa de origem (Domínio A: dígitos 0-4)
modelo_origem = CNNCompleta(n_classes=5)                       # Cria CNN

#######
treinar(modelo_origem, XA_tr_t, yA_tr_t, epocas=40, lr=1e-2)   # Treina modelo
         
acc_A = calcular_acuracia(modelo_origem, XA_te_t, yA_te_t)     # Mede acurácia
                          
print(f"Acurácia no domínio de origem A "                      # Exibe resultado
      f"(dígitos 0-4, {len(XA_te_t)} testes): " f"{acc_A:.4f}")

# 2. Transferência de Aprendizado (Extrator Congelado)
modelo_transferencia = CNNCompleta(n_classes=5)        # Cria CNN
modelo_transferencia.extrator.load_state_dict(         # Copia extrator
    modelo_origem.extrator.state_dict())

for p in modelo_transferencia.extrator.parameters():   # Percorre extrator
    p.requires_grad = False                            # Congela pesos

params_cabeca = list(modelo_transferencia.fc1.parameters())  # FC1
params_cabeca += list(modelo_transferencia.fc2.parameters()) # +FC2

#######
treinar(modelo_transferencia, XB_tr_t, yB_tr_t,              # Treina cabeça
         epocas=40, lr=1e-2, parametros=params_cabeca)

acc_transferencia = calcular_acuracia(modelo_transferencia, XB_te_t, yB_te_t) # Mede acurácia

# 3. Treinamento do Zero no Domínio B
modelo_do_zero = CNNCompleta(n_classes=5)                     # Cria CNN

#######
treinar(modelo_do_zero, XB_tr_t, yB_tr_t, epocas=40, lr=1e-2) # Treina modelo
         
acc_do_zero = calcular_acuracia(modelo_do_zero,  XB_te_t, yB_te_t) # Mede acurácia
                               

print(f"Domínio de destino B (dígitos 5-9), apenas {len(XB_tr_poucos)} ", 
      f"exemplos de treino ({len(XB_te_t)} testes):")
print(f"  Com transferência (extrator congelado): {acc_transferencia:.4f}")
print(f"  Treinando do zero (mesmos dados/épocas): {acc_do_zero:.4f}")

# Visualização comparativa
plt.figure(figsize=(4.5, 4))
plt.bar(["Do zero", "Transferência"], [acc_do_zero, acc_transferencia], 
        color=["#dc2626", "#16a34a"])
plt.ylim(0, max([acc_do_zero, acc_transferencia]) + 0.1)
plt.ylabel("Acurácia no domínio B (teste)")
plt.title(f"Efeito da Transferência ({len(XB_tr_poucos)} exemplos de treino)")

for i, v in enumerate([acc_do_zero, acc_transferencia]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

=== Detalhamento do Tamanho das Bases ===
Domínio A (0-4) — Treino: 675 amostras | Teste: 226 amostras
Domínio B (5-9) — Treino completo: 672 | Treino reduzido: 20 | Teste: 224 amostras



Acurácia no domínio de origem A (dígitos 0-4, 226 testes): 1.0000


Domínio de destino B (dígitos 5-9), apenas 20  exemplos de treino (224 testes):
  Com transferência (extrator congelado): 0.7277
  Treinando do zero (mesmos dados/épocas): 0.7679


<Figure size 1350x1200 with 1 Axes>

**Figura 9.21:** Comparação de acurácia no conjunto de teste do Domínio B (dígitos 5 a 9) sob restrição de dados (20 exemplos de treino): demonstração do impacto do congelamento rígido e da transferência negativa em redes de baixa capacidade.


###### Bloco 4: Visualizzazione del Flusso di Attivazioni con `mm.showNet`

Per confermare che l'estrazione delle caratteristiche riutilizzata preserva le trasformazioni dimensionali studiate nel progetto precedente, si utilizza nuovamente la funzione `mm.showNet` della libreria `morph`. La [Figura 9.22](#fig-09-transfer-shownet) mostra il flusso di attivazioni del `modello_transferimento` durante l'elaborazione di un campione del Dominio B (cifra $7$, reindicizzata per la classe $2$).

1. **Preservazione del Flusso Convoluzionale:** Poiché l'architettura `EstrazioneConv` replica gli stessi strati di convoluzione e *pooling* della `CNNCifre` del progetto precedente, le dimensioni dei *tensor* intermedi rimangono in $(1, 8, 4, 4)$ nel primo blocco.
2. **Ispezione della Testa Adattata:** La differenza rispetto al progetto precedente emerge nello strato di uscita (`fc2`): mentre il modello del progetto precedente proiettava il vettore intermedio in $10$ *logit* (classi da $0$ a $9$), il modello di trasferimento proietta il vettore in $5$ *logit* (classi da $0$ a $4$), catturando le probabilità relative del Dominio B.

In [23]:
# Seleziona il primo campione di test del Dominio B
x_amostra_B = XB_te_t[0:1]  # Tensore di dimensione (1, 1, 8, 8)
classe_verdadeira = yB_te_t[0].item()
classe_original = classe_verdadeira + 5

# Ispezione del flusso di attivazioni nel modello di transfer learning
acts_transfer = mm.showNet(
    modelo_transferencia,
    x_amostra_B,
    titulo="Fluxo de ativações no modelo de transferência (Domínio B)",
    subtitulo=f"Amostra do dígito {classe_original} (rótulo reindexado: {classe_verdadeira})",
)

print("Livelli catturati nel modello di transfer learning:\n", list(acts_transfer.keys()))

<Figure size 3213x578 with 7 Axes>

**Figura 9.22:** Flusso di attivazioni e trasformazioni dimensionali dei tensori nel modello di transfer learning durante l


Livelli catturati nel modello di transfer learning:
 ['extrator.conv1', 'extrator.pool', 'extrator.conv2', 'extrator.pool #2', 'fc1', 'fc2']


###### Analisi dell'Esperimento 1

Il modello addestrato da zero raggiunge un'accuratezza superiore rispetto al modello con transfer learning ed estrattore congelato. Questo risultato caratterizza un caso di **trasferimento negativo** (*negative transfer*) e deriva da tre fattori:

1. **Bassa capacità:** L'estrattore possiede solo $16$ filtri $3 \times 3$, insufficienti per apprendere rappresentazioni generalizzabili.

2. **Specializzazione nel dominio:** L'addestramento con le cifre da $0$ a $4$ produce filtri poco discriminativi per le cifre da $5$ a $9$.

3. **Assenza di adattamento:** Il congelamento impedisce all'estrattore di adattare i propri filtri al nuovo compito.

> ### 📝 Nota
>
> ###### 💡 Provocazione Pedagogica
>
> Questo esperimento utilizza un estrattore di piccole dimensioni addestrato su un dominio ristretto. Il risultato sarebbe diverso se l'estrattore avesse appreso le proprie rappresentazioni su una base con milioni di immagini e grande diversità di oggetti?

##### Esperimento 2 — Quando il Transfer Learning Funziona Davvero (*ResNet-18* Pre-addestrata)

Il secondo esperimento replica la stessa struttura del primo — pochi esempi di addestramento, due classi, confronto tra strategie —, ma sostituisce l'estrattore artigianale di $16$ filtri con la **_ResNet-18_**, un'architettura di $18$ strati pre-addestrata su *ImageNet* ($1,4$ milioni di immagini, $1.000$ categorie), e il *dataset* sintetico di cifre con fotografie reali del **Oxford-IIIT Pet Dataset** (PARKHI, 2012).

Il compito: distinguere due razze canine — **Carlino** e **Boxer** — a partire da sole $15$ fotografie di addestramento per classe.

> ### 💡 Dica
>
> ###### 🐶 Perché questo scenario?
> La sfida qui non è la somiglianza visiva tra le razze — Carlino e Boxer hanno corporature e proporzioni ben distinte —, bensì la scarsità di dati: solo $30$ fotografie reali in totale, senza alcuna immagine sintetica. È il tipo di problema con budget di dati ridotto che, nella pratica, motiva l'uso di reti pre-addestrate: non c'è tempo né risorse per fotografare ed etichettare migliaia di cani prima di addestrare un classificatore da zero.

###### Blocco 1: Caricamento del *Dataset* Reale e Campionamento Sparso

1. **Fonte:** il *Oxford-IIIT Pet Dataset* (PARKHI, 2012) viene caricato tramite `torchvision.datasets.OxfordIIITPet`, che scarica automaticamente le $7.349$ fotografie e le relative etichette di razza alla prima esecuzione.
2. **Filtraggio:** vengono mantenute solo le due razze di interesse (`Pug`, `Boxer`).
3. **Sparsità deliberata:** vengono selezionate solo $15$ fotografie di addestramento per classe ($30$ in totale) — il resto costituisce il set di test, utilizzato esclusivamente per la valutazione.

La [Figura 9.23](#fig-09-pets-amostras) mostra campioni di addestramento di ciascuna razza.

In [24]:
import random

RACAS_ALVO = ["Pug", "Boxer"]
N_TREINO_POR_CLASSE = 15
N_TESTE_POR_CLASSE = 20

# 1. Download del dataset completo (37 razze) — licenza CC BY-SA 4.0
pets_completo = OxfordIIITPet(
    root="dados_pets", split="trainval", target_types="category", download=True
)
nomes_racas = pets_completo.classes
indices_alvo = [nomes_racas.index(r) for r in RACAS_ALVO]

# 2. Filtraggio delle due razze di interesse, separate per classe
por_classe = {idx: [] for idx in indices_alvo}
for img, lbl in pets_completo:
    if lbl in indices_alvo:
        por_classe[lbl].append(img)

# 3. Campionamento: poche immagini di addestramento, più immagini di test
rng = random.Random(42)
imgs_treino, y_treino, imgs_teste, y_teste = [], [], [], []
for classe_idx, idx_original in enumerate(indices_alvo):
    imgs_raca = por_classe[idx_original][:]
    rng.shuffle(imgs_raca)
    imgs_treino += imgs_raca[:N_TREINO_POR_CLASSE]
    y_treino += [classe_idx] * N_TREINO_POR_CLASSE
    imgs_teste += imgs_raca[N_TREINO_POR_CLASSE : N_TREINO_POR_CLASSE + N_TESTE_POR_CLASSE]
    y_teste += [classe_idx] * N_TESTE_POR_CLASSE

print(f"Addestramento: {len(imgs_treino)} immagini | Test: {len(imgs_teste)} immagini")

amostras_pil = imgs_treino[:4] + imgs_treino[N_TREINO_POR_CLASSE:N_TREINO_POR_CLASSE + 4]
amostras_exibicao = [np.array(img.convert("RGB")) for img in amostras_pil]  # PIL -> ndarray
titulos_exibicao = [RACAS_ALVO[0]] * 4 + [RACAS_ALVO[1]] * 4
mm.show(amostras_exibicao, titles=titulos_exibicao, cols=4, figsize=(11, 6))

**Figura 9.23:** Campioni reali di addestramento dell


###### Bloco 2: Tre Strategie sulla Stessa Architettura

Per isolare l'effetto del trasferimento di apprendimento, le tre strategie riutilizzano **esattamente la stessa architettura** (*ResNet-18*), variando solo l'origine dei pesi e quali parametri rimangono addestrabili:

1. **`do_zero`:** pesi casuali (`weights=None`) — equivalente ad addestrare l'architettura della *ResNet-18* interamente da zero, come nel Blocco 2 dell'Esperimento 1.
2. **`congelado`:** pesi pre-addestrati su *ImageNet*, con `requires_grad = False` in tutti i livelli convoluzionali — solo il nuovo livello finale viene addestrato.
3. **`fine_tuning`:** pesi pre-addestrati su *ImageNet* come punto di partenza, ma **senza** congelamento — l'intera rete si adatta al nuovo dominio, con un tasso di apprendimento basso per non distruggere la conoscenza precedente.

In tutti i casi, il livello finale `fc` viene sostituito da `nn.Linear(fc.in_features, 2)`, corrispondente alle due razze di destinazione.

In [25]:
transformacao_resnet = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def prepara_tensores(imgs, labels):
    X = torch.stack([transformacao_resnet(img.convert("RGB")) for img in imgs])
    y = torch.tensor(labels, dtype=torch.long)
    return X, y

X_tr, y_tr = prepara_tensores(imgs_treino, y_treino)
X_te, y_te = prepara_tensores(imgs_teste, y_teste)

def cria_modelo_pets(estrategia):
    pesos = None if estrategia == "do_zero" else models.ResNet18_Weights.DEFAULT
    modelo = models.resnet18(weights=pesos)
    if estrategia == "congelado":
        for p in modelo.parameters():
            p.requires_grad = False
    modelo.fc = nn.Linear(modelo.fc.in_features, len(RACAS_ALVO))
    return modelo

modelo_do_zero = cria_modelo_pets("do_zero")
modelo_congelado = cria_modelo_pets("congelado")
modelo_fine_tuning = cria_modelo_pets("fine_tuning")

###### Blocco 3: Addestramento Comparativo e Analisi dell'Accuratezza

Riutilizzando le funzioni generiche `treinar` e `calcular_acuracia`, definite nel Blocco 2 dell'Esperimento 1, i tre modelli vengono addestrati sullo stesso insieme di $30$ fotografie e valutati sul set di test (immagini mai viste durante l'addestramento):

* Il modello `do_zero` tende a **sovra-adattarsi** rapidamente alle $30$ fotografie di addestramento, senza generalizzare al set di test — $30$ esempi sono drasticamente insufficienti per regolare gli $11$ milioni di parametri della *ResNet-18* da zero.
* Il modello `congelado` dovrebbe già raggiungere un'accuratezza considerevolmente superiore, poiché riutilizza, senza alcuna regolazione, caratteristiche visive generiche (bordi, trame, contorni) apprese su *ImageNet* — solo il nuovo strato lineare deve essere adattato alle $30$ fotografie.
* Il modello `fine_tuning` tende a eguagliare o superare l'estrattore congelato, poiché parte dalla stessa conoscenza preliminare, ma consente inoltre una messa a punto fine dell'intera rete sulle peculiarità visive delle razze.

La [Figura 9.24](#fig-09-pets-comparativo) riassume i tre risultati.

In [26]:
torch.manual_seed(42)

configuracoes = [
    ("Do zero",              modelo_do_zero,      None, 2e-3),
    ("Extrator congelado",   modelo_congelado,    "fc", 1e-3),
    ("Fine-tuning completo", modelo_fine_tuning,  None, 1e-4),
]

resultados_pets = {}
for nome, modelo, alvo_params, taxa in configuracoes:
    parametros = modelo.fc.parameters() if alvo_params == "fc" else None
    treinar(modelo, X_tr, y_tr, epocas=15, lr=taxa, tam_lote=8, parametros=parametros)
    resultados_pets[nome] = calcular_acuracia(modelo, X_te, y_te)
    print(f"{nome}: {resultados_pets[nome]*100:.1f}%")

plt.figure(figsize=(5.5, 4))
cores = ["#dc2626", "#f59e0b", "#16a34a"]
plt.bar(resultados_pets.keys(), resultados_pets.values(), color=cores)
plt.ylim(0, 1.05)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Chute aleatório (50%)")
plt.ylabel("Acurácia no teste")
plt.title("Pug vs. Boxer — 15 fotos de treino/classe")
for i, v in enumerate(resultados_pets.values()):
    plt.text(i, v + 0.02, f"{v*100:.1f}%", ha="center")
plt.xticks(rotation=10)
plt.legend()
plt.tight_layout()
plt.show()

**Figura 9.24:** Confronto dell


###### Bloco 4: Ispezione Qualitativa delle Previsioni

Come nell'Esperimento 1 e nella sezione successiva sulla diagnostica fogliare, è istruttivo osservare singolarmente alcune previsioni del modello migliore (tipicamente `fine_tuning` o `congelato`) su fotografie reali di test, confrontando l'etichetta prevista con la razza effettiva.

In [27]:
melhor_modelo = modelo_fine_tuning  # oppure modello_congelato, in base al risultato del Blocco 3
melhor_modelo.eval()

idx_amostras = list(range(4)) + list(range(N_TESTE_POR_CLASSE, N_TESTE_POR_CLASSE + 4))

imgs_pred, titulos_pred = [], []
with torch.no_grad():
    for idx in idx_amostras:
        entrada = X_te[idx].unsqueeze(0)
        pred_idx = melhor_modelo(entrada).argmax(dim=1).item()
        real_idx = y_te[idx].item()
        marcador = "✓" if pred_idx == real_idx else "✗"
        imgs_pred.append(np.array(imgs_teste[idx].convert("RGB")))  # PIL -> ndarray
        titulos_pred.append(f"{marcador} previsto: {RACAS_ALVO[pred_idx]}\n"
                             f"real: {RACAS_ALVO[real_idx]}")

mm.show(imgs_pred, titles=titulos_pred, cols=4, figsize=(12, 7))

**Figura 9.25:** Previsioni del modello con estrattore pre-addestrato (ResNet-18) su fotografie reali di test: etichetta prevista vs. razza reale, quattro campioni per ciascuna classe.


In [28]:
 
# Pulizia esplicita dei dati scaricati e liberazione della memoria
if FLAG_LIMPAR_DADOS:
    if os.path.exists('./dados_pets'):
        shutil.rmtree('./dados_pets')
        print('🧹 Directory dati temporanea ./dati_pets rimossa con successo.')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

##### Esperimento 3 — Diagnostico Fitopatologico con Transferimento di Apprendimento

L'esperimento precedente ha mostrato che una *ResNet-18* preaddestrata su **ImageNet** può adattarsi a un nuovo compito utilizzando poche campioni. Ora, la stessa strategia viene applicata a un problema di diagnostica fitopatologica. La *ResNet-18* deve classificare immagini di foglie in tre categorie: `["foglia_sana", "foglia_malata", "sintomo_sconosciuto"]`.

- **`foglia_sana`:** foglia senza lesioni visibili.
- **`foglia_malata`:** foglia con macchie scure che simulano una malattia fungina.
- **`sintomo_sconosciuto`:** foglia con clorosi giallastra, che rappresenta un pattern diverso dalla malattia nota.

> ### 💡 Dica
>
> ###### 🌱 Perché questo scenario?
>
> La diagnostica fitopatologica costituisce un'importante applicazione della visione artificiale nell'agricoltura di precisione. Un modello preaddestrato su **ImageNet** può riutilizzare caratteristiche come bordi, texture e pattern di colore per apprendere questo nuovo compito con poche immagini.

###### Blocco 1: Generazione del *Dataset* Sintetico

Questo blocco genera un insieme sintetico con **30** immagini per classe per l'addestramento e **8** per la validazione, per un totale rispettivamente di **90** e **24** immagini.

1. **Generazione della foglia:** La funzione `desenha_folha_base` crea il contorno della foglia, variando dimensioni, orientamento e tonalità di verde.

2. **Simulazione della malattia:** La funzione `aplica_manchas_doenca` aggiunge macchie scure irregolari che simulano lesioni fungine.

3. **Simulazione di un altro sintomo:** La funzione `aplica_sintoma_desconhecido` aggiunge regioni giallastre che rappresentano un pattern distinto dalla malattia nota.

4. **Visualizzazione dei campioni:** La [Figura 9.26](#fig-09-folhas-amostras) presenta esempi delle tre categorie dell'insieme sintetico.

In [29]:
CLASSES_FOLHA = ["folha_saudavel", "folha_doente", "sintoma_desconhecido"]


def desenha_folha_base(tam_img, rng):
    '''Disegna il contorno ovale di una foglia verde con nervatura centrale,
    con piccole variazioni di tonalità, dimensione e orientamento tra i campioni.'''
    img = np.full((tam_img, tam_img, 3), 245, dtype=np.uint8)  # sfondo chiaro
    cx, cy = tam_img // 2, tam_img // 2
    eixo_a = rng.randint(int(tam_img * 0.30), int(tam_img * 0.38))
    eixo_b = rng.randint(int(tam_img * 0.20), int(tam_img * 0.26))
    angulo = rng.uniform(-15, 15)
    verde = (rng.randint(40, 70), rng.randint(120, 160), rng.randint(40, 70))
    cv2.ellipse(img, (cx, cy), (eixo_a, eixo_b), angulo, 0, 360, verde, -1, cv2.LINE_AA)
    ang_rad = np.deg2rad(angulo)
    dx, dy = np.cos(ang_rad), np.sin(ang_rad)
    p1 = (int(cx - eixo_a * dx), int(cy - eixo_a * dy))
    p2 = (int(cx + eixo_a * dx), int(cy + eixo_a * dy))
    cv2.line(img, p1, p2, (25, 90, 25), 2, cv2.LINE_AA)  # nervatura centrale
    return img, (cx, cy, eixo_a, eixo_b, angulo)


def aplica_manchas_doenca(img, centro_folha, rng, n_manchas=(4, 8)):
    '''Simula lesioni fogliari: macchie scure dai bordi irregolari
    (pattern tipico di malattie fungine).'''
    cx, cy, eixo_a, eixo_b, _ = centro_folha
    for _ in range(rng.randint(*n_manchas)):
        raio = rng.randint(1, 3)
        px = cx + rng.randint(-int(eixo_a * 0.7), int(eixo_a * 0.7))
        py = cy + rng.randint(-int(eixo_b * 0.7), int(eixo_b * 0.7))
        cor_mancha = (rng.randint(50, 90), rng.randint(25, 45), rng.randint(10, 25))
        cv2.circle(img, (px, py), raio, cor_mancha, -1, cv2.LINE_AA)
        cv2.circle(img, (px, py), raio + 1, (120, 85, 30), 1, cv2.LINE_AA)  # alone
    return img


def aplica_sintoma_desconhecido(img, centro_folha, rng):
    '''Simula un pattern distinto (marmorizzato giallastro/clorosi), diverso
    dalle macchie scure della malattia nota.'''
    cx, cy, eixo_a, eixo_b, _ = centro_folha
    for _ in range(rng.randint(3, 5)):
        eixo_m = (rng.randint(1, 3), rng.randint(2, 3))
        px = cx + rng.randint(-int(eixo_a * 0.6), int(eixo_a * 0.6))
        py = cy + rng.randint(-int(eixo_b * 0.6), int(eixo_b * 0.6))
        cor_clorose = (rng.randint(200, 235), rng.randint(195, 225), rng.randint(50, 90))
        ang_m = rng.uniform(0, 180)
        cv2.ellipse(img, (px, py), eixo_m, ang_m, 0, 360, cor_clorose, -1, cv2.LINE_AA)
    return img


def aplica_ruido_sal_pimenta(img, prop_ruido=0.02, rng=None):
    '''Applica rumore sale (punti bianchi) e pepe (punti neri) casuali.
    prop_rumore: frazione di pixel modificati (es: 0.02 = 2% dei pixel).'''
    if prop_ruido <= 0:
        return img
    
    img_ruido = img.copy()
    num_pixels = int(prop_ruido * img.shape[0] * img.shape[1])
    n_sal = num_pixels // 2
    n_pimenta = num_pixels - n_sal

    # Applica Sale (Bianco - [255, 255, 255])
    for _ in range(n_sal):
        y = rng.randint(0, img.shape[0] - 1)
        x = rng.randint(0, img.shape[1] - 1)
        img_ruido[y, x] = [255, 255, 255]

    # Applica Pepe (Nero - [0, 0, 0])
    for _ in range(n_pimenta):
        y = rng.randint(0, img.shape[0] - 1)
        x = rng.randint(0, img.shape[1] - 1)
        img_ruido[y, x] = [0, 0, 0]

    return img_ruido


def gera_folha(classe_idx, tam_img=128, rng=None, prop_ruido=0.02):
    rng = rng or random.Random()
    img, geometria = desenha_folha_base(tam_img, rng)
    nome = CLASSES_FOLHA[classe_idx]
    
    if nome == "folha_doente":
        img = aplica_manchas_doenca(img, geometria, rng)
    elif nome == "sintoma_desconhecido":
        img = aplica_sintoma_desconhecido(img, geometria, rng)
        
    ruido_exp = rng.randint(-3, 3)  # leggera variazione di esposizione
    img = np.clip(img.astype(np.int16) + ruido_exp, 0, 255).astype(np.uint8)
    
    # Applicazione del rumore Sale e Pepe
    img = aplica_ruido_sal_pimenta(img, prop_ruido=prop_ruido, rng=rng)
    
    return img


def gera_conjunto(n_por_classe, tam_img=128, seed=0, prop_ruido=0.02):
    rng = random.Random(seed)
    imgs, labels = [], []
    for classe_idx in range(len(CLASSES_FOLHA)):
        for _ in range(n_por_classe):
          imgs.append(gera_folha(classe_idx, tam_img=tam_img, rng=rng, prop_ruido=prop_ruido))
          labels.append(classe_idx)
    return imgs, labels


N_POR_CLASSE_TREINO, N_POR_CLASSE_VAL = 30, 8

imgs_treino, labels_treino = gera_conjunto(n_por_classe=N_POR_CLASSE_TREINO, seed=42, 
                                           prop_ruido=0.02)
imgs_val, labels_val = gera_conjunto(n_por_classe=N_POR_CLASSE_VAL, seed=123, prop_ruido=0.02)

print(f"Addestramento: {len(imgs_treino)} immagini ({N_POR_CLASSE_TREINO} per classe) | "
      f"Validazione: {len(imgs_val)} immagini ({N_POR_CLASSE_VAL} per classe)")

# Visualizzazione di 2 campioni per ogni classe (6 immagini in totale)
amostras_exibir, titulos_exibir = [], []
for classe_idx, nome in enumerate(CLASSES_FOLHA):
    for k in range(2):
        idx = classe_idx * N_POR_CLASSE_TREINO + k
        amostras_exibir.append(imgs_treino[idx])
        titulos_exibir.append(nome)

mm.show(amostras_exibir, titles=titulos_exibir, cols=3, figsize=(10, 7))

Addestramento: 90 immagini (30 per classe) | Validazione: 24 immagini (8 per classe)


<Figure size 1500x1050 with 6 Axes>

**Figura 9.26:** Campioni sintetici del *dataset* di diagnosi fogliare: foglia sana, foglia malata (macchie scure) e sintomo sconosciuto (clorosi giallastra) con rumore sale e pepe.


> ### 📝 Nota
>
> ###### 🧠 Trappola Comune
>
> Il trasferimento dell’apprendimento richiede che ogni classe presenti pattern visivi distinti. Ripetere la stessa immagine con etichette diverse impedisce al livello classificatore di apprendere una frontiera di decisione, poiché l’estrattore genera praticamente le stesse caratteristiche per tutti i campioni.
>
> In questo esperimento, ogni immagine viene generata in modo indipendente, con pattern visivi compatibili con la propria classe (foglia sana, lesioni fungine o clorosi), fornendo informazioni sufficienti per l’addestramento del livello classificatore.

###### Bloco 2: Preparazione dei Tensori e Adattamento dell'Architettura

Modelli come la *ResNet-18* richiedono immagini a colori di $224 \times 224$ *pixel* normalizzate secondo le statistiche della *ImageNet* ($\mu = [0,485; 0,456; 0,406]$ e $\sigma = [0,229; 0,224; 0,225]$).

1. **Trasformazione di Input (`transforms.Compose`):** si applicano il ridimensionamento e la normalizzazione standard a ciascuna immagine del *dataset* sintetico, producendo i tensori `X_treino`/`X_val` e le etichette `y_treino`/`y_val` — ogni esempio è un'immagine genuinamente distinta, associata all'etichetta corretta della propria classe.
2. **Congelamento dell'Estrattore:** il ciclo `for p in modelo_resnet.parameters(): p.requires_grad = False` disattiva i gradienti nei layer convoluzionali pre-addestrati.
3. **Nuovo Layer Finale:** il layer `modelo_resnet.fc` viene sostituito da una nuova istanza `nn.Linear(modelo_resnet.fc.in_features, n_classes_destino)`, appena inizializzata e con gradienti attivi per impostazione predefinita. Il numero di caratteristiche di input viene ottenuto dinamicamente dal layer originale stesso (`in_features`, pari a $512$ nella ResNet-18), invece di essere fissato manualmente nel codice — pratica raccomandata, poiché rende il frammento riutilizzabile per altre varianti dell'architettura senza modifiche.

In [30]:
# 1. Pipeline di trasformazioni previste dalla ResNet
transformacao_resnet = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def prepara_tensores(imgs, labels):
    tensores = [transformacao_resnet(T.functional.to_tensor(img)) for img in imgs]
    X = torch.stack(tensores)
    y = torch.tensor(labels, dtype=torch.long)
    return X, y


# 2. Conversione del dataset sintetico (Blocco 1) in tensori normalizzati
X_treino, y_treino = prepara_tensores(imgs_treino, labels_treino)
X_val, y_val = prepara_tensores(imgs_val, labels_val)

loader_treino = DataLoader(TensorDataset(X_treino, y_treino), batch_size=16, shuffle=True)
loader_val = DataLoader(TensorDataset(X_val, y_val), batch_size=16, shuffle=False)

# 3. Caricamento della ResNet-18 pre-addestrata e congelamento dell'estrattore
n_classes_destino = len(CLASSES_FOLHA)
modelo_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for parametro in modelo_resnet.parameters():
    parametro.requires_grad = False

# 4. Sostituzione del layer finale per le 3 nuove classi di destinazione
modelo_resnet.fc = nn.Linear(modelo_resnet.fc.in_features, n_classes_destino)

print(f"Nuovo layer finale: {modelo_resnet.fc}")

###### Bloco 3: Ciclo di Addestramento e Valutazione

Con l'estrattore congelato e il nuovo strato di uscita correttamente collegato, si esegue il *fine-tuning* della nuova testa di classificazione.

1. **Ottimizzazione Mirata:** l'ottimizzatore *Adam* riceve strettamente `modelo_resnet.fc.parameters()`, aggiornando solo il nuovo strato di uscita — il resto della rete rimane congelato, come definito nel Blocco 2.
2. **Esecuzione del Ciclo:** a ogni epoca, il modello itera sui lotti di addestramento, calcola la perdita tramite Entropia Incrociata e regola i pesi dello strato finale; successivamente, si valuta l'accuratezza sul set di validazione (immagini mai viste durante l'addestramento).
3. **Curve di Addestramento:** la [Figura 9.27](#fig-09-resnet-treino) monitora l'evoluzione della perdita di addestramento e dell'accuratezza di validazione lungo le epoche — poiché le tre classi sono visivamente distinte tra loro, ci si aspetta una convergenza genuina, ben al di sopra della soglia del $33\%$ corrispondente a una scelta casuale tra $3$ classi.

In [31]:
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo_resnet = modelo_resnet.to(dispositivo)

criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_resnet.fc.parameters(), lr=1e-3)

historico_perda, historico_acc = [], []
epocas = 10

for epoca in range(epocas):
    modelo_resnet.train()
    perda_acumulada, n_batches = 0.0, 0

    for X_batch, y_batch in loader_treino:
        X_batch, y_batch = X_batch.to(dispositivo), y_batch.to(dispositivo)

        otimizador.zero_grad()
        saidas = modelo_resnet(X_batch)
        perda = criterio(saidas, y_batch)
        perda.backward()
        otimizador.step()

        perda_acumulada += perda.item()
        n_batches += 1
    historico_perda.append(perda_acumulada / n_batches)

    modelo_resnet.eval()
    acertos, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader_val:
            X_batch, y_batch = X_batch.to(dispositivo), y_batch.to(dispositivo)
            predicoes = modelo_resnet(X_batch).argmax(dim=1)
            acertos += (predicoes == y_batch).sum().item()
            total += y_batch.size(0)
    acc = acertos / total
    historico_acc.append(acc)

    print(f"Epoca {epoca+1}/{epocas} — perdita: {historico_perda[-1]:.4f} — ",
          f" accuratezza_val: {acc*100:.1f}%")

f=mm.showTrainCurves(
    historico_perda, historico_acc,
    titulo="Fine-Tuning da ResNet-18 — Diagnóstico Foliar",
    subtitulo="Apenas a nova camada linear (fc) é treinada; o extrator permanece congelado",
)

**Figura 9.27:** Curve di training del fine-tuning della ResNet-18 sul *dataset* sintetico di diagnosi fogliare: perdita di training e accuratezza di validazione lungo le epoche.


###### Blocco 4: Ispezione Qualitativa delle Previsioni

Oltre alla curva di accuratezza aggregata, è istruttivo osservare individualmente alcune previsioni del modello sul set di validazione, confrontando l'etichetta prevista con quella reale. La [Figura 9.28](#fig-09-resnet-predicoes) mostra due campioni per ciascuna classe.

In [32]:
modelo_resnet.eval()

# Due campioni di ciascuna classe nell'insieme di validazione
idx_amostras = [0, N_POR_CLASSE_VAL, 2 * N_POR_CLASSE_VAL,
                1, N_POR_CLASSE_VAL + 1, 2 * N_POR_CLASSE_VAL + 1]

imgs_pred, titulos_pred = [], []
with torch.no_grad():
    for idx in idx_amostras:
        entrada = X_val[idx].unsqueeze(0).to(dispositivo)
        pred_idx = modelo_resnet(entrada).argmax(dim=1).item()
        real_idx = y_val[idx].item()
        marcador = "✓" if pred_idx == real_idx else "✗"
        imgs_pred.append(imgs_val[idx])
        titulos_pred.append(f"{marcador} previsto: {CLASSES_FOLHA[pred_idx]}\n"+
                            f"real: {CLASSES_FOLHA[real_idx]}")

mm.show(imgs_pred, titles=titulos_pred, cols=3, figsize=(10, 7))

**Figura 9.28:** Previsioni della ResNet-18 messa a punto su campioni di validazione: etichetta prevista vs. etichetta reale, con due campioni per classe.


###### Analisi dell'Esperimento 3

La *ResNet-18* raggiunge un'elevata accuratezza anche utilizzando un insieme ridotto di immagini sintetiche. Questo risultato mostra che le rappresentazioni apprese su **ImageNet** rimangono utili in un dominio completamente diverso, richiedendo solo l'adattamento del livello classificatore.

L'esperimento illustra inoltre una situazione comune nelle applicazioni reali, in cui la disponibilità di dati etichettati è limitata. In questi scenari, il transfer learning riduce i tempi di addestramento e consente di ottenere modelli con buone prestazioni anche senza addestrare l'intera rete.

> ### 📝 Nota
>
> ###### 🧠 Sintesi Comparativa — Quando Funziona il Transfer Learning?
>
> I tre esperimenti mostrano che il transfer learning dipende dalla capacità di generalizzazione dell'estrattore di caratteristiche.
>
> - **Esperimento 1:** un estrattore di piccole dimensioni, addestrato su un dominio ristretto, apprende rappresentazioni poco generalizzabili e può produrre **transfer negativo**.
>
> - **Esperimento 2:** una *ResNet-18* pre-addestrata su **ImageNet** trasferisce rappresentazioni generali a un compito di classificazione delle razze di cani e gatti, raggiungendo un'elevata accuratezza con poche immagini.
>
> - **Esperimento 3:** la stessa strategia adatta il modello a un problema di diagnosi fitosanitaria, dimostrando che un unico estrattore può servire come base per diversi domini applicativi.

##### Confronto tra Approcci di Trasferimento

La [Tabela 9.2](#tbl-comparativo-transferencia) riassume i risultati ottenuti nei tre esperimenti.

<a id="tbl-comparativo-transferencia"></a>

**Tabela 9.2:** Confronto tra i tre scenari di trasferimento dell'apprendimento presentati in questa sezione.

| Aspetto | Esperimento 1 | Esperimento 2 | Esperimento 3 |
| --- | --- | --- | --- |
| **Estrattore** | Piccola CNN | *ResNet-18* | *ResNet-18* |
| **Addestramento dell'estrattore** | Cifre ($0$–$4$) | **ImageNet** | **ImageNet** |
| **Capacità di generalizzazione** | Bassa | Alta | Alta |
| **Nuovo compito** | Cifre ($5$–$9$) | Razze di cani e gatti | Diagnosi fitosanitaria |
| **Risultato** | Trasferimento negativo | Trasferimento positivo | Trasferimento positivo |


### 9.5.2 Rilevamento di Oggetti

Gli esperimenti precedenti hanno mostrato come il transfer learning adatti modelli pre-addestrati per compiti di classificazione delle immagini. Lo stesso principio è alla base anche delle architetture di rilevamento degli oggetti, in cui un estrattore di caratteristiche pre-addestrato fornisce rappresentazioni visive generali, mentre moduli specializzati localizzano e classificano gli oggetti nell'immagine.

Le sezioni successive presentano la **Faster R-CNN** come esempio di rilevatore pre-addestrato utilizzato direttamente per l'inferenza e, successivamente, un esperimento completo di fine-tuning con l'architettura **YOLO**.

#### 9.5.2.1 Faster R-CNN: Rilevatore Pre-addestrato

La rilevazione degli oggetti estende l'uso di modelli pre-addestrati a un compito più complesso della classificazione. La **Faster R-CNN** utilizza una CNN pre-addestrata, come la *ResNet-50*, come **estrattore di caratteristiche** (*backbone*) e aggiunge moduli specializzati per localizzare e classificare gli oggetti.

Riguardo a questo estrattore, l'architettura incorpora due "teste" principali:

- ***Region Proposal Network* (RPN):** propone regioni dell'immagine con alta probabilità di contenere oggetti.
- **Testa classificatrice:** affina queste regioni, assegna una classe a ciascun oggetto e regola le sue scatole delimitatrici.

Il codice seguente utilizza una Faster R-CNN con pesi pre-addestrati su **COCO** per rilevare oggetti in un'immagine, producendo le loro classi, coordinate e punteggi di confidenza.

> ### 📝 Nota
>
> ##### 🔍 Dove si trova qui il transfer learning?
>
> A differenza degli esperimenti precedenti, questo esempio **non esegue il fine-tuning**. Il modello esegue solo l'inferenza (`eval()`), riutilizzando direttamente i pesi del *backbone*, della RPN e della testa classificatrice addestrati su **COCO**.
>
> L'adattamento a un nuovo dominio richiederebbe di sostituire il layer `box_predictor` con una nuova testa di classificazione, compatibile con le classi dell'applicazione, e di addestrarla su un insieme di immagini annotate. Questa procedura segue lo stesso principio presentato nella sezione sul transfer learning e costituisce il flusso usuale per applicazioni specifiche, come la rilevazione di parassiti, difetti di fabbricazione o veicoli.

1. **Categorie COCO:** Il codice recupera i nomi delle classi dalle meta-informazioni dei pesi (`FasterRCNN_ResNet50_FPN_Weights.DEFAULT.meta["categories"]`). Sebbene questo elenco contenga $91$ voci per ragioni storiche del formato di annotazione di COCO, solo $80$ corrispondono a categorie di oggetti.

2. **Inferenza:** L'immagine caricata da `mm.read()` viene convertita in *tensor* e processata dal modello in modalità di valutazione (`eval()`). Il codice mantiene solo le rilevazioni con confidenza superiore all'$80\%$.

3. **Annotazione dell'immagine:** Per ogni oggetto rilevato, il codice disegna la scatola delimitatrice (`cv2.rectangle`) e scrive la classe predetta e la sua confidenza (`cv2.putText`).

4. **Visualizzazione:** La [Figura 9.29](#fig-09-deteccao-pretreinada) presenta l'immagine annotata con le rilevazioni effettuate dal modello.

In [33]:
# 1. Caricamento dell'immagine e dei nomi delle categorie di COCO
url_imagem = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
url_imagem = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = mm.read(url_imagem)

pesos_coco = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
categorias_coco = pesos_coco.meta["categories"]  # Mappatura indice -> nome della classe

# 2. Caricamento del modello Faster R-CNN pre-addestrato
modelo_detection = fasterrcnn_resnet50_fpn(weights=pesos_coco).eval()

# 3. Esecuzione dell'inferenza senza calcolo dei gradienti
with torch.no_grad():
    predicao = modelo_detection([to_tensor(img)])[0]

# 4. Filtraggio delle rilevazioni con confidenza superiore all'80%
limiar_confianca = 0.8
mascara_confianca = predicao["scores"] >= limiar_confianca

caixas_filtradas = predicao["boxes"][mascara_confianca].numpy()
scores_filtrados = predicao["scores"][mascara_confianca].numpy()
labels_filtrados = predicao["labels"][mascara_confianca].numpy()

img_com_caixas = img.copy()

# 5. Disegno delle bounding box e delle etichette di classe
for box, score, label_idx in zip(caixas_filtradas, scores_filtrados, labels_filtrados):
    x1, y1, x2, y2 = box.astype(int)
    nome_classe = categorias_coco[label_idx]
    texto_rotulo = f"{nome_classe}: {score:.2f}"

    # Disegna il rettangolo rosso (RGB: 255, 0, 0) con spessore di 3 pixel
    cv2.rectangle(img_com_caixas, (x1, y1), (x2, y2), (255, 0, 0), 3)

    # Scrive la classe e la confidenza sopra la bounding box
    cv2.putText(
        img_com_caixas,
        texto_rotulo,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 0, 0),
        2,
        cv2.LINE_AA,
    )

# 6. Visualizzazione grafica dell'immagine risultante
mm.show(img_com_caixas, title="Faster R-CNN (COCO) — Rilevamento con Classe e Confidenza")

**Figura 9.29:** Risultato dell


#### 9.5.2.2 Rilevamento di Oggetti e Trasferimento dell'Apprendimento con YOLO

Le sezioni precedenti hanno applicato il trasferimento dell'apprendimento a problemi di **classificazione delle immagini**, in cui il modello associa un'unica etichetta all'intera immagine. In questa sezione, lo stesso principio viene esteso al **rilevamento di oggetti**, un compito che richiede di identificare simultaneamente **cosa** è presente nell'immagine e **dove** si trova ciascun oggetto.

La sottosezione precedente ha presentato la **Faster R-CNN** come esempio di rilevatore pre-addestrato utilizzato direttamente per l'inferenza, senza alcun adattamento al nuovo dominio. In questo esperimento, il modello viene sottoposto a una fase di **ottimizzazione fine** (*fine-tuning*): si parte da un'architettura **YOLO** (*You Only Look Once*) pre-addestrata sul dataset **COCO** e si adatta la rete per rilevare e classificare oggetti di un nuovo dominio.

A differenza della Faster R-CNN, che esegue il rilevamento in due fasi, la famiglia **YOLO** adotta un'architettura a stadio singolo (*single-stage detector*), stimando, in un'unica propagazione attraverso la rete, le scatole delimitatrici (*bounding boxes*), la confidenza di ciascun rilevamento e la classe corrispondente. Questa strategia riduce il costo computazionale e rende possibili applicazioni in tempo reale.

Come esempio, l'esperimento utilizza un insieme sintetico di forme geometriche (triangoli, quadrati, stelle, tra le altre), con variazioni di colore, dimensione, rotazione e degrado dovuto a rumore di tipo sale e pepe.

##### Bloco 1: Generazione del *Dataset* Sintetico

Il blocco seguente genera un insieme sintetico per l'addestramento e la valutazione del rivelatore. Ogni immagine contiene da uno a tre oggetti appartenenti a una delle nove classi:

```python
CLASSES = [
    'Triangle', 'Square', 'Pentagon', 'Hexagon',
    'Heptagon', 'Circle', 'Ellipse', 'Star', 'Cross'
]
```

1. **Generazione delle forme:** Le funzioni `poligono_regular`, `poligono_estrela` e `poligono_cruz` costruiscono le coordinate degli oggetti. La funzione `desenha_objeto` disegna ogni forma con posizione, dimensione, orientamento e colore casuali e calcola la sua *bounding box*.

2. **Annotazione nel formato YOLO:** La funzione `gera_imagem_ruidosa` genera da uno a tre oggetti per immagine e converte ogni *bounding box* nel formato YOLO, rappresentato dalla classe e dalle coordinate normalizzate del centro, larghezza e altezza.

3. **Degradazione dell'immagine:** La funzione `adiciona_ruido_sal_pimenta` aggiunge rumore impulsivo, simulando imperfezioni di acquisizione.

4. **Visualizzazione dei campioni:** La [Figura 9.30](#fig-09-yolo-amostras-iniciais) presenta esempi dell'insieme sintetico con le *bounding boxes* sovrapposte tramite la funzione `mm.showBoundBox()`.

In [34]:
CLASSES = [
    'Triangle', 'Square', 'Pentagon', 'Hexagon',
    'Heptagon', 'Circle', 'Ellipse', 'Star', 'Cross'
]
N_LADOS = {'Triangle': 3, 'Square': 4, 'Pentagon': 5, 'Hexagon': 6, 'Heptagon': 7}

def poligono_regular(cx, cy, r, n_lados, rot_graus):
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + 2 * np.pi * np.arange(n_lados) / n_lados
    return np.stack([cx + r * np.cos(angs), cy + r * np.sin(angs)], axis=1)

def poligono_estrela(cx, cy, r_externo, rot_graus, n_pontas=5):
    r_interno = r_externo * 0.45
    ang0 = np.deg2rad(rot_graus - 90)
    angs = ang0 + np.pi * np.arange(2 * n_pontas) / n_pontas
    raios = np.where(np.arange(2 * n_pontas) % 2 == 0, r_externo, r_interno)
    return np.stack([cx + raios * np.cos(angs), cy + raios * np.sin(angs)], axis=1)

def poligono_cruz(cx, cy, r, rot_graus, espessura_rel=0.35):
    w = r * espessura_rel
    base = np.array([
        (-w, -r), (w, -r), (w, -w), (r, -w), (r, w), (w, w),
        (w, r), (-w, r), (-w, w), (-r, w), (-r, -w), (-w, -w),
    ])
    theta = np.deg2rad(rot_graus)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return base @ R.T + np.array([cx, cy])

def desenha_objeto(img, classe_idx, cx, cy, tamanho, rotacao, cor):
    nome = CLASSES[classe_idx]
    if nome in N_LADOS:
        pts = poligono_regular(cx, cy, tamanho, N_LADOS[nome], rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Star':
        pts = poligono_estrela(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Cross':
        pts = poligono_cruz(cx, cy, tamanho, rotacao)
        cv2.fillPoly(img, [pts.astype(np.int32)], cor)
        xs, ys = pts[:, 0], pts[:, 1]
    elif nome == 'Circle':
        cv2.circle(img, (int(cx), int(cy)), int(tamanho), cor, -1)
        xs, ys = np.array([cx - tamanho, cx + tamanho]), np.array([cy - tamanho, cy + tamanho])
    else:  # Ellisse
        eixo = (int(tamanho), int(tamanho * 0.6))
        cv2.ellipse(img, (int(cx), int(cy)), eixo, rotacao, 0, 360, cor, -1)
        ang = np.deg2rad(rotacao)
        dx = np.hypot(eixo[0] * np.cos(ang), eixo[1] * np.sin(ang))
        dy = np.hypot(eixo[0] * np.sin(ang), eixo[1] * np.cos(ang))
        xs, ys = np.array([cx - dx, cx + dx]), np.array([cy - dy, cy + dy])
    return xs.min(), ys.min(), xs.max(), ys.max()

def adiciona_ruido_sal_pimenta(img, quantidade=0.05):
    img_ruidosa = img.copy()
    h, w, c = img_ruidosa.shape
    num_ruido = int(quantidade * h * w)
    
    # Sale (255, 255, 255)
    coords_sal = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_sal[0], coords_sal[1]] = [255, 255, 255]
    
    # Pepe (0, 0, 0)
    coords_pimenta = [np.random.randint(0, i - 1, num_ruido) for i in (h, w)]
    img_ruidosa[coords_pimenta[0], coords_pimenta[1]] = [0, 0, 0]
    
    return img_ruidosa

def gera_imagem_ruidosa(tam_img=160, n_objetos=(1, 3), taxa_ruido=0.01, rng=None):
    rng = rng or random.Random()
    img_limpa = np.full((tam_img, tam_img, 3), 255, dtype=np.uint8)
    anotacoes = []
    
    for _ in range(rng.randint(*n_objetos)):
        classe_idx = rng.randrange(len(CLASSES))
        tamanho = rng.randint(tam_img // 10, tam_img // 5)
        cx = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        cy = rng.randint(tamanho + 2, tam_img - tamanho - 2)
        rotacao = rng.uniform(0, 360)
        cor = tuple(rng.sample(range(30, 226), 3))
        
        x0, y0, x1, y1 = desenha_objeto(img_limpa, classe_idx, cx, cy, tamanho, rotacao, cor)
        x0, y0 = max(x0, 0), max(y0, 0)
        x1, y1 = min(x1, tam_img), min(y1, tam_img)
        
        # Formato YOLO: (classe, centro_x, centro_y, larghezza, altezza) normalizzati
        xc, yc = (x0 + x1) / 2 / tam_img, (y0 + y1) / 2 / tam_img
        w, h = (x1 - x0) / tam_img, (y1 - y0) / tam_img
        anotacoes.append((classe_idx, xc, yc, w, h)) 
        
    img_ruidosa = adiciona_ruido_sal_pimenta(img_limpa, quantidade=taxa_ruido)
    return img_ruidosa, anotacoes

In [35]:
# Generazione di 5 campioni per la visualizzazione iniziale in cima al progetto
n_amostras_iniciais = 5
rng_demo = random.Random(42)

imgs_demo = []
titulos_demo = []

for idx in range(n_amostras_iniciais):
    img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_demo)
    
    # Scrittura temporanea dell'annotazione per la lettura nativa tramite mm.showBoundBox
    filename_temp = f"temp_label_{idx}.txt"
    with open(filename_temp, "w") as f:
        for c, xc, yc, w, h in anotacoes:
            f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")
            
    img_anotada = mm.showBoundBox(img_ruid, filename=filename_temp, fmt="yolo", show=False)
    imgs_demo.append(img_anotada)
    titulos_demo.append(f"Amostra {idx+1}")

# Visualizzazione del pannello di 5 campioni
mm.show(
    imgs_demo,
    titles=titulos_demo,
    cols=n_amostras_iniciais,
    figsize=(14, 3)
)

<Figure size 2100x450 with 5 Axes>

**Figura 9.30:** Campioni iniziali del *dataset* sintetico rumoroso di oggetti geometrici con le bounding box in formato YOLO sovrapposte.


##### Bloco 2: Organizzazione del *Dataset* e Creazione del File `data.yaml`

Questo blocco organizza il set di dati nel formato previsto dalla libreria **Ultralytics YOLO**. Le immagini e le annotazioni sono distribuite in directory separate per l'addestramento e la validazione, mentre il file `data.yaml` raccoglie le informazioni necessarie per l'addestramento del rilevatore.

```text
shapes_dataset/
├── data.yaml
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

1. **Generazione del set di dati:** Il codice crea **90** immagini per l'addestramento e **20** per la validazione. Per ciascuna immagine, scrive un file `.txt` contenente una riga per oggetto, nel formato YOLO (`classe`, `x_c`, `y_c`, `larghezza`, `altezza`), con tutte le coordinate normalizzate.

2. **Organizzazione dei file:** Le immagini sono archiviate in `images/train` e `images/val`, mentre le annotazioni corrispondenti vengono salvate in `labels/train` e `labels/val`, preservando lo stesso nome del file.

3. **Creazione del file `data.yaml`:** Il codice genera automaticamente il file di configurazione contenente il percorso del *dataset*, le directory di addestramento e validazione e la mappatura tra gli indici numerici e i nomi delle nove classi.

In [36]:
base_dir = "shapes_dataset"
rng_global = random.Random(42)

for split, n_imgs in [("train", 90), ("val", 20)]:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)
    
    for i in range(n_imgs):
        img_ruid, anotacoes = gera_imagem_ruidosa(rng=rng_global)
        cv2.imwrite(f"{base_dir}/images/{split}/{i:04d}.jpg", img_ruid)
        
        with open(f"{base_dir}/labels/{split}/{i:04d}.txt", "w") as f:
            for c, xc, yc, w, h in anotacoes:
                f.write(f"{c} {xc:.4f} {yc:.4f} {w:.4f} {h:.4f}\n")

with open(f"{base_dir}/data.yaml", "w") as f:
    f.write(
        f"path: {os.path.abspath(base_dir)}\n"
        "train: images/train\nval: images/val\nnames:\n"
    )
    for i, nome in enumerate(CLASSES):
        f.write(f"  {i}: {nome}\n")

print("Set di dati rumoroso generato con successo: 90 immagini di addestramento e 20 di validazione.")

Set di dati rumoroso generato con successo: 90 immagini di addestramento e 20 di validazione.


##### Bloco 3: Pre-elaborazione con Filtro Mediano

Questo blocco applica una pre-elaborazione per ridurre l'effetto del rumore di tipo sale e pepe introdotto nella generazione del *dataset*. Il **Filtro Mediano** (`cv2.medianBlur`) rimuove questo tipo di degradazione preservando meglio i bordi degli oggetti rispetto ai filtri di smoothing convenzionali.

1. **Filtraggio dell'immagine:** Il codice applica un filtro mediano con finestra $3 \times 3$ all'immagine rumorosa, riducendo i *pixel* impulsivi senza alterare le annotazioni del dataset.

2. **Visualizzazione comparativa:** La [Figura 9.31](#fig-09-yolo-pre-processamento) confronta l'immagine originale e l'immagine filtrata, mantenendo le *bounding boxes* sovrapposte tramite la funzione `mm.showBoundBox()`.

In [37]:
# 1. Caricamento del primo campione rumoroso del dataset
caminho_img = f"{base_dir}/images/train/0000.jpg"
caminho_label = f"{base_dir}/labels/train/0000.txt"

img_ruidosa = mm.read(caminho_img)

# 2. Pre-elaborazione con Filtro Mediano (finestra 3x3)
img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)

# 3. Sovrapposizione delle bounding box con mm.showBoundBox
img_ruid_anotada = mm.showBoundBox(img_ruidosa, filename=caminho_label, fmt="yolo", show=False)
img_filt_anotada = mm.showBoundBox(img_filtrada, filename=caminho_label, fmt="yolo", show=False)

# 4. Visualizzazione comparativa con mm.show
mm.show(
    [img_ruid_anotada, img_filt_anotada],
    titles=[
        "1. Immagine Rumorosa Originale (Sale e Pepe)",
        "2. Pre-elaborata (Filtro Mediano 3x3)"
    ],
    cols=2,
    figsize=(9, 4)
)

<Figure size 1350x600 with 2 Axes>

**Figura 9.31:** Confronto tra l


##### Bloco 4: Pre-elaborazione del *Dataset* e Fine-Tuning di YOLOv8

Questo blocco applica il Filtro Mediano all'insieme di immagini ed esegue il *fine-tuning* del rilevatore **YOLOv8n** pre-addestrato sull'insieme **COCO**. La filtrazione riduce l'effetto del rumore impulsivo introdotto nella generazione delle immagini, mentre l'addestramento adatta i parametri della rete al nuovo dominio delle forme geometriche.

1. **Filtraggio in batch:** Il codice scorre le directory `train` e `val` e applica `cv2.medianBlur` con finestra $3 \times 3$ su tutte le immagini, mantenendo le annotazioni YOLO originali.

2. **Fine-tuning del rilevatore:** La rete `YOLO("yolov8n.pt")`, inizialmente addestrata su COCO, viene adattata all'insieme geometrico tramite la funzione `.train()`. L'addestramento utilizza immagini con risoluzione $320 \times 320$ *pixel* per $30$ epoche.

3. **Valutazione del modello:** La funzione `.val()` calcola le metriche di rilevamento sull'insieme di validazione, includendo **precisione** (*precision*), **revocazione** (*recall*) e **mAP50** (*mean Average Precision* con soglia IoU pari a $0,5$).

In [38]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

base_dir = "shapes_dataset"
base_dir_filt = "shapes_dataset_filtrado"

# 1. Crea una CÓPIA filtrata del dataset in una cartella separata
#    (il dataset originale in base_dir rimane rumoroso, intatto)
for split in ["train", "val"]:
    pasta_imgs_orig = f"{base_dir}/images/{split}"
    pasta_labels_orig = f"{base_dir}/labels/{split}"
    pasta_imgs_filt = f"{base_dir_filt}/images/{split}"
    pasta_labels_filt = f"{base_dir_filt}/labels/{split}"

    os.makedirs(pasta_imgs_filt, exist_ok=True)
    os.makedirs(pasta_labels_filt, exist_ok=True)

    for nome_arq in os.listdir(pasta_imgs_orig):
        if nome_arq.endswith(".jpg"):
            img_ruidosa = cv2.imread(f"{pasta_imgs_orig}/{nome_arq}")
            img_filtrada = cv2.medianBlur(img_ruidosa, ksize=3)
            # salva la versione filtrata nella cartella NUOVA, non sovrascrive l'originale
            cv2.imwrite(f"{pasta_imgs_filt}/{nome_arq}", img_filtrada)

    # copia le etichette (non cambiano con il filtro)
    for nome_arq in os.listdir(pasta_labels_orig):
        shutil.copy(f"{pasta_labels_orig}/{nome_arq}", f"{pasta_labels_filt}/{nome_arq}")

# 2. data.yaml che punta al dataset FILTRATO
with open(f"{base_dir_filt}/data.yaml", "w") as f:
    f.write(
        f"path: {os.path.abspath(base_dir_filt)}\n"
        "train: images/train\nval: images/val\nnames:\n"
    )
    for i, nome in enumerate(CLASSES):
        f.write(f"  {i}: {nome}\n")

print("Pre-elaborazione completata: dataset filtrato salvato in una cartella separata.\n")

# Scaricare il modello YOLOv8 pre-addestrato (yolov8n.pt) se non è già presente
with contextlib.redirect_stdout(io.StringIO()), \
    contextlib.redirect_stderr(io.StringIO()):
    modelo_yolo = YOLO("yolov8n.pt")
print("Modello YOLOv8 caricato.")

Pre-elaborazione completata: dataset filtrato salvato in una cartella separata.

Modello YOLOv8 caricato.


In [39]:
# 3. Callback personalizzato per stampare solo l'epoca in esecuzione
def on_train_epoch_start(trainer):
    epoch_atual = trainer.epoch + 1
    total_epochs = trainer.epochs
    # Scrive direttamente sullo stdout originale (bypassando il silenziatore)
    sys.__stdout__.write(f"🔄 Processando Época {epoch_atual}/{total_epochs}...\n")
    sys.__stdout__.flush()

# Aggiunge il callback al modello
modelo_yolo.add_callback("on_train_epoch_start", on_train_epoch_start)

# 4. Gestore di contesto per silenziare la spazzatura di Ultralytics (C/C++ e Python)
@contextlib.contextmanager
def silenciar_logs():
    logger = logging.getLogger("ultralytics")
    disabled_state = logger.disabled
    logger.disabled = True
    
    with open(os.devnull, "w") as fnull:
        old_stdout_fd = os.dup(1)
        old_stderr_fd = os.dup(2)
        try:
            os.dup2(fnull.fileno(), 1)
            os.dup2(fnull.fileno(), 2)
            with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
                yield
        finally:
            os.dup2(old_stdout_fd, 1)
            os.dup2(old_stderr_fd, 2)
            os.close(old_stdout_fd)
            os.close(old_stderr_fd)
            logger.disabled = disabled_state

# Esecuzione dell'Addestramento
print("--- Avvio Addestramento YOLOv8 ---")
with silenciar_logs():
    resultados_treino = modelo_yolo.train(
        data=f"{base_dir_filt}/data.yaml",   # <-- addestra sul dataset filtrato
        epochs=30,
        imgsz=320,
        batch=16,
        device=device,
        verbose=False,
        plots=False
    )
    metricas = modelo_yolo.val(verbose=False)

# 5. Metriche Finali
precision = metricas.results_dict["metrics/precision(B)"]
recall = metricas.results_dict["metrics/recall(B)"]
map50 = metricas.results_dict["metrics/mAP50(B)"]

print("\n--- Prestazioni Ottimizzate del Modello YOLOv8 ---")
print(f"Precisione: {precision*100:.2f}%")
print(f"Revocazione (Recall): {recall*100:.2f}%")
print(f"mAP al 50% (IoU 0.50): {map50*100:.2f}%")

--- Avvio Addestramento YOLOv8 ---



--- Prestazioni Ottimizzate del Modello YOLOv8 ---
Precisione: 63.92%
Revocazione (Recall): 79.22%
mAP al 50% (IoU 0.50): 81.81%


##### Blocco 5: Confronto di Inferenza: Immagine Rumorosa e Immagine Ripristinata

Dopo la messa a punto della YOLOv8 sull'insieme ripristinato, si esegue un confronto visivo tra il rilevamento applicato direttamente a un'immagine degradata dal rumore sale e pepe e la stessa immagine dopo il Filtro Mediano.

L'obiettivo è osservare come una semplice fase di preelaborazione possa influenzare la qualità delle previsioni di un rilevatore già adattato al nuovo dominio.

1. **Inferenza con la YOLO:** La funzione `modello_yolo.predict()` esegue il rilevamento nelle due versioni dell'immagine, utilizzando una soglia di confidenza del $25\%$ (`conf=0.25`).

2. **Visualizzazione delle Previsioni:** La funzione `plot()` genera le immagini annotate con i riquadri delimitatori e le etichette previste dal modello. La [Figura 9.32](#fig-09-yolo-inferencia-comparativa) presenta il confronto tra i due scenari.

In [40]:
# 1. Caricamento di un campione di test originale (senza il filtro salvato in batch)
caminho_teste = f"{base_dir}/images/val/0002.jpg"
img_ruidosa_teste = mm.read(caminho_teste)

# 2. Applicazione puntuale del Filtro Mediano (3x3) per il confronto
img_filtrada_teste = cv2.medianBlur(img_ruidosa_teste, ksize=3)

# 3. Inferenza con il modello YOLOv8 addestrato
pred_ruidosa = modelo_yolo.predict(img_ruidosa_teste, conf=0.25, verbose=False)[0]
pred_filtrada = modelo_yolo.predict(img_filtrada_teste, conf=0.25, verbose=False)[0]

# 4. Estrazione delle matrici annotate dal generatore di YOLO (conversione BGR -> RGB)
img_pred_ruid = cv2.cvtColor(pred_ruidosa.plot(), cv2.COLOR_BGR2RGB)
img_pred_filt = cv2.cvtColor(pred_filtrada.plot(), cv2.COLOR_BGR2RGB)

# 5. Visualizzazione comparativa standardizzata tramite mm.show
mm.show(
    [img_pred_ruid, img_pred_filt],
    titles=[
        f"Inferenza sull'Immagine Rumorosa ({len(pred_ruidosa.boxes)} oggetti)",
        f"Inferenza sull'Immagine Filtrata ({len(pred_filtrada.boxes)} oggetti)"
    ],
    cols=2,
    figsize=(10, 4)
)

**Figura 9.32:** Confronto dell


In [41]:
# Pulizia esplicita dei dati scaricati e liberazione della memoria
if FLAG_LIMPAR_DADOS:
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
        print('🧹 Diretório de dados temporários {} removido com sucesso.'.format(base_dir))

    if os.path.exists(base_dir_filt):
        shutil.rmtree(base_dir_filt)
        print('🧹 Diretório de dados temporários {} removido com sucesso.'.format(base_dir_filt))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

> ### 📝 Nota
>
> ###### 🧠 Un dominio molto più distante di quello delle cifre
>
> Nell'esperimento di transfer learning tra cifre scritte a mano (domini A e B), il compito di origine e quello di destinazione condividevano statistiche visive molto simili: entrambi erano tratti in scala di grigi su sfondo uniforme. Qui, la distanza tra i domini è molto maggiore — YOLO è stato pre-addestrato su **fotografie naturali a colori** di COCO (persone, animali, veicoli, oggetti quotidiani), e il compito di destinazione consiste in **forme geometriche sintetiche, di colore pieno e contorno ben definito**, senza texture, illuminazione o sfondo complesso.
>
> Ciononostante, il transfer learning risulta vantaggioso: i primi strati di un rilevatore addestrato su COCO apprendono filtri generici — rilevatori di bordi, angoli e regioni di contrasto — che restano utili per delimitare il contorno di un triangolo o di una stella, anche se il contenuto visivo finale è piuttosto diverso. È per questo che consentire il *fine-tuning* di tutti gli strati, combinato con un preprocessamento coerente tra addestramento e inferenza per attenuare il rumore sale e pepe, rende possibili tassi di accuratezza elevati nel rilevamento di oggetti del nuovo dominio.

##### Utilizzando un Set di Dati Reale

Il *pipeline* sopra descritto è stato costruito interamente attorno al **formato di annotazione YOLO** (classe, $x_{centro}$, $y_{centro}$, larghezza, altezza, normalizzati per larghezza e altezza dell'immagine) esattamente affinché possa essere riutilizzato senza modifiche qualora il lettore abbia accesso a un set di immagini reali annotate nello stesso modo — ad esempio, un set di immagini di oggetti geometrici fotografati o renderizzati, ciascuno con un file `.txt` corrispondente nello stesso formato usato qui. A tal fine, basterebbe:

1. Organizzare le immagini reali in `shapes_dataset/images/train` e `shapes_dataset/images/val`, e i file `.txt` di annotazione corrispondenti nelle cartelle `labels/train` e `labels/val` (un file di annotazione per immagine, stesso nome di base, estensione `.txt`);
2. Modificare il file `data.yaml` qualora il numero o i nomi delle classi fossero diversi;
3. Eseguire le stesse celle di addestramento, ottimizzazione e visualizzazione già presentate, senza alcun'altra modifica al codice.

Questa separazione tra **generazione/organizzazione dei dati** e **addestramento del modello** è, in pratica, il motivo per cui formati di annotazione standardizzati (come quello di YOLO) sono così ampiamente adottati: essi consentono di sostituire il set di dati di input — da sintetico a reale, da un dominio a un altro — mantenendo invariato tutto il resto del *pipeline* di transfer learning.

### 9.5.3 Segmentazione degli Oggetti

Lo stesso principio di trasferimento dell'apprendimento è alla base anche delle architetture di segmentazione delle immagini, in cui un estrattore di caratteristiche pre-addestrato fornisce rappresentazioni visive generali, mentre una testa specializzata esegue la classificazione densa, pixel per pixel.

Le sezioni successive presentano **DeepLabV3** come esempio di segmentatore pre-addestrato utilizzato direttamente per l'inferenza e, successivamente, l'architettura **U-Net**, addestrata da zero e confrontata con una baseline morfologica classica.

#### 9.5.3.1 DeepLabV3: Segmentatore Pre-addestrato

Nella segmentazione semantica, l’obiettivo non si limita alla localizzazione degli oggetti tramite bounding box. La rete assegna una classe a ogni pixel dell’immagine, producendo una mappa di etichette con la stessa risoluzione dell’input. Architetture come la **DeepLabV3**, con backbone **ResNet-50**, utilizzano un estrattore di caratteristiche pre-addestrato e una testa specializzata per eseguire questa classificazione densa.

1. **Caricamento e inferenza:** Il modello `deeplabv3_resnet50(weights="DEFAULT")` carica pesi pre-addestrati sul dataset **Pascal VOC**, che definisce $21$ classi di segmentazione. Il codice converte l’immagine in tensore, aggiunge la dimensione del batch (`unsqueeze(0)`) ed esegue l’inferenza.

2. **Mappa delle classi:** L’output del modello ha dimensioni $(1, 21, H, W)$, contenente un valore per ogni classe in ogni pixel. L’operazione `.argmax(dim=1)` seleziona la classe con la risposta più alta in ciascuna posizione, generando una matrice bidimensionale di etichette con dimensioni $(H, W)$.

3. **Visualizzazione:** Il codice converte la mappa delle etichette nel formato atteso da `mm.show()`, che mostra il risultato della segmentazione nella [Figura 9.33](#fig-09-segmentacao-pretreinada).

In [42]:
# 1. Caricamento dell'immagine (restituisce numpy.ndarray)
url_imagem = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
url_imagem = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = mm.read(url_imagem)

# 2. Caricamento del modello DeepLabV3 pre-addestrato in modalità di valutazione
modelo_segmentacao = deeplabv3_resnet50(weights="DEFAULT").eval()

# 3. Esecuzione dell'inferenza senza calcolo dei gradienti
with torch.no_grad():
    tensor_entrada = to_tensor(img).unsqueeze(0)  # Formato (1, C, H, W)
    saida = modelo_segmentacao(tensor_entrada)["out"]
    
    # Selezione della classe con probabilità maggiore per pixel (argmax sull'asse dei canali)
    mapa_classes = saida.argmax(dim=1).squeeze(0).byte().cpu().numpy()

# 4. Visualizzazione della mappa di segmentazione semantica
mm.show(
    [img,mapa_classes],
    title=["Immagine originale","Segmentazione Semantica (DeepLabV3)"]
)

**Figura 9.33:** Mappa di segmentazione semantica generato dal modello DeepLabV3 pre-addestrato: classificazione pixel per pixel rappresentata in matrice 2D visualizzata.


#### 9.5.3.2 Segmentazione Semantica con Architettura U-Net

La sottosezione precedente ha presentato un modello pre-addestrato (*DeepLabV3*) che produce direttamente un'etichetta di classe per *pixel*. Questa sezione completa la sequenza di progetti pratici — classificazione e rilevamento — affrontando la segmentazione semantica implementata e addestrata da zero con la **U-Net**, l'architettura di riferimento introdotta da Ronneberger (2015).

Nella classificazione, la mappa delle caratteristiche finale veniva appiattita (*flatten*) in un vettore, scartando l'informazione spaziale a favore di un'unica etichetta per immagine. La U-Net, invece, produce un'uscita con la **stessa risoluzione spaziale dell'ingresso**: una mappa bidimensionale in cui ogni *pixel* riceve la propria classificazione. Questo requisito — preservare il dettaglio spaziale ad alta risoluzione mentre si costruisce il contesto semantico negli strati profondi — motiva l'architettura encoder-decoder con connessioni di salto (*skip connections*).

Lo scenario utilizzato simula la segmentazione di noduli in esami medici sintetici: immagini in scala di grigi contengono una regione circolare ("nodulo") sovrapposta a uno sfondo, entrambi contaminati da rumore gaussiano e con medie di intensità molto vicine — una sfida intenzionale a basso contrasto, ideale per dimostrare il guadagno dell'apprendimento spaziale rispetto alla sogliatura puntuale.

##### Bloco 1: Generatore dell'Insieme Sintetico di Noduli e Visualizzazione Iniziale

Il generatore seguente produce coppie (immagine, maschera): l'immagine contiene una regione circolare di intensità leggermente superiore a quella dello sfondo, entrambe affette dalla stessa deviazione standard del rumore gaussiano. La maschera binaria delimita esattamente la regione del nodulo e funge da verità di riferimento (*ground truth*).

1. **Costruzione del Nodulo:** La funzione `gera_imagem_com_nodulo` sovrappone il nodulo allo sfondo su una matrice $64 \times 64$ e applica rumore gaussiano.
2. **Visualizzazione con `mm.show`:** La [Figura 9.34](#fig-09-unet-dataset) illustra i primi due campioni e le rispettive maschere.

In [43]:
TAM_IMG = 64


def gera_imagem_com_nodulo(
    tam=TAM_IMG,
    raio_min=7,
    raio_max=15,
    media_fundo=95,
    media_nodulo=118,
    sigma_ruido=26,
    rng=None,
):
    rng = rng or np.random.default_rng()
    fundo = rng.normal(media_fundo, sigma_ruido, (tam, tam))
    nodulo = rng.normal(media_nodulo, sigma_ruido, (tam, tam))
    mascara = np.zeros((tam, tam), dtype=np.uint8)

    raio = int(rng.integers(raio_min, raio_max))
    cx = int(rng.integers(raio + 4, tam - raio - 4))
    cy = int(rng.integers(raio + 4, tam - raio - 4))

    cv2.circle(mascara, (cx, cy), raio, 255, -1)
    imagem = np.clip(np.where(mascara > 0, nodulo, fundo), 0, 255).astype(
        np.uint8
    )
    return imagem, mascara


# Generazione dei set di training e validazione
rng_dados = np.random.default_rng(42)
N_TREINO, N_VAL = 160, 40

imgs_treino, masks_treino = zip(
    *[gera_imagem_com_nodulo(rng=rng_dados) for _ in range(N_TREINO)]
)
imgs_val, masks_val = zip(
    *[gera_imagem_com_nodulo(rng=rng_dados) for _ in range(N_VAL)]
)

print(
    f"Set di training: {N_TREINO} immagini | Set di validazione: {N_VAL} immagini\n"
)

# Visualizzazione dei campioni iniziali 
mm.show(
    [imgs_treino[0], masks_treino[0], imgs_treino[1], masks_treino[1]],
    titles=["Immagine 1", "Maschera 1", "Immagine 2", "Maschera 2"],
    cols=4,
    figsize=(11, 3),
)

Set di training: 160 immagini | Set di validazione: 40 immagini



<Figure size 1650x450 with 4 Axes>

**Figura 9.34:** Campioni del dataset sintetico di noduli: immagine in scala di grigi sotto rumore e rispettiva maschera binaria di riferimento mostrate.


##### Bloco 2: Baseline Classica (Filtraggio, Otsu e Morfologia)

Prima di impiegare la U-Net, si valuta le prestazioni di un *pipeline* morfologico classico costruito con la libreria `morph`: uno **smorzatore gaussiano** (`mm.blur`), una **soglia di Otsu** (`mm.threshold`) e un'**apertura morfologica** (`mm.open`) per l'eliminazione di rumori isolati.

1. **Metrica di Intersezione su Unione (IoU):** La funzione `iou_mascaras` calcola il grado di sovrapposizione *pixel per pixel* tra la predizione e la maschera reale.
2. **Esecuzione e Confronto:** La [Figura 9.35](#fig-09-unet-classico) mostra il risultato della segmentazione classica su un'immagine di test, evidenziando le limitazioni della soglia globale in condizioni di basso contrasto.

In [44]:
def iou_mascaras(predita, referencia):
    p, r = predita > 0, referencia > 0
    intersecao = np.logical_and(p, r).sum()
    uniao = np.logical_or(p, r).sum()
    return intersecao / uniao if uniao else 1.0


def segmenta_classico(imagem, elemento_estrutural):
    suavizada = mm.blur(imagem, 7)
    binaria = mm.threshold(suavizada)
    return mm.open(binaria, elemento_estrutural)


elemento_estrutural = mm.sedisk(5)
ious_classico = [
    iou_mascaras(segmenta_classico(img, elemento_estrutural), mask)
    for img, mask in zip(imgs_val, masks_val)
]
iou_classico_medio = float(np.mean(ious_classico))
print(
    f"IoU medio (linea di base classica) sulla validazione: {iou_classico_medio:.4f}\n"
)

predicao_classica_exemplo = segmenta_classico(imgs_val[0], elemento_estrutural)

mm.show(
    [imgs_val[0], masks_val[0], predicao_classica_exemplo],
    titles=["Immagine", "Maschera di Riferimento", "Predizione Classica"],
    cols=3,
    figsize=(9, 3.2),
)

IoU medio (linea di base classica) sulla validazione: 0.7516



<Figure size 1350x480 with 3 Axes>

**Figura 9.35:** Linea di base classica di segmentazione: smoothing, sogliatura di Otsu e apertura morfologica mostrate.


##### Bloco 3: Costruzione dell'architettura U-Net e funzioni di perdita

La U-Net è un'architettura a forma di "U" (da cui il nome), pensata specificamente per la segmentazione delle immagini. È composta da due percorsi che lavorano insieme:

- 🔽 **Codificatore (encoder):** scende attraverso l'immagine, riducendo la risoluzione spaziale a ogni passo mentre estrae caratteristiche sempre più astratte (bordi → trame → forme → contesto).
- 🔼 **Decodificatore (decoder):** risale, ricostruendo la risoluzione originale attraverso convoluzioni trasposte (*upsampling*), fino a generare una maschera delle stesse dimensioni dell'immagine di input.

L'elemento che rende la U-Net speciale sono le **connessioni di salto** (*skip connections*): esse trasportano le mappe delle caratteristiche dal codificatore direttamente alla fase corrispondente del decodificatore, alla stessa risoluzione. Questo evita che i dettagli fini — come contorni e bordi — si perdano durante la compressione spaziale.

**Flusso generale dell'architettura:**

```text
Input
  │
  ▼
Codificatore (Conv → Conv → Pool) × 3
  │
  ├──── le connessioni di salto reintroducono mappe ad alta risoluzione  ────┐
  ▼                                                                          │
Base (collo di bottiglia)                                                    │
  │                                                                          │
  ▼                                                                          │
Decodificatore (Upsample → Concat → Conv → Conv) × 3  ◄──────────────────────┘
  │
  ▼
Output 1×1 (logits)
```

**Componenti principali:**

1. **Blocco convoluzionale base (`BloccoConv`)**
   L'unità fondamentale ripetuta in tutta la rete. Applica due convoluzioni $3 \times 3$ in sequenza, ciascuna seguita da attivazione ReLU, con *padding* che preserva le dimensioni spaziali dell'input. È questo blocco che appare sia nel codificatore che nel decodificatore.

2. **Convoluzione trasposta (`nn.ConvTranspose2d`)**
   È l'operazione responsabile dell'*upsampling* nel decodificatore: invece di ridurre la risoluzione spaziale (come fa il `MaxPool2d` nel codificatore), la aumenta, apprendendo i pesi necessari per "annullare" la compressione e recuperare gradualmente la dimensione originale dell'immagine.

3. **Perdita combinata (BCE + Dice)**
   La funzione `perdita_segmentazione` somma due metriche complementari:
   - **Entropia incrociata binaria (BCE):** valuta l'accuratezza *pixel per pixel*.
   - **Coefficiente di Dice:** valuta la *sovrapposizione globale* tra la maschera prevista e quella reale.

   Insieme, esse bilanciano la precisione locale con la fedeltà della forma segmentata nel suo complesso.

In [45]:
class BlocoConv(nn.Module):

    def __init__(self, canais_entrada, canais_saida):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Conv2d(canais_entrada, canais_saida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(canais_saida, canais_saida, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.rede(x)


class UNetCompacta(nn.Module):

    def __init__(self, canais_entrada=1, base=8):
        super().__init__()
        self.enc1 = BlocoConv(canais_entrada, base)
        self.enc2 = BlocoConv(base, base * 2)
        self.enc3 = BlocoConv(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)

        self.fundo = BlocoConv(base * 4, base * 8)

        self.up3 = nn.ConvTranspose2d(
            base * 8, base * 4, kernel_size=2, stride=2
        )
        self.dec3 = BlocoConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(
            base * 4, base * 2, kernel_size=2, stride=2
        )
        self.dec2 = BlocoConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(
            base * 2, base, kernel_size=2, stride=2
        )
        self.dec1 = BlocoConv(base * 2, base)

        self.saida = nn.Conv2d(base, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        f = self.fundo(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(f), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.saida(d1)


def para_tensores(imagens, mascaras):
    X = (
        torch.tensor(np.stack(imagens), dtype=torch.float32).unsqueeze(1)
        / 255.0
    )
    Y = (
        torch.tensor(np.stack(mascaras), dtype=torch.float32).unsqueeze(1)
        / 255.0
    )
    return X, Y


def perda_dice(logits, alvo, eps=1e-6):
    probs = torch.sigmoid(logits)
    intersecao = (probs * alvo).sum(dim=(1, 2, 3))
    uniao = probs.sum(dim=(1, 2, 3)) + alvo.sum(dim=(1, 2, 3))
    dice = (2 * intersecao + eps) / (uniao + eps)
    return 1 - dice.mean()


def perda_segmentacao(logits, alvo):
    return nn.functional.binary_cross_entropy_with_logits(
        logits, alvo
    ) + perda_dice(logits, alvo)

##### Blocco 4: Addestramento della U-Net e Valutazione delle Prestazioni

L'addestramento viene eseguito per $35$ epoche utilizzando l'ottimizzatore *Adam*. Ad ogni epoca, si monitorano la Perdita di Addestramento e l'indice IoU medio sul set di validazione.

1. **Ciclo di Addestramento:** I parametri vengono aggiornati con *mini-batches* di $16$ campioni.
2. **Evoluzione Grafica:** La [Figura 9.36](#fig-09-unet-treinamento) mostra il grafico con il progresso della perdita e dell'IoU.

In [46]:
modelo_unet = UNetCompacta()
print(
    f"Parametri addestrabili della U-Net: {sum(p.numel() for p in modelo_unet.parameters())}"
)

X_treino_unet, Y_treino_unet = para_tensores(imgs_treino, masks_treino)
X_val_unet, Y_val_unet = para_tensores(imgs_val, masks_val)

otimizador_unet = optim.Adam(modelo_unet.parameters(), lr=1e-3)
n = X_treino_unet.size(0)
tam_lote = 16
epocas_unet = 35
historico_perda_unet, historico_iou_unet = [], []

for epoca in range(epocas_unet):
    if epoca % 5 == 0:  # Stampa ogni 5 epoche
        print(epoca + 1, "/", epocas_unet)
    modelo_unet.train()
    perm = torch.randperm(n)
    perda_epoca = 0.0

    for i in range(0, n, tam_lote):
        idx = perm[i : i + tam_lote]
        otimizador_unet.zero_grad()
        logits = modelo_unet(X_treino_unet[idx])
        perda = perda_segmentacao(logits, Y_treino_unet[idx])
        perda.backward()
        otimizador_unet.step()
        perda_epoca += perda.item() * len(idx)

    modelo_unet.eval()
    with torch.no_grad():
        predicao_val = (torch.sigmoid(modelo_unet(X_val_unet)) > 0.5).float()
        intersecao = (predicao_val * Y_val_unet).sum(dim=(1, 2, 3))
        uniao = ((predicao_val + Y_val_unet) > 0).float().sum(dim=(1, 2, 3))
        iou_epoca = (intersecao / uniao.clamp(min=1e-6)).mean().item()

    historico_perda_unet.append(perda_epoca / n)
    historico_iou_unet.append(iou_epoca)

iou_unet_final = historico_iou_unet[-1]
print(f"IoU medio finale della U-Net nella validazione: {iou_unet_final:.4f}")

# Grafico dell'evoluzione dell'allenamento
r = mm.showTrainCurves(historico_perda_unet, historico_iou_unet,
        titulo="Treinamento da U-Net — Segmentação de Nódulos Sintéticos",
        subtitulo="Perda no conjunto de treino e IoU médio no conjunto de validação \
            ao longo de 35 épocas")

Parametri addestrabili della U-Net: 120681
1 / 35


6 / 35


11 / 35


16 / 35


21 / 35


26 / 35


31 / 35


IoU medio finale della U-Net nella validazione: 0.8876


<Figure size 1190x714 with 2 Axes>

**Figura 9.36:** Evoluzione dell


##### Blocco 5: Confronto Quantitativo e Qualitativo (Classico vs. U-Net)

Il confronto tra l'approccio classico e la U-Net rende esplicita la superiorità dell'apprendimento di rappresentazioni in scenari di basso contrasto.

1. **Pannello delle metriche:** Il grafico a barre in [Figura 9.37](#fig-09-unet-comparativo) contrappone lo IoU medio di entrambi i metodi in validazione.
2. **Visualizzazione qualitativa:** Il confronto visivo su tre campioni dimostra come le connessioni di shortcut recuperino il contorno del nodulo anche in presenza di rumore accentuato.

In [47]:
# 1.  confronto 
print(f"IoU medio (classico Smoothing + Otsu): {iou_classico_medio:.4f}")
print(f"IoU medio (U-Net): {iou_unet_final:.4f}")

# 2. Visualizzazione qualitativa affiancata su 3 campioni tramite mm.show
imgs_comparativas = []
titulos_comparativos = []

for i in range(3):
    with torch.no_grad():
        pred_unet = (
            (torch.sigmoid(modelo_unet(X_val_unet[i : i + 1])) > 0.5)
            .float()
            .squeeze()
            .numpy()
            * 255
        )

    pred_classico = segmenta_classico(imgs_val[i], elemento_estrutural)

    imgs_comparativas.extend(
        [imgs_val[i], masks_val[i], pred_classico, pred_unet.astype(np.uint8)]
    )

    t_prefix = f"Amostra {i+1}"
    titulos_comparativos.extend(
        [
            f"{t_prefix}: Imagem",
            f"{t_prefix}: Referência",
            f"{t_prefix}: Clássico",
            f"{t_prefix}: U-Net",
        ]
    )

mm.show(
    imgs_comparativas,
    titles=titulos_comparativos,
    cols=4,
    figsize=(11, 7.5),
)

IoU medio (classico Smoothing + Otsu): 0.7516
IoU medio (U-Net): 0.8876


<Figure size 1650x1125 with 12 Axes>

**Figura 9.37:** Confronto quantitativo (IoU medio) e qualitativo tra l


> ### 📝 Nota
>
> ###### 🧠 Perché la U-Net supera la sogliatura fissa?
>
> La sogliatura di Otsu applica un valore di taglio globale sull'intensità locale. Quando la differenza di media tra il nodulo e lo sfondo è piccola rispetto al rumore gaussiano, questa regola commette errori sistematici ai bordi.
>
> La U-Net supera questa limitazione combinando l'ampio contesto semantico estratto dall'encoder con i dettagli spaziali fini preservati dalle connessioni di salto. Ciò consente di identificare la presenza del nodulo e di delinearne i contorni con precisione, anche in presenza di rumore intenso.

### 9.5.4 Ingegneria dei Dati per la Visione Artificiale (*Roboflow*)

Una U-Net può essere addestrata a partire da immagini annotate su piattaforme di ingegneria dei dati per la Visione Artificiale (VA), come *Roboflow*. Queste piattaforme consentono di organizzare set di dati, eseguire annotazioni, applicare fasi di pre-elaborazione e *data augmentation*, addestrare modelli ed esportare i dati in diversi formati. In questa sezione, tuttavia, l'esempio riprende la **rilevazione di oggetti geometrici**, utilizzando set di dati già presentati in questo libro.

> ### ❗ Dipendenza da connessione e chiave API
>
> Le celle di questa sezione richiedono una connessione a Internet e una chiave API gratuita di *Roboflow* (`app.roboflow.com`). Nel *Workspace* del progetto, accedi a **⚙ → Roboflow API** e copia la **Private API Key**.
>
> Crea, in questa cartella, il file `chave_roboflow.txt` contenente **solo la chiave**, senza virgolette. Un modello di questo file è disponibile in `chave_roboflow.txt.exemplo`.
>
> Aggiungi `chave_roboflow.txt` al file `.gitignore`, poiché contiene una credenziale di accesso che non deve essere versionata né condivisa.

#### 9.5.4.1 Rilevamento di oggetti utilizzando *Roboflow*

*Roboflow* consente di eseguire inferenze su immagini locali, permettendo
di valutare le prestazioni del modello ospitato nell'identificazione degli
oggetti di interesse.

Oltre all'inferenza, il *dataset* può essere esportato nel formato
`png-mask-semantic`, in cui ogni immagine è accompagnata da una maschera di
segmentazione semantica. In questa maschera, ogni pixel rappresenta la classe a cui
appartiene l'oggetto corrispondente. Le coppie immagine-maschera vengono utilizzate
come dati di addestramento per la `UNetCompacta`.

#### 9.5.4.2 Connessione, *download* e verifica del *dataset*

Il codice stabilisce una connessione con il *Workspace* `mctest` e il progetto
`geometric-test00`, versione 6, ed esegue il *download* del *dataset* in
`dados/datasetRoboFlow`. Successivamente, verifica la *shape* delle immagini in ciascuno
*split*. La funzione di verifica viene riutilizzata successivamente nella sezione.

In [48]:
from roboflow import Roboflow
from pathlib import Path
from PIL import Image
from collections import Counter

def contar_shapes(raiz, splits=("train", "valid", "test")):
    """Conta la shape (altezza, larghezza, canali) delle immagini per split."""
    for split in splits:
        pasta = raiz / split / "images"
        if not pasta.exists():
            print(f"{split}: cartella non trovata")
            continue

        shapes = Counter()
        for arquivo in pasta.iterdir():
            if arquivo.is_file():
                with Image.open(arquivo) as img:
                    shapes[(img.height, img.width, len(img.getbands()))] += 1

        txt = ", ".join(f"{s}: {n}" for s, n in shapes.items())
        print(f"{split}: {txt}")


chave = Path("chave_roboflow.txt")

if not chave.exists():
    print("Chiave Roboflow non trovata: chiave_roboflow.txt")
else:
    with open(chave) as f:
        api_key = f.read().strip()

    # Progetto:
    # https://app.roboflow.com/mctest/geometric-test00/models
    # geometric-test00/6

    rf = Roboflow(api_key=api_key)
    projeto = rf.workspace("mctest").project("geometric-test00")
    versao = projeto.version(6)

    print(
        f"ID: {versao.version} | Nome: {versao.name} | "
        f"Immagini: {versao.images}"
    )

    raiz = Path("dados/datasetRoboFlow")
    versao.download("yolov8", location=str(raiz))

    print("Dataset:", raiz)
    contar_shapes(raiz)

#### 9.5.4.3 Scaricare il *dataset* in modo riproducibile (alternativa)

In alternativa al *dataset* ottenuto tramite *Roboflow*, è possibile utilizzare un
*dataset* reso disponibile in un repository GitHub, anch'esso organizzato negli
stessi *splits* e contenente le stesse classi di oggetti. I set di
dati, tuttavia, non sono identici: le immagini su GitHub hanno una risoluzione
di `608×608` pixel, mentre le immagini esportate da *Roboflow* hanno
`640×640` pixel.

Il codice seguente esegue il *download* del *dataset* da GitHub, nel caso in cui
non sia già disponibile localmente, e riutilizza `contar_shapes` per verificare
le dimensioni delle immagini in ciascuno *split*.

In [49]:
import os

if not os.path.exists("dados/dataset"):
    cmd = (
        "git clone --no-checkout --depth 1 --filter=blob:none "
        "https://github.com/fzampirolli/pdi-vc.git tmp_repo && "
        "cd tmp_repo && git sparse-checkout set all/cap09/dados/dataset "
        "&& git checkout && cd .. && mkdir -p dados && "
        "cp -r tmp_repo/all/cap09/dados/dataset dados/dataset && "
        "rm -rf tmp_repo"
    )
    !{cmd}

if not chave.exists():
    print("Chave do Roboflow não encontrada: chave_roboflow.txt")
else:
  versao_recente = projeto.versions()[-1]
  print(f"Versão mais recente: {versao_recente.version.split('/')[-1]}")

  contar_shapes(Path("dados/dataset"))

#### 9.5.4.4 Confronto di un'immagine da ciascun *dataset*

I due *dataset* contengono immagini con risoluzioni diverse: `608×608` su
GitHub e `640×640` su *Roboflow*. Per un confronto visivo diretto, le
immagini vengono ridimensionate alla stessa dimensione prima di essere
mostrate affiancate ([Figura 9.38](#fig-09-datasets-comparacao)).

In [50]:
import cv2

if not chave.exists():
    print("Chiave Roboflow non trovata: chave_roboflow.txt")
else:
    caminho1 = next((raiz / "train/images").iterdir())
    caminho2 = next((Path("dados/dataset") / "train/images").iterdir())

    img1 = mm.read(str(caminho1))
    img2 = mm.read(str(caminho2))

    # Ridimensiona entrambe alla stessa dimensione (la più piccola tra le due)
    largura = min(img1.shape[1], img2.shape[1])
    altura = min(img1.shape[0], img2.shape[0])
    img1_r = cv2.resize(img1, (largura, altura))
    img2_r = cv2.resize(img2, (largura, altura))

    mm.show(
        [img1_r, img2_r],
        title=[f"Roboflow {img1.shape}", f"GitHub {img2.shape}"],
    )

**Figura 9.38:** Un


#### 9.5.4.5 Inferenza con il modello addestrato

Il modello addestrato nella versione 6 viene utilizzato per eseguire l'inferenza su
un'immagine di test locale. La previsione considera le soglie di confidenza e
di sovrapposizione impiegate dalla soppressione non massima (*Non-Maximum
Suppression*, NMS), e il risultato viene presentato nell'immagine annotata della
[Figura 9.39](#fig-09-roboflow-detection).

In [51]:
# version.model è deprecato; usare version.models()

if not chave.exists():
    print("Chiave Roboflow non trovata: chave_roboflow.txt")
else:
    modelo = versao.models()[0]

    img_path = "dados/dataset/test/images/00001.jpg"
    pred = modelo.predict(img_path, confidence=40, overlap=30)
    resp = pred.json()  # include le caselle rilevate in resp["predictions"]

    altura, largura, _ = mm.read(img_path).shape
    print("Dimensioni dell'immagine di test:", (altura, largura))

    pred.save("resultado.jpg")  # immagine annotata

    mm.show(
        mm.read("resultado.jpg"),
        title="Risultato dell'inferenza con il modello di Roboflow",
        figsize=(6, 6)
    )

**Figura 9.39:** Risultato dell


#### 9.5.4.6 Valutando le predizioni con IoU e classe

*Roboflow* restituisce ogni riquadro con le coordinate del centro (`x`, `y`) in *pixel* e la classe prevista, mentre le etichette locali (`dati/dataset/test/labels/00001.txt`) seguono il formato YOLO, con centro e dimensioni normalizzati nell'intervallo `[0, 1]`. Prima del confronto tramite `mm.IoU`, i riquadri devono essere convertiti nello stesso formato, con le coordinate dell'angolo superiore sinistro e le dimensioni espresse in *pixel*.

Una predizione è considerata corretta solo quando la classe prevista coincide con la classe del riquadro reale e la sua *Intersection over Union* (IoU) è maggiore o uguale alla soglia definita, adottando, in questo esempio, `0,5` (50%) come valore predefinito.

> ### ❗ Importante
>
> ##### Il `class_id` di *Roboflow* non corrisponde all'indice delle *label* locali
>
> Nell'esportazione, *Roboflow* riordina le classi in **ordine alfabetico** nel `data.yaml`, indipendentemente dall'ordine utilizzato nel progetto originale, mantenuto nel `data.yaml` del GitHub. Così, `class_id = 0` corrisponde a `Circulo` nella risposta dell'API, mentre lo stesso indice corrisponde a `Triangulo` nelle *label* locali.
>
> Pertanto, il confronto deve essere effettuato tramite il **nome della classe** (`p["class"]`), convertendolo successivamente nell'indice corrispondente nella lista locale. I `class_id` non devono essere confrontati direttamente.

In [52]:
# resp, largura e altura sono stati definiti nella cella precedente

# Ordine delle classi usato nelle etichette locali (dati/dataset/*/labels/*.txt)
CLASSES_LOCAIS = ["Triangulo", "Quadrado", "Pentagono", "Hexagono",
                   "Heptagono", "Circulo", "Elipse"]

def predicao_correta(pred: tuple, real: tuple, limiar: float = 0.5) -> bool:
    """Vero se stessa classe e mm.IoU(scatola_pred, scatola_reale) >= soglia."""
    classe_pred, caixa_pred = pred
    classe_real, caixa_real = real
    return classe_pred == classe_real and mm.IoU(caixa_pred, caixa_real) >= limiar

def caixa_roboflow(p: dict) -> tuple:
    """Converte la predizione di Roboflow in (classe, (x, y, w, h))."""
    caixa = (p["x"] - p["width"] / 2, p["y"] - p["height"] / 2,
             p["width"], p["height"])
    # usa il nome della classe (non il class_id!) per corrispondere all'ordine locale
    classe = CLASSES_LOCAIS.index(p["class"])
    return (classe, caixa)

def carrega_labels_yolo(caminho_txt: str, largura: int, altura: int) -> list:
    """Legge etichette YOLO (normalizzate) e le converte in (classe, (x, y, w, h))."""
    caixas = []
    with open(caminho_txt) as f:
        for linha in f:
            classe, xc, yc, w, h = map(float, linha.split())
            w_px, h_px = w * largura, h * altura
            x_px = xc * largura - w_px / 2
            y_px = yc * altura - h_px / 2
            caixas.append((int(classe), (x_px, y_px, w_px, h_px)))
    return caixas

if not chave.exists():
    print("Chiave Roboflow non trovata: chiave_roboflow.txt")
else:
    caixas_pred = [caixa_roboflow(p) for p in resp["predictions"]]
    caixas_real = carrega_labels_yolo(
        "dados/dataset/test/labels/00001.txt", largura, altura
    )

    acertos = sum(
        any(predicao_correta(cp, cr, limiar=0.5) for cr in caixas_real)
        for cp in caixas_pred
    )
    print(f"{acertos}/{len(caixas_pred)} predizioni corrette (classe + IoU >= 50%)")

Per visualizzare il risultato, ogni previsione viene disegnata sull'immagine:
in verde quando è un successo (classe + IoU ≥ soglia) e in rosso quando è un
errore, come mostra la [Figura 9.40](#fig-09-roboflow-acertos).

In [53]:
import cv2

VERDE, VERMELHO = (0, 255, 0), (255, 0, 0)

if not chave.exists():
    print("Chave do Roboflow não encontrada: chave_roboflow.txt")
else:
    img_acertos = mm.read(img_path).copy()

    if not chave.exists():
        print("Chave do Roboflow não encontrada: chave_roboflow.txt")
    else:
        for cp in caixas_pred:
            classe_pred, (x, y, w, h) = cp
            correta = any(predicao_correta(cp, cr, limiar=0.5) for cr in caixas_real)
            cor = VERDE if correta else VERMELHO

            p1, p2 = (int(x), int(y)), (int(x + w), int(y + h))
            cv2.rectangle(img_acertos, p1, p2, cor, 2)
            cv2.putText(img_acertos, str(classe_pred), (p1[0], p1[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, cor, 2)

    mm.show(
        img_acertos,
        title=f"{acertos}/{len(caixas_pred)} predições corretas "
            f"(classe + IoU >= 50%)",
        figsize=(6, 6)
    )

**Figura 9.40:** Predições corretas (verde) e incorretas (vermelho)


### 9.5.5 Applicazioni Geometriche e Integrate

Il capitolo si conclude integrando due pilastri della VC: la **geometria proiettiva** (studiata nei Capitoli 6 e 8) e l'**apprendimento profondo**. La combinazione di questi approcci supporta applicazioni pratiche nel mondo reale, come illustrato di seguito.

#### 9.5.5.1 Realtà Aumentata con Marcatori e Omografia

Immagina una telecamera puntata su un tavolo dove qualcuno ha incollato un piccolo marker ArUco. A seconda dell'angolazione della telecamera, questo marker appare **ruotato, inclinato, in prospettiva** — mai perfettamente quadrato. È proprio questa distorsione che l'omografia sa "leggere" e annullare (o, nel nostro caso, replicare per una nuova immagine).

Il flusso completo di un'applicazione di RA basata su marcatori segue tre passaggi:

1. **Ambientazione:** il marker viene inserito in una scena reale, subendo una trasformazione di prospettiva (simulando l'angolazione della telecamera).
2. **Rilevamento:** l'algoritmo localizza il marker nella scena e recupera le coordinate esatte dei suoi 4 angoli con `cv2.aruco.ArucoDetector`.
3. **Sostituzione:** con l'omografia tra il marker "ideale" e il marker "rilevato", si proietta una **nuova immagine virtuale** esattamente sull'area del marker — come se si fosse trasformato in una finestra verso un altro contenuto, come mostrato in [Figura 9.41](#fig-09-realidade-aumentada).

> ### 📝 Nota
>
> **Dettaglio tecnico importante:** l'ArUco necessita di un margine bianco attorno al pattern (la "zona di silenzio") affinché il rilevatore possa distinguere il marker dallo sfondo. Per questo motivo, il codice seguente aggiunge un bordo con `cv2.copyMakeBorder` e utilizza l'interpolazione `cv2.INTER_NEAREST` quando deforma l'immagine, evitando che la rotazione sfumi i piccoli quadrati bianchi e neri e impedisca il rilevamento.

In [54]:
# 1. Generazione del marker ArUco sintetico, già con margine bianco (quiet zone)
# → questo margine è essenziale affinché il rilevatore riesca a "vedere" il marker
dic = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_bruto = cv2.aruco.generateImageMarker(dic, 7, 200)
aruco_gray = cv2.copyMakeBorder(
    aruco_bruto, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=255
)  # 200 -> 280px, con 40px di cornice bianca su ogni lato
aruco = cv2.cvtColor(aruco_gray, cv2.COLOR_GRAY2BGR)
lado = aruco.shape[0]  # 280

# 2. Scena reale di sfondo, usando un'immagine di esempio da skimage.data
fundo = cv2.cvtColor(skdata.coffee(), cv2.COLOR_RGB2BGR)
cena = cv2.resize(fundo, (640, 480))

# Angoli del marker "frontale" (src) e la sua posizione ruotata/inclinata nella scena (dst)
src = np.float32([[0, 0], [lado, 0], [lado, lado], [0, lado]])
dst = np.float32([[190, 160], [420, 70], [470, 330], [150, 370]])  # rotazione + prospettiva

# 3. Proietta il marker (con bordi nitidi) sulla scena reale
H_cena, _ = cv2.findHomography(src, dst)
mask_cena = cv2.warpPerspective(
    np.full((lado, lado), 255, np.uint8), H_cena, (640, 480),
    flags=cv2.INTER_NEAREST,
)
cena[mask_cena > 0] = cv2.warpPerspective(
    aruco, H_cena, (640, 480), flags=cv2.INTER_NEAREST
)[mask_cena > 0]

# 4. Rilevamento del marker all'interno della scena (come farebbe una telecamera)
det = cv2.aruco.ArucoDetector(dic, cv2.aruco.DetectorParameters())
corners, ids, _ = det.detectMarkers(cena)

assert ids is not None and len(corners) > 0, \
    "Marcador não detectado — confira iluminação/contraste da cena."

# 5. Immagine virtuale (un'altra immagine da skimage.data) che "sostituirà" il marker
# Nota: usiamo il quadrato INTERNO del marker (senza margine) come area di proiezione,
# quindi l'omografia dell'immagine virtuale usa 'src' originale (200x200), non il 'lato' con bordo
src_interno = np.float32([[0, 0], [200, 0], [200, 200], [0, 200]])
virtual = cv2.resize(
    cv2.cvtColor(skdata.camera(), cv2.COLOR_RGB2BGR), (200, 200)
)

# Gli angoli rilevati corrispondono al marker CON il margine (280x280),
# quindi ricalcoliamo H_ra usando 'src' con margine, per mantenere la proiezione coerente
H_ra, _ = cv2.findHomography(src, corners[0][0])

# L'omografia rilevata viene applicata all'immagine virtuale (ridimensionata a 'lato')
virtual_grande = cv2.resize(virtual, (lado, lado))
ra = cena.copy()
mask_ra = cv2.warpPerspective(
    np.full((lado, lado), 255, np.uint8), H_ra, (640, 480), flags=cv2.INTER_NEAREST
)
ra[mask_ra > 0] = cv2.warpPerspective(
    virtual_grande, H_ra, (640, 480), flags=cv2.INTER_NEAREST
)[mask_ra > 0]

# Visualizzazione: scena con marker vs. scena con Realtà Aumentata applicata
mm.show(
    [cv2.cvtColor(cena, cv2.COLOR_BGR2RGB), cv2.cvtColor(ra, cv2.COLOR_BGR2RGB)],
    titles=["Marker nella Scena Reale (ruotato)", "Sovrapposizione Virtuale tramite Omografia"],
    cols=2,
    figsize=(9, 4),
)

<Figure size 1350x600 with 2 Axes>

**Figura 9.41:** Realtà aumentata basata su marker ArUco: marker inserito in scena reale e ruotato, rilevato e sostituito con immagine virtuale tramite omografia.


**Riepilogo del *pipeline*:**

| Fase | Funzione | Funzione chiave |
|---|---|---|
| 1. Generazione | Marker ArUco con margine bianco | `generateImageMarker` + `copyMakeBorder` |
| 2. Ambientazione | Inserisce marker distorto nella scena | `findHomography` + `warpPerspective` |
| 3. Rilevamento | Localizza marker e restituisce angoli | `ArucoDetector.detectMarkers` |
| 4. Sostituzione | Proietta immagine virtuale sul marker | `findHomography` + `warpPerspective` |

> 💡 **Lezione:** è comune nella visione artificiale che il rilevatore "non trovi nulla". Chiediti sempre: **"ho fornito abbastanza contrasto e spazio?"** — vale per ArUco, codici QR e riconoscimento facciale.

#### 9.5.5.2 Fotogrammetria e Riferimento di Scala

La **fotogrammetria** consente di stimare le dimensioni fisiche degli oggetti a partire da immagini digitali. A tal fine, si utilizza un oggetto di riferimento con dimensioni note, posizionato nella stessa scena dell’oggetto di interesse. Questa procedura stabilisce una relazione tra le distanze misurate in *pixel* e le corrispondenti dimensioni nel mondo reale.

Si consideri, ad esempio, una carta di credito, le cui dimensioni seguono lo standard internazionale ISO/IEC 7810. Poiché la sua larghezza è esattamente **8,56 cm**, è sufficiente determinare quanti *pixel* occupa tale larghezza nell’immagine per calcolare il fattore di conversione tra *pixel* e centimetri. Se la carta corrisponde a 140 *pixel*, allora ogni *pixel* rappresenterà approssimativamente **0,061 cm**. Questo stesso fattore di scala può essere applicato per stimare le dimensioni di qualsiasi altro oggetto situato nello stesso piano della scena, come illustrato in [Figura 9.42](#fig-09-fotogrametria)..

La procedura può essere suddivisa in due fasi principali:

1. **Segmentazione e *bounding box*:** individuare nell’immagine sia l’oggetto di riferimento sia l’oggetto di interesse, utilizzando tecniche come la segmentazione per colore, la sogliatura, il rilevamento dei contorni o metodi di rilevamento degli oggetti.
2. **Conversione in dimensioni fisiche:** calcolare il rapporto $\mathrm{cm/pixel}$ a partire dalla larghezza nota dell’oggetto di riferimento e utilizzarlo per convertire le misure dell’oggetto di interesse da *pixel* a centimetri.

> ### 📝 Nota
>
> **Condizione per misurazioni affidabili**
>
> La conversione tra *pixel* e centimetri presuppone che l’oggetto di riferimento e l’oggetto di interesse si trovino approssimativamente sullo stesso piano e alla stessa distanza dalla fotocamera. In queste condizioni, la scala rimane pressoché costante in tutta l’immagine. Differenze di profondità, inclinazione della fotocamera o distorsioni dell’obiettivo possono introdurre errori nelle misure stimate.

In [55]:
COR_REFERENCIA = (200, 200, 200)  # Carta di riferimento (grigia)
COR_OBJETO = (60, 60, 220)  # Oggetto bersaglio (rosso)

# Disegno della scena sintetica
cena_medicao = np.full((300, 500, 3), 255, dtype=np.uint8)
cv2.rectangle(
    cena_medicao, (30, 200), (30 + 140, 200 + 88), COR_REFERENCIA, -1
)
cv2.rectangle(cena_medicao, (250, 100), (250 + 220, 100 + 150), COR_OBJETO, -1)


def caixa_delimitadora_por_cor(imagem_bgr, cor_bgr, tolerancia=40):
    diferenca = np.abs(imagem_bgr.astype(int) - np.array(cor_bgr)).sum(axis=2)
    mascara = (diferenca < tolerancia).astype(np.uint8) * 255
    contornos, _ = cv2.findContours(
        mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    maior_contorno = max(contornos, key=cv2.contourArea)
    return cv2.boundingRect(maior_contorno)


x_ref, y_ref, w_ref_px, h_ref_px = caixa_delimitadora_por_cor(
    cena_medicao, COR_REFERENCIA
)
x_obj, y_obj, w_obj_px, h_obj_px = caixa_delimitadora_por_cor(
    cena_medicao, COR_OBJETO
)

# Calcolo della scala fisica
LARGURA_REFERENCIA_CM = 8.56
razao_cm_por_px = LARGURA_REFERENCIA_CM / w_ref_px
largura_obj_cm = w_obj_px * razao_cm_por_px
altura_obj_cm = h_obj_px * razao_cm_por_px

# Disegno delle scatole e delle misurazioni stimate
resultado = cena_medicao.copy()
cv2.rectangle(
    resultado, (x_ref, y_ref), (x_ref + w_ref_px, y_ref + h_ref_px), (0, 180, 0), 2
)
cv2.rectangle(
    resultado, (x_obj, y_obj), (x_obj + w_obj_px, y_obj + h_obj_px), (0, 180, 0), 2
)

cv2.putText(
    resultado,
    f"{LARGURA_REFERENCIA_CM:.2f} cm",
    (x_ref, y_ref - 8),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.55,
    (0, 120, 0),
    2,
)

cv2.putText(
    resultado,
    f"{largura_obj_cm:.1f} x {altura_obj_cm:.1f} cm",
    (x_obj, y_obj - 8),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.6,
    (0, 120, 0),
    2,
)

mm.show(
    [
        cv2.cvtColor(cena_medicao, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB),
    ],
    titles=[
        "Scena Originale",
        "Misurazione tramite Riferimento di Scala (Centimetri)",
    ],
    cols=2,
    figsize=(9, 4),
)

<Figure size 1350x600 with 2 Axes>

**Figura 9.42:** Misurazione di dimensioni fisiche reali utilizzando una carta di riferimento di scala nota (8,56 cm) renderizzata. Il rettangolo grigio rappresenta la carta di riferimento e il rettangolo blu rappresenta l


**Sintesi del *pipeline*:**

| Fase | Funzione | Funzione chiave |
|---|---|---|
| 1. Scena sintetica | Disegna la carta di riferimento e l'oggetto target con colori distinti | `cv2.rectangle` |
| 2. Segmentazione | Isola ciascun oggetto per colore e ne estrae il contorno | `cv2.findContours` |
| 3. Bounding Box | Ottiene il riquadro delimitatore (posizione e dimensioni in pixel) di ciascun oggetto | `cv2.boundingRect` |
| 4. Ridimensionamento | Converte i pixel in centimetri usando la larghezza nota della carta | Regola del tre: $\text{cm/pixel} = \dfrac{8{,}56}{w_{ref\_px}}$ |
| 5. Annotazione | Disegna i riquadri e mostra le misure stimate sull'immagine | `cv2.rectangle` + `cv2.putText` |

> 💡 **Applicazione nel mondo reale:** questa è esattamente la tecnica usata dalle *app* di *e-commerce* che stimano la dimensione di un prodotto a partire da una foto scattata accanto a una carta, da sistemi agricoli che misurano frutta su nastri trasportatori, e persino da perizie forensi che calcolano le dimensioni di tracce su scene del crimine — tutto con la stessa idea: **un righello noto all'interno della stessa foto.**

## 9.6 Riassunto

Questo capitolo, che chiude la Parte II del libro, ha presentato:

* **Convoluzione e *pooling* appresi:** la stessa operazione matematica di
  convoluzione del Capitolo 3, ma con *kernel* trattati come parametri
  regolati mediante addestramento, anziché definiti manualmente;

* **Condivisione dei pesi e gerarchia delle caratteristiche** come
  proprietà che rendono le CNN efficienti e capaci di apprendere
  rappresentazioni sempre più astratte in livelli successivi;

* **Addestramento di una CNN da zero**, con prestazioni paragonabili — e non
  necessariamente superiori — ai classificatori classici del Capitolo 7
  su una base piccola e semplice, a sottolineare che la scelta del metodo
  deve essere proporzionale alla complessità reale del problema;

* **Transfer learning**, dimostrato sperimentalmente come una
  strategia efficace per compiti con pochi dati etichettati, riutilizzando
  un estrattore di caratteristiche già addestrato su un compito correlato, e
  le sue limitazioni, evidenziate dal transfer negativo tra domini molto
  diversi;

* **Applicazioni su larga scala** con modelli pre-addestrati di classificazione,
  rilevamento (*Faster R-CNN*) e segmentazione (*DeepLabV3*), seguendo lo
  stesso principio di transfer learning su scala industriale;

* **Transfer learning applicato al rilevamento di oggetti**, adattando
  un YOLO pre-addestrato su COCO per localizzare e classificare oggetti
  geometrici sintetici, illustrando lo stesso principio di congelamento
  parziale in un dominio di origine e destinazione ancora più distanti tra loro;

* **Segmentazione semantica con U-Net**, implementata e addestrata da zero su
  un insieme sintetico a basso contrasto, superando una linea di base
  classica di sogliatura grazie alle connessioni di skip tra codificatore e
  decodificatore;

* **Ingegneria dei dati per la Visione Artificiale con *Roboflow***, utilizzando
  un *dataset* e un modello pre-addestrato per eseguire inferenza nel rilevamento
  di oggetti geometrici, oltre a confrontare insiemi di dati con diverse
  risoluzioni, ma con le stesse classi di oggetti;

* L'integrazione di **geometria computazionale** (omografia e calibrazione) e
  **deep learning** in due applicazioni reali che chiudono il libro:
  **realtà aumentata** e **fotogrammetria**.

## 9.7 🤖 Uso del Gemini Notebook come Tutor Complementare

In questa edizione, l'uso del **Gemini Notebook** è incoraggiato come strumento
complementare di apprendimento. Basato sull'intelligenza artificiale, il sistema
utilizza esclusivamente i documenti forniti dall'autore come fonte di
conoscenza, producendo risposte allineate ai contenuti e all'approccio
adottato nel corso di questo capitolo.

> ### ❗ 🎓 Studia con il Tutor Intelligente
>
> [🚀 ACCEDI AL GEMINI NOTEBOOK: CAPITOLO 09](https://notebooklm.google.com/notebook/88495fa2-9139-45da-8d1b-9a9b2e609d36)
>
> #### 🌐 Lingua e Linguaggio di Programmazione
>
> Il progetto di questo capitolo nel Gemini Notebook è stato realizzato esclusivamente con il testo in **portoghese** e gli esempi di codice in **Python**. Se stai studiando utilizzando l'edizione in inglese o francese, o seguendo il percorso in C++, le risposte del tutor potrebbero non corrispondere esattamente alla versione che stai leggendo.
>
> #### ⚠️ Avvertenza sul Contenuto Generato dall'IA
>
> Sebbene sia uno strumento prezioso di supporto allo studio, il Gemini Notebook può
> talvolta produrre risposte incomplete, imprecise o errate.
> Si consiglia di validare le informazioni consultando il materiale del capitolo,
> libri, articoli scientifici e altre fonti accademiche affidabili. Quando
> possibile, esegui e sperimenta gli esempi pratici presentati nel
> corso del testo per consolidare la comprensione dei concetti.

## 9.8 Elenco di Esercizi

Gli esercizi seguenti consolidano i concetti presentati in questo capitolo tramite adattamenti, esperimenti ed estensioni degli algoritmi sviluppati nel testo, utilizzando le librerie `PyTorch`, `ultralytics` e la libreria didattica `morph`.

1. **(10%)** Indagare l'impatto della profondità in un'architettura convoluzionale. Partendo dalla rete a due strati del Progetto Pratico 1, aggiungere un terzo strato convoluzionale con 32 filtri prima degli strati completamente connessi. Addestrare la nuova architettura mantenendo lo stesso numero di epoche e la stessa suddivisione dei dati. Confrontare l'accuratezza sul set di test e il numero totale di parametri addestrabili rispetto alla rete originale, discutendo se l'aumento di profondità abbia apportato un beneficio misurabile per immagini di dimensione $8 \times 8$.

2. **(15%)** Valutare la soglia di dati necessari nel dominio di destinazione affinché l'addestramento di una CNN da zero diventi competitivo con il **transfer learning**. Variando il numero di campioni di addestramento disponibili nel dominio B tra $\{5, 10, 20, 40, 80\}$, misurare l'accuratezza di test per entrambe le strategie. Presentare i risultati in un grafico a linee e determinare a partire da quale volume di dati l'addestramento da zero raggiunge prestazioni equivalenti all'estrattore pre-addestrato.

3. **(15%)** Indagare la strategia di **fine-tuning parziale** rispetto al congelamento totale dei pesi. Nello scenario di transfer learning tra domini di cifre, scongelare il secondo strato convoluzionale (`conv2`) dell'estrattore affinché venga aggiornato insieme alla testa di classificazione durante l'addestramento nel dominio B. Confrontare l'accuratezza ottenuta con il congelamento totale e con l'addestramento da zero, discutendo il compromesso tra capacità di adattamento e rischio di overfitting.

4. **(20%)** Valutare l'influenza della profondità del congelamento (*freeze*) sulle prestazioni dei rilevatori **YOLO** sottoposti a transfer learning. Utilizzando il set di dati sintetico di forme geometriche, eseguire il fine-tuning variando il parametro di congelamento del *backbone* per $\{0, 5, 10, 15\}$. Registrare la metrica $\text{mAP}_{50}$ sul set di validazione per ogni configurazione, presentare i dati in una tabella e discutere se il congelamento parziale sia vantaggioso quando i domini di origine (COCO) e di destinazione (forme geometriche) sono significativamente distinti.

5. **(20%)** Studiare l'importanza delle **connessioni di salto** (*skip connections*) nell'architettura **U-Net** per la segmentazione semantica. Implementare una variazione `UNetSenzaSalti` che funzioni come un *autoencoder* convoluzionale tradizionale, rimuovendo le concatenazioni tra gli stadi dell'encoder e del decoder. Addestrare entrambi i modelli sulla stessa base di noduli sintetici, confrontare l'IoU medio sul set di validazione e presentare visivamente la differenza nella precisione dei bordi segmentati da ciascun metodo.

6. **(20%) — Sfida: segmentazione semantica con *Roboflow*.** Utilizzare un progetto *Roboflow* di tipo *instance segmentation*, contenente le sette classi di forme geometriche utilizzate in questo capitolo. Esportare il *dataset* nel formato `coco-segmentation` e sviluppare una procedura per convertire i poligoni memorizzati nei file `_annotations.coco.json` in maschere semantiche multiclasse, in cui ogni pixel riceve l'indice della classe corrispondente e il valore `0` rappresenta lo sfondo. Utilizzare le immagini e le maschere risultanti per addestrare la `UNetCompatta`. Valutare l'IoU medio sul set di test e confrontare visivamente le maschere previste con le annotazioni originali. Discutere le principali difficoltà incontrate nella conversione delle annotazioni COCO in maschere e gli effetti di oggetti sovrapposti o appartenenti a classi diverse.

7. **(Bonus – 10%)** Sviluppare un sistema interattivo che combini **rilevamento di oggetti (YOLO)** con **misurazione tramite riferimento di scala (fotogrammetria)**. Addestrare il rilevatore per identificare due classi in una scena: una "Carta di Riferimento" (dimensione nota di $8{,}56\text{ cm} \times 5{,}39\text{ cm}$) e un "Oggetto Target". Durante l'inferenza su una nuova immagine, utilizzare la dimensione in pixel della *bounding box* della carta rilevata per convertire le dimensioni della scatola dell'oggetto target in centimetri. Visualizzare l'immagine processata con le etichette di classe, la probabilità di confidenza e le dimensioni fisiche stimate sovrapposte.

8. **(Bonus – 10%)** Sviluppare un sistema di **stima della profondità tramite visione stereo** a partire da due immagini della stessa scena acquisite da posizioni diverse, simulando una coppia di telecamere stereo. Considerare nota la distanza tra le due posizioni di acquisizione (*baseline*).

   Utilizzare uno dei metodi di **rilevamento di oggetti** presentati nel capitolo, come YOLO, per localizzare gli oggetti di interesse nelle due immagini. Per ogni rilevamento, stabilire la corrispondenza tra lo stesso oggetto nelle due posizioni e determinare la sua **disparità**. A partire dalla disparità, dalla *baseline* e dai parametri della telecamera, utilizzare la geometria stereo per stimare la distanza di ciascun oggetto rispetto alle telecamere.

   Come estensione della libreria didattica `morph`, modificare il metodo `showBoundBox` affinché, oltre alla **classe** e alla **confidenza del rilevamento**, presenti su ogni *bounding box* la **distanza stimata dell'oggetto**. Il risultato deve consentire di visualizzare, direttamente nelle immagini, la classe, l'accuratezza (confidenza) e la profondità di ciascun oggetto rilevato.

   Presentare le due immagini con i rilevamenti, le corrispondenze tra gli oggetti, l'immagine di disparità e una rappresentazione della profondità stimata. Discutere come la distanza tra le telecamere, la precisione del rilevamento e della corrispondenza, la risoluzione delle immagini e la posizione dell'oggetto nella scena influenzino la qualità della stima.

   Per la validazione, utilizzare almeno un oggetto la cui distanza dalla telecamera sia nota. Confrontare la profondità stimata con il valore reale e riportare l'**errore assoluto** e l'**errore relativo**. Discutere inoltre le limitazioni del metodo quando un oggetto non viene rilevato correttamente in entrambe le immagini o quando la corrispondenza tra le regioni osservate è ambigua.

## 9.9 Chiusura della Parte II

Questo capitolo conclude la Parte II del libro e chiude la sequenza di contenuti iniziata nel Capitolo 6, dedicata alla rappresentazione, rilevazione, descrizione e corrispondenza delle caratteristiche nelle immagini. Nel corso di questi capitoli, sono stati presentati metodi classici di CV basati su **caratteristiche progettate manualmente**, come Sobel, LBP, HOG, ORB e Haar Cascade, nonché metodi fondati su **caratteristiche apprese automaticamente**, rappresentati dalle CNN.

Gli esempi e gli esperimenti sviluppati evidenziano che nessuno di questi approcci è universalmente superiore. La scelta della tecnica più adeguata dipende dalle caratteristiche del problema, dalla disponibilità di dati per l'addestramento, dai requisiti di precisione e dai vincoli computazionali dell'applicazione. In problemi ben strutturati e con pochi dati, i descrittori classici offrono spesso soluzioni semplici ed efficienti. Al contrario, compiti più complessi tendono a trarre beneficio dalla capacità di apprendimento offerta dalle CNN.

Diverse direzioni di studio possono approfondire i concetti presentati in questa parte del libro, tra le quali si distinguono:

- **Architetture moderne di CNN**, come ResNet, EfficientNet e *Vision Transformers*, che ampliano la capacità di rappresentazione e le prestazioni in compiti di classificazione e riconoscimento visivo;
- **Rilevazione e segmentazione di oggetti**, con particolare attenzione alla famiglia YOLO e ai modelli di segmentazione basati su *prompt*, come *Segment Anything*;
- **Ricostruzione tridimensionale e SLAM** (*Simultaneous Localization and Mapping*), che utilizzano più immagini per stimare la geometria della scena e la traiettoria di telecamere in movimento;
- **Modelli generativi di immagini**, come le reti generative avversarie (GAN) e i modelli di diffusione, capaci di sintetizzare immagini realistiche a partire da esempi o descrizioni testuali.

Le basi sviluppate nel corso della Parte II costituiscono il fondamento per queste e altre aree avanzate della CV, nelle quali la rappresentazione adeguata delle informazioni visive rimane l'elemento centrale per l'analisi e la comprensione delle immagini.

## Riferimenti del Capitolo

I concetti e gli algoritmi presentati in questo capitolo sono stati fondati su riferimenti classici e contemporanei della letteratura sul Deep Learning applicato alla VC:

* Mcculloch (1943), per la proposta della prima astrazione matematica e logica del neurone artificiale, che getta le basi concettuali dell'elaborazione neurale computazionale.
* Rosenblatt (1958), per la formulazione originale del **Perceptron**, modello precursore del neurone artificiale utilizzato nelle architetture moderne di deep learning.
* Goodfellow (2016) e Lecun (2015), per i fondamenti delle reti neurali, della convoluzione, delle funzioni di attivazione e dell'addestramento di modelli profondi.
* Bishop (2006), per i concetti rigorosi di riconoscimento di pattern, probabilità, stima di massima verosimiglianza e metodi statistici applicati al machine learning.
* Ronneberger (2015) per l'architettura **U-Net**, utilizzata nella segmentazione semantica con connessioni di skip tra codificatore e decodificatore.
* Redmon (2016), per l'architettura **YOLO** (*You Only Look Once*), utilizzata negli esperimenti di rilevamento degli oggetti con transfer learning.
* Ren (2015), per l'architettura **Faster R-CNN**, impiegata come modello pre-addestrato per il rilevamento degli oggetti.
* He (2016), per l'architettura **ResNet**, base di diversi estrattori di caratteristiche pre-addestrati utilizzati in questo capitolo.
* Chen (2018), per l'architettura **DeepLabV3**, utilizzata come modello pre-addestrato per la segmentazione semantica.
* Kirillov (2023), per il modello **Segment Anything (SAM)**, menzionato come direzione di studio per la segmentazione promptable.
* {google} (2025), relativo allo strumento **Gemini Notebook**, utilizzato nell'elaborazione dell'infografica di sintesi del capitolo e reso disponibile come supporto complementare allo studio.

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap09/cap09.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 9.10 💻 **Parte Pratica con Esercizi di Programmazione**

La presente lista di **Esercizi di Programmazione (EP)** consolida le formulazioni teoriche presentate nel Capitolo 9 — Deep Learning per la Visione Artificiale — tramite un percorso pratico applicato. Diversamente dall'addestramento di reti neurali complete con PyTorch, che richiede tempi di esecuzione e, talvolta, GPU, gli EP di questo capitolo isolano le **grandezze intermedie** di una *pipeline* reale di deep learning — l'output di un singolo strato convoluzionale, il risultato di un'operazione di *pooling*, il conteggio dei parametri addestrabili di un'architettura, la sovrapposizione tra bounding box candidate, la qualità di una maschera di segmentazione e il filtro di soppressione non-massima — consentendo di validare manualmente ogni fase del ragionamento senza dipendere da librerie di machine learning né da un addestramento reale.

L'incatenamento degli esercizi riproduce il flusso concettuale del capitolo e cresce in difficoltà a ogni passo: si inizia con il calcolo manuale dell'output di uno **strato convoluzionale addestrato** (🟢), a partire da un *kernel* e un bias già addestrati; si prosegue con l'operazione di ***pooling*** (🟢, massimo e media), che riduce la risoluzione spaziale tra blocchi convoluzionali; si continua con il **conteggio dei parametri addestrabili** (🟡) di un'architettura CNN completa, evidenziando perché la condivisione dei pesi rende queste reti così più economiche rispetto a uno strato completamente connesso equivalente; si approfondisce il calcolo dell'**Intersezione su Unione (IoU)** e della **Soppressione Non-Massima (NMS)** (🟡), fase di post-elaborazione comune a rilevatori come Faster R-CNN e YOLO; si passa alla **valutazione delle maschere di segmentazione** (🟠) con le stesse metriche IoU e Dice utilizzate per confrontare U-Net con la baseline morfologica classica; e si conclude con una ***pipeline* integrata** (🔴), unendo l'output di un rilevatore di oggetti (dopo NMS) a una misurazione del mondo reale tramite riferimento di scala — lo stesso principio della fotogrammetria studiato nell'integrazione finale del capitolo.

Ogni volta che ha senso, ciascun esercizio indica i metodi della libreria didattica `morph.py` (la stessa utilizzata nel capitolo, importata come `mm`) che risolvono una fase del problema o che servono da riferimento per verificare i propri calcoli — senza, tuttavia, sostituire il ragionamento che devi implementare.

> ### ❗ Linee Guida per la Risoluzione degli Esercizi di Programmazione
>
> In tutti gli esercizi di questo capitolo, le fasi di discretizzazione o arrotondamento numerico devono impiegare l'arrotondamento standard all'intero più vicino (*round half away from zero*), mitigando le ambiguità in valori con frazione esattamente uguale a $0{,}5$. Salvo indicazione esplicita contraria: (i) l'operazione di "convoluzione" segue la convenzione adottata dai *framework* di deep learning — **correlazione incrociata**, senza inversione spaziale del *kernel*, esattamente come presentato nella Sezione "Strato Convoluzionale"; (ii) il riempimento (*padding*) è effettuato con zeri; (iii) le bounding box sono specificate nel formato angolo-a-angolo $(x_1, y_1, x_2, y_2)$, con $x_1 < x_2$ e $y_1 < y_2$; e (iv) vettori/matrici seguono l'indicizzazione a partire da $0$, con la convenzione `[riga][colonna]` per strutture bidimensionali.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EPs)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

#### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella sottostante:

In [56]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

✅ Ambiente pronto. Morph: 1.1.9 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Esecuzione dei Test
Per valutare i test, esegui `TestSuite("EP09_01.extensão").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola automaticamente il voto.

Per testare direttamente il codice Python, senza salvare il file, usa `run_code(codigo)` passando il codice come *stringa* in una variabile `codigo`:

```python
codigo = """
# 9 ... il tuo codice qui ...
"""
TestSuite("EP09_01").run_code(codigo)
```

### 9.0.1 EP09_01 🟢 Convoluzione 2D Manuale (*Forward* di un Livello Appreso)

Il PyTorch, presentato in questo capitolo, esegue `nn.Conv2d(x)` in un'unica chiamata — ma dietro di essa c'è solo la correlazione incrociata tra un *kernel* (già addestrato) e un intorno dell'input, seguita dalla somma di un bias e di un'attivazione, esattamente come formalizzato nella Sezione "Livello Convoluzionale". La differenza essenziale rispetto alla convoluzione con *kernel* fissi del Capitolo 3 è che, qui, i valori del *kernel* e del bias **sono già pronti** (come se fossero stati appresi per gradiente), e spetta a te riprodurre manualmente il passaggio diretto (*forward pass*) che il *framework* esegue internamente.

Prima di addestrare una vera CNN, ti è stato affidato il compito di implementare questo passaggio diretto da zero, per un singolo livello convoluzionale con un singolo canale di input e un singolo filtro di output, incluso il supporto a *padding* e *stride* arbitrari.

#### 9.0.1.1 📋 Linee Guida di Implementazione

1. **Input:** Leggere le dimensioni $H \times W$ della mappa delle caratteristiche di input e, successivamente, i suoi $H \times W$ valori reali.
   
2. ***Kernel* e bias:** Leggere le dimensioni $k_h \times k_w$ del *kernel* (già addestrato), i suoi valori reali, e il bias $b$ (reale, scalare).
   
3. **Iperparametri:** Leggere il *padding* $p$ (intero, numero di zeri aggiunti su ciascun bordo) e lo *stride* $s$ (intero, passo dello scorrimento).
   
4. **Riempimento:** Aggiungere $p$ zeri su ciascuno dei quattro bordi della mappa di input prima della correlazione.
   
5. **Correlazione incrociata:** Per ogni posizione di output $(i, j)$, calcolare
   $$
   z(i,j) = b + \sum_{u=0}^{k_h-1} \sum_{v=0}^{k_w-1} K(u,v) \cdot X_{pad}(i \cdot s + u,\; j \cdot s + v),
   $$
   scorrendo l'input **senza** invertire il *kernel* (convenzione dei *framework* di deep learning, diversa dalla convoluzione matematica classica).

6. **Attivazione:** Applicare ReLU a ogni valore: $a(i,j) = \max(0, z(i,j))$.

7. **Dimensioni di output:** $O_h = \lfloor (H + 2p - k_h)/s \rfloor + 1$ e $O_w = \lfloor (W + 2p - k_w)/s \rfloor + 1$.

8. **Output:** Stampare $O_h$ e $O_w$ nella prima riga, seguiti da $O_h$ righe con $O_w$ valori reali ciascuna (la mappa delle caratteristiche di output, già con ReLU applicata), formattati con 4 cifre decimali.

#### 9.0.1.2 📌 Vincoli Computazionali

* **Un canale di input, un filtro di output:** non è necessario gestire più canali o più filtri in questa versione semplificata.
* **Senza inversione del *kernel*:** implementare la correlazione incrociata, non la convoluzione matematica classica con *kernel* invertito — è questa l'operazione che PyTorch (e la maggior parte dei *framework*) chiama "convoluzione".
* **Riempimento con zeri:** i $p$ pixel aggiunti su ciascun bordo valgono sempre $0$.
* **Formattazione:** tutti i valori di output devono avere esattamente 4 cifre decimali, anche quando il valore è un intero (es.: `2.0000`).

#### 9.0.1.3 🧠 Fondamenti Teorici

| Elemento | Ruolo nel livello convoluzionale |
|---|---|
| *Kernel* $K$ | Parametri appresi per gradiente, analoghi ai coefficienti di un filtro fisso del Capitolo 3, ma regolati tramite backpropagation |
| Bias $b$ | Offset appreso, sommato dopo la correlazione — consente al neurone di "attivarsi" anche con input nullo |
| *Padding* | Controlla la dimensione spaziale dell'output e previene la perdita di informazioni ai bordi a ogni livello |
| *Stride* | Controlla il passo dello scorrimento; valori $> 1$ riducono la risoluzione spaziale, come una forma di sottocampionamento integrato nella convoluzione stessa |
| ReLU | Introduce non linearità dopo la combinazione lineare, esattamente come nella Sezione "Funzione di Attivazione" |

#### 9.0.1.4 🧩 Metodi di `morph.py` che possono aiutare

* `mm.readImg(h, w, dtype='float')` — legge direttamente una matrice $h \times w$ di valori reali dall'input standard, evitando il *parsing* manuale della mappa delle caratteristiche e del *kernel*.
* `mm.correlacao0(f, kernel, bias)` — implementa la stessa somma di correlazione incrociata + bias che dovrai calcolare a mano, ma **senza** supporto per *padding* o *stride*, e converte il risultato in `uint8` (tronca valori negativi e decimali). Può servire come riferimento concettuale o per verificare il caso più semplice ($p=0$, $s=1$), ma non sostituisce la tua implementazione completa — che deve preservare segno, cifre decimali, *padding*, *stride* e ReLU.

#### 9.0.1.5 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ valori reali ciascuna (mappa di input).
* Riga successiva: Interi $k_h$ e $k_w$.
* Prossime $k_h$ righe: $k_w$ valori reali ciascuna (*kernel*).
* Riga successiva: Reale $b$ (bias).
* Riga successiva: Interi $p$ e $s$.

**Output:**

* Riga 1: Interi $O_h$ e $O_w$.
* Prossime $O_h$ righe: $O_w$ valori reali ciascuna, con 4 cifre decimali.

#### 9.0.1.6 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 1<br>1 1<br>-2<br>0 1 | 2 2<br>2.0000 3.0000<br>0.0000 2.0000 | *Padding* 0, *stride* 1: output $2\times2$ senza riempimento. |
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 0<br>0 1<br>0<br>1 2 | 2 2<br>1.0000 0.0000<br>1.0000 2.0000 | *Padding* 1, *stride* 2: input riempito con zeri prima della correlazione. |

In [57]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0901" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Convoluzione 2D Manuale</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 correlazione incrociata + bias + ReLU</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ingresso 4×4 fisso, kernel 2×2 fisso (evidenziato in blu) &mdash; regola <em>padding</em> (p), <em>stride</em> (s) e bias (b), esattamente i parametri che l'EP09_01 richiede in ingresso, e osserva come cambiano la dimensione e i valori dell'uscita.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Padding (p)</div>
        <div id="ep0901_pad_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0901_stride_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Bias (b)</div>
        <input id="ep0901_bias" type="number" step="0.5" value="0.5" style="width:70px;font-family:monospace;text-align:center;border:1px solid #ccc;border-radius:6px;padding:3px;">
      </div>
    </div>

    <div id="ep0901_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Posizione di uscita (i,j)</label>
        <span id="ep0901_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0901_sl" style="width:100%;accent-color:#2980b9;" max="8" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Ingresso X imbottito (con padding)</div>
        <div id="ep0901_grid" style="display:grid;gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> originale</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#f5f5f5;border:1px dashed #ccc;border-radius:2px;vertical-align:middle;"></span> padding (0)</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> finestra corrente</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Kernel K (2×2)</div>
        <div id="ep0901_kernel" style="display:grid;grid-template-columns:repeat(2,44px);gap:3px;"></div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Uscita Y = ReLU(X⊛K + b)</div>
        <div id="ep0901_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0901_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ riavvia l'esplorazione</button>
    </div>
    <div id="ep0901_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Ogni posizione dello slider rivela una cella della matrice di uscita. Percorri tutte le posizioni per completare la mappa di uscita. Cambiare p, s o b riavvia l'esplorazione, perché la mappa di uscita cambia dimensione e/o valori.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4, kh = 2, kw = 2;
    var X = [[1,3,2,0],[0,1,4,1],[2,0,1,3],[1,2,0,1]];
    var K = [[1,0],[0,-1]];

    var state = { p: 0, s: 1, bias: 0.5 };
    var visited = {};

    var slEl = root.querySelector('#ep0901_sl');
    var vlEl = root.querySelector('#ep0901_vl');
    var gridEl = root.querySelector('#ep0901_grid');
    var kernelEl = root.querySelector('#ep0901_kernel');
    var outEl = root.querySelector('#ep0901_out');
    var dbg = root.querySelector('#ep0901_debug');
    var formulaEl = root.querySelector('#ep0901_formula');
    var resetBtn = root.querySelector('#ep0901_reset');
    var padBtnsEl = root.querySelector('#ep0901_pad_btns');
    var strideBtnsEl = root.querySelector('#ep0901_stride_btns');
    var biasInput = root.querySelector('#ep0901_bias');

    kernelEl.innerHTML = '';
    for(var u=0; u<kh; u++) for(var v=0; v<kw; v++){
      var kd = document.createElement('div');
      kd.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;background:#bbdefb;border:1px solid #64b5f6;border-radius:6px;font-family:monospace;font-weight:bold;color:#0d47a1;';
      kd.textContent = K[u][v];
      kernelEl.appendChild(kd);
    }

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function buildPadded(p){
      var size = H + 2*p;
      var Xp = [];
      for(var r=0; r<size; r++){
        var row = [];
        for(var c=0; c<size; c++){
          var origR = r-p, origC = c-p;
          var isPad = !(origR>=0 && origR<H && origC>=0 && origC<W);
          row.push({ val: isPad ? 0 : X[origR][origC], pad: isPad });
        }
        Xp.push(row);
      }
      return Xp;
    }

    function computeAll(p, s, bias){
      var Xp = buildPadded(p);
      var size = H + 2*p;
      var Oh = Math.floor((size - kh)/s) + 1;
      var Ow = Math.floor((size - kw)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var soma = 0;
          for(var u=0; u<kh; u++) for(var v=0; v<kw; v++) soma += K[u][v]*Xp[i*s+u][j*s+v].val;
          var z = soma + bias;
          var a = Math.max(0, z);
          vals[i].push({ soma: soma, z: z, a: a });
        }
      }
      return { Xp: Xp, size: size, Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.p, state.s, state.bias);
      gridEl.style.gridTemplateColumns = 'repeat(' + model.size + ', ' + Math.min(44, Math.floor(360/model.size)) + 'px)';
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 44px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' + 2·' + state.p + ' − ' + kh + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' + 2·' + state.p + ' − ' + kw + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;

      gridEl.innerHTML = '';
      var cellPx = Math.min(44, Math.floor(360/model.size));
      for(var r=0; r<model.size; r++){
        for(var c=0; c<model.size; c++){
          var cell = model.Xp[r][c];
          var dentroJanela = (r>=winRowStart && r<winRowStart+kh && c>=winColStart && c<winColStart+kw);
          var d = document.createElement('div');
          var base = 'width:'+cellPx+'px;height:'+cellPx+'px;display:flex;align-items:center;justify-content:center;border-radius:5px;font-family:monospace;font-size:11px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(cell.pad){
            base += 'background:#f5f5f5;border:1px dashed #ccc;color:#bbb;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = cell.val;
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          var isCurrent = (oi===i && oj===j);
          var wasVisited = !!visited[oi+','+oj];
          var od = document.createElement('div');
          var style = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi][oj].a.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela em ('+i+','+j+'), topo-esquerda em X_pad('+winRowStart+','+winColStart+')  |  soma(X⊙K)='+cur.soma.toFixed(2)+'  +  viés='+state.bias.toFixed(2)+'  =  z='+cur.z.toFixed(2)+'  →  ReLU(z)='+cur.a.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setPadding(val){
      state.p = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    function setStride(val){
      state.s = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
    buildButtons(strideBtnsEl, [1,2], state.s, setStride);

    biasInput.addEventListener('change', function(){
      var v = parseFloat(biasInput.value);
      state.bias = isNaN(v) ? 0 : v;
      rebuildModel(true);
    });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0901');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.43:** Simulatore EP09_01: Convoluzione 2D Manuale (correlazione incrociata + bias + ReLU, con *padding* e *stride* regolabili)


<figure id="fig-09-sim-ep0901">
  <img src="imagens/fig-09-sim-ep0901.png" alt=" Simulatore EP09_01: Convoluzione 2D Manuale (correlazione incrociata + bias + ReLU, con *padding* e *stride* regolabili) " style="max-width:80%" />
  <figcaption><strong>Figura 9.43:</strong>  Simulatore EP09_01: Convoluzione 2D Manuale (correlazione incrociata + bias + ReLU, con *padding* e *stride* regolabili) </figcaption>
</figure>

In [58]:
%%writefile EP09_01.py
# Codice Python

Overwriting EP09_01.py


In [59]:
TestSuite("EP09_01.py").run()

### 9.0.2 EP09_02 🟢 *Pooling* Manuale (Massimo e Media)

Tra i blocchi convoluzionali, l'architettura tipica di una CNN intercala livelli di ***pooling***, che riducono la risoluzione spaziale della mappa delle caratteristiche senza introdurre nuovi parametri addestrabili — a differenza della convoluzione, il *pooling* non ha pesi: si limita a riassumere ogni finestra dell'ingresso in un singolo valore, tramite un massimo o una media, esattamente come formalizzato nella Sezione "*Pooling*".

Sei stato incaricato di implementare questa operazione a partire da una finestra scorrevole quadrata, senza sovrapposizione parziale sui bordi (solo finestre complete), supportando i due tipi più comuni: `max` (preserva il valore più saliente, tipicamente usato per mantenere bordi e texture forti) e `avg` (smussa la regione, preservando l'informazione di intensità media).

#### 9.0.2.1 📋 Linee Guida di Implementazione

1. **Ingresso:** Leggere le dimensioni $H \times W$ della mappa delle caratteristiche di ingresso e i suoi $H \times W$ valori reali.
2. **Finestra:** Leggere gli interi $k$ (dimensione della finestra quadrata $k \times k$) e $s$ (*stride*).
3. **Tipo:** Leggere una *stringa*, `max` o `avg`, che indica il tipo di *pooling*.
4. **Senza riempimento:** Questa operazione **non** utilizza *padding*; le finestre che supererebbero il bordo dell'ingresso vengono scartate.
5. **Calcolo:** Per ogni posizione di uscita $(i,j)$, calcolare il massimo o la media dei $k \times k$ valori della finestra corrispondente, iniziando da $(i \cdot s,\, j \cdot s)$.
6. **Dimensioni di uscita:** $O_h = \lfloor (H - k)/s \rfloor + 1$ e $O_w = \lfloor (W - k)/s \rfloor + 1$.
7. **Uscita:** Stampare $O_h$ e $O_w$ nella prima riga, seguiti da $O_h$ righe con $O_w$ valori reali ciascuna, formattati con 4 cifre decimali.

#### 9.0.2.2 📌 Vincoli Computazionali

* **Finestra quadrata:** $k \times k$, senza supporto per finestre rettangolari in questa versione.
* **Senza *padding*:** solo le finestre interamente contenute nell'ingresso sono considerate — le dimensioni che "avanzeranno" sono semplicemente scartate.
* **`avg` usa divisione reale:** la media è sempre $\text{somma}/k^2$, anche quando il risultato ha molte cifre decimali — arrotondare solo nella formattazione finale, secondo la linea guida generale del capitolo.
* **Formattazione:** tutti i valori di uscita con esattamente 4 cifre decimali.

#### 9.0.2.3 🧠 Fondamento Teorico

| Elemento | Ruolo nell'architettura |
|---|---|
| *Pooling* massimo | Preserva l'attivazione più forte della finestra; comune dopo livelli convoluzionali per mantenere bordi e texture salienti |
| *Pooling* medio | Smussa la regione, preservando l'intensità media; comune nei livelli finali (*global average pooling*) |
| Assenza di parametri | Differenzia il *pooling* dalla convoluzione: riduce la risoluzione spaziale senza costi aggiuntivi di addestramento |
| Riduzione della risoluzione | Contribuisce all'invarianza rispetto a piccole traslazioni e alla riduzione del costo computazionale dei livelli successivi |

#### 9.0.2.4 🧩 Metodi di `morph.py` che possono aiutare

Il `morph.py` non implementa il *pooling* con sottocampionamento direttamente, ma due famiglie di operazioni mostrano la stessa idea sotto un'altra ottica, utile per verificare la tua intuizione:

* `mm.dil(f, Bc)` / `mm.dil0(f, B)` — dilatazione morfologica: sostituisce ogni pixel con il **massimo** del suo intorno definito dall'elemento strutturante $B$ (es.: `mm.sebox(n)` per una finestra $(2n+1)\times(2n+1)$). È concettualmente un "*max-pooling* senza sottocampionamento" (produce un'immagine della stessa dimensione, invece che ridotta).
* `mm.blur(f, N)` — smussatura per media in una finestra $N \times N$, analoga all'*avg-pooling*, anch'essa senza riduzione della risoluzione.
* `mm.readImg(h, w, dtype='float')` — utile per leggere la mappa di ingresso in virgola mobile.

#### 9.0.2.5 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ valori reali ciascuna.
* Riga successiva: Interi $k$ e $s$.
* Riga successiva: `max` o `avg`.

**Uscita:**

* Riga 1: Interi $O_h$ e $O_w$.
* Prossime $O_h$ righe: $O_w$ valori reali ciascuna, con 4 cifre decimali.

#### 9.0.2.6 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>max | 2 2<br>6.0000 4.0000<br>4.0000 5.0000 | *Pooling* massimo, finestra $2\times2$, *stride* 2. |
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>avg | 2 2<br>3.7500 2.2500<br>2.2500 2.2500 | *Pooling* medio sulle stesse finestre. |

In [60]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0902" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Pooling Manuale</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 senza padding, finestre complete</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ingresso 4×4 fisso &mdash; regola la dimensione della finestra (k), lo stride (s) e il tipo, esattamente i parametri che EP09_02 legge in ingresso, e guarda come cambiano la dimensione e i valori dell'uscita.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Finestra (k)</div>
        <div id="ep0902_k_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0902_s_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Tipo</div>
        <div style="display:flex;gap:8px;">
          <button id="ep0902_max" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #f0ad4e;background:#fff3cd;color:#7a5c00;font-weight:bold;font-size:11px;">max</button>
          <button id="ep0902_avg" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #ddd;background:#f3f4f6;color:#555;font-weight:bold;font-size:11px;">avg</button>
        </div>
      </div>
    </div>

    <div id="ep0902_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Posizione di uscita (i,j)</label>
        <span id="ep0902_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0902_sl" style="width:100%;accent-color:#2980b9;" max="3" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Ingresso X (4×4)</div>
        <div id="ep0902_grid" style="display:grid;grid-template-columns:repeat(4,44px);gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> fuori dalla finestra</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> finestra corrente</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#eee;border:1px dashed #bbb;border-radius:2px;vertical-align:middle;"></span> scartato (avanzo)</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Uscita Y (pooling)</div>
        <div id="ep0902_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0902_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ riavvia esplorazione</button>
    </div>
    <div id="ep0902_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Ogni posizione dello slider rivela una cella della matrice di uscita. Le celle grigio tratteggiate nell'ingresso sono "avanzi" che nessuna finestra raggiunge &mdash; nota come ciò accade quando (H&minus;k) non è multiplo di s. Cambiare k, s o il tipo riavvia l'esplorazione.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4;
    var X = [[1,3,2,4],[5,6,1,2],[2,1,0,3],[4,2,5,1]];

    var state = { k: 2, s: 2, tipo: 'max' };
    var visited = {};

    var slEl = root.querySelector('#ep0902_sl');
    var vlEl = root.querySelector('#ep0902_vl');
    var gridEl = root.querySelector('#ep0902_grid');
    var outEl = root.querySelector('#ep0902_out');
    var dbg = root.querySelector('#ep0902_debug');
    var formulaEl = root.querySelector('#ep0902_formula');
    var resetBtn = root.querySelector('#ep0902_reset');
    var kBtnsEl = root.querySelector('#ep0902_k_btns');
    var sBtnsEl = root.querySelector('#ep0902_s_btns');
    var btnMax = root.querySelector('#ep0902_max');
    var btnAvg = root.querySelector('#ep0902_avg');

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function estiloTipoBotoes(){
      btnMax.style.background = state.tipo==='max' ? '#fff3cd' : '#f3f4f6';
      btnMax.style.borderColor = state.tipo==='max' ? '#f0ad4e' : '#ddd';
      btnMax.style.color = state.tipo==='max' ? '#7a5c00' : '#555';
      btnAvg.style.background = state.tipo==='avg' ? '#fff3cd' : '#f3f4f6';
      btnAvg.style.borderColor = state.tipo==='avg' ? '#f0ad4e' : '#ddd';
      btnAvg.style.color = state.tipo==='avg' ? '#7a5c00' : '#555';
    }

    function computeAll(k, s, tipo){
      var Oh = Math.floor((H - k)/s) + 1;
      var Ow = Math.floor((W - k)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var janela = [];
          for(var r=i*s; r<i*s+k; r++) for(var c=j*s; c<j*s+k; c++) janela.push(X[r][c]);
          var resultado = tipo === 'max'
            ? Math.max.apply(null, janela)
            : janela.reduce(function(a,b){return a+b;},0)/janela.length;
          vals[i].push({ janela: janela, resultado: resultado });
        }
      }
      return { Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.k, state.s, state.tipo);
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 48px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      estiloTipoBotoes();
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;
      var alcancavel = []; // marca quais células de X são alcançadas por ALGUMA janela válida
      for(var r=0;r<H;r++){ alcancavel.push(new Array(W).fill(false)); }
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          for(var r=oi*state.s; r<oi*state.s+state.k; r++)
            for(var c=oj*state.s; c<oj*state.s+state.k; c++)
              alcancavel[r][c] = true;
        }
      }

      gridEl.innerHTML = '';
      for(var r=0; r<H; r++){
        for(var c=0; c<W; c++){
          var dentroJanela = (r>=winRowStart && r<winRowStart+state.k && c>=winColStart && c<winColStart+state.k);
          var d = document.createElement('div');
          var base = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(!alcancavel[r][c]){
            base += 'background:#eee;border:1px dashed #bbb;color:#999;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = X[r][c];
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi2=0; oi2<model.Oh; oi2++){
        for(var oj2=0; oj2<model.Ow; oj2++){
          var isCurrent = (oi2===i && oj2===j);
          var wasVisited = !!visited[oi2+','+oj2];
          var od = document.createElement('div');
          var style = 'width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi2][oj2].resultado.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela=['+cur.janela.join(', ')+']  |  tipo='+state.tipo+'  →  resultado='+cur.resultado.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setK(val){
      state.k = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    function setS(val){
      state.s = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
    buildButtons(sBtnsEl, [1,2,3], state.s, setS);

    btnMax.addEventListener('click', function(){ state.tipo='max'; rebuildModel(true); });
    btnAvg.addEventListener('click', function(){ state.tipo='avg'; rebuildModel(true); });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0902');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figura 9.44:** Simulatore EP09_02: Pooling Manuale (massimo vs. media, con finestra k e stride s regolabili)


<figure id="fig-09-sim-ep0902">
  <img src="imagens/fig-09-sim-ep0902.png" alt=" Simulatore EP09_02: Pooling Manuale (massimo vs. media, con finestra k e stride s regolabili) " style="max-width:80%" />
  <figcaption><strong>Figura 9.44:</strong>  Simulatore EP09_02: Pooling Manuale (massimo vs. media, con finestra k e stride s regolabili) </figcaption>
</figure>

In [61]:
%%writefile EP09_02.py
# Codice Python

Overwriting EP09_02.py


In [62]:
TestSuite("EP09_02.py").run()

### 9.0.3 EP09_03 🟡 Conteggio dei Parametri Addestrabili di una CNN

Questo EP formalizza il conteggio dei parametri addestrabili di una *CNN*. Data la descrizione testuale di una piccola architettura, composta da layer convoluzionali, di *pooling* e completamente connessi, determinare, per ogni layer, il numero di parametri addestrabili e il totale della rete.

L'architettura deve essere interpretata **sequenzialmente**: l'uscita di un layer convoluzionale diventa l'ingresso del layer successivo compatibile. Pertanto, il numero di canali prodotti da un layer `CONV` determina il numero di canali di ingresso (`cin`) del layer convoluzionale successivo.

In un layer convoluzionale, è importante distinguere **canali di ingresso** e **canali di uscita**:

* $c_{in}$ (*channels in*) è il numero di **canali che entrano nel layer**. Un'immagine in scala di grigi ha $c_{in}=1$, mentre un'immagine RGB ha $c_{in}=3$. In un layer convoluzionale intermedio, `cin` è normalmente uguale al numero di canali prodotti dal layer `CONV` precedente.
* $c_{out}$ (*channels out*) è il numero di **canali prodotti dal layer**. È uguale al numero di filtri utilizzati. Pertanto, se un layer ha 16 filtri, produce $c_{out}=16$ canali.

Ad esempio, si consideri la sequenza:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
POOL
FC 784 10 1
```

La prima convoluzione riceve un'immagine con un canale e produce 8 canali. Dopo il *pooling*, la seconda convoluzione riceve questi 8 canali e ne produce 16. Il layer `POOL` non altera il numero di canali, può solo ridurre le dimensioni spaziali. Il layer `FC` riceve la quantità di ingressi indicata nella propria descrizione.

Ogni filtro convoluzionale ha dimensioni

$$
k_h \times k_w \times c_{in}.
$$

Pertanto, un layer con $c_{out}$ filtri ha

$$
k_h \cdot k_w \cdot c_{in} \cdot c_{out}
$$

pesi. Se c'è un bias, si aggiunge un parametro per ogni filtro, totalizzando altri $c_{out}$ parametri.

Il punto centrale di questo esercizio è osservare che la quantità di parametri di un layer convoluzionale **non dipende dalle dimensioni spaziali** ($H \times W$) della mappa delle caratteristiche. Ciò è dovuto alla **condivisione dei pesi**: lo stesso filtro viene riutilizzato in diverse posizioni dell'ingresso.

#### 9.0.3.1 📋 Linee Guida di Implementazione

1. **Ingresso:** Leggere l'intero $L$ (numero di layer dell'architettura, nell'ordine in cui vengono applicati).

2. **Layer:** Leggere $L$ righe, ciascuna descrive un layer in uno dei tre formati:

   * `CONV kh kw cin cout bias` — layer convoluzionale con *kernel* $k_h \times k_w$, $c_{in}$ canali di ingresso, $c_{out}$ canali di uscita e `bias` (0 o 1), che indica se c'è un bias per filtro;
   * `POOL` — layer di *pooling* (massimo o medio), che non ha parametri addestrabili e preserva il numero di canali;
   * `FC in out bias` — layer completamente connesso con `in` ingressi, `out` uscite e `bias` (0 o 1), che indica se c'è un bias per neurone.

3. **Coerenza tra layer `CONV`:** in una sequenza di layer convoluzionali, il `cin` di un layer deve corrispondere al `cout` del layer convoluzionale precedente. Un layer `POOL` non altera questo numero di canali.

   Ad esempio:

   ```text
   CONV 3 3 1 8 1
   POOL
   CONV 3 3 8 16 1
   ```

   La prima `CONV` produce 8 canali, che vengono ricevuti dalla seconda `CONV`. Pertanto, nel secondo layer, `cin=8` e `cout=16`.

4. **Parametri di un layer `CONV`:**

   Ciascuno dei $c_{out}$ filtri ha $k_h \cdot k_w \cdot c_{in}$ pesi. Pertanto,

   $$
   P_{\mathrm{CONV}} =
   k_h \cdot k_w \cdot c_{in} \cdot c_{out}
   +
   c_{out}\cdot\text{bias}.
   $$

5. **Parametri di un layer `FC`:**

   $$
   P_{\mathrm{FC}} = 
   \text{in}\cdot\text{out}
   +
   \text{out}\cdot\text{bias}.
   $$

6. **Parametri di un layer `POOL`:** sempre $0$.

7. **Totale della rete:** sommare i parametri addestrabili di tutti i layer.

8. **Uscita:** Per ogni layer, nell'ordine di lettura, stampare `Camada i: P`, dove $i$ inizia da $1$ e $P$ è il numero di parametri di quel layer. Alla fine, stampare `Total: T`.

#### 9.0.3.2 📐 Esempio per capire `cin` e `cout`

Si consideri la sequenza:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
```

Nel primo layer:

* `cin=1`: entra un canale;
* `cout=8`: ci sono 8 filtri e, quindi, escono 8 canali.

Ogni filtro ha

$$
3\cdot3\cdot1=9
$$

pesi. Poiché ci sono 8 filtri:

$$
9\cdot8=72
$$

pesi. Con un bias per filtro:

$$
72+8=80.
$$

Nel secondo layer:

* `cin=8`: entrano gli 8 canali prodotti dalla prima `CONV`;
* `cout=16`: ci sono 16 filtri e, quindi, escono 16 canali.

Ogni filtro ha

$$
3\cdot3\cdot8=72
$$

pesi. Poiché ci sono 16 filtri:

$$
72\cdot16=1152
$$

pesi. Con 16 bias:

$$
1152+16=1168.
$$

Pertanto, i due layer hanno, rispettivamente, **80** e **1168 parametri addestrabili**.

Si noti che `cout` **non è** $cin$ moltiplicato per il numero di filtri. Il numero di filtri è esattamente `cout`: ogni filtro combina tutti i canali di ingresso e produce **un singolo canale di uscita**.

#### 9.0.3.3 📌 Vincoli Computazionali

* **Indipendenza dalla dimensione spaziale:** l'ingresso non fornisce $H \times W$. Il conteggio di un layer `CONV` dipende solo da `kh`, `kw`, `cin` e `cout`.
* **Coerenza dei canali:** per due layer `CONV` consecutivi, il `cin` del secondo deve essere uguale al `cout` del primo. Un layer `POOL` preserva il numero di canali.
* **`bias` sempre 0 o 1:** moltiplicare direttamente il termine di bias per questo valore.
* **Layer `POOL` senza argomenti aggiuntivi:** la riga contiene solo la parola `POOL`.
* **Layer `FC`:** il numero di ingressi `in` è fornito esplicitamente. Non è necessario calcolare le dimensioni spaziali prodotte dai layer precedenti.
* Tutti i valori numerici di ingresso sono interi non negativi.

#### 9.0.3.4 🧠 Fondamenti Teorici

| Elemento                  | Ruolo nel conteggio dei parametri                                                                               |
| ------------------------- | -------------------------------------------------------------------------------------------------------------- |
| $c_{in}$                  | Numero di canali ricevuti dal layer                                                                             |
| $c_{out}$                 | Numero di filtri e, quindi, di canali prodotti dal layer                                                        |
| Filtro convoluzionale     | Ogni filtro ha $k_h \cdot k_w \cdot c_{in}$ pesi e produce un canale di uscita                                 |
| Condivisione dei pesi     | Lo stesso filtro viene riutilizzato in diverse posizioni dell'ingresso, rendendo il conteggio indipendente da $H \times W$ |
| Bias                      | Un singolo parametro aggiuntivo per filtro (`CONV`) o per neurone (`FC`)                                        |
| *Pooling*                 | Può alterare $H \times W$, ma non ha parametri addestrabili e preserva il numero di canali                      |
| Layer `FC`                | Ha un peso per ogni combinazione tra ingresso e neurone di uscita                                               |

#### 9.0.3.5 🧩 Metodi di `morph.py` che possono aiutare

Questo esercizio è puramente aritmetico e non utilizza direttamente le funzioni di `morph.py`. Il conteggio può, tuttavia, essere verificato in un'architettura reale implementata in PyTorch tramite:

```python
sum(p.numel() for p in modelo.parameters())
```

Questa espressione calcola i parametri del modello, inclusi pesi e bias.

#### 9.0.3.6 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $L$.
* Prossime $L$ righe: descrizione di ogni layer, nel formato `CONV kh kw cin cout bias`, `POOL` o `FC in out bias`.

**Uscita:**

* $L$ righe nel formato `Camada i: P`.
* Ultima riga: `Total: T`.

#### 9.0.3.7 📌 Esempi

| Ingresso                                                                            | Uscita                                                                                                         | Osservazione                                                                                                        |
| ----------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------ |
| 3<br>CONV 3 3 1 8 1<br>POOL<br>FC 1352 10 1                                         | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 13530<br>Total: 13610                                                 | Rete semplice con una convoluzione, *pooling* e layer di classificazione.                                          |
| 5<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 400 10 1               | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 4010<br>Total: 5258                  | Piccola CNN con due convoluzioni, due *pooling* e un layer completamente connesso.                                 |
| 6<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 256 32 1<br>FC 32 10 1 | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 8224<br>Camada 6: 330<br>Total: 9802 | Piccola CNN con due convoluzioni, *pooling* intermedio e due layer completamente connessi per la classificazione.  |

In [63]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0903" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Conteggio dei Parametri</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 condivisione dei pesi</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>CONV</b> Blocco blu
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:50%;display:inline-block;"></span>
        <b>POOL</b> Cilindro verde
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;display:inline-block;transform:rotate(45deg);"></span>
        <b>FC</b> Rombo arancione
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;display:inline-block;"></span>
        <b>BATCH</b> Pila rossa
      </span>
      <span style="display:flex;align-items:center;gap:3px;color:#666;">
        🖱️ Trascina per spostare i livelli
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">H×W</label>
            <span id="ep0903_hw_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">32×32</span>
          </div>
          <input id="ep0903_hw" style="width:100%;accent-color:#2980b9;height:4px;" max="64" min="8" step="2" type="range" value="32">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Canali</label>
            <span id="ep0903_cin_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">1</span>
          </div>
          <input id="ep0903_cin" style="width:100%;accent-color:#2980b9;height:4px;" max="3" min="1" step="1" type="range" value="1">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Batch</label>
            <span id="ep0903_batch_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">4</span>
          </div>
          <input id="ep0903_batch" style="width:100%;accent-color:#2980b9;height:4px;" max="16" min="1" step="1" type="range" value="4">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Livelli</label>
            <span id="ep0903_nlayers_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">3</span>
          </div>
          <input id="ep0903_nlayers" style="width:100%;accent-color:#2980b9;height:4px;" max="6" min="1" step="1" type="range" value="3">
        </div>
      </div>
      
      <!-- Configuração das camadas compacta -->
      <div id="ep0903_layers_config" style="margin-bottom:8px;display:flex;flex-wrap:wrap;gap:6px;">
        <!-- Gerado dinamicamente -->
      </div>
      
      <div style="display:flex;gap:12px;align-items:center;font-size:10px;">
        <label style="display:flex;align-items:center;gap:4px;cursor:pointer;">
          <input id="ep0903_bias" type="checkbox" checked style="accent-color:#2980b9;width:14px;height:14px;">
          <span style="font-weight:bold;color:#2980b9;">Usa bias</span>
        </label>
      </div>
    </div>
    
    <!-- Visualização 3D -->
    <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;margin-bottom:12px;min-height:400px;">
      <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">
        🧠 Visualizzazione 3D
      </div>
      
      <div style="position:absolute;top:8px;right:8px;display:flex;gap:4px;z-index:10;">
        <button id="ep0903_pause_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          ⏸️ Pausa
        </button>
        <button id="ep0903_reset_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          🔄 Ripristina
        </button>
        <button id="ep0903_auto_layout_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          📐 Automatico
        </button>
      </div>
      
      <canvas id="ep0903_canvas" style="width:100%;height:340px;display:block;cursor:grab;"></canvas>
      
      <div style="position:absolute;bottom:6px;right:8px;color:white;font-size:9px;background:rgba(0,0,0,0.5);padding:3px 8px;border-radius:14px;">
        🖱️ Trascina livelli | Scroll zoom | P pausa
      </div>
    </div>
    
    <!-- Resumo compacto -->
    <div id="ep0903_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var hwEl = root.querySelector('#ep0903_hw'), hwvEl = root.querySelector('#ep0903_hw_v');
    var cinEl = root.querySelector('#ep0903_cin'), cinvEl = root.querySelector('#ep0903_cin_v');
    var batchEl = root.querySelector('#ep0903_batch'), batchvEl = root.querySelector('#ep0903_batch_v');
    var nlayersEl = root.querySelector('#ep0903_nlayers'), nlayersvEl = root.querySelector('#ep0903_nlayers_v');
    var layersConfigEl = root.querySelector('#ep0903_layers_config');
    var biasEl = root.querySelector('#ep0903_bias');
    var summaryEl = root.querySelector('#ep0903_summary');
    var canvas = root.querySelector('#ep0903_canvas');
    var ctx = canvas.getContext('2d');
    var pauseBtn = root.querySelector('#ep0903_pause_btn');
    var resetBtn = root.querySelector('#ep0903_reset_btn');
    var autoLayoutBtn = root.querySelector('#ep0903_auto_layout_btn');
    
    // Estado da visualização
    var rotationX = -0.3;
    var rotationY = 0.5;
    var zoom = 1;
    var isDragging = false;
    var isDraggingLayer = false;
    var selectedLayer = null;
    var lastX = 0;
    var lastY = 0;
    var autoRotate = true;
    var isPaused = false;
    var lastInteractionTime = Date.now();
    var animationId = null;
    var time = 0;
    
    // Posições das camadas
    var layerPositions = [];
    var batchPosition = { x: -6, y: -0.5, z: 0 };
    
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', resizeCanvas);
    
    function autoLayout() {
      var nlayers = parseInt(nlayersEl.value);
      var spacing = 4;
      var startX = -((nlayers) * spacing) / 2;
      
      batchPosition = { x: startX - spacing / 2, y: -0.5, z: 0 };
      
      layerPositions = [];
      for (var i = 0; i < nlayers; i++) {
        layerPositions.push({
          x: startX + (i + 0.5) * spacing,
          y: i * 1.5,
          z: 0
        });
      }
    }
    
    function togglePause() {
      isPaused = !isPaused;
      if (isPaused) {
        pauseBtn.textContent = '▶️';
        pauseBtn.style.background = 'rgba(76, 175, 80, 0.4)';
        autoRotate = false;
      } else {
        pauseBtn.textContent = '⏸️';
        pauseBtn.style.background = 'rgba(255,255,255,0.2)';
        autoRotate = true;
        lastInteractionTime = Date.now();
      }
    }
    
    function resetView() {
      rotationX = -0.3;
      rotationY = 0.5;
      zoom = 1;
      isPaused = false;
      autoRotate = true;
      pauseBtn.textContent = '⏸️';
      pauseBtn.style.background = 'rgba(255,255,255,0.2)';
      lastInteractionTime = Date.now();
      autoLayout();
    }
    
    pauseBtn.addEventListener('click', togglePause);
    resetBtn.addEventListener('click', resetView);
    autoLayoutBtn.addEventListener('click', autoLayout);
    
    document.addEventListener('keydown', function(e) {
      if (e.key === 'p' || e.key === 'P') togglePause();
      if (e.key === 'r' || e.key === 'R') resetView();
      if (e.key === 'a' || e.key === 'A') autoLayout();
    });
    
    function findLayerAt(mouseX, mouseY, layers) {
      var minDist = Infinity;
      var foundLayer = null;
      
      var batchProj = project(batchPosition);
      var batchDist = Math.sqrt(Math.pow(batchProj.x - mouseX, 2) + Math.pow(batchProj.y - mouseY, 2));
      if (batchDist < 50) {
        minDist = batchDist;
        foundLayer = { type: 'batch', index: -1 };
      }
      
      for (var i = 0; i < layerPositions.length && i < layers.length; i++) {
        var proj = project(layerPositions[i]);
        var dist = Math.sqrt(Math.pow(proj.x - mouseX, 2) + Math.pow(proj.y - mouseY, 2));
        
        if (dist < minDist && dist < 60) {
          minDist = dist;
          foundLayer = { type: 'layer', index: i };
        }
      }
      
      return foundLayer;
    }
    
    canvas.addEventListener('mousedown', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      var layers = getCurrentLayersInfo();
      var clickedLayer = findLayerAt(mouseX, mouseY, layers);
      
      if (clickedLayer) {
        isDraggingLayer = true;
        selectedLayer = clickedLayer;
        canvas.style.cursor = 'grabbing';
      } else {
        isDragging = true;
        canvas.style.cursor = 'grabbing';
      }
      
      autoRotate = false;
      lastX = e.clientX;
      lastY = e.clientY;
      lastInteractionTime = Date.now();
    });
    
    canvas.addEventListener('mousemove', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      if (isDraggingLayer && selectedLayer) {
        var deltaX = (e.clientX - lastX) * 0.05;
        var deltaY = -(e.clientY - lastY) * 0.05;
        
        if (selectedLayer.type === 'batch') {
          batchPosition.x += deltaX;
          batchPosition.y += deltaY;
        } else if (selectedLayer.type === 'layer') {
          layerPositions[selectedLayer.index].x += deltaX;
          layerPositions[selectedLayer.index].y += deltaY;
        }
        
        lastX = e.clientX;
        lastY = e.clientY;
      } else if (isDragging) {
        var deltaX = e.clientX - lastX;
        var deltaY = e.clientY - lastY;
        rotationY += deltaX * 0.01;
        rotationX += deltaY * 0.01;
        rotationX = Math.max(-1.5, Math.min(1.5, rotationX));
        lastX = e.clientX;
        lastY = e.clientY;
      }
      
      if (!isDragging && !isDraggingLayer) {
        var layers = getCurrentLayersInfo();
        var hoveredLayer = findLayerAt(mouseX, mouseY, layers);
        canvas.style.cursor = hoveredLayer ? 'pointer' : 'grab';
      }
    });
    
    canvas.addEventListener('mouseup', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    canvas.addEventListener('mouseleave', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
    });
    
    canvas.addEventListener('wheel', function(e) {
      e.preventDefault();
      zoom *= (1 + e.deltaY * 0.001);
      zoom = Math.max(0.5, Math.min(2, zoom));
      lastInteractionTime = Date.now();
      autoRotate = false;
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    function rotateX(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x, y: point.y * cos - point.z * sin, z: point.y * sin + point.z * cos };
    }
    
    function rotateY(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x * cos - point.z * sin, y: point.y, z: point.x * sin + point.z * cos };
    }
    
    function project(point) {
      var rotated = rotateX(point, rotationX);
      rotated = rotateY(rotated, rotationY);
      var scale = zoom * 30;
      return { x: canvas.width / 2 + rotated.x * scale, y: canvas.height / 2 - rotated.y * scale, z: rotated.z };
    }
    
    function shadeColor(color, percent) {
      var num = parseInt(color.replace('#', ''), 16);
      var amt = Math.round(2.55 * percent);
      var R = (num >> 16) + amt;
      var G = (num >> 8 & 0x00FF) + amt;
      var B = (num & 0x0000FF) + amt;
      return '#' + (0x1000000 + (R < 255 ? R < 1 ? 0 : R : 255) * 0x10000 + (G < 255 ? G < 1 ? 0 : G : 255) * 0x100 + (B < 255 ? B < 1 ? 0 : B : 255)).toString(16).slice(1);
    }
    
    function draw3DBox(x, y, z, width, height, depth, color, opacity, label, shape) {
      shape = shape || 'box';
      if (shape === 'cylinder') { draw3DCylinder(x, y, z, width, height, depth, color, opacity, label); return; }
      if (shape === 'diamond') { draw3DDiamond(x, y, z, width, height, depth, color, opacity, label); return; }
      
      var vertices = [
        {x: x - width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y + height/2, z: z + depth/2},
        {x: x - width/2, y: y + height/2, z: z + depth/2}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2, 3], color: shadeColor(color, -20)},
        {vertices: [4, 5, 6, 7], color: shadeColor(color, 20)},
        {vertices: [0, 1, 5, 4], color: shadeColor(color, -40)},
        {vertices: [2, 3, 7, 6], color: shadeColor(color, 40)},
        {vertices: [1, 2, 6, 5], color: shadeColor(color, -10)},
        {vertices: [0, 3, 7, 4], color: shadeColor(color, 10)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DCylinder(x, y, z, width, height, depth, color, opacity, label) {
      var segments = 12;
      var topVertices = [];
      var bottomVertices = [];
      
      for (var i = 0; i < segments; i++) {
        var angle = (i / segments) * Math.PI * 2;
        var cx = x + Math.cos(angle) * width / 2;
        var cz = z + Math.sin(angle) * depth / 2;
        topVertices.push({x: cx, y: y + height/2, z: cz});
        bottomVertices.push({x: cx, y: y - height/2, z: cz});
      }
      
      var projectedTop = topVertices.map(function(v) { return project(v); });
      var projectedBottom = bottomVertices.map(function(v) { return project(v); });
      
      for (var i = 0; i < segments; i++) {
        var next = (i + 1) % segments;
        ctx.beginPath();
        ctx.moveTo(projectedTop[i].x, projectedTop[i].y);
        ctx.lineTo(projectedTop[next].x, projectedTop[next].y);
        ctx.lineTo(projectedBottom[next].x, projectedBottom[next].y);
        ctx.lineTo(projectedBottom[i].x, projectedBottom[i].y);
        ctx.closePath();
        ctx.fillStyle = shadeColor(color, (i % 2 === 0) ? -10 : 10);
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      }
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DDiamond(x, y, z, width, height, depth, color, opacity, label) {
      var vertices = [
        {x: x, y: y + height/2, z: z},
        {x: x + width/2, y: y, z: z},
        {x: x, y: y, z: z + depth/2},
        {x: x - width/2, y: y, z: z},
        {x: x, y: y, z: z - depth/2},
        {x: x, y: y - height/2, z: z}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2], color: shadeColor(color, -20)},
        {vertices: [0, 2, 3], color: shadeColor(color, 20)},
        {vertices: [0, 3, 4], color: shadeColor(color, -10)},
        {vertices: [0, 4, 1], color: shadeColor(color, 10)},
        {vertices: [5, 1, 2], color: shadeColor(color, -30)},
        {vertices: [5, 2, 3], color: shadeColor(color, 30)},
        {vertices: [5, 3, 4], color: shadeColor(color, -20)},
        {vertices: [5, 4, 1], color: shadeColor(color, 20)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function drawImageBatch(x, y, z, width, height, numImages, color) {
      var imageDepth = 0.3;
      var gap = 0.1;
      var totalDepth = numImages * (imageDepth + gap);
      var startZ = z - totalDepth / 2;
      
      for (var i = 0; i < numImages; i++) {
        var imageZ = startZ + i * (imageDepth + gap);
        var alpha = 0.3 + (i / numImages) * 0.5;
        draw3DBox(x, y, imageZ, width, height, imageDepth, color, alpha, null, 'box');
      }
    }
    
    function drawConnection(x1, y1, z1, x2, y2, z2, animated) {
      var start = project({x: x1, y: y1, z: z1});
      var end = project({x: x2, y: y2, z: z2});
      var midX = (start.x + end.x) / 2;
      var midY = Math.min(start.y, end.y) - 20;
      
      if (animated && !isPaused) {
        var pulse = Math.sin(time * 0.002) * 0.5 + 0.5;
        ctx.strokeStyle = 'rgba(255, 255, 255, ' + (0.3 + pulse * 0.3) + ')';
        ctx.lineWidth = 1.5 + pulse;
      } else {
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1.5;
      }
      
      ctx.setLineDash([4, 4]);
      ctx.beginPath();
      ctx.moveTo(start.x, start.y);
      ctx.quadraticCurveTo(midX, midY, end.x, end.y);
      ctx.stroke();
      ctx.setLineDash([]);
    }
    
    function getCurrentLayersInfo() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var layersInfo = [];
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect ? typeSelect.value : 'CONV';
        var inputStr = '';
        var outputStr = '';
        
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          // Se a camada anterior era FC, usa a saída dela
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            // Se veio de CONV/POOL, faz flatten
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          outputStr = fcout;
          currentFCInput = fcout;
          // Após FC, não há mais dimensões espaciais
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
      }
      
      return layersInfo;
    }
    
    function render3D(layers, batchSize) {
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      if (!isPaused) time += 16;
      if (autoRotate && !isDragging && !isPaused && Date.now() - lastInteractionTime > 3000) rotationY += 0.005;
      
      // Grid
      ctx.strokeStyle = 'rgba(255, 255, 255, 0.08)';
      ctx.lineWidth = 0.5;
      for (var i = -10; i <= 10; i++) {
        var start = project({x: i * 2, y: -2, z: -10 * 2});
        var end = project({x: i * 2, y: -2, z: 10 * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
        start = project({x: -10 * 2, y: -2, z: i * 2});
        end = project({x: 10 * 2, y: -2, z: i * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
      }
      
      var colors = ['#4a90e2', '#50e3c2', '#f5a623', '#d0021b', '#8b572a', '#9013fe'];
      
      // Batch (apenas se a primeira camada for CONV ou POOL)
      if (layers.length > 0 && (layers[0].type === 'CONV' || layers[0].type === 'POOL')) {
        var inputWidth = Math.max(1, Math.min(4, layers[0].h / 8));
        var inputHeight = Math.max(1, Math.min(4, layers[0].w / 8));
        var labelPos = project({x: batchPosition.x, y: batchPosition.y + inputHeight/2 + 0.7, z: batchPosition.z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 9px Arial';
        ctx.textAlign = 'center';
        ctx.fillText('BATCH: ' + batchSize, labelPos.x, labelPos.y);
        drawImageBatch(batchPosition.x, batchPosition.y, batchPosition.z, inputWidth, inputHeight, batchSize, '#ff6b6b');
        
        // Conexão batch -> primeira camada
        drawConnection(batchPosition.x + inputWidth/2, batchPosition.y, batchPosition.z, layerPositions[0].x - Math.max(1, Math.min(4, layers[0].h / 8))/2, layerPositions[0].y, layerPositions[0].z, true);
      }
      
      // Conexões entre camadas
      for (var i = 0; i < layers.length - 1 && i < layerPositions.length - 1; i++) {
        drawConnection(layerPositions[i].x + 1, layerPositions[i].y, layerPositions[i].z, layerPositions[i + 1].x - 1, layerPositions[i + 1].y, layerPositions[i + 1].z, true);
      }
      
      // Camadas
      for (var i = 0; i < layers.length && i < layerPositions.length; i++) {
        var layer = layers[i];
        var pos = layerPositions[i];
        
        var color = colors[i % colors.length];
        var label = '';
        
        if (layer.type === 'CONV') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'box');
        } else if (layer.type === 'POOL') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'cylinder');
        } else if (layer.type === 'FC') {
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, 1, 1, 1, color, 0.7, label, 'diamond');
        }
      }
      
      animationId = requestAnimationFrame(function() { render3D(layers, batchSize); });
    }
    
    function generateLayerConfig() {
      var nlayers = parseInt(nlayersEl.value);
      var html = '';
      
      for (var i = 0; i < nlayers; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:6px 8px;display:flex;gap:6px;align-items:center;flex-wrap:wrap;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">' + (i+1) + ':</span>';
        html += '<select id="ep0903_type_' + i + '" style="padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">';
        html += '<option value="CONV"' + (i < 2 ? ' selected' : '') + '>CONV</option>';
        html += '<option value="POOL">POOL</option>';
        html += '<option value="FC"' + (i >= 2 ? ' selected' : '') + '>FC</option>';
        html += '</select>';
        html += '<div id="ep0903_params_' + i + '" style="display:flex;gap:3px;flex-wrap:wrap;"></div>';
        html += '</div>';
      }
      
      layersConfigEl.innerHTML = html;
      
      for (var i = 0; i < nlayers; i++) {
        (function(index) {
          var typeSelect = root.querySelector('#ep0903_type_' + index);
          typeSelect.addEventListener('change', function() {
            updateLayerParams(index);
            render();
          });
          updateLayerParams(index);
        })(i);
      }
      
      autoLayout();
    }
    
    function updateLayerParams(index) {
      var typeSelect = root.querySelector('#ep0903_type_' + index);
      var paramsDiv = root.querySelector('#ep0903_params_' + index);
      var type = typeSelect.value;
      
      if (type === 'CONV') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_kh_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel h">' +
          '<span style="font-size:8px;">×</span>' +
          '<input type="number" id="ep0903_kw_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel w">' +
          '<input type="number" id="ep0903_cout_' + index + '" value="' + (index === 0 ? '8' : '16') + '" min="1" max="64" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="filtri">';
      } else if (type === 'POOL') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_pool_size_' + index + '" value="2" min="2" max="4" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="pool size">';
      } else if (type === 'FC') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_fcout_' + index + '" value="10" min="1" max="100" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="uscite">';
      }
      
      var inputs = paramsDiv.querySelectorAll('input');
      inputs.forEach(function(input) {
        input.addEventListener('input', render);
        input.addEventListener('change', render);
      });
    }
    
    function render() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var batchSize = parseInt(batchEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var bias = biasEl.checked ? 1 : 0;
      
      hwvEl.textContent = hw + '×' + hw;
      cinvEl.textContent = cin;
      batchvEl.textContent = batchSize;
      nlayersvEl.textContent = nlayers;
      
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var totalParams = 0;
      var layersInfo = [];
      var summaryHTML = '';
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect.value;
        var params = 0;
        var inputStr = '';
        var outputStr = '';
        
        // Determinar entrada
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        // Processar camada
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          params = kh * kw * currentCin * cout + cout * bias;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          params = 0;
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          var fcin = parseInt(inputStr);
          params = fcin * fcout + fcout * bias;
          outputStr = fcout;
          currentFCInput = fcout;
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        totalParams += params;
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          params: params,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
        
        summaryHTML += '<span style="color:' + (type === 'CONV' ? '#2980b9' : type === 'POOL' ? '#666' : '#009933') + ';font-weight:bold;">' + (i+1) + ' (' + type + '):</span> ';
        summaryHTML += inputStr + ' → ' + outputStr;
        summaryHTML += ' [' + params.toLocaleString('pt-BR') + ']<br>';
      }
      
      summaryHTML += '<b>Total: ' + totalParams.toLocaleString('pt-BR') + ' parâmetros</b>';
      summaryEl.innerHTML = summaryHTML;
      
      if (animationId) cancelAnimationFrame(animationId);
      render3D(layersInfo, batchSize);
    }
    
    // Inicializar
    autoLayout();
    generateLayerConfig();
    render();
    
    // Event listeners
    hwEl.addEventListener('input', render);
    cinEl.addEventListener('input', render);
    batchEl.addEventListener('input', render);
    nlayersEl.addEventListener('input', function() { generateLayerConfig(); render(); });
    biasEl.addEventListener('change', render);
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0903');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.45:** Simulatore EP09_03: Conteggio dei Parametri — Convoluzione vs. Strato Totalmente Connesso


<figure id="fig-09-sim-ep0903">
  <img src="imagens/fig-09-sim-ep0903.png" alt=" Simulatore EP09_03: Conteggio dei Parametri — Convoluzione vs. Strato Totalmente Connesso " style="max-width:80%" />
  <figcaption><strong>Figura 9.45:</strong>  Simulatore EP09_03: Conteggio dei Parametri — Convoluzione vs. Strato Totalmente Connesso </figcaption>
</figure>

In [64]:
%%writefile EP09_03.py
# Codice Python

Overwriting EP09_03.py


In [65]:
TestSuite("EP09_03.py").run()

### 9.0.4 EP09_04 🟡 Intersezione su Unione (IoU) e Soppressione dei Non-Massimi (NMS)

I modelli di rilevamento degli oggetti possono produrre **diverse bounding box candidate** per lo stesso oggetto, con posizioni e punteggi di confidenza differenti. La fase di post-elaborazione responsabile dell'eliminazione di queste rilevazioni ridondanti è la **Soppressione dei Non-Massimi (NMS)**, la cui operazione fondamentale utilizza la metrica di **Intersezione su Unione (IoU)**.

La NMS utilizza questa misura per decidere quali box devono essere mantenute. In generale, la box con la confidenza più alta viene selezionata per prima; successivamente, le box che presentano un'IoU superiore a una certa soglia con la box selezionata sono considerate ridondanti e vengono rimosse. Il processo viene ripetuto finché non rimangono box candidate.

In questo esercizio, dovrai implementare l'algoritmo NMS da zero, calcolando l'IoU tra le box e applicando successivamente il criterio di selezione e soppressione per produrre l'insieme finale di rilevazioni.

#### 9.0.4.1 📋 Linee Guida di Implementazione

1. **Input:** Leggere l'intero $N$ (numero di box candidate) e la soglia reale $\tau$ (soglia IoU per la soppressione), sulla stessa riga.

2. **Box:** Leggere $N$ righe, ciascuna con cinque valori reali:

   `x1 y1 x2 y2 score`

   dove $(x_1,y_1)$ rappresenta l'angolo superiore sinistro, $(x_2,y_2)$ l'angolo inferiore destro e `score` il punteggio di confidenza.

3. **Intersezione su Unione:** Per due box $A$ e $B$,

   $$
   IoU(A,B)=
   \frac{\operatorname{Area}(A\cap B)}
   {\operatorname{Area}(A\cup B)}.
   $$

   L'area di intersezione deve essere calcolata dalla sovrapposizione degli intervalli in $x$ e $y$. Se non c'è sovrapposizione, l'area di intersezione è zero.

4. **Algoritmo greedy di NMS:**

   a. Ordina le box per `score` decrescente. In caso di parità, mantieni l'ordine originale di lettura.

   b. Seleziona la box con il punteggio più alto tra le box rimanenti e aggiungila all'insieme di output.

   c. Calcola l'IoU tra la box selezionata e **tutte le box ancora rimanenti**. Sopprimi le box per cui

   $$
   \text{IoU} > \tau.
   $$

   d. Ripeti i passaggi (b) e (c) finché non rimangono box.

5. **Output:** Per ogni box mantenuta, nell'ordine in cui è stata selezionata, stampa il suo indice originale (posizione di lettura, a partire da $0$) e il suo `score`, formattato con 4 cifre decimali. Alla fine, stampa:

   `Totale mantenute: X`

#### 9.0.4.2 📌 Vincoli Computazionali

* **Soppressione stretta:** solo le box con $\text{IoU} > \tau$ vengono soppresse. Le box con $\text{IoU}=\tau$ vengono mantenute.
* **Indici originali:** l'output fa riferimento alla posizione in cui ogni box è stata letta nell'input (a partire da $0$), non alla sua posizione dopo l'ordinamento.
* **Ordinamento stabile:** in caso di `score` uguali, deve essere preservato l'ordine originale di lettura.
* **Rettangoli allineati agli assi:** tutte le box sono specificate da due angoli, con $x_1 < x_2$ e $y_1 < y_2$ garantiti nell'input.
* **Coordinate e punteggi:** i valori reali possono essere positivi o negativi, secondo i limiti definiti dall'input, ma le dimensioni delle box sono sempre positive.

#### 9.0.4.3 🧠 Fondamento Teorico

| Elemento                | Ruolo nella post-elaborazione della rilevazione                                                                                          |
| ----------------------- | ---------------------------------------------------------------------------------------------------------------------------------------- |
| IoU                     | Quantifica la sovrapposizione spaziale tra due box; $\text{IoU}=1$ per box identiche e $\text{IoU}=0$ per box senza sovrapposizione |
| Ordinamento per confidenza | Fa sì che la box con `score` più alto venga analizzata per prima                                                                        |
| Soglia $\tau$           | Definisce la quantità di sovrapposizione necessaria affinché una box sia considerata ridondante                                           |
| Soppressione            | Rimuove le box che presentano una grande sovrapposizione con una box già selezionata                                                     |
| Box distanti            | Hanno IoU vicina a zero e, in generale, non vengono soppresse da questa regola                                                           |

#### 9.0.4.4 🧩 Metodi di `morph.py` che possono aiutare

* `mm.IoU(boxA, boxB)` — calcola la metrica IoU, ma si aspetta le box nel formato $(x,y,w,h)$, cioè angolo superiore sinistro, larghezza e altezza. L'input di questo esercizio utilizza il formato $(x_1,y_1,x_2,y_2)$. La conversione è diretta:

  $$
  w=x_2-x_1,\qquad h=y_2-y_1.
  $$

  L'uso di questa funzione è facoltativo. L'obiettivo principale dell'esercizio è implementare correttamente il processo di selezione e soppressione della NMS.

#### 9.0.4.5 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: intero $N$ e reale $\tau$.
* Prossime $N$ righe: $x_1\ y_1\ x_2\ y_2\ \text{score}$.

**Output:**

* Una riga per box mantenuta, nell'ordine di selezione: `indice score`.
* Ultima riga: `Totale mantenute: X`.

In [66]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0904" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: IoU e Soppressione Non-Massimale</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 NMS</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Casella selezionata</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Casella mantenuta</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Casella soppressa</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Casella candidata</b>
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Numero di caselle</label>
            <span id="ep0904_n_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5</span>
          </div>
          <input id="ep0904_n" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="2" step="1" type="range" value="5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Soglia τ (IoU)</label>
            <span id="ep0904_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0904_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Esempio</label>
            <span id="ep0904_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Predefinito</span>
          </div>
          <select id="ep0904_example" style="width:100%;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">
            <option value="padrao">Esempio predefinito</option>
            <option value="agrupado">Caselle raggruppate</option>
            <option value="disperso">Caselle sparse</option>
            <option value="aninhado">Caselle annidate</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0904_run_btn" style="background:#2980b9;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;transition:all 0.3s;">
            ▶️ Esegui NMS
          </button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0904_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:8px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:300px;">
        <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">
          🎯 Visualizzazione delle caselle
        </div>
        <canvas id="ep0904_canvas" style="width:100%;height:280px;display:block;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:310px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📋 Procedura passo passo della NMS
        </div>
        <div id="ep0904_steps" style="font-family:monospace;font-size:10px;line-height:1.6;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo -->
    <div id="ep0904_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var nEl = root.querySelector('#ep0904_n'), nvEl = root.querySelector('#ep0904_n_v');
    var tauEl = root.querySelector('#ep0904_tau'), tauvEl = root.querySelector('#ep0904_tau_v');
    var exampleEl = root.querySelector('#ep0904_example');
    var boxesConfigEl = root.querySelector('#ep0904_boxes_config');
    var canvas = root.querySelector('#ep0904_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0904_steps');
    var summaryEl = root.querySelector('#ep0904_summary');
    var runBtn = root.querySelector('#ep0904_run_btn');
    
    // Estado
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    
    // Exemplos pré-definidos
    var examples = {
      padrao: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 30, y1: 15, x2: 70, y2: 55, score: 0.7},
        {x1: 80, y1: 80, x2: 120, y2: 120, score: 0.6},
        {x1: 85, y1: 85, x2: 125, y2: 125, score: 0.5}
      ],
      agrupado: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 15, y1: 15, x2: 55, y2: 55, score: 0.85},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 25, y1: 25, x2: 65, y2: 65, score: 0.75},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7}
      ],
      disperso: [
        {x1: 10, y1: 10, x2: 40, y2: 40, score: 0.9},
        {x1: 80, y1: 10, x2: 110, y2: 40, score: 0.8},
        {x1: 10, y1: 80, x2: 40, y2: 110, score: 0.7},
        {x1: 80, y1: 80, x2: 110, y2: 110, score: 0.6},
        {x1: 45, y1: 45, x2: 75, y2: 75, score: 0.5}
      ],
      aninhado: [
        {x1: 10, y1: 10, x2: 90, y2: 90, score: 0.9},
        {x1: 20, y1: 20, x2: 80, y2: 80, score: 0.8},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7},
        {x1: 40, y1: 40, x2: 60, y2: 60, score: 0.6},
        {x1: 45, y1: 45, x2: 55, y2: 55, score: 0.5}
      ]
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', function() {
      resizeCanvas();
      render();
    });
    
    // Carregar exemplo
    function loadExample(name) {
      boxes = JSON.parse(JSON.stringify(examples[name]));
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas carregadas: ' + boxes.length + '. Clique em "Executar NMS".';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      for (var i = 0; i < boxes.length; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">#' + i + ':</span>';
        html += '<input type="number" id="ep0904_x1_' + i + '" value="' + boxes[i].x1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x1">';
        html += '<input type="number" id="ep0904_y1_' + i + '" value="' + boxes[i].y1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y1">';
        html += '<input type="number" id="ep0904_x2_' + i + '" value="' + boxes[i].x2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x2">';
        html += '<input type="number" id="ep0904_y2_' + i + '" value="' + boxes[i].y2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y2">';
        html += '<input type="number" id="ep0904_score_' + i + '" value="' + boxes[i].score + '" step="0.05" min="0" max="1" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="punteggio">';
        html += '</div>';
      }
      boxesConfigEl.innerHTML = html;
      
      // Adicionar event listeners
      for (var i = 0; i < boxes.length; i++) {
        (function(index) {
          ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
            var input = root.querySelector('#ep0904_' + field + '_' + index);
            if (input) {
              input.addEventListener('input', function() {
                boxes[index][field] = parseFloat(input.value) || 0;
                selectedBoxes = [];
                suppressedBoxes = [];
                render();
              });
            }
          });
        })(i);
      }
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar NMS
    function runNMS() {
      var tau = parseFloat(tauEl.value);
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      // Ordenar por score decrescente (estável)
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) {
          return b.box.score - a.box.score;
        }
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push({
          type: 'select',
          box: selected,
          remaining: remaining.slice()
        });
        
        var newRemaining = [];
        var suppressed = [];
        
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressed.push({ box: remaining[i], iou: iou });
            suppressedBoxes.push(remaining[i]);
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        
        if (suppressed.length > 0) {
          steps.push({
            type: 'suppress',
            box: selected,
            suppressed: suppressed,
            remaining: newRemaining.slice()
          });
        }
        
        remaining = newRemaining;
      }
      
      return steps;
    }
    
    // Renderizar visualização
    function render() {
      var tau = parseFloat(tauEl.value);
      nvEl.textContent = boxes.length;
      tauvEl.textContent = tau.toFixed(2);
      
      // Limpar canvas
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      
      // Desenhar todas as caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        
        var color = '#f5a623'; // candidata
        if (isSelected) color = '#4a90e2'; // selecionada
        if (isSuppressed) color = '#ff6b6b'; // suprimida
        
        drawBox(box, color, index);
      });
      
      // Atualizar resumo
      var summaryHTML = '';
      if (selectedBoxes.length > 0) {
        summaryHTML += '<b>Caixas selecionadas (em ordem):</b><br>';
        selectedBoxes.forEach(function(s) {
          summaryHTML += '#' + s.originalIndex + ' (score: ' + s.box.score.toFixed(4) + ')<br>';
        });
        summaryHTML += '<b>Total mantidas: ' + selectedBoxes.length + '</b>';
        summaryEl.innerHTML = summaryHTML;
      }
    }
    
    // Desenhar caixa
    function drawBox(box, color, index) {
      var scale = 2.0;
      var offsetX = 30;
      var offsetY = 30;
      
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      // Desenhar caixa
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      // Preenchimento translúcido
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      // Label
      ctx.fillStyle = color;
      ctx.font = 'bold 12px Arial';
      ctx.textAlign = 'center';
      ctx.fillText('#' + index, x + w/2, y - 5);
      
      // Score
      ctx.fillStyle = '#666';
      ctx.font = '10px Arial';
      ctx.fillText('punteggio: ' + box.score.toFixed(2), x + w/2, y + h/2);
    }
    
    // Mostrar passo a passo
    function showSteps(steps) {
      var html = '';
      
      steps.forEach(function(step, index) {
        if (step.type === 'select') {
          html += '<div style="color:#4a90e2;font-weight:bold;margin-top:4px;">';
          html += 'Passo ' + (index + 1) + ': Selecionar caixa #' + step.box.originalIndex;
          html += ' (score: ' + step.box.box.score.toFixed(4) + ')';
          html += '</div>';
        } else if (step.type === 'suppress') {
          html += '<div style="color:#ff6b6b;margin-left:10px;">';
          html += '↳ Suprimir: ';
          step.suppressed.forEach(function(s, i) {
            if (i > 0) html += ', ';
            html += '#' + s.box.originalIndex;
            html += ' (IoU: ' + s.iou.toFixed(3) + ')';
          });
          html += '</div>';
        }
      });
      
      if (selectedBoxes.length > 0) {
        html += '<div style="color:#50e3c2;font-weight:bold;margin-top:8px;">';
        html += '✓ Resultado: ' + selectedBoxes.length + ' caixa(s) mantida(s)';
        html += '</div>';
      }
      
      stepsEl.innerHTML = html;
    }
    
    // Executar NMS
    function executeNMS() {
      var steps = runNMS();
      showSteps(steps);
      render();
      
      // Animação do botão
      runBtn.textContent = '✓ Executado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Executar NMS';
        runBtn.style.background = '#2980b9';
      }, 1000);
    }
    
    // Event listeners
    runBtn.addEventListener('click', executeNMS);
    
    nEl.addEventListener('input', function() {
      var n = parseInt(nEl.value);
      var currentN = boxes.length;
      
      if (n > currentN) {
        for (var i = currentN; i < n; i++) {
          boxes.push({
            x1: 10 + i * 5,
            y1: 10 + i * 5,
            x2: 50 + i * 5,
            y2: 50 + i * 5,
            score: 0.9 - i * 0.1
          });
        }
      } else if (n < currentN) {
        boxes = boxes.slice(0, n);
      }
      
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas atualizadas: ' + boxes.length + '. Clique em "Executar NMS".';
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Limiar atualizado. Clique em "Executar NMS".';
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0904');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.46:** Simulatore EP09_04: IoU e Soppressione Non-Massima (NMS)


<figure id="fig-09-sim-ep0904">
  <img src="imagens/fig-09-sim-ep0904.png" alt=" Simulatore EP09_04: IoU e Soppressione Non-Massima (NMS) " style="max-width:80%" />
  <figcaption><strong>Figura 9.46:</strong>  Simulatore EP09_04: IoU e Soppressione Non-Massima (NMS) </figcaption>
</figure>

In [67]:
%%writefile EP09_04.py
# Codice Python


Overwriting EP09_04.py


In [68]:
TestSuite("EP09_04.py").run()

### 9.0.5 EP09_05 🟠 Valutazione della Segmentazione: IoU e Dice Pixel per Pixel

Il Blocco 2 della sezione "Segmentazione Semantica con Architettura U-Net" definisce, in poche righe, la funzione `iou_mascaras`, utilizzata per misurare la qualità della baseline morfologica classica (smoothing + Otsu + apertura) e, più avanti, della stessa U-Net addestrata. Diversamente dall'IoU dell'EP09_04 — calcolato su **bounding box** (regioni rettangolari descritte da quattro numeri) —, l'IoU di segmentazione è calcolato **pixel per pixel**: ogni posizione dell'immagine viene confrontata individualmente tra la maschera predetta e la maschera di riferimento.

Ti è stato affidato il compito di generalizzare questa valutazione, implementando non solo l'IoU pixel per pixel, ma anche il **coefficiente di Dice**, un'altra metrica di sovrapposizione ampiamente utilizzata in segmentazione medica (inclusa nella funzione `perda_dice`, menzionata nello stesso blocco del capitolo come base della funzione di perdita utilizzata per addestrare la U-Net).

#### 9.0.5.1 📋 Linee Guida di Implementazione

1. **Input:** Leggere le dimensioni $H \times W$ delle maschere.

2. **Maschera predetta:** Leggere $H$ righe con $W$ valori interi (0 o 1) ciascuna — ad esempio, l'output di una U-Net dopo la sogliatura a $0{,}5$ sulla sigmoide, come nel Blocco 4 del capitolo.

3. **Maschera di riferimento:** Leggere altre $H$ righe con $W$ valori interi (0 o 1) ciascuna — il *ground truth*.

4. **Intersezione e unione:** Considerando ogni pixel come appartenente all'oggetto quando il suo valore è diverso da zero,
   $$
   \text{intersezione} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \wedge R_{ij}=1], \qquad
   \text{unione} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \vee R_{ij}=1].
   $$

5. **IoU pixel per pixel:**
   $$
   \text{IoU} = \frac{\text{intersezione}}{\text{unione}}.
   $$

6. **Coefficiente di Dice:**
   $$
   \text{Dice} = \frac{2 \cdot \text{intersezione}}{|P| + |R|},
   $$
   dove $|P|$ e $|R|$ sono il numero totale di pixel dell'oggetto in ciascuna maschera.

7. **Convenzione per maschere vuote:** se **entrambe** le maschere non hanno alcun pixel dell'oggetto (unione $= 0$ e $|P|+|R|=0$), considera la corrispondenza banalmente perfetta: $\text{IoU} = \text{Dice} = 1{,}0$.

8. **Output:** Due righe, `IoU: X.XXXX` e `Dice: X.XXXX`, ciascun valore con 4 cifre decimali.

#### 9.0.5.2 📌 Vincoli Computazionali

* **Qualsiasi valore non nullo conta come oggetto:** tratta i valori diversi da $0$ (non solo $1$) come appartenenti alla maschera, replicando il controllo `predita > 0` usato in `iou_mascaras` nel capitolo.
* **Stesse dimensioni:** le due maschere hanno sempre esattamente $H \times W$ elementi.
* **Convenzione del vuoto:** applica la regola del punto 7 **solo** quando entrambe le maschere sono completamente vuote; se solo una è vuota, l'intersezione è $0$ e l'IoU/Dice risultante sarà anch'esso $0$.

#### 9.0.5.3 🧠 Fondamento Teorico

| Elemento | Ruolo nella valutazione della segmentazione |
|---|---|
| IoU pixel per pixel | Generalizza la metrica dell'EP09_04 a regioni di forma arbitraria — non solo rettangoli — confrontando la maschera predetta e il riferimento posizione per posizione |
| Coefficiente di Dice | Metrica correlata all'IoU (sempre $\text{Dice} \ge \text{IoU}$), più sensibile a piccole intersezioni e ampiamente utilizzata come funzione di perdita in segmentazione (funzione `perda_dice` del capitolo) |
| Convenzione per maschere vuote | Evita la divisione per zero e riconosce che "nessun oggetto previsto, nessun oggetto reale" è, per definizione, un successo |
| Confronto classico vs. U-Net | Il capitolo usa esattamente questo tipo di metrica per giustificare, numericamente, perché la U-Net supera la baseline morfologica in scenari a basso contrasto |

#### 9.0.5.4 🧩 Metodi di `morph.py` che possono aiutare

* `mm.readImg(h, w, dtype='uint8')` — legge direttamente ogni maschera binaria $h \times w$ dall'input standard (i valori $0/1$ rientrano perfettamente nel tipo intero standard).
* La stessa funzione `iou_mascaras`, definita nel Blocco 2 della sezione U-Net del capitolo (non fa parte di `morph.py`, ma del codice del capitolo), è l'ispirazione diretta di questo esercizio — vale la pena rileggere quelle poche righe prima di programmare.
* Per un'estensione opzionale (non richiesta da questo EP), `mm.connectedComponents` o `mm.label0` (visti nel contesto dell'analisi delle componenti connesse) permetterebbero di etichettare ogni nodulo individualmente e calcolare l'IoU **per componente**, invece che sull'intera maschera.

#### 9.0.5.5 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ valori interi (0 o 1) — maschera predetta.
* Prossime $H$ righe: $W$ valori interi (0 o 1) — maschera di riferimento.

**Output:**

* Riga 1: `IoU: X.XXXX`.
* Riga 2: `Dice: X.XXXX`.

> ### 💡 Dica
>
> ##### 💡 Esempio Illustrativo
>
> Considera una maschera predetta con un quadrato $2\times2$ di pixel attivi e un riferimento spostato di una colonna, sovrapponendosi solo per metà dell'area:
>
> ```
> Predetta        Riferimento
> 0 0 0 0         0 0 0 0
> 0 1 1 0         0 0 1 1
> 0 1 1 0         0 0 1 1
> 0 0 0 0         0 0 0 0
> ```
>
> Intersezione $=2$ pixel, unione $=6$ pixel ($4+4-2$), quindi $\text{IoU}=2/6\approx0{,}3333$ e $\text{Dice}=2\cdot2/(4+4)=0{,}5000$ — nota che il Dice è sempre uguale o maggiore dell'IoU per la stessa sovrapposizione.

#### 9.0.5.6 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4 4<br>0 0 0 0<br>0 1 1 0<br>0 1 1 0<br>0 0 0 0<br>0 0 0 0<br>0 0 1 1<br>0 0 1 1<br>0 0 0 0 | IoU: 0.3333<br>Dice: 0.5000 | Maschere $4\times4$ con sovrapposizione parziale di 2 pixel. |

In [69]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0905" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: IoU e Dice Pixel per Pixel</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟠 Segmentazione</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Intersezione</b> (VP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Solo previsto</b> (FP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Solo riferimento</b> (FN)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f0f0f0;border:2px solid #ccc;border-radius:2px;display:inline-block;"></span>
        <b>Sfondo</b> (VN)
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:flex;gap:10px;margin-bottom:8px;flex-wrap:wrap;">
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Dimensioni</label>
            <span id="ep0905_dim_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5×5</span>
          </div>
          <input id="ep0905_dim" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="3" step="1" type="range" value="5">
        </div>
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Esempio</label>
            <span id="ep0905_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Quadrato</span>
          </div>
          <select id="ep0905_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="quadrado">Quadrato 2×2</option>
            <option value="deslocado">Spostato</option>
            <option value="perfeito">Perfetto</option>
            <option value="vazio">Maschere Vuote</option>
            <option value="parcial">Sovrapposizione Parziale</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;gap:8px;">
          <button id="ep0905_clear_btn" style="background:#666;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🗑️ Pulisci</button>
          <button id="ep0905_random_btn" style="background:#f5a623;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🎲 Casuale</button>
        </div>
      </div>
      
      <!-- Grids de máscaras -->
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:10px;">
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🔵 Maschera Prevista
          </div>
          <div id="ep0905_pred_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🟡 Maschera di Riferimento
          </div>
          <div id="ep0905_ref_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Confronto Visivo
        </div>
        <canvas id="ep0905_canvas" style="width:100%;height:220px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Métricas -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:280px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📊 Calcoli e Formule
        </div>
        <div id="ep0905_metrics" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo final -->
    <div id="ep0905_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var dimEl = root.querySelector('#ep0905_dim');
    var dimvEl = root.querySelector('#ep0905_dim_v');
    var exampleEl = root.querySelector('#ep0905_example');
    var predGridEl = root.querySelector('#ep0905_pred_grid');
    var refGridEl = root.querySelector('#ep0905_ref_grid');
    var canvas = root.querySelector('#ep0905_canvas');
    var ctx = canvas.getContext('2d');
    var metricsEl = root.querySelector('#ep0905_metrics');
    var summaryEl = root.querySelector('#ep0905_summary');
    var clearBtn = root.querySelector('#ep0905_clear_btn');
    var randomBtn = root.querySelector('#ep0905_random_btn');
    
    // Estado
    var predMask = [];
    var refMask = [];
    var H = 5;
    var W = 5;
    
    // Exemplos pré-definidos
    var examples = {
      quadrado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      deslocado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      perfeito: {
        pred: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]],
        ref: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]]
      },
      vazio: {
        pred: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      parcial: {
        pred: [[0,0,0,0,0],[0,1,1,1,0],[0,1,1,1,0],[0,1,1,1,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0]]
      }
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      predMask = example.pred.map(function(row) { return row.slice(); });
      refMask = example.ref.map(function(row) { return row.slice(); });
      H = predMask.length;
      W = predMask[0].length;
      dimEl.value = H;
      dimvEl.textContent = H + '×' + W;
      generateGrids();
      render();
    }
    
    // Gerar grids clicáveis
    function generateGrids() {
      var predHTML = '<table style="border-collapse:collapse;">';
      var refHTML = '<table style="border-collapse:collapse;">';
      
      for (var i = 0; i < H; i++) {
        predHTML += '<tr>';
        refHTML += '<tr>';
        for (var j = 0; j < W; j++) {
          predHTML += '<td data-type="pred" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (predMask[i][j] ? '#4a90e2' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (predMask[i][j] ? 'white' : '#999') + ';">' + (predMask[i][j] ? '1' : '0') + '</td>';
          refHTML += '<td data-type="ref" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (refMask[i][j] ? '#f5a623' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (refMask[i][j] ? 'white' : '#999') + ';">' + (refMask[i][j] ? '1' : '0') + '</td>';
        }
        predHTML += '</tr>';
        refHTML += '</tr>';
      }
      
      predHTML += '</table>';
      refHTML += '</table>';
      
      predGridEl.innerHTML = predHTML;
      refGridEl.innerHTML = refHTML;
      
      // Adicionar event listeners
      predGridEl.querySelectorAll('td[data-type="pred"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          predMask[i][j] = predMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
      
      refGridEl.querySelectorAll('td[data-type="ref"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          refMask[i][j] = refMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
    }
    
    // Calcular métricas
    function calculateMetrics() {
      var TP = 0, FP = 0, FN = 0, TN = 0;
      
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          if (p && r) TP++;
          else if (p && !r) FP++;
          else if (!p && r) FN++;
          else TN++;
        }
      }
      
      var intersection = TP;
      var union = TP + FP + FN;
      var predCount = TP + FP;
      var refCount = TP + FN;
      
      var iou, dice;
      
      if (union === 0) {
        iou = 1.0;
        dice = 1.0;
      } else {
        iou = intersection / union;
        dice = (predCount + refCount === 0) ? 1.0 : (2 * intersection) / (predCount + refCount);
      }
      
      return { TP, FP, FN, TN, intersection, union, predCount, refCount, iou, dice };
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      var m = calculateMetrics();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var cellSize = Math.min(35, (canvas.width - 40) / W);
      var offsetX = (canvas.width - W * cellSize) / 2;
      var offsetY = (canvas.height - H * cellSize) / 2;
      
      // Desenhar grid
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          var color = '#f0f0f0';
          if (p && r) color = '#4a90e2';
          else if (p && !r) color = '#ff6b6b';
          else if (!p && r) color = '#f5a623';
          
          var x = offsetX + j * cellSize;
          var y = offsetY + i * cellSize;
          
          ctx.fillStyle = color;
          ctx.fillRect(x, y, cellSize - 2, cellSize - 2);
          ctx.strokeStyle = '#999';
          ctx.lineWidth = 1;
          ctx.strokeRect(x, y, cellSize - 2, cellSize - 2);
        }
      }
      
      // Fórmulas e cálculos
      var html = '';
      html += '<div style="margin-bottom:6px;"><b>1. Contagem de pixels:</b></div>';
      html += '<div style="color:#4a90e2;">TP (interseção) = ' + m.TP + '</div>';
      html += '<div style="color:#ff6b6b;">FP (só predita) = ' + m.FP + '</div>';
      html += '<div style="color:#f5a623;">FN (só referência) = ' + m.FN + '</div>';
      html += '<div style="color:#999;">TN (fundo) = ' + m.TN + '</div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>2. Interseção e União:</b></div>';
      html += '<div>Interseção = TP = <b>' + m.intersection + '</b></div>';
      html += '<div>União = TP + FP + FN = ' + m.TP + ' + ' + m.FP + ' + ' + m.FN + ' = <b>' + m.union + '</b></div>';
      html += '<div>|P| = TP + FP = ' + m.TP + ' + ' + m.FP + ' = <b>' + m.predCount + '</b></div>';
      html += '<div>|R| = TP + FN = ' + m.TP + ' + ' + m.FN + ' = <b>' + m.refCount + '</b></div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>3. Fórmulas:</b></div>';
      
      if (m.union === 0) {
        html += '<div style="color:#888;">IoU = 1.0 (máscaras vazias)</div>';
        html += '<div style="color:#888;">Dice = 1.0 (máscaras vazias)</div>';
      } else {
        html += '<div>IoU = Interseção / União = ' + m.intersection + ' / ' + m.union + ' = <b style="color:#2980b9;">' + m.iou.toFixed(4) + '</b></div>';
        html += '<div>Dice = 2·Interseção / (|P| + |R|) = 2·' + m.intersection + ' / (' + m.predCount + ' + ' + m.refCount + ') = ' + (2 * m.intersection) + ' / ' + (m.predCount + m.refCount) + ' = <b style="color:#50e3c2;">' + m.dice.toFixed(4) + '</b></div>';
      }
      
      metricsEl.innerHTML = html;
      
      // Resumo final
      summaryEl.innerHTML = '<b>IoU: ' + m.iou.toFixed(4) + '</b> &nbsp;&nbsp;|&nbsp;&nbsp; <b>Dice: ' + m.dice.toFixed(4) + '</b>';
    }
    
    // Limpar máscaras
    function clearMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = 0;
          refMask[i][j] = 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Gerar máscaras aleatórias
    function randomMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = Math.random() > 0.5 ? 1 : 0;
          refMask[i][j] = Math.random() > 0.5 ? 1 : 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Event listeners
    clearBtn.addEventListener('click', clearMasks);
    randomBtn.addEventListener('click', randomMasks);
    
    dimEl.addEventListener('input', function() {
      H = parseInt(dimEl.value);
      W = H;
      dimvEl.textContent = H + '×' + W;
      
      var newPred = [];
      var newRef = [];
      for (var i = 0; i < H; i++) {
        newPred.push([]);
        newRef.push([]);
        for (var j = 0; j < W; j++) {
          newPred[i].push(i < predMask.length && j < predMask[0].length ? predMask[i][j] : 0);
          newRef[i].push(i < refMask.length && j < refMask[0].length ? refMask[i][j] : 0);
        }
      }
      predMask = newPred;
      refMask = newRef;
      generateGrids();
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('quadrado');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0905');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.47:** Simulatore EP09_05: Valutazione della Segmentazione — IoU e Dice Pixel per Pixel


<figure id="fig-09-sim-ep0905">
  <img src="imagens/fig-09-sim-ep0905.png" alt=" Simulatore EP09_05: Valutazione della Segmentazione — IoU e Dice Pixel per Pixel " style="max-width:80%" />
  <figcaption><strong>Figura 9.47:</strong>  Simulatore EP09_05: Valutazione della Segmentazione — IoU e Dice Pixel per Pixel </figcaption>
</figure>

In [70]:
%%writefile EP09_05.py
# Codice Python


Overwriting EP09_05.py


In [71]:
TestSuite("EP09_05.py").run()

### 9.0.6 EP09_06 🔴 *Pipeline* Integrato: Dalla Rilevazione alla Misurazione nel Mondo Reale

Questo esercizio finale integra i due esercizi di rilevazione e il principio di **fotogrammetria** presentato nella sezione "Fotogrammetria e Riferimento di Scala" — esattamente lo stesso calcolo implementato nella figura di misurazione per riferimento di scala di questo capitolo. Lo scenario riproduce una situazione realistica: un rilevatore (Faster R-CNN o YOLO) genera **diverse scatole candidate sovrapposte** per lo stesso oggetto di interesse; dopo averle filtrate tramite NMS, la scatola sopravvissuta con maggiore confidenza viene utilizzata, insieme a una scatola di riferimento di larghezza reale nota (come la carta da $8{,}56$ cm), per stimare le dimensioni reali dell'oggetto rilevato.

#### 9.0.6.1 📋 Linee Guida di Implementazione

1. **Riferimento noto:** Leggere il valore reale $L_{ref}$ (larghezza reale dell'oggetto di riferimento, in cm) e successivamente i quattro reali $x_1\ y_1\ x_2\ y_2$ della sua scatola delimitante in pixel (già nota, senza necessità di rilevazione).
2. **Candidati dell'oggetto da misurare:** Leggere l'intero $N$ (numero di scatole candidate prodotte dal rilevatore per l'oggetto di interesse) e la soglia reale $\tau$; successivamente, leggere le $N$ righe di scatole candidate, ciascuna con $x_1\ y_1\ x_2\ y_2\ \text{score}$.
3. **Fase 1 — NMS:** Applicare esattamente l'algoritmo di Soppressione Non Massima dell'EP09_04 alle $N$ scatole candidate, utilizzando la soglia $\tau$, per eliminare rilevazioni ridondanti dello stesso oggetto.
4. **Fase 2 — Selezione della scatola finale:** Dopo il NMS, la scatola con il `score` più alto tra quelle mantenute è la rilevazione finale dell'oggetto (l'input garantisce che tutte le scatole candidate corrispondano a un singolo oggetto fisico, quindi la prima scatola selezionata dal NMS è già il risultato finale).
5. **Fase 3 — Misurazione per riferimento di scala:** Calcolare il rapporto $\text{cm/pixel} = L_{ref} / \text{larghezza del riferimento in pixel}$ e applicarlo sia alla larghezza che all'altezza (in pixel) della scatola finale dell'oggetto, ottenendo le sue dimensioni reali stimate in centimetri.
6. **Output:** Prima, una riga per ogni scatola mantenuta dopo il NMS (stesso formato dell'EP09_04): `índice score`. Successivamente, la riga `Total mantidas: X`. Infine, la riga `Objeto: L x A cm`, dove $L$ e $A$ sono la larghezza e l'altezza stimate dell'oggetto, ciascuna con 2 cifre decimali.

#### 9.0.6.2 📌 Vincoli Computazionali

* **Riutilizzare integralmente il NMS dell'EP09_04** — stessa regola di parità, stesso criterio di soppressione ($\text{IoU} > \tau$).
* **Il riferimento non passa attraverso il NMS:** la sua scatola è data direttamente, senza candidati concorrenti.
* **Rapporto unico per larghezza e altezza:** così come nella figura di fotogrammetria del capitolo, lo stesso rapporto cm/pixel (derivato dalla larghezza del riferimento) viene applicato sia alla larghezza che all'altezza dell'oggetto — non vi è calibrazione verticale separata.

#### 9.0.6.3 🧠 Fondamenti Teorici

| Fase | Concetto del capitolo |
|---|---|
| Multiple scatole candidate | Output grezzo di un rilevatore come Faster R-CNN o YOLO, prima della post-elaborazione |
| NMS (EP09_04) | Filtra le rilevazioni ridondanti, preservando solo la più affidabile per l'oggetto |
| Riferimento di scala noto | Stesso principio della carta da $8{,}56$ cm utilizzata nella sezione "Fotogrammetria e Riferimento di Scala" |
| Conversione pixel → centimetro | Regola del tre semplice: $\text{cm/pixel} = L_{ref} / w_{ref\_px}$, applicata alla scatola finale dell'oggetto |

#### 9.0.6.4 🧩 Metodi di `morph.py` che possono aiutare

* `mm.IoU(boxA, boxB)` — la stessa funzione suggerita nell'EP09_04, qui riutilizzata all'interno della fase di NMS di questo *pipeline* integrato (ricordarsi della conversione di formato: $w = x_2-x_1$, $h = y_2-y_1$).
* Se hai già risolto l'EP09_04 incapsulando il NMS in una funzione propria, questo è il momento ideale per **riutilizzare quel codice** — l'integrazione di moduli già testati singolarmente è esattamente la pratica ingegneristica che questo esercizio vuole rafforzare.

#### 9.0.6.5 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Reale $L_{ref}$.
* Riga 2: $x_1\ y_1\ x_2\ y_2$ della scatola di riferimento.
* Riga 3: Intero $N$ e reale $\tau$.
* Prossime $N$ righe: $x_1\ y_1\ x_2\ y_2\ \text{score}$ delle scatole candidate dell'oggetto.

**Output:**

* Una riga per ogni scatola mantenuta dopo il NMS: `índice score`.
* Riga successiva: `Total mantidas: X`.
* Ultima riga: `Objeto: L x A cm`.


#### 9.0.6.6 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 8.56<br>30 200 170 288<br>3 0.5<br>250 100 470 250 0.92<br>255 105 468 245 0.88<br>600 600 650 650 0.40 | 0 0.9200<br>2 0.4000<br>Total mantidas: 2<br>Objeto: 13.45 x 9.17 cm | La scatola 1 viene soppressa perché si sovrappone fortemente alla scatola 0; la rilevazione finale dell'oggetto è la scatola 0. |

In [72]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0906" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulatore: Pipeline Integrato — Dal Rilevamento alla Misurazione</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 Fotogrammetria</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Casella selezionata</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Casella soppressa</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Riferimento</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Oggetto finale</b>
      </span>
    </div>
    
    <!-- Controles -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">L_ref (cm)</label>
            <span id="ep0906_lref_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">8.56</span>
          </div>
          <input id="ep0906_lref" style="width:100%;accent-color:#2980b9;height:4px;" max="20" min="1" step="0.01" type="range" value="8.56">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Soglia τ</label>
            <span id="ep0906_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0906_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Esempio</label>
            <span id="ep0906_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Modello</span>
          </div>
          <select id="ep0906_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="padrao">Esempio Modello</option>
            <option value="multiplos">Oggetti Multipli</option>
            <option value="agrupado">Caselle Raggruppate</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0906_run_btn" style="background:#2980b9;color:white;border:none;padding:8px 16px;border-radius:16px;cursor:pointer;font-size:11px;font-weight:bold;transition:all 0.3s;">
            ▶️ Esegui Pipeline
          </button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0906_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;font-size:9px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:3fr 2fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:350px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Visualizzazione del Pipeline
        </div>
        <canvas id="ep0906_canvas" style="width:100%;height:300px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:350px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📋 Pipeline Passo dopo Passo
        </div>
        <div id="ep0906_steps" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resultado final -->
    <div id="ep0906_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var lrefEl = root.querySelector('#ep0906_lref');
    var lrefvEl = root.querySelector('#ep0906_lref_v');
    var tauEl = root.querySelector('#ep0906_tau');
    var tauvEl = root.querySelector('#ep0906_tau_v');
    var exampleEl = root.querySelector('#ep0906_example');
    var boxesConfigEl = root.querySelector('#ep0906_boxes_config');
    var canvas = root.querySelector('#ep0906_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0906_steps');
    var summaryEl = root.querySelector('#ep0906_summary');
    var runBtn = root.querySelector('#ep0906_run_btn');
    
    // Estado
    var refBox = { x1: 30, y1: 200, x2: 170, y2: 288 };
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    var finalBox = null;
    var cmPerPixel = 0;
    
    // Exemplos
    var examples = {
      padrao: {
        refBox: { x1: 30, y1: 200, x2: 170, y2: 288 },
        boxes: [
          { x1: 250, y1: 100, x2: 470, y2: 250, score: 0.92 },
          { x1: 255, y1: 105, x2: 468, y2: 245, score: 0.88 },
          { x1: 600, y1: 600, x2: 650, y2: 650, score: 0.40 }
        ]
      },
      multiplos: {
        refBox: { x1: 20, y1: 50, x2: 100, y2: 130 },
        boxes: [
          { x1: 200, y1: 150, x2: 350, y2: 280, score: 0.85 },
          { x1: 210, y1: 160, x2: 360, y2: 290, score: 0.75 },
          { x1: 400, y1: 300, x2: 550, y2: 420, score: 0.70 },
          { x1: 410, y1: 310, x2: 560, y2: 430, score: 0.65 }
        ]
      },
      agrupado: {
        refBox: { x1: 50, y1: 50, x2: 150, y2: 150 },
        boxes: [
          { x1: 300, y1: 200, x2: 500, y2: 350, score: 0.95 },
          { x1: 310, y1: 210, x2: 490, y2: 340, score: 0.90 },
          { x1: 320, y1: 220, x2: 480, y2: 330, score: 0.85 },
          { x1: 330, y1: 230, x2: 470, y2: 320, score: 0.80 }
        ]
      }
    };
    
    // Ajustar canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      refBox = JSON.parse(JSON.stringify(example.refBox));
      boxes = JSON.parse(JSON.stringify(example.boxes));
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Processar Pipeline" para executar.';
      summaryEl.innerHTML = 'Aguardando processamento...';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      html += '<div style="background:#e8f5e9;border:1px solid #a5d6a7;border-radius:6px;padding:4px 6px;">';
      html += '<b>Referência:</b> ';
      html += '<input type="number" id="ep0906_ref_x1" value="' + refBox.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y1" value="' + refBox.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_x2" value="' + refBox.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y2" value="' + refBox.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '</div>';
      
      boxes.forEach(function(box, i) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;">';
        html += '<b>#' + i + ':</b> ';
        html += '<input type="number" id="ep0906_x1_' + i + '" value="' + box.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y1_' + i + '" value="' + box.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_x2_' + i + '" value="' + box.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y2_' + i + '" value="' + box.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_score_' + i + '" value="' + box.score + '" step="0.05" min="0" max="1" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '</div>';
      });
      
      boxesConfigEl.innerHTML = html;
      
      // Event listeners para referência
      ['x1', 'y1', 'x2', 'y2'].forEach(function(field) {
        var input = root.querySelector('#ep0906_ref_' + field);
        if (input) {
          input.addEventListener('input', function() {
            refBox[field] = parseFloat(input.value) || 0;
            render();
          });
        }
      });
      
      // Event listeners para caixas
      boxes.forEach(function(box, i) {
        ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
          var input = root.querySelector('#ep0906_' + field + '_' + i);
          if (input) {
            input.addEventListener('input', function() {
              boxes[i][field] = parseFloat(input.value) || 0;
              selectedBoxes = [];
              suppressedBoxes = [];
              finalBox = null;
              render();
            });
          }
        });
      });
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar pipeline
    function runPipeline() {
      var tau = parseFloat(tauEl.value);
      var lref = parseFloat(lrefEl.value);
      
      // Etapa 1: NMS
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) return b.box.score - a.box.score;
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      steps.push('<div style="font-weight:bold;color:#333;">Etapa 1: NMS</div>');
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push('<div style="color:#4a90e2;">Selecionar caixa #' + selected.originalIndex + ' (score: ' + selected.box.score.toFixed(4) + ')</div>');
        
        var newRemaining = [];
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressedBoxes.push(remaining[i]);
            steps.push('<div style="color:#ff6b6b;margin-left:10px;">↳ Suprimir #' + remaining[i].originalIndex + ' (IoU: ' + iou.toFixed(3) + ')</div>');
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        remaining = newRemaining;
      }
      
      steps.push('<div style="margin-top:4px;">Total mantidas: <b>' + selectedBoxes.length + '</b></div>');
      
      // Etapa 2: Seleção da caixa final
      if (selectedBoxes.length > 0) {
        finalBox = selectedBoxes[0];
        steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 2: Caixa Final</div>');
        steps.push('<div>Caixa selecionada: #' + finalBox.originalIndex + ' (score: ' + finalBox.box.score.toFixed(4) + ')</div>');
      }
      
      // Etapa 3: Medição
      steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 3: Medição por Referência</div>');
      
      var refWidthPx = refBox.x2 - refBox.x1;
      cmPerPixel = lref / refWidthPx;
      
      steps.push('<div>Largura da referência: ' + refWidthPx + ' pixels</div>');
      steps.push('<div>cm/pixel = ' + lref + ' / ' + refWidthPx + ' = <b>' + cmPerPixel.toFixed(6) + '</b></div>');
      
      if (finalBox) {
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        steps.push('<div>Largura do objeto: ' + objWidthPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objWidthCm.toFixed(2) + ' cm</b></div>');
        steps.push('<div>Altura do objeto: ' + objHeightPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objHeightCm.toFixed(2) + ' cm</b></div>');
        
        stepsEl.innerHTML = steps.join('');
        
        summaryEl.innerHTML = '<b>Objeto: ' + objWidthCm.toFixed(2) + ' x ' + objHeightCm.toFixed(2) + ' cm</b>';
      } else {
        stepsEl.innerHTML = steps.join('');
        summaryEl.innerHTML = 'Nenhum objeto detectado.';
      }
      
      render();
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var scale = Math.min(canvas.width / 700, canvas.height / 700);
      var offsetX = 20;
      var offsetY = 20;
      
      // Desenhar caixa de referência
      drawBoxOnCanvas(refBox, '#50e3c2', 'Ref', scale, offsetX, offsetY);
      
      // Desenhar caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        var isFinal = finalBox && finalBox.originalIndex === index;
        
        var color = '#f5a623';
        var label = '#' + index;
        
        if (isFinal) {
          color = '#f5a623';
          label = '#' + index + ' ✓';
        } else if (isSelected) {
          color = '#4a90e2';
        } else if (isSuppressed) {
          color = '#ff6b6b';
          label = '#' + index + ' ✗';
        }
        
        drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY);
      });
      
      // Desenhar linhas de medição
      if (finalBox && cmPerPixel > 0) {
        var refWidthPx = refBox.x2 - refBox.x1;
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        // Linha de largura do objeto
        var y = offsetY + finalBox.box.y1 * scale - 10;
        var x1 = offsetX + finalBox.box.x1 * scale;
        var x2 = offsetX + finalBox.box.x2 * scale;
        
        ctx.strokeStyle = '#f5a623';
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(x1, y);
        ctx.lineTo(x2, y);
        ctx.stroke();
        
        ctx.fillStyle = '#f5a623';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(objWidthCm.toFixed(2) + ' cm', (x1 + x2) / 2, y - 3);
      }
    }
    
    // Desenhar caixa no canvas
    function drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY) {
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      ctx.fillStyle = color;
      ctx.font = 'bold 11px Arial';
      ctx.textAlign = 'center';
      ctx.fillText(label, x + w/2, y - 5);
    }
    
    // Event listeners
    runBtn.addEventListener('click', function() {
      runPipeline();
      runBtn.textContent = '✓ Processado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Processar Pipeline';
        runBtn.style.background = '#2980b9';
      }, 1000);
    });
    
    lrefEl.addEventListener('input', function() {
      lrefvEl.textContent = parseFloat(lrefEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0906');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 9.48:** Simulatore EP09_06: Pipeline Integrato — Rilevamento alla Misurazione del Mondo Reale


<figure id="fig-09-sim-ep0906">
  <img src="imagens/fig-09-sim-ep0906.png" alt=" Simulatore EP09_06: Pipeline Integrato — Rilevamento alla Misurazione del Mondo Reale " style="max-width:80%" />
  <figcaption><strong>Figura 9.48:</strong>  Simulatore EP09_06: Pipeline Integrato — Rilevamento alla Misurazione del Mondo Reale </figcaption>
</figure>

In [73]:
%%writefile EP09_06.py
# Codice Python

Overwriting EP09_06.py


In [74]:
TestSuite("EP09_06.py").run()

## Referências do Capítulo


BISHOP, C. M. **Pattern Recognition and Machine Learning**. New York, NY, USA, Springer, 2006.

CHEN, Liang-Chieh *et al*. **Encoder-Decoder with Atrous Separable Convolution for Semantic Image Segmentation**. Cham, Springer International Publishing, 2018.

GOODFELLOW, Ian; BENGIO, Yoshua; COURVILLE, Aaron. **Deep Learning**. Cambridge, MA, USA, MIT Press, 2016.

HE, Kaiming *et al*. **Deep Residual Learning for Image Recognition**. 2016.

KINGMA, D. P.; BA, J. **Adam: A Method for Stochastic Optimization**. San Diego, CA, USA, 2015.

KIRILLOV, Alexander *et al*. **Segment Anything**. 2023.

LECUN, Y.; BENGIO, Y.; HINTON, G. **Deep learning**. Nature Publishing Group, 2015.

MCCULLOCH, W. S.; PITTS, W. **A Logical Calculus of the Ideas Immanent in Nervous Activity**. 1943.

PARKHI, Omkar M. *et al*. **Cats and Dogs**. IEEE, 2012.

REDMON, Joseph *et al*. **You Only Look Once: Unified, Real-Time Object Detection**. 2016.

REN, Shaoqing *et al*. **Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks**. Curran Associates, Inc., 2015.

RONNEBERGER, O.; FISCHER, P.; BROX, T. **U-Net: Convolutional Networks for Biomedical Image Segmentation**. Cham, Switzerland, Springer, 2015.

ROSENBLATT, F. **The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain**. 1958.

{GOOGLE}. **{NotebookLM}**. 2025.

*Referência não encontrada para: 50*